**Apply CNN on Max Dataset**

In [1]:
# ================================================================
# OPTIMIZED 1D CNN CLASSIFICATION
# MAX DATASET
# PYTORCH GPU SUPPORT
#
# Preferred device:
# Intel XPU
#
# Fallback:
# CUDA
# CPU
#
# EXPERIMENT DESIGN
#
# Complete dataset
# Person wise 70 / 20 / 10 split
# Fold specific StandardScaler
# 10 Optuna trials
# 5 fold GroupKFold
# 5 stochastic final runs
# Full Leave One Group Out
# Validation based early stopping
# Validation based best run selection
# Balanced CrossEntropy loss
# AdamW optimization
# Learning rate scheduler
# Gradient clipping
# Training and validation curves
# Complete classification metrics
# ROC AUC
# Confusion matrices
# Test predictions
#
# RESUME SUPPORT
#
# Persistent Optuna database
# Five run checkpoints
# Full LOGO checkpoint
#
# TEST DATA ARE USED ONLY FOR FINAL EVALUATION
# ================================================================


# ================================================================
# IMPORTS
# ================================================================

import os
import gc
import json
import time
import copy
import random
import hashlib
import warnings

from pathlib import Path

import joblib
import psutil
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.model_selection import (
    GroupKFold,
    LeaveOneGroupOut
)

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.utils.class_weight import (
    compute_class_weight
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ================================================================
# WARNINGS
# ================================================================

warnings.filterwarnings("ignore")

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# ================================================================
# BASIC CONFIGURATION
# ================================================================

SEED = 42

N_TRIALS = 10

N_RUNS = 5

GROUP_K_FOLDS = 5


# ================================================================
# CNN TRAINING CONFIGURATION
# ================================================================

CNN_EPOCHS = 50

CNN_BATCH_SIZE = 1024

CNN_PATIENCE = 6

GRADIENT_CLIP = 5.0


# ================================================================
# FULL LOGO
#
# None means all training participants.
# ================================================================

LOGO_MAX_FOLDS = None


# ================================================================
# CPU THREAD LIMIT
# ================================================================

CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)


torch.set_num_threads(
    CPU_THREADS
)


# ================================================================
# INPUT PATH
# ================================================================

FILE_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset/"
    "Cleaned_oN_Selected_Max.csv"
)


if not FILE_PATH.exists():

    raise FileNotFoundError(
        f"Input dataset not found: {FILE_PATH}"
    )


# ================================================================
# OUTPUT ROOT
# ================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset"
)


# ================================================================
# ALGORITHM RESULT FOLDER
# ================================================================

OUTPUT_DIR = (
    OUTPUT_ROOT
    / "CNN_Results"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# CHECKPOINT FOLDER
# ================================================================

CHECKPOINT_DIR = (
    OUTPUT_DIR
    / "checkpoints"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# IMPORTANT FILES
# ================================================================

OPTUNA_DATABASE = (
    OUTPUT_DIR
    / "cnn_optuna.db"
)


OPTUNA_STORAGE = (
    f"sqlite:///{OPTUNA_DATABASE}"
)


OPTUNA_STUDY_NAME = (
    "Max_Dataset_Optimized_CNN_v1"
)


FIVE_RUNS_PATH = (
    OUTPUT_DIR
    / "five_runs.csv"
)


LOGO_RESULT_PATH = (
    OUTPUT_DIR
    / "full_logo_results.csv"
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "cnn_best_model.pt"
)


# ================================================================
# MEMORY MONITOR
# ================================================================

PROC = psutil.Process(
    os.getpid()
)


def mem_mb():

    return (
        PROC.memory_info().rss
        / 1024
        / 1024
    )


# ================================================================
# ATOMIC CSV SAVE
# ================================================================

def atomic_csv(
    dataframe,
    path
):

    path = Path(path)

    temporary = path.with_name(
        path.name + ".tmp"
    )


    dataframe.to_csv(
        temporary,
        index=False
    )


    os.replace(
        temporary,
        path
    )


# ================================================================
# ATOMIC TORCH SAVE
# ================================================================

def atomic_torch_save(
    obj,
    path
):

    path = Path(path)

    temporary = path.with_name(
        path.name + ".tmp"
    )


    torch.save(
        obj,
        temporary
    )


    os.replace(
        temporary,
        path
    )


# ================================================================
# RANDOM SEED
# ================================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if (
        hasattr(torch, "xpu")
        and
        torch.xpu.is_available()
    ):

        try:

            torch.xpu.manual_seed_all(
                seed
            )

        except Exception:

            pass


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    SEED
)


# ================================================================
# DEVICE DETECTION
#
# Priority:
#
# Intel XPU
# CUDA
# CPU
# ================================================================

if (
    hasattr(torch, "xpu")
    and
    torch.xpu.is_available()
):

    DEVICE = torch.device(
        "xpu:0"
    )

    DEVICE_TYPE = "xpu"

    DEVICE_NAME = torch.xpu.get_device_name(
        0
    )


elif torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda:0"
    )

    DEVICE_TYPE = "cuda"

    DEVICE_NAME = torch.cuda.get_device_name(
        0
    )


else:

    DEVICE = torch.device(
        "cpu"
    )

    DEVICE_TYPE = "cpu"

    DEVICE_NAME = "CPU"


# ================================================================
# DEVICE HELPERS
# ================================================================

def synchronize():

    if DEVICE_TYPE == "xpu":

        torch.xpu.synchronize()

    elif DEVICE_TYPE == "cuda":

        torch.cuda.synchronize()


def clear_device():

    gc.collect()


    if DEVICE_TYPE == "xpu":

        torch.xpu.empty_cache()


    elif DEVICE_TYPE == "cuda":

        torch.cuda.empty_cache()


# ================================================================
# DEVICE INFORMATION
# ================================================================

print("=" * 80)

print("GPU AND DEVICE CONFIGURATION")

print("=" * 80)

print(
    "PyTorch version:",
    torch.__version__
)

print(
    "Selected device:",
    DEVICE
)

print(
    "Device type:",
    DEVICE_TYPE
)

print(
    "Device name:",
    DEVICE_NAME
)

print(
    "CPU thread limit:",
    CPU_THREADS
)

print("=" * 80)


# ================================================================
# TOTAL TIMER
# ================================================================

TOTAL_START = time.time()


# ================================================================
# EXPERIMENT INFORMATION
# ================================================================

print()

print("=" * 80)

print("OPTIMIZED 1D CNN")

print("MAX DATASET")

print("=" * 80)

print(
    "Input dataset:"
)

print(
    FILE_PATH
)

print()

print(
    "Result folder:"
)

print(
    OUTPUT_DIR
)

print()

print(
    "Optuna trials:",
    N_TRIALS
)

print(
    "GroupKFold folds:",
    GROUP_K_FOLDS
)

print(
    "Final runs:",
    N_RUNS
)

print(
    "Maximum epochs:",
    CNN_EPOCHS
)

print(
    "Batch size:",
    CNN_BATCH_SIZE
)

print(
    "Early stopping patience:",
    CNN_PATIENCE
)

print(
    "LOGO cap:",
    LOGO_MAX_FOLDS
)

print(
    "Data subsampling: NONE"
)

print("=" * 80)


# ================================================================
# LOAD DATASET
# ================================================================

print()

print("=" * 80)

print("LOADING COMPLETE MAX DATASET")

print("=" * 80)


load_start = time.time()


df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)


print(
    "Raw dataset shape:",
    df.shape
)


print(
    "RAM after loading:",
    f"{mem_mb():.2f} MB"
)


# ================================================================
# COLUMN DEFINITIONS
# ================================================================

metadata_cols = [
    "PersonSrial",
    "Window",
    "Window_Start",
    "Window_End",
    "Window_Samples"
]


target_col = "Class"


required_columns = [
    "PersonSrial",
    target_col
]


for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"Required column missing: {column}"
        )


# ================================================================
# FEATURE COLUMNS
# ================================================================

feature_cols = [
    column
    for column in df.columns
    if column not in metadata_cols
    and column != target_col
]


if len(feature_cols) == 0:

    raise ValueError(
        "No feature columns were detected."
    )


NUMBER_FEATURES = len(
    feature_cols
)


print(
    "Feature count:",
    NUMBER_FEATURES
)


# ================================================================
# FEATURE MATRIX
# ================================================================

X_all = (
    df[
        feature_cols
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(
        dtype=np.float32
    )
)


# ================================================================
# INVALID VALUE CHECK
# ================================================================

invalid_count = int(
    (
        ~np.isfinite(
            X_all
        )
    ).sum()
)


print(
    "Invalid feature cells:",
    invalid_count
)


if invalid_count > 0:

    raise ValueError(
        "Feature matrix contains NaN or infinite values."
    )


# ================================================================
# LABEL ENCODING
# ================================================================

label_encoder = LabelEncoder()


y_all = label_encoder.fit_transform(
    df[
        target_col
    ].astype(str)
).astype(
    np.int64
)


class_names = list(
    label_encoder.classes_
)


NUMBER_CLASSES = len(
    class_names
)


# ================================================================
# PARTICIPANTS
# ================================================================

groups_all = (
    df[
        "PersonSrial"
    ]
    .to_numpy()
)


print()

print(
    "Feature matrix:",
    X_all.shape
)

print(
    "Classes:",
    class_names
)

print(
    "Number of classes:",
    NUMBER_CLASSES
)

print(
    "Participants:",
    len(
        np.unique(
            groups_all
        )
    )
)

print(
    "Loading time:",
    f"{time.time() - load_start:.2f}",
    "seconds"
)

print(
    "RAM:",
    f"{mem_mb():.2f}",
    "MB"
)


# ================================================================
# CLASS DISTRIBUTION
# ================================================================

print()

print("CLASS DISTRIBUTION")


class_distribution_records = []


for class_index, class_name in enumerate(
    class_names
):

    count = int(
        np.sum(
            y_all == class_index
        )
    )


    percentage = (
        count
        / len(y_all)
        * 100.0
    )


    print(
        f"{class_name}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


    class_distribution_records.append(
        {
            "Class":
                class_name,

            "Count":
                count,

            "Percentage":
                percentage
        }
    )


pd.DataFrame(
    class_distribution_records
).to_csv(
    OUTPUT_DIR
    / "class_distribution.csv",
    index=False
)


# ================================================================
# RELEASE DATAFRAME
# ================================================================

del df

gc.collect()


# ================================================================
# PERSON WISE 70 / 20 / 10 SPLIT
# ================================================================

print()

print("=" * 80)

print("PERSON WISE 70 / 20 / 10 SPLIT")

print("=" * 80)


persons = np.unique(
    groups_all
)


rng = np.random.RandomState(
    SEED
)


rng.shuffle(
    persons
)


number_persons = len(
    persons
)


number_test = max(
    1,
    int(
        round(
            0.20
            * number_persons
        )
    )
)


number_validation = max(
    1,
    int(
        round(
            0.10
            * number_persons
        )
    )
)


# ================================================================
# PARTICIPANT SPLITS
# ================================================================

test_persons = persons[
    :number_test
]


validation_persons = persons[
    number_test:
    number_test
    + number_validation
]


train_persons = persons[
    number_test
    + number_validation:
]


# ================================================================
# VERIFY NO PARTICIPANT OVERLAP
# ================================================================

assert (
    np.intersect1d(
        train_persons,
        validation_persons
    ).size == 0
)


assert (
    np.intersect1d(
        train_persons,
        test_persons
    ).size == 0
)


assert (
    np.intersect1d(
        validation_persons,
        test_persons
    ).size == 0
)


# ================================================================
# MASKS
# ================================================================

train_mask = np.isin(
    groups_all,
    train_persons
)


validation_mask = np.isin(
    groups_all,
    validation_persons
)


test_mask = np.isin(
    groups_all,
    test_persons
)


# ================================================================
# SPLIT ARRAYS
# ================================================================

X_train_raw = np.ascontiguousarray(
    X_all[
        train_mask
    ],
    dtype=np.float32
)


y_train = np.ascontiguousarray(
    y_all[
        train_mask
    ],
    dtype=np.int64
)


g_train = np.ascontiguousarray(
    groups_all[
        train_mask
    ]
)


X_validation_raw = np.ascontiguousarray(
    X_all[
        validation_mask
    ],
    dtype=np.float32
)


y_validation = np.ascontiguousarray(
    y_all[
        validation_mask
    ],
    dtype=np.int64
)


g_validation = np.ascontiguousarray(
    groups_all[
        validation_mask
    ]
)


X_test_raw = np.ascontiguousarray(
    X_all[
        test_mask
    ],
    dtype=np.float32
)


y_test = np.ascontiguousarray(
    y_all[
        test_mask
    ],
    dtype=np.int64
)


g_test = np.ascontiguousarray(
    groups_all[
        test_mask
    ]
)


TRAIN_SAMPLES = len(
    X_train_raw
)


VALIDATION_SAMPLES = len(
    X_validation_raw
)


TEST_SAMPLES = len(
    X_test_raw
)


print(
    "Training participants:",
    len(
        train_persons
    )
)

print(
    "Validation participants:",
    len(
        validation_persons
    )
)

print(
    "Testing participants:",
    len(
        test_persons
    )
)

print()

print(
    "Training:",
    X_train_raw.shape
)

print(
    "Validation:",
    X_validation_raw.shape
)

print(
    "Testing:",
    X_test_raw.shape
)


# ================================================================
# SAVE PARTICIPANT SPLIT
# ================================================================

split_table = pd.DataFrame(
    [
        {
            "Split":
                "Training",

            "Participants":
                len(
                    train_persons
                ),

            "Rows":
                TRAIN_SAMPLES
        },

        {
            "Split":
                "Validation",

            "Participants":
                len(
                    validation_persons
                ),

            "Rows":
                VALIDATION_SAMPLES
        },

        {
            "Split":
                "Testing",

            "Participants":
                len(
                    test_persons
                ),

            "Rows":
                TEST_SAMPLES
        }
    ]
)


split_table.to_csv(
    OUTPUT_DIR
    / "split_summary.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "participant_split.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        {
            "training":
                [
                    str(x)
                    for x in train_persons
                ],

            "validation":
                [
                    str(x)
                    for x in validation_persons
                ],

            "testing":
                [
                    str(x)
                    for x in test_persons
                ]
        },
        file,
        indent=4
    )


# ================================================================
# RELEASE FULL DATA
# ================================================================

del X_all
del y_all
del groups_all

del train_mask
del validation_mask
del test_mask

gc.collect()


print()

print(
    "RAM after split:",
    f"{mem_mb():.2f} MB"
)


# ================================================================
# GROUPKFOLD
# ================================================================

gkf_splits = list(
    GroupKFold(
        n_splits=
            GROUP_K_FOLDS
    ).split(
        X_train_raw,
        y_train,
        g_train
    )
)


FULL_LOGO_FOLDS = len(
    np.unique(
        g_train
    )
)


print()

print(
    "GroupKFold folds:",
    len(
        gkf_splits
    )
)

print(
    "Available LOGO folds:",
    FULL_LOGO_FOLDS
)


# ================================================================
# FOLD SPECIFIC STANDARDIZATION
# ================================================================

def prepare_fold(
    training_index,
    validation_index
):

    scaler = StandardScaler()


    X_fold_training = scaler.fit_transform(
        X_train_raw[
            training_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    X_fold_validation = scaler.transform(
        X_train_raw[
            validation_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    return (
        np.ascontiguousarray(
            X_fold_training
        ),
        np.ascontiguousarray(
            X_fold_validation
        )
    )


# ================================================================
# FINAL STANDARDIZATION
#
# Scaler is fitted only on training participants.
# ================================================================

print()

print("=" * 80)

print("FINAL TRAINING SCALER")

print("=" * 80)


final_scaler = StandardScaler()


X_train = final_scaler.fit_transform(
    X_train_raw
).astype(
    np.float32,
    copy=False
)


X_validation = final_scaler.transform(
    X_validation_raw
).astype(
    np.float32,
    copy=False
)


X_test = final_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


X_train = np.ascontiguousarray(
    X_train
)


X_validation = np.ascontiguousarray(
    X_validation
)


X_test = np.ascontiguousarray(
    X_test
)


print(
    "Training:",
    X_train.shape
)

print(
    "Validation:",
    X_validation.shape
)

print(
    "Testing:",
    X_test.shape
)

print(
    "RAM:",
    f"{mem_mb():.2f} MB"
)


# ================================================================
# SAVE PREPROCESSING
# ================================================================

joblib.dump(
    final_scaler,
    OUTPUT_DIR
    / "standard_scaler.joblib"
)


joblib.dump(
    label_encoder,
    OUTPUT_DIR
    / "label_encoder.joblib"
)


with open(
    OUTPUT_DIR
    / "feature_columns.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        feature_cols,
        file,
        indent=4
    )


# ================================================================
# CLASS WEIGHTS
# ================================================================

def get_class_weights(
    y
):

    present_classes = np.unique(
        y
    )


    calculated_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y
    )


    weights = np.ones(
        NUMBER_CLASSES,
        dtype=np.float32
    )


    weights[
        present_classes
    ] = calculated_weights.astype(
        np.float32
    )


    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


# ================================================================
# CNN MODEL
#
# Input:
# batch x 232
#
# Conv1D input:
# batch x 1 x 232
# ================================================================

class CNN1D(
    nn.Module
):

    def __init__(
        self,
        params
    ):

        super().__init__()


        number_blocks = int(
            params[
                "n_conv_blocks"
            ]
        )


        base_filters = int(
            params[
                "base_filters"
            ]
        )


        kernel_size = int(
            params[
                "kernel_size"
            ]
        )


        conv_dropout = float(
            params[
                "conv_dropout"
            ]
        )


        padding = (
            kernel_size
            // 2
        )


        layers_list = []


        input_channels = 1


        for block_index in range(
            number_blocks
        ):

            output_channels = (
                base_filters
                * (
                    2
                    ** block_index
                )
            )


            layers_list.extend(
                [
                    nn.Conv1d(
                        input_channels,
                        output_channels,
                        kernel_size=
                            kernel_size,
                        padding=
                            padding,
                        bias=False
                    ),

                    nn.BatchNorm1d(
                        output_channels
                    ),

                    nn.GELU(),

                    nn.Conv1d(
                        output_channels,
                        output_channels,
                        kernel_size=
                            kernel_size,
                        padding=
                            padding,
                        bias=False
                    ),

                    nn.BatchNorm1d(
                        output_channels
                    ),

                    nn.GELU(),

                    nn.MaxPool1d(
                        kernel_size=2
                    ),

                    nn.Dropout(
                        conv_dropout
                    )
                ]
            )


            input_channels = (
                output_channels
            )


        self.features = nn.Sequential(
            *layers_list
        )


        self.global_pool = nn.AdaptiveAvgPool1d(
            1
        )


        dense_units = int(
            params[
                "dense_units"
            ]
        )


        dense_dropout = float(
            params[
                "dense_dropout"
            ]
        )


        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                input_channels,
                dense_units
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                dense_units,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        x = x.unsqueeze(
            1
        )


        x = self.features(
            x
        )


        x = self.global_pool(
            x
        )


        x = self.classifier(
            x
        )


        return x


# ================================================================
# BUILD CNN
# ================================================================

def build_cnn(
    params,
    seed=SEED
):

    set_seed(
        seed
    )


    model = CNN1D(
        params
    )


    return model.to(
        DEVICE
    )


# ================================================================
# GPU DATA HOLDER
# ================================================================

class DeviceData:

    def __init__(
        self,
        X,
        y
    ):

        X_array = np.ascontiguousarray(
            X,
            dtype=np.float32
        )


        y_array = np.ascontiguousarray(
            y,
            dtype=np.int64
        )


        self.X = torch.from_numpy(
            X_array
        ).to(
            DEVICE
        )


        self.y = torch.from_numpy(
            y_array
        ).to(
            DEVICE
        )


    def __len__(
        self
    ):

        return int(
            self.y.shape[
                0
            ]
        )


# ================================================================
# BATCH INDEX GENERATOR
# ================================================================

def batch_indices(
    number_samples,
    shuffle=True
):

    if shuffle:

        indices = torch.randperm(
            number_samples,
            device=DEVICE
        )


    else:

        indices = torch.arange(
            number_samples,
            device=DEVICE
        )


    for start in range(
        0,
        number_samples,
        CNN_BATCH_SIZE
    ):

        yield indices[
            start:
            start
            + CNN_BATCH_SIZE
        ]


# ================================================================
# TRAIN ONE EPOCH
# ================================================================

def train_one_epoch(
    model,
    data,
    optimizer,
    criterion
):

    model.train()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    for indices in batch_indices(
        len(data),
        shuffle=True
    ):

        X_batch = data.X[
            indices
        ]


        y_batch = data.y[
            indices
        ]


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            X_batch
        )


        loss = criterion(
            logits,
            y_batch
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )


        optimizer.step()


        batch_size = int(
            y_batch.shape[
                0
            ]
        )


        total_loss += (
            float(
                loss.detach().item()
            )
            * batch_size
        )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


    synchronize()


    return (
        total_loss
        / total_samples,

        total_correct
        / total_samples
    )


# ================================================================
# EVALUATION
# ================================================================

@torch.inference_mode()
def evaluate_model(
    model,
    data,
    criterion=None,
    return_probabilities=False
):

    model.eval()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    probability_parts = []


    for indices in batch_indices(
        len(data),
        shuffle=False
    ):

        X_batch = data.X[
            indices
        ]


        y_batch = data.y[
            indices
        ]


        logits = model(
            X_batch
        )


        batch_size = int(
            y_batch.shape[
                0
            ]
        )


        if criterion is not None:

            loss = criterion(
                logits,
                y_batch
            )


            total_loss += (
                float(
                    loss.item()
                )
                * batch_size
            )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


        if return_probabilities:

            probability_parts.append(
                torch.softmax(
                    logits,
                    dim=1
                )
                .cpu()
            )


    synchronize()


    accuracy = (
        total_correct
        / total_samples
    )


    if criterion is None:

        mean_loss = np.nan


    else:

        mean_loss = (
            total_loss
            / total_samples
        )


    if return_probabilities:

        probabilities = (
            torch.cat(
                probability_parts,
                dim=0
            )
            .numpy()
        )


        return (
            mean_loss,
            accuracy,
            probabilities
        )


    return (
        mean_loss,
        accuracy
    )


# ================================================================
# FIT CNN
# ================================================================

def fit_cnn(
    params,
    X_training,
    y_training,
    X_validation=None,
    y_validation=None,
    seed=SEED,
    epochs=CNN_EPOCHS,
    use_early_stopping=True,
    verbose=False
):

    clear_device()


    model = build_cnn(
        params,
        seed
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(
            params[
                "learning_rate"
            ]
        ),
        weight_decay=float(
            params[
                "weight_decay"
            ]
        )
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )


    class_weights = get_class_weights(
        y_training
    )


    criterion = nn.CrossEntropyLoss(
        weight=
            class_weights,
        label_smoothing=
            float(
                params[
                    "label_smoothing"
                ]
            )
    )


    training_data = DeviceData(
        X_training,
        y_training
    )


    if X_validation is not None:

        validation_data = DeviceData(
            X_validation,
            y_validation
        )


    else:

        validation_data = None


    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": []
    }


    best_loss = float(
        "inf"
    )


    best_state = None

    best_epoch = 1

    patience_counter = 0


    for epoch in range(
        1,
        epochs + 1
    ):

        train_loss, train_accuracy = train_one_epoch(
            model,
            training_data,
            optimizer,
            criterion
        )


        if validation_data is not None:

            validation_loss, validation_accuracy = evaluate_model(
                model,
                validation_data,
                criterion
            )


            monitored_loss = (
                validation_loss
            )


        else:

            validation_loss = np.nan

            validation_accuracy = np.nan

            monitored_loss = (
                train_loss
            )


        scheduler.step(
            monitored_loss
        )


        current_lr = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )


        history[
            "loss"
        ].append(
            train_loss
        )


        history[
            "accuracy"
        ].append(
            train_accuracy
        )


        history[
            "val_loss"
        ].append(
            validation_loss
        )


        history[
            "val_accuracy"
        ].append(
            validation_accuracy
        )


        history[
            "learning_rate"
        ].append(
            current_lr
        )


        if verbose:

            print(
                f"Epoch "
                f"{epoch:02d}/"
                f"{epochs} | "
                f"Train Loss "
                f"{train_loss:.4f} | "
                f"Train Acc "
                f"{train_accuracy:.4f} | "
                f"Val Loss "
                f"{validation_loss:.4f} | "
                f"Val Acc "
                f"{validation_accuracy:.4f} | "
                f"LR "
                f"{current_lr:.2e}"
            )


        if use_early_stopping:

            if (
                monitored_loss
                < best_loss
                - 1e-5
            ):

                best_loss = (
                    monitored_loss
                )


                best_epoch = (
                    epoch
                )


                best_state = {
                    key:
                        value.detach()
                        .cpu()
                        .clone()

                    for key, value
                    in model.state_dict().items()
                }


                patience_counter = 0


            else:

                patience_counter += 1


                if (
                    patience_counter
                    >= CNN_PATIENCE
                ):

                    if verbose:

                        print(
                            "Early stopping."
                        )


                    break


    # ============================================================
    # RESTORE BEST MODEL
    # ============================================================

    if (
        use_early_stopping
        and
        best_state is not None
    ):

        model.load_state_dict(
            best_state
        )


    del optimizer
    del scheduler
    del criterion
    del class_weights
    del training_data


    if validation_data is not None:

        del validation_data


    if best_state is not None:

        del best_state


    clear_device()


    return (
        model,
        history,
        best_epoch
    )


# ================================================================
# PREDICT PROBABILITIES
# ================================================================

def predict_probabilities(
    model,
    X,
    y
):

    data = DeviceData(
        X,
        y
    )


    _, _, probabilities = evaluate_model(
        model,
        data,
        return_probabilities=True
    )


    del data

    clear_device()


    return probabilities


# ================================================================
# OPTUNA OBJECTIVE
# ================================================================

def cnn_objective(
    trial
):

    params = {
        "n_conv_blocks":

            trial.suggest_int(
                "n_conv_blocks",
                2,
                3
            ),

        "base_filters":

            trial.suggest_categorical(
                "base_filters",
                [
                    32,
                    64
                ]
            ),

        "kernel_size":

            trial.suggest_categorical(
                "kernel_size",
                [
                    3,
                    5,
                    7
                ]
            ),

        "conv_dropout":

            trial.suggest_float(
                "conv_dropout",
                0.05,
                0.35
            ),

        "dense_units":

            trial.suggest_categorical(
                "dense_units",
                [
                    64,
                    128,
                    256
                ]
            ),

        "dense_dropout":

            trial.suggest_float(
                "dense_dropout",
                0.10,
                0.50
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                3e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.10
            )
    }


    print()

    print(
        f"OPTUNA TRIAL "
        f"{trial.number + 1}"
    )


    print(
        params
    )


    fold_scores = []


    for fold_number, (
        training_index,
        validation_index
    ) in enumerate(
        gkf_splits,
        start=1
    ):

        fold_start = time.time()


        (
            X_fold_training,
            X_fold_validation
        ) = prepare_fold(
            training_index,
            validation_index
        )


        model, history, best_epoch = fit_cnn(
            params,
            X_fold_training,
            y_train[
                training_index
            ],
            X_fold_validation,
            y_train[
                validation_index
            ],
            seed=
                SEED
                + fold_number,
            epochs=
                CNN_EPOCHS,
            use_early_stopping=
                True,
            verbose=
                False
        )


        probabilities = predict_probabilities(
            model,
            X_fold_validation,
            y_train[
                validation_index
            ]
        )


        predictions = np.argmax(
            probabilities,
            axis=1
        )


        fold_accuracy = accuracy_score(
            y_train[
                validation_index
            ],
            predictions
        )


        fold_scores.append(
            fold_accuracy
        )


        print(
            f"Trial "
            f"{trial.number + 1}, "
            f"Fold "
            f"{fold_number}/"
            f"{GROUP_K_FOLDS}, "
            f"Accuracy = "
            f"{fold_accuracy:.4f}, "
            f"Best Epoch = "
            f"{best_epoch}, "
            f"Time = "
            f"{(time.time() - fold_start) / 60:.2f} min, "
            f"RAM = "
            f"{mem_mb():.2f} MB"
        )


        del model
        del history
        del probabilities
        del predictions
        del X_fold_training
        del X_fold_validation


        clear_device()


    mean_score = float(
        np.mean(
            fold_scores
        )
    )


    std_score = float(
        np.std(
            fold_scores
        )
    )


    print(
        f"Trial "
        f"{trial.number + 1} "
        f"Mean Accuracy = "
        f"{mean_score:.4f} "
        f"plus or minus "
        f"{std_score:.4f}"
    )


    return mean_score


# ================================================================
# OPTUNA
# ================================================================

print()

print("=" * 80)

print("CNN OPTUNA")

print("=" * 80)


optuna_start = time.time()


study = optuna.create_study(
    study_name=
        OPTUNA_STUDY_NAME,
    storage=
        OPTUNA_STORAGE,
    direction=
        "maximize",
    sampler=
        optuna.samplers.TPESampler(
            seed=SEED
        ),
    load_if_exists=
        True
)


completed_trials = sum(
    1
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
)


remaining_trials = max(
    0,
    N_TRIALS
    - completed_trials
)


print(
    "Previously completed trials:",
    completed_trials
)

print(
    "Remaining trials:",
    remaining_trials
)


if remaining_trials > 0:

    study.optimize(
        cnn_objective,
        n_trials=
            remaining_trials,
        show_progress_bar=
            False,
        gc_after_trial=
            True
    )


complete_trials = [
    trial
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
]


if len(
    complete_trials
) == 0:

    raise RuntimeError(
        "No Optuna trial completed successfully."
    )


optuna_time = (
    time.time()
    - optuna_start
)


best_params = dict(
    study.best_params
)


print()

print("=" * 80)

print("OPTUNA COMPLETED")

print("=" * 80)


print(
    "Best GroupKFold Accuracy:",
    f"{study.best_value:.4f}"
)

print(
    "Best Parameters:"
)

print(
    best_params
)

print(
    "Optuna Time:",
    f"{optuna_time / 60:.2f}",
    "minutes"
)


study.trials_dataframe().to_csv(
    OUTPUT_DIR
    / "optuna_trials.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "best_parameters.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        best_params,
        file,
        indent=4
    )


# ================================================================
# CONFIGURATION ID
#
# Prevents old checkpoints from another experiment
# from being reused.
# ================================================================

configuration_text = json.dumps(
    {
        "best_params":
            best_params,

        "features":
            NUMBER_FEATURES,

        "classes":
            class_names,

        "seed":
            SEED
    },
    sort_keys=True
)


CONFIG_ID = hashlib.sha256(
    configuration_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


print(
    "Experiment configuration ID:",
    CONFIG_ID
)


# ================================================================
# FIVE FINAL RUN CHECKPOINTS
# ================================================================

print()

print("=" * 80)

print("CNN FIVE FINAL RUNS")

print("=" * 80)


run_records = []

completed_runs = set()


# ================================================================
# LOAD PREVIOUS FIVE RUN RESULTS
# ================================================================

if FIVE_RUNS_PATH.exists():

    previous_runs = pd.read_csv(
        FIVE_RUNS_PATH
    )


    if (
        "Config_ID"
        in previous_runs.columns
    ):

        previous_runs = previous_runs[
            previous_runs[
                "Config_ID"
            ].astype(str)
            == CONFIG_ID
        ]


        for _, row in previous_runs.iterrows():

            run_number = int(
                row[
                    "Run"
                ]
            )


            run_checkpoint = (
                CHECKPOINT_DIR
                / f"cnn_run_{run_number:02d}.pt"
            )


            if run_checkpoint.exists():

                run_records.append(
                    row.to_dict()
                )

                completed_runs.add(
                    run_number
                )


print(
    "Previously completed final runs:",
    len(
        completed_runs
    )
)


# ================================================================
# RUN REMAINING FINAL RUNS
# ================================================================

five_run_start = time.time()


for run_number in range(
    1,
    N_RUNS + 1
):

    if run_number in completed_runs:

        print(
            f"Run "
            f"{run_number}/"
            f"{N_RUNS} "
            f"already completed."
        )

        continue


    print()

    print(
        f"RUN "
        f"{run_number}/"
        f"{N_RUNS}"
    )


    run_start = time.time()


    run_seed = (
        SEED
        + run_number
        - 1
    )


    model, history, best_epoch = fit_cnn(
        best_params,
        X_train,
        y_train,
        X_validation,
        y_validation,
        seed=
            run_seed,
        epochs=
            CNN_EPOCHS,
        use_early_stopping=
            True,
        verbose=
            True
    )


    # ============================================================
    # TRAIN PREDICTION
    # ============================================================

    train_probabilities = predict_probabilities(
        model,
        X_train,
        y_train
    )


    train_predictions = np.argmax(
        train_probabilities,
        axis=1
    )


    train_accuracy = accuracy_score(
        y_train,
        train_predictions
    )


    # ============================================================
    # VALIDATION PREDICTION
    # ============================================================

    validation_probabilities = predict_probabilities(
        model,
        X_validation,
        y_validation
    )


    validation_predictions = np.argmax(
        validation_probabilities,
        axis=1
    )


    validation_accuracy = accuracy_score(
        y_validation,
        validation_predictions
    )


    run_time = (
        time.time()
        - run_start
    )


    record = {
        "Config_ID":
            CONFIG_ID,

        "Run":
            run_number,

        "Seed":
            run_seed,

        "Training_Accuracy":
            float(
                train_accuracy
            ),

        "Validation_Accuracy":
            float(
                validation_accuracy
            ),

        "Epochs_Run":
            int(
                len(
                    history[
                        "loss"
                    ]
                )
            ),

        "Best_Epoch":
            int(
                best_epoch
            ),

        "Execution_Time_Seconds":
            float(
                run_time
            )
    }


    # ============================================================
    # COPY MODEL STATE TO CPU
    # ============================================================

    model_state = {
        key:
            value.detach()
            .cpu()
            .clone()

        for key, value
        in model.state_dict().items()
    }


    # ============================================================
    # SAVE RUN CHECKPOINT
    # ============================================================

    run_checkpoint = (
        CHECKPOINT_DIR
        / f"cnn_run_{run_number:02d}.pt"
    )


    atomic_torch_save(
        {
            "Config_ID":
                CONFIG_ID,

            "Run":
                run_number,

            "Seed":
                run_seed,

            "best_params":
                best_params,

            "record":
                record,

            "model_state_dict":
                model_state,

            "history":
                history
        },
        run_checkpoint
    )


    run_records.append(
        record
    )


    run_table_temp = pd.DataFrame(
        run_records
    )


    run_table_temp = (
        run_table_temp
        .sort_values(
            "Run"
        )
        .drop_duplicates(
            subset=[
                "Run"
            ],
            keep="last"
        )
    )


    atomic_csv(
        run_table_temp,
        FIVE_RUNS_PATH
    )


    print()

    print(
        f"[CNN {run_number}] "
        f"Train = "
        f"{train_accuracy:.4f}, "
        f"Validation = "
        f"{validation_accuracy:.4f}, "
        f"Best Epoch = "
        f"{best_epoch}, "
        f"Time = "
        f"{run_time / 60:.2f} min"
    )


    del model
    del history
    del model_state
    del train_probabilities
    del train_predictions
    del validation_probabilities
    del validation_predictions


    clear_device()


five_run_time = (
    time.time()
    - five_run_start
)


# ================================================================
# LOAD COMPLETE FIVE RUN TABLE
# ================================================================

run_table = pd.read_csv(
    FIVE_RUNS_PATH
)


run_table = run_table[
    run_table[
        "Config_ID"
    ].astype(str)
    == CONFIG_ID
].copy()


run_table = (
    run_table
    .sort_values(
        "Run"
    )
    .drop_duplicates(
        subset=[
            "Run"
        ],
        keep="last"
    )
)


if len(
    run_table
) != N_RUNS:

    raise RuntimeError(
        "Five final CNN runs are not complete."
    )


print()

print(
    run_table[
        [
            "Run",
            "Seed",
            "Training_Accuracy",
            "Validation_Accuracy",
            "Best_Epoch",
            "Execution_Time_Seconds"
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ================================================================
# SELECT BEST RUN USING VALIDATION ACCURACY
# ================================================================

selected_row = (
    run_table
    .sort_values(
        by=[
            "Validation_Accuracy",
            "Run"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[
        0
    ]
)


best_run_number = int(
    selected_row[
        "Run"
    ]
)


best_run_checkpoint = (
    CHECKPOINT_DIR
    / f"cnn_run_{best_run_number:02d}.pt"
)


best_bundle = torch.load(
    best_run_checkpoint,
    map_location="cpu",
    weights_only=False
)


best_record = (
    best_bundle[
        "record"
    ]
)


best_history = (
    best_bundle[
        "history"
    ]
)


print()

print(
    "Selected Run:",
    best_run_number
)

print(
    "Selected Seed:",
    best_record[
        "Seed"
    ]
)

print(
    "Selected Validation Accuracy:",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Selected Best Epoch:",
    best_record[
        "Best_Epoch"
    ]
)


# ================================================================
# BUILD SELECTED MODEL
# ================================================================

best_model = build_cnn(
    best_params,
    seed=
        int(
            best_record[
                "Seed"
            ]
        )
)


best_model.load_state_dict(
    best_bundle[
        "model_state_dict"
    ]
)


best_model.eval()


# ================================================================
# SAVE BEST MODEL PACKAGE
# ================================================================

atomic_torch_save(
    {
        "Algorithm":
            "Optimized 1D CNN",

        "Dataset":
            "Max Dataset",

        "Device_Type":
            DEVICE_TYPE,

        "Device_Name":
            DEVICE_NAME,

        "Config_ID":
            CONFIG_ID,

        "best_params":
            best_params,

        "selected_run":
            best_run_number,

        "selected_seed":
            int(
                best_record[
                    "Seed"
                ]
            ),

        "best_epoch":
            int(
                best_record[
                    "Best_Epoch"
                ]
            ),

        "model_state_dict":
            best_bundle[
                "model_state_dict"
            ],

        "feature_columns":
            feature_cols,

        "class_names":
            class_names,

        "number_features":
            NUMBER_FEATURES,

        "number_classes":
            NUMBER_CLASSES
    },
    BEST_MODEL_PATH
)


# ================================================================
# TRAINING HISTORY
# ================================================================

history_table = pd.DataFrame(
    {
        "Epoch":
            np.arange(
                1,
                len(
                    best_history[
                        "loss"
                    ]
                )
                + 1
            ),

        "Training_Loss":
            best_history[
                "loss"
            ],

        "Training_Accuracy":
            best_history[
                "accuracy"
            ],

        "Validation_Loss":
            best_history[
                "val_loss"
            ],

        "Validation_Accuracy":
            best_history[
                "val_accuracy"
            ],

        "Learning_Rate":
            best_history[
                "learning_rate"
            ]
    }
)


history_table.to_csv(
    OUTPUT_DIR
    / "training_history.csv",
    index=False
)


# ================================================================
# LOSS CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Loss"
    ],
    label=
        "Training Loss"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Loss"
    ],
    label=
        "Validation Loss"
)


ax.set_xlabel(
    "Epoch"
)


ax.set_ylabel(
    "Cross Entropy Loss"
)


ax.set_title(
    "CNN Loss Curve"
)


ax.legend()


ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "cnn_loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# ACCURACY CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Accuracy"
    ],
    label=
        "Training Accuracy"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Accuracy"
    ],
    label=
        "Validation Accuracy"
)


ax.set_xlabel(
    "Epoch"
)


ax.set_ylabel(
    "Accuracy"
)


ax.set_title(
    "CNN Accuracy Curve"
)


ax.legend()


ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "cnn_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# LEARNING RATE CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Learning_Rate"
    ]
)


ax.set_xlabel(
    "Epoch"
)


ax.set_ylabel(
    "Learning Rate"
)


ax.set_title(
    "CNN Learning Rate Schedule"
)


ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "cnn_learning_rate_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# FIVE RUN STATISTICS
# ================================================================

run_train_mean = float(
    run_table[
        "Training_Accuracy"
    ].mean()
)


run_train_std = float(
    run_table[
        "Training_Accuracy"
    ].std(
        ddof=0
    )
)


run_validation_mean = float(
    run_table[
        "Validation_Accuracy"
    ].mean()
)


run_validation_std = float(
    run_table[
        "Validation_Accuracy"
    ].std(
        ddof=0
    )
)


# ================================================================
# RELEASE FINAL TRAIN AND VALIDATION SCALED ARRAYS
#
# LOGO needs X_train_raw.
#
# Final test needs X_test.
# ================================================================

del X_train
del X_validation

gc.collect()


# ================================================================
# FULL LEAVE ONE GROUP OUT
#
# Held out LOGO participant is never used
# for early stopping.
#
# Epoch count comes from the independent
# validation participant set.
# ================================================================

print()

print("=" * 80)

print("FULL LEAVE ONE GROUP OUT")

print("=" * 80)


FIXED_LOGO_EPOCHS = max(
    1,
    int(
        best_record[
            "Best_Epoch"
        ]
    )
)


if LOGO_MAX_FOLDS is None:

    LOGO_FOLDS_TO_USE = (
        FULL_LOGO_FOLDS
    )


else:

    LOGO_FOLDS_TO_USE = min(
        int(
            LOGO_MAX_FOLDS
        ),
        FULL_LOGO_FOLDS
    )


print(
    "Available LOGO folds:",
    FULL_LOGO_FOLDS
)

print(
    "LOGO folds requested:",
    LOGO_FOLDS_TO_USE
)

print(
    "Fixed LOGO epochs:",
    FIXED_LOGO_EPOCHS
)


# ================================================================
# LOGO CONFIGURATION ID
# ================================================================

logo_configuration_text = json.dumps(
    {
        "Config_ID":
            CONFIG_ID,

        "fixed_epochs":
            FIXED_LOGO_EPOCHS,

        "selected_seed":
            int(
                best_record[
                    "Seed"
                ]
            )
    },
    sort_keys=True
)


LOGO_CONFIG_ID = hashlib.sha256(
    logo_configuration_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


# ================================================================
# LOAD LOGO CHECKPOINT
# ================================================================

logo_records = []

completed_logo_persons = set()


if LOGO_RESULT_PATH.exists():

    previous_logo = pd.read_csv(
        LOGO_RESULT_PATH
    )


    if (
        "Config_ID"
        in previous_logo.columns
    ):

        matching_logo = previous_logo[
            previous_logo[
                "Config_ID"
            ].astype(str)
            == LOGO_CONFIG_ID
        ].copy()


        if len(
            matching_logo
        ) > 0:

            logo_records = matching_logo.to_dict(
                orient="records"
            )


            completed_logo_persons = set(
                matching_logo[
                    "Held_Out_Person"
                ].astype(str)
            )


print(
    "Previously completed LOGO folds:",
    len(
        completed_logo_persons
    )
)


# ================================================================
# RUN LOGO
# ================================================================

logo_start = time.time()


logo_splitter = LeaveOneGroupOut()


for fold_number, (
    training_index,
    validation_index
) in enumerate(
    logo_splitter.split(
        X_train_raw,
        y_train,
        g_train
    ),
    start=1
):

    if fold_number > LOGO_FOLDS_TO_USE:

        break


    held_out_persons = np.unique(
        g_train[
            validation_index
        ]
    )


    if len(
        held_out_persons
    ) != 1:

        raise RuntimeError(
            "Unexpected LOGO participant grouping."
        )


    held_out_person = str(
        held_out_persons[
            0
        ]
    )


    if held_out_person in completed_logo_persons:

        print(
            f"LOGO "
            f"{fold_number}/"
            f"{LOGO_FOLDS_TO_USE}, "
            f"Participant "
            f"{held_out_person} "
            f"already completed."
        )

        continue


    fold_start = time.time()


    # ============================================================
    # FOLD SPECIFIC SCALING
    # ============================================================

    (
        X_logo_training,
        X_logo_validation
    ) = prepare_fold(
        training_index,
        validation_index
    )


    # ============================================================
    # FIXED EPOCH TRAINING
    #
    # No held out LOGO participant is used
    # for early stopping.
    # ============================================================

    logo_model, _, _ = fit_cnn(
        best_params,
        X_logo_training,
        y_train[
            training_index
        ],
        X_validation=None,
        y_validation=None,
        seed=
            int(
                best_record[
                    "Seed"
                ]
            ),
        epochs=
            FIXED_LOGO_EPOCHS,
        use_early_stopping=
            False,
        verbose=
            False
    )


    logo_probabilities = predict_probabilities(
        logo_model,
        X_logo_validation,
        y_train[
            validation_index
        ]
    )


    logo_predictions = np.argmax(
        logo_probabilities,
        axis=1
    )


    fold_accuracy = accuracy_score(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_precision = precision_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_recall = recall_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_f1 = f1_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_time = (
        time.time()
        - fold_start
    )


    logo_record = {
        "Config_ID":
            LOGO_CONFIG_ID,

        "Fold":
            fold_number,

        "Held_Out_Person":
            held_out_person,

        "Rows":
            int(
                len(
                    validation_index
                )
            ),

        "Accuracy":
            float(
                fold_accuracy
            ),

        "Precision_Weighted":
            float(
                fold_precision
            ),

        "Recall_Weighted":
            float(
                fold_recall
            ),

        "F1_Weighted":
            float(
                fold_f1
            ),

        "Fixed_Epochs":
            int(
                FIXED_LOGO_EPOCHS
            ),

        "Execution_Time_Seconds":
            float(
                fold_time
            )
    }


    logo_records.append(
        logo_record
    )


    logo_table_temp = pd.DataFrame(
        logo_records
    )


    atomic_csv(
        logo_table_temp,
        LOGO_RESULT_PATH
    )


    completed_logo_persons.add(
        held_out_person
    )


    print(
        f"LOGO "
        f"{fold_number}/"
        f"{LOGO_FOLDS_TO_USE}, "
        f"Participant = "
        f"{held_out_person}, "
        f"Accuracy = "
        f"{fold_accuracy:.4f}, "
        f"F1 = "
        f"{fold_f1:.4f}, "
        f"Time = "
        f"{fold_time / 60:.2f} min"
    )


    del logo_model
    del logo_probabilities
    del logo_predictions
    del X_logo_training
    del X_logo_validation


    clear_device()


logo_time = (
    time.time()
    - logo_start
)


# ================================================================
# LOGO SUMMARY
# ================================================================

logo_table = pd.read_csv(
    LOGO_RESULT_PATH
)


logo_table = logo_table[
    logo_table[
        "Config_ID"
    ].astype(str)
    == LOGO_CONFIG_ID
].copy()


logo_table = (
    logo_table
    .sort_values(
        "Fold"
    )
    .drop_duplicates(
        subset=[
            "Held_Out_Person"
        ],
        keep="last"
    )
)


logo_scores = logo_table[
    "Accuracy"
].astype(float).to_numpy()


logo_mean = float(
    np.mean(
        logo_scores
    )
)


logo_std = float(
    np.std(
        logo_scores
    )
)


print()

print(
    "LOGO Accuracy:",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Completed LOGO folds:",
    len(
        logo_table
    )
)

print(
    "LOGO time this session:",
    f"{logo_time / 60:.2f}",
    "minutes"
)


# ================================================================
# FINAL UNTOUCHED TEST EVALUATION
# ================================================================

print()

print("=" * 80)

print("FINAL UNTOUCHED TEST EVALUATION")

print("=" * 80)


test_start = time.time()


test_probabilities = predict_probabilities(
    best_model,
    X_test,
    y_test
)


test_predictions = np.argmax(
    test_probabilities,
    axis=1
)


test_time = (
    time.time()
    - test_start
)


# ================================================================
# TEST METRICS
# ================================================================

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)


test_precision = precision_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_recall = recall_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_weighted = f1_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_macro = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0
)


test_kappa = cohen_kappa_score(
    y_test,
    test_predictions
)


test_mcc = matthews_corrcoef(
    y_test,
    test_predictions
)


# ================================================================
# ROC AUC
# ================================================================

try:

    test_roc_auc = roc_auc_score(
        y_test,
        test_probabilities,
        multi_class="ovr",
        average="weighted"
    )


except Exception:

    test_roc_auc = np.nan


# ================================================================
# CLASSIFICATION REPORT
# ================================================================

report = classification_report(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    ),
    target_names=
        class_names,
    digits=4,
    zero_division=0
)


print()

print("CLASSIFICATION REPORT")

print()

print(
    report
)


with open(
    OUTPUT_DIR
    / "classification_report.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        report
    )


report_table = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        labels=np.arange(
            NUMBER_CLASSES
        ),
        target_names=
            class_names,
        output_dict=True,
        zero_division=0
    )
).transpose()


report_table.to_csv(
    OUTPUT_DIR
    / "classification_report.csv"
)


# ================================================================
# CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    )
)


pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
).to_csv(
    OUTPUT_DIR
    / "confusion_matrix.csv"
)


# ================================================================
# CONFUSION MATRIX FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "CNN Confusion Matrix"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            str(
                cm[
                    row,
                    column
                ]
            ),
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# NORMALIZED CONFUSION MATRIX
# ================================================================

row_sums = cm.sum(
    axis=1,
    keepdims=True
)


normalized_cm = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(
        cm,
        dtype=np.float64
    ),
    where=(
        row_sums != 0
    )
)


pd.DataFrame(
    normalized_cm,
    index=class_names,
    columns=class_names
).to_csv(
    OUTPUT_DIR
    / "normalized_confusion_matrix.csv"
)


# ================================================================
# NORMALIZED CONFUSION MATRIX FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    normalized_cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "Normalized CNN Confusion Matrix"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            f"{normalized_cm[row, column]:.2f}",
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "normalized_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# SAVE TEST PREDICTIONS
# ================================================================

prediction_table = pd.DataFrame(
    {
        "PersonSrial":
            g_test,

        "True_Class":
            label_encoder.inverse_transform(
                y_test
            ),

        "Predicted_Class":
            label_encoder.inverse_transform(
                test_predictions.astype(
                    np.int32
                )
            )
    }
)


for class_index, class_name in enumerate(
    class_names
):

    prediction_table[
        f"Probability_{class_name}"
    ] = test_probabilities[
        :,
        class_index
    ]


prediction_table.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False
)


# ================================================================
# TOTAL EXECUTION TIME
# ================================================================

total_execution_time = (
    time.time()
    - TOTAL_START
)


# ================================================================
# FINAL METRICS
# ================================================================

final_metrics = {
    "Algorithm":
        "Optimized 1D CNN",

    "Framework":
        "PyTorch",

    "Dataset":
        "Max Dataset",

    "Input_File":
        str(
            FILE_PATH
        ),

    "Output_Directory":
        str(
            OUTPUT_DIR
        ),

    "Device_Type":
        DEVICE_TYPE,

    "Device_Name":
        DEVICE_NAME,

    "Feature_Count":
        int(
            NUMBER_FEATURES
        ),

    "Number_of_Classes":
        int(
            NUMBER_CLASSES
        ),

    "Classes":
        class_names,

    "Training_Participants":
        int(
            len(
                train_persons
            )
        ),

    "Validation_Participants":
        int(
            len(
                validation_persons
            )
        ),

    "Testing_Participants":
        int(
            len(
                test_persons
            )
        ),

    "Training_Samples":
        int(
            TRAIN_SAMPLES
        ),

    "Validation_Samples":
        int(
            VALIDATION_SAMPLES
        ),

    "Testing_Samples":
        int(
            TEST_SAMPLES
        ),

    "Optuna_Trials":
        int(
            N_TRIALS
        ),

    "Optuna_GroupKFold_Accuracy":
        float(
            study.best_value
        ),

    "Best_Parameters":
        best_params,

    "Selected_Run":
        int(
            best_record[
                "Run"
            ]
        ),

    "Selected_Seed":
        int(
            best_record[
                "Seed"
            ]
        ),

    "Selected_Best_Epoch":
        int(
            best_record[
                "Best_Epoch"
            ]
        ),

    "Training_Accuracy":
        float(
            best_record[
                "Training_Accuracy"
            ]
        ),

    "Validation_Accuracy":
        float(
            best_record[
                "Validation_Accuracy"
            ]
        ),

    "Testing_Accuracy":
        float(
            test_accuracy
        ),

    "Five_Run_Training_Mean":
        float(
            run_train_mean
        ),

    "Five_Run_Training_STD":
        float(
            run_train_std
        ),

    "Five_Run_Validation_Mean":
        float(
            run_validation_mean
        ),

    "Five_Run_Validation_STD":
        float(
            run_validation_std
        ),

    "Full_LOGO_Accuracy":
        float(
            logo_mean
        ),

    "Full_LOGO_STD":
        float(
            logo_std
        ),

    "Full_LOGO_Folds_Available":
        int(
            FULL_LOGO_FOLDS
        ),

    "Full_LOGO_Folds_Completed":
        int(
            len(
                logo_table
            )
        ),

    "Precision_Weighted":
        float(
            test_precision
        ),

    "Recall_Weighted":
        float(
            test_recall
        ),

    "F1_Weighted":
        float(
            test_f1_weighted
        ),

    "F1_Macro":
        float(
            test_f1_macro
        ),

    "Cohen_Kappa":
        float(
            test_kappa
        ),

    "MCC":
        float(
            test_mcc
        ),

    "ROC_AUC_Weighted_OVR":
        (
            None
            if np.isnan(
                test_roc_auc
            )
            else float(
                test_roc_auc
            )
        ),

    "Optuna_Time_Seconds":
        float(
            optuna_time
        ),

    "Five_Run_Time_This_Session_Seconds":
        float(
            five_run_time
        ),

    "LOGO_Time_This_Session_Seconds":
        float(
            logo_time
        ),

    "Test_Time_Seconds":
        float(
            test_time
        ),

    "Total_Execution_Time_Seconds":
        float(
            total_execution_time
        ),

    "Current_RAM_MB":
        float(
            mem_mb()
        )
}


with open(
    OUTPUT_DIR
    / "final_metrics.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        final_metrics,
        file,
        indent=4
    )


# ================================================================
# FINAL SUMMARY TEXT
# ================================================================

summary_lines = [
    "OPTIMIZED 1D CNN",
    "MAX DATASET",

    "",

    f"Input File: "
    f"{FILE_PATH}",

    f"Output Directory: "
    f"{OUTPUT_DIR}",

    "",

    f"Device Type: "
    f"{DEVICE_TYPE}",

    f"Device Name: "
    f"{DEVICE_NAME}",

    "",

    f"Feature Count: "
    f"{NUMBER_FEATURES}",

    f"Classes: "
    f"{class_names}",

    "",

    f"Training Participants: "
    f"{len(train_persons)}",

    f"Validation Participants: "
    f"{len(validation_persons)}",

    f"Testing Participants: "
    f"{len(test_persons)}",

    "",

    f"Training Samples: "
    f"{TRAIN_SAMPLES}",

    f"Validation Samples: "
    f"{VALIDATION_SAMPLES}",

    f"Testing Samples: "
    f"{TEST_SAMPLES}",

    "",

    f"Best Parameters: "
    f"{best_params}",

    "",

    f"Selected Run: "
    f"{best_record['Run']}",

    f"Selected Seed: "
    f"{best_record['Seed']}",

    f"Best Epoch: "
    f"{best_record['Best_Epoch']}",

    "",

    f"Optuna GroupKFold Accuracy: "
    f"{study.best_value:.4f}",

    f"Training Accuracy: "
    f"{best_record['Training_Accuracy']:.4f}",

    f"Validation Accuracy: "
    f"{best_record['Validation_Accuracy']:.4f}",

    f"Testing Accuracy: "
    f"{test_accuracy:.4f}",

    "",

    f"Five Run Training Accuracy: "
    f"{run_train_mean:.4f} "
    f"plus or minus "
    f"{run_train_std:.4f}",

    f"Five Run Validation Accuracy: "
    f"{run_validation_mean:.4f} "
    f"plus or minus "
    f"{run_validation_std:.4f}",

    "",

    f"Full LOGO Accuracy: "
    f"{logo_mean:.4f} "
    f"plus or minus "
    f"{logo_std:.4f}",

    f"Full LOGO Folds Completed: "
    f"{len(logo_table)}",

    "",

    f"Precision Weighted: "
    f"{test_precision:.4f}",

    f"Recall Weighted: "
    f"{test_recall:.4f}",

    f"F1 Weighted: "
    f"{test_f1_weighted:.4f}",

    f"F1 Macro: "
    f"{test_f1_macro:.4f}",

    f"Cohen Kappa: "
    f"{test_kappa:.4f}",

    f"MCC: "
    f"{test_mcc:.4f}",

    (
        "ROC AUC Weighted OVR: N/A"
        if np.isnan(
            test_roc_auc
        )
        else
        f"ROC AUC Weighted OVR: "
        f"{test_roc_auc:.4f}"
    ),

    "",

    f"Optuna Time: "
    f"{optuna_time / 60:.2f} minutes",

    f"Five Run Time This Session: "
    f"{five_run_time / 60:.2f} minutes",

    f"LOGO Time This Session: "
    f"{logo_time / 60:.2f} minutes",

    f"Test Prediction Time: "
    f"{test_time:.2f} seconds",

    f"Total Execution Time: "
    f"{total_execution_time / 60:.2f} minutes"
]


with open(
    OUTPUT_DIR
    / "final_summary.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        "\n".join(
            summary_lines
        )
    )


# ================================================================
# FINAL SCREEN OUTPUT
# ================================================================

print()

print("#" * 80)

print("FINAL CNN RESULTS")

print("MAX DATASET")

print("#" * 80)


print(
    "Device                    :",
    DEVICE
)

print(
    "Device Name               :",
    DEVICE_NAME
)

print(
    "Feature Count             :",
    NUMBER_FEATURES
)

print()

print(
    "Best Conv Blocks          :",
    best_params[
        "n_conv_blocks"
    ]
)

print(
    "Best Base Filters         :",
    best_params[
        "base_filters"
    ]
)

print(
    "Best Kernel Size          :",
    best_params[
        "kernel_size"
    ]
)

print(
    "Best Conv Dropout         :",
    f"{best_params['conv_dropout']:.4f}"
)

print(
    "Best Dense Units          :",
    best_params[
        "dense_units"
    ]
)

print(
    "Best Dense Dropout        :",
    f"{best_params['dense_dropout']:.4f}"
)

print(
    "Best Learning Rate        :",
    f"{best_params['learning_rate']:.2e}"
)

print(
    "Best Weight Decay         :",
    f"{best_params['weight_decay']:.2e}"
)

print(
    "Best Label Smoothing      :",
    f"{best_params['label_smoothing']:.4f}"
)

print()

print(
    "Selected Run              :",
    best_record[
        "Run"
    ]
)

print(
    "Selected Seed             :",
    best_record[
        "Seed"
    ]
)

print(
    "Best Epoch                :",
    best_record[
        "Best_Epoch"
    ]
)

print()

print(
    "Optuna GroupKFold         :",
    f"{study.best_value:.4f}"
)

print(
    "Training Accuracy         :",
    f"{best_record['Training_Accuracy']:.4f}"
)

print(
    "Validation Accuracy       :",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Testing Accuracy          :",
    f"{test_accuracy:.4f}"
)

print()

print(
    "Five Run Train            :",
    f"{run_train_mean:.4f}",
    "plus or minus",
    f"{run_train_std:.4f}"
)

print(
    "Five Run Validation       :",
    f"{run_validation_mean:.4f}",
    "plus or minus",
    f"{run_validation_std:.4f}"
)

print()

print(
    "Full LOGO Accuracy        :",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Full LOGO Folds           :",
    len(
        logo_table
    ),
    "of",
    FULL_LOGO_FOLDS
)

print()

print(
    "Precision Weighted        :",
    f"{test_precision:.4f}"
)

print(
    "Recall Weighted           :",
    f"{test_recall:.4f}"
)

print(
    "F1 Weighted               :",
    f"{test_f1_weighted:.4f}"
)

print(
    "F1 Macro                  :",
    f"{test_f1_macro:.4f}"
)

print(
    "Cohen Kappa               :",
    f"{test_kappa:.4f}"
)

print(
    "MCC                       :",
    f"{test_mcc:.4f}"
)


if np.isnan(
    test_roc_auc
):

    print(
        "ROC AUC Weighted OVR      : N/A"
    )


else:

    print(
        "ROC AUC Weighted OVR      :",
        f"{test_roc_auc:.4f}"
    )


print()

print(
    "Optuna Time               :",
    f"{optuna_time / 60:.2f}",
    "minutes"
)

print(
    "Five Run Time             :",
    f"{five_run_time / 60:.2f}",
    "minutes"
)

print(
    "Full LOGO Time            :",
    f"{logo_time / 60:.2f}",
    "minutes"
)

print(
    "Test Prediction Time      :",
    f"{test_time:.2f}",
    "seconds"
)

print(
    "Total Execution Time      :",
    f"{total_execution_time / 60:.2f}",
    "minutes"
)

print(
    "Current RAM               :",
    f"{mem_mb():.2f}",
    "MB"
)

print()

print(
    "No dataset rows were subsampled."
)

print(
    "All five GroupKFold folds were used."
)

print(
    "Fold specific StandardScaler fitting was used."
)

print(
    "Best run was selected using validation participants."
)

print(
    "Test participants were not used for tuning or model selection."
)

print(
    "LOGO participants were not used for early stopping."
)

print(
    "Optuna and LOGO support resume after interruption."
)

print()

print(
    "ALL RESULTS SAVED IN:"
)

print(
    OUTPUT_DIR
)

print("#" * 80)

/home/rakib/Desktop/ml_apply/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU AND DEVICE CONFIGURATION
PyTorch version: 2.14.1+xpu
Selected device: xpu:0
Device type: xpu
Device name: Intel(R) Arc(TM) A750 Graphics
CPU thread limit: 8

OPTIMIZED 1D CNN
MAX DATASET
Input dataset:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/Cleaned_oN_Selected_Max.csv

Result folder:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/CNN_Results

Optuna trials: 10
GroupKFold folds: 5
Final runs: 5
Maximum epochs: 50
Batch size: 1024
Early stopping patience: 6
LOGO cap: None
Data subsampling: NONE

LOADING COMPLETE MAX DATASET
Raw dataset shape: (149350, 238)
RAM after loading: 982.94 MB
Feature count: 232
Invalid feature cells: 0

Feature matrix: (149350, 232)
Classes: ['class_lecture', 'music', 'normal', 'reading']
Number of classes: 4
Participants: 60
Loading time: 8.14 seconds
RAM: 1119.87 MB

CLASS DISTRIBUTION
class_lecture: 39857 (26.69%)
music: 32276 (21.61%)
normal: 39327 (26.33%)
reading: 37890 (25.37%)

PERSON WISE 70 / 20 / 10 SPLIT
Training participants: 42
Va

**Apply EEG-Conformer + 1D-TCN (Dual-Stream Cross-Attention Network) on Max Dataset**

In [2]:
# ================================================================
# EEG CONFORMER + 1D TCN
# DUAL STREAM CROSS ATTENTION NETWORK
# MAX DATASET
#
# PYTORCH
# INTEL XPU GPU SUPPORT
#
# COMPLETE DATASET
# PERSON WISE 70 / 20 / 10
# FOLD SPECIFIC STANDARDIZATION
#
# OPTUNA 10 TRIALS
# GROUPKFOLD 5
# FIVE STOCHASTIC RUNS
# FULL LOGO
#
# VALIDATION BASED EARLY STOPPING
# VALIDATION BASED MODEL SELECTION
#
# ADAMW
# CLASS WEIGHTED CROSS ENTROPY
# LABEL SMOOTHING
# GRADIENT CLIPPING
# LEARNING RATE REDUCTION
#
# CONFORMER STREAM
# TCN STREAM
# BIDIRECTIONAL CROSS ATTENTION
#
# RESUMABLE OPTUNA
# RESUMABLE FIVE RUNS
# RESUMABLE FULL LOGO
#
# TEST DATA USED ONLY FOR FINAL EVALUATION
# ================================================================


# ================================================================
# IMPORTS
# ================================================================

import os
import gc
import json
import time
import copy
import math
import random
import hashlib
import warnings

from pathlib import Path

import joblib
import psutil
import optuna

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import (
    GroupKFold,
    LeaveOneGroupOut
)

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.utils.class_weight import (
    compute_class_weight
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ================================================================
# WARNINGS
# ================================================================

warnings.filterwarnings(
    "ignore"
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# ================================================================
# GENERAL CONFIGURATION
# ================================================================

SEED = 42

N_TRIALS = 10

N_RUNS = 5

GROUP_K_FOLDS = 5


# ================================================================
# TRAINING CONFIGURATION
# ================================================================

MAX_EPOCHS = 40

TRAIN_BATCH_SIZE = 512

EARLY_STOPPING_PATIENCE = 5

LR_SCHEDULER_PATIENCE = 2

GRADIENT_CLIP = 1.0


# ================================================================
# TOKENIZATION CONFIGURATION
#
# 232 feature values are converted into feature tokens.
#
# Stride 4 greatly reduces attention memory consumption.
# ================================================================

PATCH_KERNEL_SIZE = 7

PATCH_STRIDE = 4


# ================================================================
# FULL LOGO
#
# None means every training participant.
# ================================================================

LOGO_MAX_FOLDS = None


# ================================================================
# CPU THREAD CONTROL
# ================================================================

CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)


torch.set_num_threads(
    CPU_THREADS
)


# ================================================================
# INPUT
# ================================================================

FILE_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset/"
    "Cleaned_oN_Selected_Max.csv"
)


if not FILE_PATH.exists():

    raise FileNotFoundError(
        f"Input dataset was not found: {FILE_PATH}"
    )


# ================================================================
# OUTPUT ROOT
# ================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset"
)


# ================================================================
# ALGORITHM OUTPUT DIRECTORY
# ================================================================

OUTPUT_DIR = (
    OUTPUT_ROOT
    / "EEG_Conformer_TCN_CrossAttention_Results"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# CHECKPOINT DIRECTORY
# ================================================================

CHECKPOINT_DIR = (
    OUTPUT_DIR
    / "checkpoints"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# IMPORTANT OUTPUT PATHS
# ================================================================

OPTUNA_DATABASE = (
    OUTPUT_DIR
    / "optuna.db"
)


OPTUNA_STORAGE = (
    f"sqlite:///{OPTUNA_DATABASE}"
)


OPTUNA_STUDY_NAME = (
    "Max_EEG_Conformer_TCN_CrossAttention_v1"
)


FIVE_RUNS_PATH = (
    OUTPUT_DIR
    / "five_runs.csv"
)


LOGO_RESULT_PATH = (
    OUTPUT_DIR
    / "full_logo_results.csv"
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "best_model.pt"
)


# ================================================================
# MEMORY MONITOR
# ================================================================

PROC = psutil.Process(
    os.getpid()
)


def mem_mb():

    return (
        PROC.memory_info().rss
        / 1024
        / 1024
    )


# ================================================================
# ATOMIC CSV SAVE
# ================================================================

def atomic_csv(
    dataframe,
    path
):

    path = Path(
        path
    )


    temporary_path = path.with_name(
        path.name
        + ".tmp"
    )


    dataframe.to_csv(
        temporary_path,
        index=False
    )


    os.replace(
        temporary_path,
        path
    )


# ================================================================
# ATOMIC TORCH SAVE
# ================================================================

def atomic_torch_save(
    obj,
    path
):

    path = Path(
        path
    )


    temporary_path = path.with_name(
        path.name
        + ".tmp"
    )


    torch.save(
        obj,
        temporary_path
    )


    os.replace(
        temporary_path,
        path
    )


# ================================================================
# RANDOM SEED
# ================================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if (
        hasattr(
            torch,
            "xpu"
        )
        and
        torch.xpu.is_available()
    ):

        try:

            torch.xpu.manual_seed_all(
                seed
            )

        except Exception:

            pass


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    SEED
)


# ================================================================
# DEVICE DETECTION
#
# PRIORITY
#
# Intel XPU
# CUDA
# CPU
# ================================================================

if (
    hasattr(
        torch,
        "xpu"
    )
    and
    torch.xpu.is_available()
):

    DEVICE = torch.device(
        "xpu:0"
    )

    DEVICE_TYPE = "xpu"

    DEVICE_NAME = torch.xpu.get_device_name(
        0
    )


elif torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda:0"
    )

    DEVICE_TYPE = "cuda"

    DEVICE_NAME = torch.cuda.get_device_name(
        0
    )


else:

    DEVICE = torch.device(
        "cpu"
    )

    DEVICE_TYPE = "cpu"

    DEVICE_NAME = "CPU"


# ================================================================
# DEVICE HELPERS
# ================================================================

def synchronize():

    if DEVICE_TYPE == "xpu":

        torch.xpu.synchronize()


    elif DEVICE_TYPE == "cuda":

        torch.cuda.synchronize()


def clear_device():

    gc.collect()


    if DEVICE_TYPE == "xpu":

        torch.xpu.empty_cache()


    elif DEVICE_TYPE == "cuda":

        torch.cuda.empty_cache()


# ================================================================
# DEVICE INFORMATION
# ================================================================

print("=" * 80)

print(
    "DEVICE CONFIGURATION"
)

print("=" * 80)

print(
    "PyTorch version:",
    torch.__version__
)

print(
    "Selected device:",
    DEVICE
)

print(
    "Device type:",
    DEVICE_TYPE
)

print(
    "Device name:",
    DEVICE_NAME
)

print(
    "CPU threads:",
    CPU_THREADS
)

print("=" * 80)


# ================================================================
# TOTAL TIMER
# ================================================================

TOTAL_START = time.time()


# ================================================================
# EXPERIMENT INFORMATION
# ================================================================

print()

print("=" * 80)

print(
    "EEG CONFORMER + 1D TCN"
)

print(
    "DUAL STREAM CROSS ATTENTION NETWORK"
)

print(
    "MAX DATASET"
)

print("=" * 80)


print(
    "Input dataset:"
)

print(
    FILE_PATH
)

print()

print(
    "Output directory:"
)

print(
    OUTPUT_DIR
)

print()

print(
    "GPU:",
    DEVICE_NAME
)

print(
    "Optuna trials:",
    N_TRIALS
)

print(
    "GroupKFold folds:",
    GROUP_K_FOLDS
)

print(
    "Final runs:",
    N_RUNS
)

print(
    "Maximum epochs:",
    MAX_EPOCHS
)

print(
    "Batch size:",
    TRAIN_BATCH_SIZE
)

print(
    "Patch kernel:",
    PATCH_KERNEL_SIZE
)

print(
    "Patch stride:",
    PATCH_STRIDE
)

print(
    "LOGO cap:",
    LOGO_MAX_FOLDS
)

print(
    "Row subsampling: NONE"
)

print("=" * 80)


# ================================================================
# LOAD MAX DATASET
# ================================================================

print()

print("=" * 80)

print(
    "LOADING COMPLETE MAX DATASET"
)

print("=" * 80)


load_start = time.time()


df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)


print(
    "Raw dataset shape:",
    df.shape
)

print(
    "RAM after loading:",
    f"{mem_mb():.2f} MB"
)


# ================================================================
# COLUMN DEFINITIONS
# ================================================================

metadata_cols = [
    "PersonSrial",
    "Window",
    "Window_Start",
    "Window_End",
    "Window_Samples"
]


target_col = "Class"


required_columns = [
    "PersonSrial",
    target_col
]


for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"Required column missing: {column}"
        )


# ================================================================
# FEATURE COLUMNS
# ================================================================

feature_cols = [
    column
    for column in df.columns
    if column not in metadata_cols
    and column != target_col
]


if len(
    feature_cols
) == 0:

    raise ValueError(
        "No feature columns were detected."
    )


NUMBER_FEATURES = len(
    feature_cols
)


print(
    "Feature count:",
    NUMBER_FEATURES
)


# ================================================================
# FEATURE MATRIX
# ================================================================

X_all = (
    df[
        feature_cols
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(
        dtype=np.float32
    )
)


# ================================================================
# INVALID VALUES
# ================================================================

invalid_count = int(
    (
        ~np.isfinite(
            X_all
        )
    ).sum()
)


print(
    "Invalid feature cells:",
    invalid_count
)


if invalid_count > 0:

    raise ValueError(
        "Feature matrix contains NaN or infinite values."
    )


# ================================================================
# LABELS
# ================================================================

label_encoder = LabelEncoder()


y_all = label_encoder.fit_transform(
    df[
        target_col
    ].astype(str)
).astype(
    np.int64
)


class_names = list(
    label_encoder.classes_
)


NUMBER_CLASSES = len(
    class_names
)


# ================================================================
# PARTICIPANTS
# ================================================================

groups_all = (
    df[
        "PersonSrial"
    ]
    .to_numpy()
)


print()

print(
    "Feature matrix:",
    X_all.shape
)

print(
    "Classes:",
    class_names
)

print(
    "Number of classes:",
    NUMBER_CLASSES
)

print(
    "Participants:",
    len(
        np.unique(
            groups_all
        )
    )
)

print(
    "Loading time:",
    f"{time.time() - load_start:.2f}",
    "seconds"
)

print(
    "RAM:",
    f"{mem_mb():.2f}",
    "MB"
)


# ================================================================
# CLASS DISTRIBUTION
# ================================================================

print()

print(
    "CLASS DISTRIBUTION"
)


class_distribution_records = []


for class_index, class_name in enumerate(
    class_names
):

    count = int(
        np.sum(
            y_all
            == class_index
        )
    )


    percentage = (
        count
        / len(
            y_all
        )
        * 100.0
    )


    print(
        f"{class_name}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


    class_distribution_records.append(
        {
            "Class":
                class_name,

            "Count":
                count,

            "Percentage":
                percentage
        }
    )


pd.DataFrame(
    class_distribution_records
).to_csv(
    OUTPUT_DIR
    / "class_distribution.csv",
    index=False
)


# ================================================================
# RELEASE DATAFRAME
# ================================================================

del df

gc.collect()


# ================================================================
# PERSON WISE 70 / 20 / 10 SPLIT
# ================================================================

print()

print("=" * 80)

print(
    "PERSON WISE 70 / 20 / 10 SPLIT"
)

print("=" * 80)


persons = np.unique(
    groups_all
)


rng = np.random.RandomState(
    SEED
)


rng.shuffle(
    persons
)


number_persons = len(
    persons
)


number_test = max(
    1,
    int(
        round(
            0.20
            * number_persons
        )
    )
)


number_validation = max(
    1,
    int(
        round(
            0.10
            * number_persons
        )
    )
)


test_persons = persons[
    :number_test
]


validation_persons = persons[
    number_test:
    number_test
    + number_validation
]


train_persons = persons[
    number_test
    + number_validation:
]


# ================================================================
# VERIFY NO OVERLAP
# ================================================================

assert (
    np.intersect1d(
        train_persons,
        validation_persons
    ).size == 0
)


assert (
    np.intersect1d(
        train_persons,
        test_persons
    ).size == 0
)


assert (
    np.intersect1d(
        validation_persons,
        test_persons
    ).size == 0
)


# ================================================================
# SPLIT MASKS
# ================================================================

train_mask = np.isin(
    groups_all,
    train_persons
)


validation_mask = np.isin(
    groups_all,
    validation_persons
)


test_mask = np.isin(
    groups_all,
    test_persons
)


# ================================================================
# CREATE SPLITS
# ================================================================

X_train_raw = np.ascontiguousarray(
    X_all[
        train_mask
    ],
    dtype=np.float32
)


y_train = np.ascontiguousarray(
    y_all[
        train_mask
    ],
    dtype=np.int64
)


g_train = np.ascontiguousarray(
    groups_all[
        train_mask
    ]
)


X_validation_raw = np.ascontiguousarray(
    X_all[
        validation_mask
    ],
    dtype=np.float32
)


y_validation = np.ascontiguousarray(
    y_all[
        validation_mask
    ],
    dtype=np.int64
)


g_validation = np.ascontiguousarray(
    groups_all[
        validation_mask
    ]
)


X_test_raw = np.ascontiguousarray(
    X_all[
        test_mask
    ],
    dtype=np.float32
)


y_test = np.ascontiguousarray(
    y_all[
        test_mask
    ],
    dtype=np.int64
)


g_test = np.ascontiguousarray(
    groups_all[
        test_mask
    ]
)


TRAIN_SAMPLES = len(
    X_train_raw
)


VALIDATION_SAMPLES = len(
    X_validation_raw
)


TEST_SAMPLES = len(
    X_test_raw
)


print(
    "Training participants:",
    len(
        train_persons
    )
)

print(
    "Validation participants:",
    len(
        validation_persons
    )
)

print(
    "Testing participants:",
    len(
        test_persons
    )
)

print()

print(
    "Training:",
    X_train_raw.shape
)

print(
    "Validation:",
    X_validation_raw.shape
)

print(
    "Testing:",
    X_test_raw.shape
)


# ================================================================
# SAVE PARTICIPANT SPLIT
# ================================================================

split_table = pd.DataFrame(
    [
        {
            "Split":
                "Training",

            "Participants":
                len(
                    train_persons
                ),

            "Rows":
                TRAIN_SAMPLES
        },

        {
            "Split":
                "Validation",

            "Participants":
                len(
                    validation_persons
                ),

            "Rows":
                VALIDATION_SAMPLES
        },

        {
            "Split":
                "Testing",

            "Participants":
                len(
                    test_persons
                ),

            "Rows":
                TEST_SAMPLES
        }
    ]
)


split_table.to_csv(
    OUTPUT_DIR
    / "split_summary.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "participant_split.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        {
            "training":
                [
                    str(x)
                    for x in train_persons
                ],

            "validation":
                [
                    str(x)
                    for x in validation_persons
                ],

            "testing":
                [
                    str(x)
                    for x in test_persons
                ]
        },
        file,
        indent=4
    )


# ================================================================
# RELEASE COMPLETE DATA
# ================================================================

del X_all
del y_all
del groups_all

del train_mask
del validation_mask
del test_mask

gc.collect()


print(
    "RAM after split:",
    f"{mem_mb():.2f} MB"
)


# ================================================================
# GROUPKFOLD
# ================================================================

gkf_splits = list(
    GroupKFold(
        n_splits=
            GROUP_K_FOLDS
    ).split(
        X_train_raw,
        y_train,
        g_train
    )
)


FULL_LOGO_FOLDS = len(
    np.unique(
        g_train
    )
)


print()

print(
    "GroupKFold folds:",
    len(
        gkf_splits
    )
)

print(
    "Available LOGO folds:",
    FULL_LOGO_FOLDS
)


# ================================================================
# FOLD SPECIFIC STANDARDIZATION
# ================================================================

def prepare_fold(
    training_index,
    validation_index
):

    scaler = StandardScaler()


    X_fold_training = scaler.fit_transform(
        X_train_raw[
            training_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    X_fold_validation = scaler.transform(
        X_train_raw[
            validation_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    return (
        np.ascontiguousarray(
            X_fold_training
        ),
        np.ascontiguousarray(
            X_fold_validation
        )
    )


# ================================================================
# FINAL STANDARDIZATION
#
# Scaler sees training participants only.
# ================================================================

print()

print("=" * 80)

print(
    "FINAL TRAINING SCALER"
)

print("=" * 80)


final_scaler = StandardScaler()


X_train = final_scaler.fit_transform(
    X_train_raw
).astype(
    np.float32,
    copy=False
)


X_validation = final_scaler.transform(
    X_validation_raw
).astype(
    np.float32,
    copy=False
)


X_test = final_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


X_train = np.ascontiguousarray(
    X_train
)


X_validation = np.ascontiguousarray(
    X_validation
)


X_test = np.ascontiguousarray(
    X_test
)


joblib.dump(
    final_scaler,
    OUTPUT_DIR
    / "standard_scaler.joblib"
)


joblib.dump(
    label_encoder,
    OUTPUT_DIR
    / "label_encoder.joblib"
)


with open(
    OUTPUT_DIR
    / "feature_columns.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        feature_cols,
        file,
        indent=4
    )


print(
    "Scaled training:",
    X_train.shape
)

print(
    "Scaled validation:",
    X_validation.shape
)

print(
    "Scaled testing:",
    X_test.shape
)


# ================================================================
# CLASS WEIGHTS
# ================================================================

def class_weight_tensor(
    y
):

    present_classes = np.unique(
        y
    )


    values = compute_class_weight(
        class_weight="balanced",
        classes=
            present_classes,
        y=y
    )


    weights = np.ones(
        NUMBER_CLASSES,
        dtype=np.float32
    )


    weights[
        present_classes
    ] = values.astype(
        np.float32
    )


    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


# ================================================================
# SINUSOIDAL POSITIONAL ENCODING
# ================================================================

class SinusoidalPositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        d_model,
        max_len=512
    ):

        super().__init__()


        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(
            1
        )


        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (
                -math.log(
                    10000.0
                )
                / d_model
            )
        )


        pe = torch.zeros(
            max_len,
            d_model,
            dtype=torch.float32
        )


        pe[
            :,
            0::2
        ] = torch.sin(
            position
            * div_term
        )


        pe[
            :,
            1::2
        ] = torch.cos(
            position
            * div_term
        )


        self.register_buffer(
            "pe",
            pe.unsqueeze(
                0
            ),
            persistent=False
        )


    def forward(
        self,
        x
    ):

        return (
            x
            + self.pe[
                :,
                :x.size(1),
                :
            ].to(
                dtype=x.dtype
            )
        )


# ================================================================
# CUSTOM MULTI HEAD ATTENTION
#
# Implemented directly using matrix multiplication.
#
# This avoids relying on backend specific optimized MHA kernels.
# ================================================================

class MultiHeadAttentionCore(
    nn.Module
):

    def __init__(
        self,
        d_model,
        num_heads,
        dropout
    ):

        super().__init__()


        if (
            d_model
            % num_heads
            != 0
        ):

            raise ValueError(
                "d_model must be divisible by num_heads."
            )


        self.d_model = (
            d_model
        )

        self.num_heads = (
            num_heads
        )

        self.head_dim = (
            d_model
            // num_heads
        )


        self.scale = (
            self.head_dim
            ** -0.5
        )


        self.q_projection = nn.Linear(
            d_model,
            d_model
        )


        self.k_projection = nn.Linear(
            d_model,
            d_model
        )


        self.v_projection = nn.Linear(
            d_model,
            d_model
        )


        self.output_projection = nn.Linear(
            d_model,
            d_model
        )


        self.attention_dropout = nn.Dropout(
            dropout
        )


        self.output_dropout = nn.Dropout(
            dropout
        )


    def _split_heads(
        self,
        x
    ):

        batch_size, sequence_length, _ = x.shape


        x = x.reshape(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        )


        return x.transpose(
            1,
            2
        )


    def forward(
        self,
        query,
        key,
        value
    ):

        q = self._split_heads(
            self.q_projection(
                query
            )
        )


        k = self._split_heads(
            self.k_projection(
                key
            )
        )


        v = self._split_heads(
            self.v_projection(
                value
            )
        )


        attention_scores = torch.matmul(
            q,
            k.transpose(
                -2,
                -1
            )
        )


        attention_scores = (
            attention_scores
            * self.scale
        )


        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )


        attention_weights = self.attention_dropout(
            attention_weights
        )


        context = torch.matmul(
            attention_weights,
            v
        )


        context = context.transpose(
            1,
            2
        ).contiguous()


        batch_size = context.shape[
            0
        ]


        sequence_length = context.shape[
            1
        ]


        context = context.reshape(
            batch_size,
            sequence_length,
            self.d_model
        )


        output = self.output_projection(
            context
        )


        output = self.output_dropout(
            output
        )


        return output


# ================================================================
# CONFORMER FEED FORWARD MODULE
# ================================================================

class ConformerFeedForward(
    nn.Module
):

    def __init__(
        self,
        d_model,
        expansion,
        dropout
    ):

        super().__init__()


        hidden_dimension = (
            d_model
            * expansion
        )


        self.network = nn.Sequential(
            nn.Linear(
                d_model,
                hidden_dimension
            ),
            nn.SiLU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dimension,
                d_model
            ),
            nn.Dropout(
                dropout
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# ================================================================
# CONFORMER CONVOLUTION MODULE
# ================================================================

class ConformerConvModule(
    nn.Module
):

    def __init__(
        self,
        d_model,
        kernel_size,
        dropout
    ):

        super().__init__()


        self.pointwise_in = nn.Conv1d(
            d_model,
            d_model * 2,
            kernel_size=1
        )


        self.depthwise = nn.Conv1d(
            d_model,
            d_model,
            kernel_size=
                kernel_size,
            padding=
                kernel_size
                // 2,
            groups=
                d_model,
            bias=False
        )


        self.batch_norm = nn.BatchNorm1d(
            d_model
        )


        self.pointwise_out = nn.Conv1d(
            d_model,
            d_model,
            kernel_size=1
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x
    ):

        x = x.transpose(
            1,
            2
        )


        x = self.pointwise_in(
            x
        )


        x = F.glu(
            x,
            dim=1
        )


        x = self.depthwise(
            x
        )


        x = self.batch_norm(
            x
        )


        x = F.silu(
            x
        )


        x = self.pointwise_out(
            x
        )


        x = self.dropout(
            x
        )


        return x.transpose(
            1,
            2
        )


# ================================================================
# CONFORMER BLOCK
# ================================================================

class ConformerBlock(
    nn.Module
):

    def __init__(
        self,
        d_model,
        num_heads,
        conv_kernel,
        dropout
    ):

        super().__init__()


        self.ffn1_norm = nn.LayerNorm(
            d_model
        )


        self.ffn1 = ConformerFeedForward(
            d_model,
            expansion=4,
            dropout=dropout
        )


        self.attention_norm = nn.LayerNorm(
            d_model
        )


        self.self_attention = MultiHeadAttentionCore(
            d_model,
            num_heads,
            dropout
        )


        self.conv_norm = nn.LayerNorm(
            d_model
        )


        self.conv_module = ConformerConvModule(
            d_model,
            conv_kernel,
            dropout
        )


        self.ffn2_norm = nn.LayerNorm(
            d_model
        )


        self.ffn2 = ConformerFeedForward(
            d_model,
            expansion=4,
            dropout=dropout
        )


        self.final_norm = nn.LayerNorm(
            d_model
        )


    def forward(
        self,
        x
    ):

        x = (
            x
            + 0.5
            * self.ffn1(
                self.ffn1_norm(
                    x
                )
            )
        )


        normalized = self.attention_norm(
            x
        )


        x = (
            x
            + self.self_attention(
                normalized,
                normalized,
                normalized
            )
        )


        x = (
            x
            + self.conv_module(
                self.conv_norm(
                    x
                )
            )
        )


        x = (
            x
            + 0.5
            * self.ffn2(
                self.ffn2_norm(
                    x
                )
            )
        )


        return self.final_norm(
            x
        )


# ================================================================
# TCN RESIDUAL BLOCK
# ================================================================

class TCNResidualBlock(
    nn.Module
):

    def __init__(
        self,
        channels,
        kernel_size,
        dilation,
        dropout
    ):

        super().__init__()


        padding = (
            dilation
            * (
                kernel_size
                - 1
            )
            // 2
        )


        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=
                kernel_size,
            dilation=
                dilation,
            padding=
                padding,
            bias=False
        )


        self.bn1 = nn.BatchNorm1d(
            channels
        )


        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=
                kernel_size,
            dilation=
                dilation,
            padding=
                padding,
            bias=False
        )


        self.bn2 = nn.BatchNorm1d(
            channels
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x
    ):

        residual = x


        x = self.conv1(
            x
        )


        x = self.bn1(
            x
        )


        x = F.gelu(
            x
        )


        x = self.dropout(
            x
        )


        x = self.conv2(
            x
        )


        x = self.bn2(
            x
        )


        x = F.gelu(
            x
        )


        x = self.dropout(
            x
        )


        return (
            x
            + residual
        )


# ================================================================
# DUAL STREAM NETWORK
# ================================================================

class EEGConformerTCNCrossAttention(
    nn.Module
):

    def __init__(
        self,
        params
    ):

        super().__init__()


        d_model = int(
            params[
                "d_model"
            ]
        )


        number_heads = int(
            params[
                "num_heads"
            ]
        )


        conformer_blocks = int(
            params[
                "n_conformer_blocks"
            ]
        )


        tcn_blocks = int(
            params[
                "n_tcn_blocks"
            ]
        )


        conformer_kernel = int(
            params[
                "conformer_kernel"
            ]
        )


        tcn_kernel = int(
            params[
                "tcn_kernel"
            ]
        )


        dropout = float(
            params[
                "dropout"
            ]
        )


        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )


        # ========================================================
        # CONFORMER STREAM STEM
        # ========================================================

        self.conformer_stem = nn.Sequential(
            nn.Conv1d(
                1,
                d_model,
                kernel_size=
                    PATCH_KERNEL_SIZE,
                stride=
                    PATCH_STRIDE,
                padding=
                    PATCH_KERNEL_SIZE
                    // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                d_model
            ),

            nn.GELU()
        )


        self.position_encoding = SinusoidalPositionalEncoding(
            d_model,
            max_len=512
        )


        self.conformer_blocks = nn.ModuleList(
            [
                ConformerBlock(
                    d_model=
                        d_model,
                    num_heads=
                        number_heads,
                    conv_kernel=
                        conformer_kernel,
                    dropout=
                        dropout
                )

                for _
                in range(
                    conformer_blocks
                )
            ]
        )


        # ========================================================
        # TCN STREAM STEM
        # ========================================================

        self.tcn_stem = nn.Sequential(
            nn.Conv1d(
                1,
                d_model,
                kernel_size=
                    PATCH_KERNEL_SIZE,
                stride=
                    PATCH_STRIDE,
                padding=
                    PATCH_KERNEL_SIZE
                    // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                d_model
            ),

            nn.GELU()
        )


        tcn_layers = []


        for block_index in range(
            tcn_blocks
        ):

            dilation = (
                2
                ** block_index
            )


            tcn_layers.append(
                TCNResidualBlock(
                    channels=
                        d_model,
                    kernel_size=
                        tcn_kernel,
                    dilation=
                        dilation,
                    dropout=
                        dropout
                )
            )


        self.tcn_blocks = nn.ModuleList(
            tcn_layers
        )


        # ========================================================
        # BIDIRECTIONAL CROSS ATTENTION
        #
        # Conformer queries TCN
        #
        # TCN queries Conformer
        # ========================================================

        self.conformer_to_tcn_norm = nn.LayerNorm(
            d_model
        )


        self.tcn_for_conformer_norm = nn.LayerNorm(
            d_model
        )


        self.tcn_to_conformer_norm = nn.LayerNorm(
            d_model
        )


        self.conformer_for_tcn_norm = nn.LayerNorm(
            d_model
        )


        self.conformer_queries_tcn = MultiHeadAttentionCore(
            d_model,
            number_heads,
            dropout
        )


        self.tcn_queries_conformer = MultiHeadAttentionCore(
            d_model,
            number_heads,
            dropout
        )


        self.conformer_fusion_norm = nn.LayerNorm(
            d_model
        )


        self.tcn_fusion_norm = nn.LayerNorm(
            d_model
        )


        self.cross_dropout = nn.Dropout(
            dropout
        )


        # ========================================================
        # FINAL FUSION
        #
        # conf
        # tcn
        # absolute difference
        # elementwise interaction
        # ========================================================

        fusion_dimension = (
            d_model
            * 4
        )


        self.classifier = nn.Sequential(
            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        # ========================================================
        # INPUT
        #
        # batch x features
        # ========================================================

        x = x.unsqueeze(
            1
        )


        # ========================================================
        # CONFORMER STREAM
        # ========================================================

        conformer = self.conformer_stem(
            x
        )


        conformer = conformer.transpose(
            1,
            2
        )


        conformer = self.position_encoding(
            conformer
        )


        for block in self.conformer_blocks:

            conformer = block(
                conformer
            )


        # ========================================================
        # TCN STREAM
        # ========================================================

        tcn = self.tcn_stem(
            x
        )


        for block in self.tcn_blocks:

            tcn = block(
                tcn
            )


        tcn = tcn.transpose(
            1,
            2
        )


        # ========================================================
        # CROSS ATTENTION
        # ========================================================

        conformer_query = self.conformer_to_tcn_norm(
            conformer
        )


        tcn_context = self.tcn_for_conformer_norm(
            tcn
        )


        conformer_cross = self.conformer_queries_tcn(
            conformer_query,
            tcn_context,
            tcn_context
        )


        conformer = self.conformer_fusion_norm(
            conformer
            + self.cross_dropout(
                conformer_cross
            )
        )


        tcn_query = self.tcn_to_conformer_norm(
            tcn
        )


        conformer_context = self.conformer_for_tcn_norm(
            conformer
        )


        tcn_cross = self.tcn_queries_conformer(
            tcn_query,
            conformer_context,
            conformer_context
        )


        tcn = self.tcn_fusion_norm(
            tcn
            + self.cross_dropout(
                tcn_cross
            )
        )


        # ========================================================
        # GLOBAL TOKEN POOLING
        # ========================================================

        conformer_vector = torch.mean(
            conformer,
            dim=1
        )


        tcn_vector = torch.mean(
            tcn,
            dim=1
        )


        difference_vector = torch.abs(
            conformer_vector
            - tcn_vector
        )


        interaction_vector = (
            conformer_vector
            * tcn_vector
        )


        fused = torch.cat(
            [
                conformer_vector,
                tcn_vector,
                difference_vector,
                interaction_vector
            ],
            dim=1
        )


        return self.classifier(
            fused
        )


# ================================================================
# BUILD MODEL
# ================================================================

def build_model(
    params,
    seed=SEED
):

    set_seed(
        seed
    )


    model = EEGConformerTCNCrossAttention(
        params
    )


    return model.to(
        DEVICE
    )


# ================================================================
# PARAMETER COUNT
# ================================================================

def count_trainable_parameters(
    model
):

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


# ================================================================
# DEVICE DATA
# ================================================================

class DeviceData:

    def __init__(
        self,
        X,
        y
    ):

        X_array = np.ascontiguousarray(
            X,
            dtype=np.float32
        )


        y_array = np.ascontiguousarray(
            y,
            dtype=np.int64
        )


        self.X = torch.from_numpy(
            X_array
        ).to(
            DEVICE
        )


        self.y = torch.from_numpy(
            y_array
        ).to(
            DEVICE
        )


    def __len__(
        self
    ):

        return int(
            self.y.shape[
                0
            ]
        )


# ================================================================
# BATCH GENERATOR
# ================================================================

def batch_indices(
    number_samples,
    shuffle=True
):

    if shuffle:

        indices = torch.randperm(
            number_samples,
            device=DEVICE
        )


    else:

        indices = torch.arange(
            number_samples,
            device=DEVICE
        )


    for start in range(
        0,
        number_samples,
        TRAIN_BATCH_SIZE
    ):

        yield indices[
            start:
            start
            + TRAIN_BATCH_SIZE
        ]


# ================================================================
# TRAIN ONE EPOCH
# ================================================================

def train_one_epoch(
    model,
    data,
    optimizer,
    criterion
):

    model.train()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    for indices in batch_indices(
        len(
            data
        ),
        shuffle=True
    ):

        X_batch = data.X[
            indices
        ]


        y_batch = data.y[
            indices
        ]


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            X_batch
        )


        loss = criterion(
            logits,
            y_batch
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )


        optimizer.step()


        batch_size = int(
            y_batch.shape[
                0
            ]
        )


        total_loss += (
            float(
                loss.detach().item()
            )
            * batch_size
        )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


    synchronize()


    return (
        total_loss
        / total_samples,

        total_correct
        / total_samples
    )


# ================================================================
# EVALUATION
# ================================================================

@torch.inference_mode()
def evaluate_model(
    model,
    data,
    criterion=None,
    return_probabilities=False
):

    model.eval()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    probability_parts = []


    for indices in batch_indices(
        len(
            data
        ),
        shuffle=False
    ):

        X_batch = data.X[
            indices
        ]


        y_batch = data.y[
            indices
        ]


        logits = model(
            X_batch
        )


        batch_size = int(
            y_batch.shape[
                0
            ]
        )


        if criterion is not None:

            loss = criterion(
                logits,
                y_batch
            )


            total_loss += (
                float(
                    loss.item()
                )
                * batch_size
            )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


        if return_probabilities:

            probability_parts.append(
                torch.softmax(
                    logits,
                    dim=1
                )
                .cpu()
            )


    synchronize()


    accuracy = (
        total_correct
        / total_samples
    )


    if criterion is None:

        mean_loss = np.nan


    else:

        mean_loss = (
            total_loss
            / total_samples
        )


    if return_probabilities:

        probabilities = (
            torch.cat(
                probability_parts,
                dim=0
            )
            .numpy()
        )


        return (
            mean_loss,
            accuracy,
            probabilities
        )


    return (
        mean_loss,
        accuracy
    )


# ================================================================
# TRAIN MODEL
# ================================================================

def fit_model(
    params,
    X_training,
    y_training,
    X_validation=None,
    y_validation=None,
    seed=SEED,
    epochs=MAX_EPOCHS,
    use_early_stopping=True,
    verbose=False
):

    clear_device()


    model = build_model(
        params,
        seed
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(
            params[
                "learning_rate"
            ]
        ),
        weight_decay=float(
            params[
                "weight_decay"
            ]
        )
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=
            LR_SCHEDULER_PATIENCE,
        min_lr=1e-6
    )


    class_weights = class_weight_tensor(
        y_training
    )


    criterion = nn.CrossEntropyLoss(
        weight=
            class_weights,
        label_smoothing=
            float(
                params[
                    "label_smoothing"
                ]
            )
    )


    training_data = DeviceData(
        X_training,
        y_training
    )


    if X_validation is not None:

        validation_data = DeviceData(
            X_validation,
            y_validation
        )


    else:

        validation_data = None


    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": []
    }


    best_loss = float(
        "inf"
    )


    best_state = None

    best_epoch = 1

    patience_counter = 0


    for epoch in range(
        1,
        epochs + 1
    ):

        train_loss, train_accuracy = train_one_epoch(
            model,
            training_data,
            optimizer,
            criterion
        )


        if validation_data is not None:

            validation_loss, validation_accuracy = evaluate_model(
                model,
                validation_data,
                criterion
            )


            monitored_loss = (
                validation_loss
            )


        else:

            validation_loss = np.nan

            validation_accuracy = np.nan

            monitored_loss = (
                train_loss
            )


        scheduler.step(
            monitored_loss
        )


        learning_rate = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )


        history[
            "loss"
        ].append(
            train_loss
        )


        history[
            "accuracy"
        ].append(
            train_accuracy
        )


        history[
            "val_loss"
        ].append(
            validation_loss
        )


        history[
            "val_accuracy"
        ].append(
            validation_accuracy
        )


        history[
            "learning_rate"
        ].append(
            learning_rate
        )


        if verbose:

            print(
                f"Epoch "
                f"{epoch:02d}/"
                f"{epochs} | "
                f"Train Loss "
                f"{train_loss:.4f} | "
                f"Train Acc "
                f"{train_accuracy:.4f} | "
                f"Val Loss "
                f"{validation_loss:.4f} | "
                f"Val Acc "
                f"{validation_accuracy:.4f} | "
                f"LR "
                f"{learning_rate:.2e}"
            )


        if use_early_stopping:

            if (
                monitored_loss
                < best_loss
                - 1e-5
            ):

                best_loss = (
                    monitored_loss
                )


                best_epoch = (
                    epoch
                )


                best_state = {
                    key:
                        value.detach()
                        .cpu()
                        .clone()

                    for key, value
                    in model.state_dict().items()
                }


                patience_counter = 0


            else:

                patience_counter += 1


                if (
                    patience_counter
                    >= EARLY_STOPPING_PATIENCE
                ):

                    if verbose:

                        print(
                            "Early stopping."
                        )


                    break


    # ============================================================
    # RESTORE BEST STATE
    # ============================================================

    if (
        use_early_stopping
        and
        best_state is not None
    ):

        model.load_state_dict(
            best_state
        )


    del optimizer
    del scheduler
    del criterion
    del class_weights
    del training_data


    if validation_data is not None:

        del validation_data


    if best_state is not None:

        del best_state


    clear_device()


    return (
        model,
        history,
        best_epoch
    )


# ================================================================
# PREDICT PROBABILITIES
# ================================================================

def predict_probabilities(
    model,
    X,
    y
):

    data = DeviceData(
        X,
        y
    )


    _, _, probabilities = evaluate_model(
        model,
        data,
        return_probabilities=True
    )


    del data

    clear_device()


    return probabilities


# ================================================================
# GPU SMOKE TEST
# ================================================================

print()

print("=" * 80)

print(
    "MODEL GPU SMOKE TEST"
)

print("=" * 80)


smoke_params = {
    "d_model":
        64,

    "num_heads":
        4,

    "n_conformer_blocks":
        1,

    "n_tcn_blocks":
        2,

    "conformer_kernel":
        15,

    "tcn_kernel":
        3,

    "dropout":
        0.20,

    "classifier_hidden":
        128,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-4,

    "label_smoothing":
        0.05
}


smoke_model = build_model(
    smoke_params,
    SEED
)


smoke_input = torch.randn(
    8,
    NUMBER_FEATURES,
    device=DEVICE
)


with torch.inference_mode():

    smoke_output = smoke_model(
        smoke_input
    )


synchronize()


print(
    "Smoke input:",
    smoke_input.shape
)

print(
    "Smoke output:",
    smoke_output.shape
)

print(
    "Smoke device:",
    smoke_output.device
)

print(
    "Smoke test passed."
)


del smoke_model
del smoke_input
del smoke_output

clear_device()


# ================================================================
# OPTUNA OBJECTIVE
# ================================================================

def objective(
    trial
):

    params = {
        "d_model":

            trial.suggest_categorical(
                "d_model",
                [
                    64,
                    96
                ]
            ),

        "num_heads":

            trial.suggest_categorical(
                "num_heads",
                [
                    4,
                    8
                ]
            ),

        "n_conformer_blocks":

            trial.suggest_int(
                "n_conformer_blocks",
                1,
                2
            ),

        "n_tcn_blocks":

            trial.suggest_int(
                "n_tcn_blocks",
                2,
                3
            ),

        "conformer_kernel":

            trial.suggest_categorical(
                "conformer_kernel",
                [
                    7,
                    15,
                    31
                ]
            ),

        "tcn_kernel":

            trial.suggest_categorical(
                "tcn_kernel",
                [
                    3,
                    5
                ]
            ),

        "dropout":

            trial.suggest_float(
                "dropout",
                0.10,
                0.35
            ),

        "classifier_hidden":

            trial.suggest_categorical(
                "classifier_hidden",
                [
                    128,
                    256
                ]
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.08
            )
    }


    print()

    print(
        f"OPTUNA TRIAL "
        f"{trial.number + 1}"
    )

    print(
        params
    )


    fold_scores = []


    for fold_number, (
        training_index,
        validation_index
    ) in enumerate(
        gkf_splits,
        start=1
    ):

        fold_start = time.time()


        (
            X_fold_training,
            X_fold_validation
        ) = prepare_fold(
            training_index,
            validation_index
        )


        model, history, best_epoch = fit_model(
            params,
            X_fold_training,
            y_train[
                training_index
            ],
            X_fold_validation,
            y_train[
                validation_index
            ],
            seed=
                SEED
                + fold_number,
            epochs=
                MAX_EPOCHS,
            use_early_stopping=
                True,
            verbose=
                False
        )


        probabilities = predict_probabilities(
            model,
            X_fold_validation,
            y_train[
                validation_index
            ]
        )


        predictions = np.argmax(
            probabilities,
            axis=1
        )


        fold_accuracy = accuracy_score(
            y_train[
                validation_index
            ],
            predictions
        )


        fold_scores.append(
            fold_accuracy
        )


        print(
            f"Trial "
            f"{trial.number + 1}, "
            f"Fold "
            f"{fold_number}/"
            f"{GROUP_K_FOLDS}, "
            f"Accuracy = "
            f"{fold_accuracy:.4f}, "
            f"Best Epoch = "
            f"{best_epoch}, "
            f"Time = "
            f"{(time.time() - fold_start) / 60:.2f} min, "
            f"RAM = "
            f"{mem_mb():.2f} MB"
        )


        del model
        del history
        del probabilities
        del predictions
        del X_fold_training
        del X_fold_validation


        clear_device()


    mean_accuracy = float(
        np.mean(
            fold_scores
        )
    )


    std_accuracy = float(
        np.std(
            fold_scores
        )
    )


    print(
        f"Trial "
        f"{trial.number + 1} "
        f"Mean Accuracy = "
        f"{mean_accuracy:.4f} "
        f"plus or minus "
        f"{std_accuracy:.4f}"
    )


    return mean_accuracy


# ================================================================
# OPTUNA
# ================================================================

print()

print("=" * 80)

print(
    "OPTUNA HYPERPARAMETER OPTIMIZATION"
)

print("=" * 80)


optuna_start = time.time()


study = optuna.create_study(
    study_name=
        OPTUNA_STUDY_NAME,

    storage=
        OPTUNA_STORAGE,

    direction=
        "maximize",

    sampler=
        optuna.samplers.TPESampler(
            seed=SEED
        ),

    load_if_exists=
        True
)


completed_trials = sum(
    1
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
)


remaining_trials = max(
    0,
    N_TRIALS
    - completed_trials
)


print(
    "Previously completed trials:",
    completed_trials
)

print(
    "Remaining trials:",
    remaining_trials
)


if remaining_trials > 0:

    study.optimize(
        objective,
        n_trials=
            remaining_trials,
        show_progress_bar=
            False,
        gc_after_trial=
            True
    )


complete_trials = [
    trial
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
]


if len(
    complete_trials
) == 0:

    raise RuntimeError(
        "No Optuna trial completed successfully."
    )


optuna_time = (
    time.time()
    - optuna_start
)


best_params = dict(
    study.best_params
)


print()

print("=" * 80)

print(
    "OPTUNA COMPLETED"
)

print("=" * 80)


print(
    "Best GroupKFold accuracy:",
    f"{study.best_value:.4f}"
)

print(
    "Best parameters:"
)

print(
    best_params
)

print(
    "Optuna time:",
    f"{optuna_time / 60:.2f}",
    "minutes"
)


study.trials_dataframe().to_csv(
    OUTPUT_DIR
    / "optuna_trials.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "best_parameters.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        best_params,
        file,
        indent=4
    )


# ================================================================
# EXPERIMENT CONFIGURATION ID
# ================================================================

configuration_text = json.dumps(
    {
        "best_params":
            best_params,

        "feature_count":
            NUMBER_FEATURES,

        "classes":
            class_names,

        "patch_kernel":
            PATCH_KERNEL_SIZE,

        "patch_stride":
            PATCH_STRIDE,

        "seed":
            SEED
    },
    sort_keys=True
)


CONFIG_ID = hashlib.sha256(
    configuration_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


print(
    "Configuration ID:",
    CONFIG_ID
)


# ================================================================
# MODEL SIZE INFORMATION
# ================================================================

temporary_model = build_model(
    best_params,
    SEED
)


TRAINABLE_PARAMETERS = count_trainable_parameters(
    temporary_model
)


print(
    "Trainable parameters:",
    f"{TRAINABLE_PARAMETERS:,}"
)


with open(
    OUTPUT_DIR
    / "model_architecture.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        str(
            temporary_model
        )
    )


del temporary_model

clear_device()


# ================================================================
# FIVE STOCHASTIC RUNS
# ================================================================

print()

print("=" * 80)

print(
    "FIVE FINAL STOCHASTIC RUNS"
)

print("=" * 80)


run_records = []

completed_runs = set()


# ================================================================
# LOAD PREVIOUS RUN CHECKPOINT
# ================================================================

if FIVE_RUNS_PATH.exists():

    previous_runs = pd.read_csv(
        FIVE_RUNS_PATH
    )


    if (
        "Config_ID"
        in previous_runs.columns
    ):

        matching_runs = previous_runs[
            previous_runs[
                "Config_ID"
            ].astype(str)
            == CONFIG_ID
        ]


        for _, row in matching_runs.iterrows():

            run_number = int(
                row[
                    "Run"
                ]
            )


            checkpoint_path = (
                CHECKPOINT_DIR
                / f"run_{run_number:02d}.pt"
            )


            if checkpoint_path.exists():

                run_records.append(
                    row.to_dict()
                )

                completed_runs.add(
                    run_number
                )


print(
    "Previously completed runs:",
    len(
        completed_runs
    )
)


five_run_start = time.time()


for run_number in range(
    1,
    N_RUNS + 1
):

    if run_number in completed_runs:

        print(
            f"Run "
            f"{run_number}/"
            f"{N_RUNS} "
            f"already completed."
        )

        continue


    print()

    print(
        f"RUN "
        f"{run_number}/"
        f"{N_RUNS}"
    )


    run_start = time.time()


    run_seed = (
        SEED
        + run_number
        - 1
    )


    model, history, best_epoch = fit_model(
        best_params,
        X_train,
        y_train,
        X_validation,
        y_validation,
        seed=
            run_seed,
        epochs=
            MAX_EPOCHS,
        use_early_stopping=
            True,
        verbose=
            True
    )


    # ============================================================
    # TRAIN PREDICTION
    # ============================================================

    training_probabilities = predict_probabilities(
        model,
        X_train,
        y_train
    )


    training_predictions = np.argmax(
        training_probabilities,
        axis=1
    )


    training_accuracy = accuracy_score(
        y_train,
        training_predictions
    )


    # ============================================================
    # VALIDATION PREDICTION
    # ============================================================

    validation_probabilities = predict_probabilities(
        model,
        X_validation,
        y_validation
    )


    validation_predictions = np.argmax(
        validation_probabilities,
        axis=1
    )


    validation_accuracy = accuracy_score(
        y_validation,
        validation_predictions
    )


    run_time = (
        time.time()
        - run_start
    )


    record = {
        "Config_ID":
            CONFIG_ID,

        "Run":
            run_number,

        "Seed":
            run_seed,

        "Training_Accuracy":
            float(
                training_accuracy
            ),

        "Validation_Accuracy":
            float(
                validation_accuracy
            ),

        "Epochs_Run":
            int(
                len(
                    history[
                        "loss"
                    ]
                )
            ),

        "Best_Epoch":
            int(
                best_epoch
            ),

        "Execution_Time_Seconds":
            float(
                run_time
            )
    }


    model_state = {
        key:
            value.detach()
            .cpu()
            .clone()

        for key, value
        in model.state_dict().items()
    }


    checkpoint_path = (
        CHECKPOINT_DIR
        / f"run_{run_number:02d}.pt"
    )


    atomic_torch_save(
        {
            "Config_ID":
                CONFIG_ID,

            "record":
                record,

            "best_params":
                best_params,

            "model_state_dict":
                model_state,

            "history":
                history
        },
        checkpoint_path
    )


    run_records.append(
        record
    )


    run_table_temp = pd.DataFrame(
        run_records
    )


    run_table_temp = (
        run_table_temp
        .sort_values(
            "Run"
        )
        .drop_duplicates(
            subset=[
                "Run"
            ],
            keep="last"
        )
    )


    atomic_csv(
        run_table_temp,
        FIVE_RUNS_PATH
    )


    print()

    print(
        f"[RUN {run_number}] "
        f"Train = "
        f"{training_accuracy:.4f}, "
        f"Validation = "
        f"{validation_accuracy:.4f}, "
        f"Best Epoch = "
        f"{best_epoch}, "
        f"Time = "
        f"{run_time / 60:.2f} min"
    )


    del model
    del history
    del model_state
    del training_probabilities
    del training_predictions
    del validation_probabilities
    del validation_predictions


    clear_device()


five_run_time = (
    time.time()
    - five_run_start
)


# ================================================================
# LOAD FIVE RUN TABLE
# ================================================================

run_table = pd.read_csv(
    FIVE_RUNS_PATH
)


run_table = run_table[
    run_table[
        "Config_ID"
    ].astype(str)
    == CONFIG_ID
].copy()


run_table = (
    run_table
    .sort_values(
        "Run"
    )
    .drop_duplicates(
        subset=[
            "Run"
        ],
        keep="last"
    )
)


if len(
    run_table
) != N_RUNS:

    raise RuntimeError(
        "Five final runs are not complete."
    )


print()

print(
    run_table[
        [
            "Run",
            "Seed",
            "Training_Accuracy",
            "Validation_Accuracy",
            "Best_Epoch",
            "Execution_Time_Seconds"
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ================================================================
# SELECT BEST VALIDATION RUN
# ================================================================

selected_row = (
    run_table
    .sort_values(
        by=[
            "Validation_Accuracy",
            "Run"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[
        0
    ]
)


best_run_number = int(
    selected_row[
        "Run"
    ]
)


best_checkpoint_path = (
    CHECKPOINT_DIR
    / f"run_{best_run_number:02d}.pt"
)


best_bundle = torch.load(
    best_checkpoint_path,
    map_location="cpu",
    weights_only=False
)


best_record = best_bundle[
    "record"
]


best_history = best_bundle[
    "history"
]


print()

print(
    "Selected run:",
    best_record[
        "Run"
    ]
)

print(
    "Selected seed:",
    best_record[
        "Seed"
    ]
)

print(
    "Selected validation accuracy:",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Selected best epoch:",
    best_record[
        "Best_Epoch"
    ]
)


# ================================================================
# SAVE BEST MODEL PACKAGE
# ================================================================

atomic_torch_save(
    {
        "Algorithm":
            "EEG Conformer + 1D TCN Dual Stream Cross Attention",

        "Dataset":
            "Max Dataset",

        "Config_ID":
            CONFIG_ID,

        "Device_Type":
            DEVICE_TYPE,

        "Device_Name":
            DEVICE_NAME,

        "best_params":
            best_params,

        "selected_run":
            int(
                best_record[
                    "Run"
                ]
            ),

        "selected_seed":
            int(
                best_record[
                    "Seed"
                ]
            ),

        "best_epoch":
            int(
                best_record[
                    "Best_Epoch"
                ]
            ),

        "model_state_dict":
            best_bundle[
                "model_state_dict"
            ],

        "feature_columns":
            feature_cols,

        "class_names":
            class_names,

        "feature_count":
            NUMBER_FEATURES,

        "patch_kernel":
            PATCH_KERNEL_SIZE,

        "patch_stride":
            PATCH_STRIDE
    },
    BEST_MODEL_PATH
)


# ================================================================
# TRAINING HISTORY
# ================================================================

history_table = pd.DataFrame(
    {
        "Epoch":
            np.arange(
                1,
                len(
                    best_history[
                        "loss"
                    ]
                )
                + 1
            ),

        "Training_Loss":
            best_history[
                "loss"
            ],

        "Training_Accuracy":
            best_history[
                "accuracy"
            ],

        "Validation_Loss":
            best_history[
                "val_loss"
            ],

        "Validation_Accuracy":
            best_history[
                "val_accuracy"
            ],

        "Learning_Rate":
            best_history[
                "learning_rate"
            ]
    }
)


history_table.to_csv(
    OUTPUT_DIR
    / "training_history.csv",
    index=False
)


# ================================================================
# LOSS CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Loss"
    ],
    label=
        "Training Loss"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Loss"
    ],
    label=
        "Validation Loss"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Cross Entropy Loss"
)

ax.set_title(
    "Dual Stream Network Loss Curve"
)

ax.legend()

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# ACCURACY CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Accuracy"
    ],
    label=
        "Training Accuracy"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Accuracy"
    ],
    label=
        "Validation Accuracy"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Accuracy"
)

ax.set_title(
    "Dual Stream Network Accuracy Curve"
)

ax.legend()

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# LEARNING RATE CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Learning_Rate"
    ]
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Learning Rate"
)

ax.set_title(
    "Learning Rate Schedule"
)

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "learning_rate_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# FIVE RUN STATISTICS
# ================================================================

run_train_mean = float(
    run_table[
        "Training_Accuracy"
    ].mean()
)


run_train_std = float(
    run_table[
        "Training_Accuracy"
    ].std(
        ddof=0
    )
)


run_validation_mean = float(
    run_table[
        "Validation_Accuracy"
    ].mean()
)


run_validation_std = float(
    run_table[
        "Validation_Accuracy"
    ].std(
        ddof=0
    )
)


# ================================================================
# RELEASE FINAL TRAIN AND VALIDATION SCALED ARRAYS
#
# LOGO still needs raw training data.
# ================================================================

del X_train
del X_validation

del X_validation_raw
del X_test_raw

gc.collect()

clear_device()


# ================================================================
# FULL LOGO
#
# Best epoch learned from external validation set.
#
# Held out LOGO participant never controls early stopping.
# ================================================================

print()

print("=" * 80)

print(
    "FULL LEAVE ONE GROUP OUT"
)

print("=" * 80)


FIXED_LOGO_EPOCHS = max(
    1,
    int(
        best_record[
            "Best_Epoch"
        ]
    )
)


if LOGO_MAX_FOLDS is None:

    LOGO_FOLDS_TO_USE = (
        FULL_LOGO_FOLDS
    )


else:

    LOGO_FOLDS_TO_USE = min(
        int(
            LOGO_MAX_FOLDS
        ),
        FULL_LOGO_FOLDS
    )


print(
    "Available LOGO folds:",
    FULL_LOGO_FOLDS
)

print(
    "LOGO folds requested:",
    LOGO_FOLDS_TO_USE
)

print(
    "Fixed LOGO epochs:",
    FIXED_LOGO_EPOCHS
)


# ================================================================
# LOGO CONFIG ID
# ================================================================

logo_config_text = json.dumps(
    {
        "Config_ID":
            CONFIG_ID,

        "Fixed_Epochs":
            FIXED_LOGO_EPOCHS,

        "Seed":
            int(
                best_record[
                    "Seed"
                ]
            )
    },
    sort_keys=True
)


LOGO_CONFIG_ID = hashlib.sha256(
    logo_config_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


# ================================================================
# LOAD PREVIOUS LOGO
# ================================================================

logo_records = []

completed_logo_persons = set()


if LOGO_RESULT_PATH.exists():

    previous_logo = pd.read_csv(
        LOGO_RESULT_PATH
    )


    if (
        "Config_ID"
        in previous_logo.columns
    ):

        matching_logo = previous_logo[
            previous_logo[
                "Config_ID"
            ].astype(str)
            == LOGO_CONFIG_ID
        ].copy()


        if len(
            matching_logo
        ) > 0:

            logo_records = matching_logo.to_dict(
                orient="records"
            )


            completed_logo_persons = set(
                matching_logo[
                    "Held_Out_Person"
                ].astype(str)
            )


print(
    "Previously completed LOGO folds:",
    len(
        completed_logo_persons
    )
)


# ================================================================
# RUN LOGO
# ================================================================

logo_start = time.time()


logo_splitter = LeaveOneGroupOut()


for fold_number, (
    training_index,
    validation_index
) in enumerate(
    logo_splitter.split(
        X_train_raw,
        y_train,
        g_train
    ),
    start=1
):

    if fold_number > LOGO_FOLDS_TO_USE:

        break


    held_out_persons = np.unique(
        g_train[
            validation_index
        ]
    )


    if len(
        held_out_persons
    ) != 1:

        raise RuntimeError(
            "Unexpected LOGO participant grouping."
        )


    held_out_person = str(
        held_out_persons[
            0
        ]
    )


    if (
        held_out_person
        in completed_logo_persons
    ):

        print(
            f"LOGO "
            f"{fold_number}/"
            f"{LOGO_FOLDS_TO_USE}, "
            f"Participant "
            f"{held_out_person} "
            f"already completed."
        )

        continue


    fold_start = time.time()


    (
        X_logo_training,
        X_logo_validation
    ) = prepare_fold(
        training_index,
        validation_index
    )


    logo_model, _, _ = fit_model(
        best_params,
        X_logo_training,
        y_train[
            training_index
        ],
        X_validation=None,
        y_validation=None,
        seed=
            int(
                best_record[
                    "Seed"
                ]
            ),
        epochs=
            FIXED_LOGO_EPOCHS,
        use_early_stopping=
            False,
        verbose=
            False
    )


    logo_probabilities = predict_probabilities(
        logo_model,
        X_logo_validation,
        y_train[
            validation_index
        ]
    )


    logo_predictions = np.argmax(
        logo_probabilities,
        axis=1
    )


    fold_accuracy = accuracy_score(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_precision = precision_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_recall = recall_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_f1 = f1_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_mcc = matthews_corrcoef(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_time = (
        time.time()
        - fold_start
    )


    logo_record = {
        "Config_ID":
            LOGO_CONFIG_ID,

        "Fold":
            fold_number,

        "Held_Out_Person":
            held_out_person,

        "Rows":
            int(
                len(
                    validation_index
                )
            ),

        "Accuracy":
            float(
                fold_accuracy
            ),

        "Precision_Weighted":
            float(
                fold_precision
            ),

        "Recall_Weighted":
            float(
                fold_recall
            ),

        "F1_Weighted":
            float(
                fold_f1
            ),

        "MCC":
            float(
                fold_mcc
            ),

        "Fixed_Epochs":
            int(
                FIXED_LOGO_EPOCHS
            ),

        "Execution_Time_Seconds":
            float(
                fold_time
            )
    }


    logo_records.append(
        logo_record
    )


    logo_table_temp = pd.DataFrame(
        logo_records
    )


    atomic_csv(
        logo_table_temp,
        LOGO_RESULT_PATH
    )


    completed_logo_persons.add(
        held_out_person
    )


    print(
        f"LOGO "
        f"{fold_number}/"
        f"{LOGO_FOLDS_TO_USE}, "
        f"Participant = "
        f"{held_out_person}, "
        f"Accuracy = "
        f"{fold_accuracy:.4f}, "
        f"F1 = "
        f"{fold_f1:.4f}, "
        f"MCC = "
        f"{fold_mcc:.4f}, "
        f"Time = "
        f"{fold_time / 60:.2f} min"
    )


    del logo_model
    del logo_probabilities
    del logo_predictions
    del X_logo_training
    del X_logo_validation


    clear_device()


logo_time = (
    time.time()
    - logo_start
)


# ================================================================
# LOGO SUMMARY
# ================================================================

logo_table = pd.read_csv(
    LOGO_RESULT_PATH
)


logo_table = logo_table[
    logo_table[
        "Config_ID"
    ].astype(str)
    == LOGO_CONFIG_ID
].copy()


logo_table = (
    logo_table
    .sort_values(
        "Fold"
    )
    .drop_duplicates(
        subset=[
            "Held_Out_Person"
        ],
        keep="last"
    )
)


logo_scores = (
    logo_table[
        "Accuracy"
    ]
    .astype(float)
    .to_numpy()
)


logo_mean = float(
    np.mean(
        logo_scores
    )
)


logo_std = float(
    np.std(
        logo_scores
    )
)


print()

print(
    "Full LOGO Accuracy:",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Completed LOGO folds:",
    len(
        logo_table
    )
)

print(
    "LOGO time this session:",
    f"{logo_time / 60:.2f}",
    "minutes"
)


# ================================================================
# RESTORE SELECTED MODEL FOR FINAL TEST
# ================================================================

best_model = build_model(
    best_params,
    seed=
        int(
            best_record[
                "Seed"
            ]
        )
)


best_model.load_state_dict(
    best_bundle[
        "model_state_dict"
    ]
)


best_model.eval()


# ================================================================
# FINAL UNTOUCHED TEST EVALUATION
# ================================================================

print()

print("=" * 80)

print(
    "FINAL UNTOUCHED TEST EVALUATION"
)

print("=" * 80)


test_start = time.time()


test_probabilities = predict_probabilities(
    best_model,
    X_test,
    y_test
)


test_predictions = np.argmax(
    test_probabilities,
    axis=1
)


test_time = (
    time.time()
    - test_start
)


# ================================================================
# TEST METRICS
# ================================================================

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)


test_precision = precision_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_recall = recall_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_weighted = f1_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_macro = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0
)


test_kappa = cohen_kappa_score(
    y_test,
    test_predictions
)


test_mcc = matthews_corrcoef(
    y_test,
    test_predictions
)


try:

    test_roc_auc = roc_auc_score(
        y_test,
        test_probabilities,
        multi_class="ovr",
        average="weighted"
    )


except Exception:

    test_roc_auc = np.nan


# ================================================================
# CLASSIFICATION REPORT
# ================================================================

report = classification_report(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    ),
    target_names=
        class_names,
    digits=4,
    zero_division=0
)


print()

print(
    "CLASSIFICATION REPORT"
)

print()

print(
    report
)


with open(
    OUTPUT_DIR
    / "classification_report.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        report
    )


report_table = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        labels=np.arange(
            NUMBER_CLASSES
        ),
        target_names=
            class_names,
        output_dict=True,
        zero_division=0
    )
).transpose()


report_table.to_csv(
    OUTPUT_DIR
    / "classification_report.csv"
)


# ================================================================
# CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    )
)


pd.DataFrame(
    cm,
    index=
        class_names,
    columns=
        class_names
).to_csv(
    OUTPUT_DIR
    / "confusion_matrix.csv"
)


# ================================================================
# CONFUSION MATRIX FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "Dual Stream Cross Attention Confusion Matrix"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            str(
                cm[
                    row,
                    column
                ]
            ),
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# NORMALIZED CONFUSION MATRIX
# ================================================================

row_sums = cm.sum(
    axis=1,
    keepdims=True
)


normalized_cm = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(
        cm,
        dtype=np.float64
    ),
    where=(
        row_sums
        != 0
    )
)


pd.DataFrame(
    normalized_cm,
    index=
        class_names,
    columns=
        class_names
).to_csv(
    OUTPUT_DIR
    / "normalized_confusion_matrix.csv"
)


# ================================================================
# NORMALIZED CONFUSION MATRIX FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    normalized_cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "Normalized Dual Stream Cross Attention Confusion Matrix"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            f"{normalized_cm[row, column]:.2f}",
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "normalized_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# SAVE TEST PREDICTIONS
# ================================================================

prediction_table = pd.DataFrame(
    {
        "PersonSrial":
            g_test,

        "True_Class":
            label_encoder.inverse_transform(
                y_test
            ),

        "Predicted_Class":
            label_encoder.inverse_transform(
                test_predictions.astype(
                    np.int32
                )
            )
    }
)


for class_index, class_name in enumerate(
    class_names
):

    prediction_table[
        f"Probability_{class_name}"
    ] = test_probabilities[
        :,
        class_index
    ]


prediction_table.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False
)


# ================================================================
# TOTAL TIME
# ================================================================

total_execution_time = (
    time.time()
    - TOTAL_START
)


# ================================================================
# FINAL METRICS
# ================================================================

final_metrics = {
    "Algorithm":
        "EEG Conformer + 1D TCN Dual Stream Cross Attention",

    "Dataset":
        "Max Dataset",

    "Framework":
        "PyTorch",

    "Input_File":
        str(
            FILE_PATH
        ),

    "Output_Directory":
        str(
            OUTPUT_DIR
        ),

    "Device_Type":
        DEVICE_TYPE,

    "Device_Name":
        DEVICE_NAME,

    "Feature_Count":
        int(
            NUMBER_FEATURES
        ),

    "Number_of_Classes":
        int(
            NUMBER_CLASSES
        ),

    "Classes":
        class_names,

    "Patch_Kernel_Size":
        int(
            PATCH_KERNEL_SIZE
        ),

    "Patch_Stride":
        int(
            PATCH_STRIDE
        ),

    "Trainable_Parameters":
        int(
            TRAINABLE_PARAMETERS
        ),

    "Training_Participants":
        int(
            len(
                train_persons
            )
        ),

    "Validation_Participants":
        int(
            len(
                validation_persons
            )
        ),

    "Testing_Participants":
        int(
            len(
                test_persons
            )
        ),

    "Training_Samples":
        int(
            TRAIN_SAMPLES
        ),

    "Validation_Samples":
        int(
            VALIDATION_SAMPLES
        ),

    "Testing_Samples":
        int(
            TEST_SAMPLES
        ),

    "Optuna_Trials":
        int(
            N_TRIALS
        ),

    "Optuna_GroupKFold_Accuracy":
        float(
            study.best_value
        ),

    "Best_Parameters":
        best_params,

    "Selected_Run":
        int(
            best_record[
                "Run"
            ]
        ),

    "Selected_Seed":
        int(
            best_record[
                "Seed"
            ]
        ),

    "Selected_Best_Epoch":
        int(
            best_record[
                "Best_Epoch"
            ]
        ),

    "Training_Accuracy":
        float(
            best_record[
                "Training_Accuracy"
            ]
        ),

    "Validation_Accuracy":
        float(
            best_record[
                "Validation_Accuracy"
            ]
        ),

    "Testing_Accuracy":
        float(
            test_accuracy
        ),

    "Five_Run_Training_Mean":
        float(
            run_train_mean
        ),

    "Five_Run_Training_STD":
        float(
            run_train_std
        ),

    "Five_Run_Validation_Mean":
        float(
            run_validation_mean
        ),

    "Five_Run_Validation_STD":
        float(
            run_validation_std
        ),

    "Full_LOGO_Accuracy":
        float(
            logo_mean
        ),

    "Full_LOGO_STD":
        float(
            logo_std
        ),

    "Full_LOGO_Folds_Available":
        int(
            FULL_LOGO_FOLDS
        ),

    "Full_LOGO_Folds_Completed":
        int(
            len(
                logo_table
            )
        ),

    "Precision_Weighted":
        float(
            test_precision
        ),

    "Recall_Weighted":
        float(
            test_recall
        ),

    "F1_Weighted":
        float(
            test_f1_weighted
        ),

    "F1_Macro":
        float(
            test_f1_macro
        ),

    "Cohen_Kappa":
        float(
            test_kappa
        ),

    "MCC":
        float(
            test_mcc
        ),

    "ROC_AUC_Weighted_OVR":
        (
            None
            if np.isnan(
                test_roc_auc
            )
            else float(
                test_roc_auc
            )
        ),

    "Optuna_Time_Seconds":
        float(
            optuna_time
        ),

    "Five_Run_Time_This_Session_Seconds":
        float(
            five_run_time
        ),

    "LOGO_Time_This_Session_Seconds":
        float(
            logo_time
        ),

    "Test_Prediction_Time_Seconds":
        float(
            test_time
        ),

    "Total_Execution_Time_Seconds":
        float(
            total_execution_time
        ),

    "Current_RAM_MB":
        float(
            mem_mb()
        )
}


with open(
    OUTPUT_DIR
    / "final_metrics.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        final_metrics,
        file,
        indent=4
    )


# ================================================================
# FINAL SUMMARY
# ================================================================

summary_lines = [
    "EEG CONFORMER + 1D TCN",
    "DUAL STREAM CROSS ATTENTION NETWORK",
    "MAX DATASET",

    "",

    f"Input File: "
    f"{FILE_PATH}",

    f"Output Directory: "
    f"{OUTPUT_DIR}",

    "",

    f"Device: "
    f"{DEVICE_NAME}",

    f"Feature Count: "
    f"{NUMBER_FEATURES}",

    f"Trainable Parameters: "
    f"{TRAINABLE_PARAMETERS}",

    "",

    f"Best Parameters: "
    f"{best_params}",

    "",

    f"Selected Run: "
    f"{best_record['Run']}",

    f"Selected Seed: "
    f"{best_record['Seed']}",

    f"Best Epoch: "
    f"{best_record['Best_Epoch']}",

    "",

    f"Optuna GroupKFold Accuracy: "
    f"{study.best_value:.4f}",

    f"Training Accuracy: "
    f"{best_record['Training_Accuracy']:.4f}",

    f"Validation Accuracy: "
    f"{best_record['Validation_Accuracy']:.4f}",

    f"Testing Accuracy: "
    f"{test_accuracy:.4f}",

    "",

    f"Five Run Training Accuracy: "
    f"{run_train_mean:.4f} "
    f"plus or minus "
    f"{run_train_std:.4f}",

    f"Five Run Validation Accuracy: "
    f"{run_validation_mean:.4f} "
    f"plus or minus "
    f"{run_validation_std:.4f}",

    "",

    f"Full LOGO Accuracy: "
    f"{logo_mean:.4f} "
    f"plus or minus "
    f"{logo_std:.4f}",

    f"Full LOGO Folds Completed: "
    f"{len(logo_table)}",

    "",

    f"Precision Weighted: "
    f"{test_precision:.4f}",

    f"Recall Weighted: "
    f"{test_recall:.4f}",

    f"F1 Weighted: "
    f"{test_f1_weighted:.4f}",

    f"F1 Macro: "
    f"{test_f1_macro:.4f}",

    f"Cohen Kappa: "
    f"{test_kappa:.4f}",

    f"MCC: "
    f"{test_mcc:.4f}",

    (
        "ROC AUC Weighted OVR: N/A"
        if np.isnan(
            test_roc_auc
        )
        else
        f"ROC AUC Weighted OVR: "
        f"{test_roc_auc:.4f}"
    ),

    "",

    f"Optuna Time: "
    f"{optuna_time / 60:.2f} minutes",

    f"Five Run Time This Session: "
    f"{five_run_time / 60:.2f} minutes",

    f"LOGO Time This Session: "
    f"{logo_time / 60:.2f} minutes",

    f"Test Prediction Time: "
    f"{test_time:.2f} seconds",

    f"Total Execution Time: "
    f"{total_execution_time / 60:.2f} minutes"
]


with open(
    OUTPUT_DIR
    / "final_summary.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        "\n".join(
            summary_lines
        )
    )


# ================================================================
# FINAL SCREEN OUTPUT
# ================================================================

print()

print("#" * 80)

print(
    "FINAL RESULTS"
)

print(
    "EEG CONFORMER + 1D TCN"
)

print(
    "DUAL STREAM CROSS ATTENTION NETWORK"
)

print("#" * 80)


print(
    "Device                    :",
    DEVICE_NAME
)

print(
    "Features                  :",
    NUMBER_FEATURES
)

print(
    "Trainable Parameters      :",
    f"{TRAINABLE_PARAMETERS:,}"
)

print()

print(
    "Best d_model              :",
    best_params[
        "d_model"
    ]
)

print(
    "Best Attention Heads      :",
    best_params[
        "num_heads"
    ]
)

print(
    "Conformer Blocks          :",
    best_params[
        "n_conformer_blocks"
    ]
)

print(
    "TCN Blocks                :",
    best_params[
        "n_tcn_blocks"
    ]
)

print(
    "Conformer Kernel          :",
    best_params[
        "conformer_kernel"
    ]
)

print(
    "TCN Kernel                :",
    best_params[
        "tcn_kernel"
    ]
)

print(
    "Dropout                   :",
    f"{best_params['dropout']:.4f}"
)

print(
    "Learning Rate             :",
    f"{best_params['learning_rate']:.2e}"
)

print(
    "Weight Decay              :",
    f"{best_params['weight_decay']:.2e}"
)

print()

print(
    "Selected Run              :",
    best_record[
        "Run"
    ]
)

print(
    "Selected Seed             :",
    best_record[
        "Seed"
    ]
)

print(
    "Best Epoch                :",
    best_record[
        "Best_Epoch"
    ]
)

print()

print(
    "Optuna GroupKFold         :",
    f"{study.best_value:.4f}"
)

print(
    "Training Accuracy         :",
    f"{best_record['Training_Accuracy']:.4f}"
)

print(
    "Validation Accuracy       :",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Testing Accuracy          :",
    f"{test_accuracy:.4f}"
)

print()

print(
    "Five Run Training         :",
    f"{run_train_mean:.4f}",
    "plus or minus",
    f"{run_train_std:.4f}"
)

print(
    "Five Run Validation       :",
    f"{run_validation_mean:.4f}",
    "plus or minus",
    f"{run_validation_std:.4f}"
)

print()

print(
    "Full LOGO Accuracy        :",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Full LOGO Folds           :",
    len(
        logo_table
    ),
    "of",
    FULL_LOGO_FOLDS
)

print()

print(
    "Precision Weighted        :",
    f"{test_precision:.4f}"
)

print(
    "Recall Weighted           :",
    f"{test_recall:.4f}"
)

print(
    "F1 Weighted               :",
    f"{test_f1_weighted:.4f}"
)

print(
    "F1 Macro                  :",
    f"{test_f1_macro:.4f}"
)

print(
    "Cohen Kappa               :",
    f"{test_kappa:.4f}"
)

print(
    "MCC                       :",
    f"{test_mcc:.4f}"
)


if np.isnan(
    test_roc_auc
):

    print(
        "ROC AUC Weighted OVR      : N/A"
    )


else:

    print(
        "ROC AUC Weighted OVR      :",
        f"{test_roc_auc:.4f}"
    )


print()

print(
    "Optuna Time               :",
    f"{optuna_time / 60:.2f}",
    "minutes"
)

print(
    "Five Run Time             :",
    f"{five_run_time / 60:.2f}",
    "minutes"
)

print(
    "Full LOGO Time            :",
    f"{logo_time / 60:.2f}",
    "minutes"
)

print(
    "Test Prediction Time      :",
    f"{test_time:.2f}",
    "seconds"
)

print(
    "Total Execution Time      :",
    f"{total_execution_time / 60:.2f}",
    "minutes"
)

print(
    "Current RAM               :",
    f"{mem_mb():.2f}",
    "MB"
)

print()

print(
    "No dataset rows were subsampled."
)

print(
    "All five GroupKFold folds were used."
)

print(
    "Full training participant LOGO was requested."
)

print(
    "Scaling was fitted independently inside every CV fold."
)

print(
    "External validation participants controlled early stopping."
)

print(
    "Test participants were not used for tuning or model selection."
)

print(
    "Held out LOGO participants were not used for early stopping."
)

print(
    "Optuna, final runs, and LOGO support resume after interruption."
)

print()

print(
    "ALL RESULTS SAVED IN:"
)

print(
    OUTPUT_DIR
)

print("#" * 80)

DEVICE CONFIGURATION
PyTorch version: 2.14.1+xpu
Selected device: xpu:0
Device type: xpu
Device name: Intel(R) Arc(TM) A750 Graphics
CPU threads: 8

EEG CONFORMER + 1D TCN
DUAL STREAM CROSS ATTENTION NETWORK
MAX DATASET
Input dataset:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/Cleaned_oN_Selected_Max.csv

Output directory:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/EEG_Conformer_TCN_CrossAttention_Results

GPU: Intel(R) Arc(TM) A750 Graphics
Optuna trials: 10
GroupKFold folds: 5
Final runs: 5
Maximum epochs: 40
Batch size: 512
Patch kernel: 7
Patch stride: 4
LOGO cap: None
Row subsampling: NONE

LOADING COMPLETE MAX DATASET
Raw dataset shape: (149350, 238)
RAM after loading: 4502.81 MB
Feature count: 232
Invalid feature cells: 0

Feature matrix: (149350, 232)
Classes: ['class_lecture', 'music', 'normal', 'reading']
Number of classes: 4
Participants: 60
Loading time: 8.35 seconds
RAM: 4635.15 MB

CLASS DISTRIBUTION
class_lecture: 39857 (26.69%)
music: 32276 (21.61%)
normal

**Apply (Cross-Attention Multimodal Transformer) on Max Dataset**

In [1]:
# ================================================================
# CROSS ATTENTION MULTIMODAL TRANSFORMER
# MAX DATASET
#
# PYTORCH
# INTEL ARC XPU SUPPORT
#
# THREE FEATURE MODALITY STREAMS
#
# Spectral / Frequency
# Statistical / Temporal
# Nonlinear / Complexity
#
# If feature names do not support reliable semantic grouping,
# three balanced feature views are created automatically.
#
# EXPERIMENT DESIGN
#
# Complete dataset
# Person wise 70 / 20 / 10 split
# Fold specific StandardScaler
# Optuna 10 trials
# GroupKFold 5
# Five stochastic final runs
# Full LOGO
# Validation based early stopping
# Validation based run selection
# Test used only once after selection
#
# RESUME SUPPORT
#
# Persistent Optuna
# Five run checkpoints
# Full LOGO checkpoints
# ================================================================


# ================================================================
# IMPORTS
# ================================================================

import os
import gc
import re
import json
import time
import math
import random
import hashlib
import warnings

from pathlib import Path

import joblib
import psutil
import optuna

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import (
    GroupKFold,
    LeaveOneGroupOut
)

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.utils.class_weight import (
    compute_class_weight
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ================================================================
# GENERAL SETTINGS
# ================================================================

warnings.filterwarnings(
    "ignore"
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


SEED = 42

N_TRIALS = 10

N_RUNS = 5

GROUP_K_FOLDS = 5


# ================================================================
# TRAINING SETTINGS
# ================================================================

MAX_EPOCHS = 40

BATCH_SIZE = 512

EARLY_STOPPING_PATIENCE = 5

LR_SCHEDULER_PATIENCE = 2

GRADIENT_CLIP = 1.0


# ================================================================
# FULL LOGO
#
# None means every training participant.
# ================================================================

LOGO_MAX_FOLDS = None


# ================================================================
# CPU THREAD CONTROL
# ================================================================

CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)


torch.set_num_threads(
    CPU_THREADS
)


# ================================================================
# INPUT
# ================================================================

FILE_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset/"
    "Cleaned_oN_Selected_Max.csv"
)


if not FILE_PATH.exists():

    raise FileNotFoundError(
        f"Input dataset was not found: {FILE_PATH}"
    )


# ================================================================
# OUTPUT
# ================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset"
)


OUTPUT_DIR = (
    OUTPUT_ROOT
    / "Cross_Attention_Multimodal_Transformer_Results"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHECKPOINT_DIR = (
    OUTPUT_DIR
    / "checkpoints"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# OUTPUT FILES
# ================================================================

OPTUNA_DATABASE = (
    OUTPUT_DIR
    / "multimodal_transformer_optuna.db"
)


OPTUNA_STORAGE = (
    f"sqlite:///{OPTUNA_DATABASE}"
)


OPTUNA_STUDY_NAME = (
    "Max_Cross_Attention_Multimodal_Transformer_v1"
)


FIVE_RUNS_PATH = (
    OUTPUT_DIR
    / "five_runs.csv"
)


LOGO_RESULT_PATH = (
    OUTPUT_DIR
    / "full_logo_results.csv"
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "best_multimodal_transformer.pt"
)


# ================================================================
# MEMORY
# ================================================================

PROCESS = psutil.Process(
    os.getpid()
)


def memory_mb():

    return (
        PROCESS.memory_info().rss
        / 1024
        / 1024
    )


# ================================================================
# SAFE CSV SAVE
# ================================================================

def atomic_csv(
    dataframe,
    path
):

    path = Path(
        path
    )

    temporary = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary,
        index=False
    )

    os.replace(
        temporary,
        path
    )


# ================================================================
# SAFE TORCH SAVE
# ================================================================

def atomic_torch_save(
    obj,
    path
):

    path = Path(
        path
    )

    temporary = path.with_name(
        path.name + ".tmp"
    )

    torch.save(
        obj,
        temporary
    )

    os.replace(
        temporary,
        path
    )


# ================================================================
# RANDOM SEED
# ================================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if (
        hasattr(
            torch,
            "xpu"
        )
        and
        torch.xpu.is_available()
    ):

        try:

            torch.xpu.manual_seed_all(
                seed
            )

        except Exception:

            pass


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    SEED
)


# ================================================================
# DEVICE DETECTION
#
# PRIORITY
#
# Intel XPU
# CUDA
# CPU
# ================================================================

if (
    hasattr(
        torch,
        "xpu"
    )
    and
    torch.xpu.is_available()
):

    DEVICE = torch.device(
        "xpu:0"
    )

    DEVICE_TYPE = "xpu"

    DEVICE_NAME = torch.xpu.get_device_name(
        0
    )


elif torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda:0"
    )

    DEVICE_TYPE = "cuda"

    DEVICE_NAME = torch.cuda.get_device_name(
        0
    )


else:

    DEVICE = torch.device(
        "cpu"
    )

    DEVICE_TYPE = "cpu"

    DEVICE_NAME = "CPU"


# ================================================================
# DEVICE HELPERS
# ================================================================

def synchronize():

    if DEVICE_TYPE == "xpu":

        torch.xpu.synchronize()


    elif DEVICE_TYPE == "cuda":

        torch.cuda.synchronize()


def clear_device():

    gc.collect()


    if DEVICE_TYPE == "xpu":

        torch.xpu.empty_cache()


    elif DEVICE_TYPE == "cuda":

        torch.cuda.empty_cache()


# ================================================================
# DEVICE INFORMATION
# ================================================================

print("=" * 80)

print(
    "DEVICE CONFIGURATION"
)

print("=" * 80)

print(
    "PyTorch version:",
    torch.__version__
)

print(
    "Selected device:",
    DEVICE
)

print(
    "Device type:",
    DEVICE_TYPE
)

print(
    "Device name:",
    DEVICE_NAME
)

print(
    "CPU threads:",
    CPU_THREADS
)

print("=" * 80)


# ================================================================
# TOTAL TIMER
# ================================================================

TOTAL_START = time.time()


# ================================================================
# LOAD DATASET
# ================================================================

print()

print("=" * 80)

print(
    "CROSS ATTENTION MULTIMODAL TRANSFORMER"
)

print(
    "MAX DATASET"
)

print("=" * 80)

print(
    "Input:"
)

print(
    FILE_PATH
)

print()

print(
    "Output:"
)

print(
    OUTPUT_DIR
)

print()

print(
    "Optuna trials:",
    N_TRIALS
)

print(
    "GroupKFold folds:",
    GROUP_K_FOLDS
)

print(
    "Final runs:",
    N_RUNS
)

print(
    "Maximum epochs:",
    MAX_EPOCHS
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Full LOGO:",
    LOGO_MAX_FOLDS is None
)

print("=" * 80)


load_start = time.time()


df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)


print(
    "Raw dataset shape:",
    df.shape
)

print(
    "RAM after loading:",
    f"{memory_mb():.2f} MB"
)


# ================================================================
# COLUMNS
# ================================================================

metadata_cols = [
    "PersonSrial",
    "Window",
    "Window_Start",
    "Window_End",
    "Window_Samples"
]


target_col = "Class"


required_columns = [
    "PersonSrial",
    target_col
]


for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"Required column missing: {column}"
        )


feature_cols = [
    column
    for column in df.columns
    if column not in metadata_cols
    and column != target_col
]


NUMBER_FEATURES = len(
    feature_cols
)


if NUMBER_FEATURES == 0:

    raise ValueError(
        "No feature columns were found."
    )


print(
    "Feature count:",
    NUMBER_FEATURES
)


# ================================================================
# FEATURE MATRIX
# ================================================================

X_all = (
    df[
        feature_cols
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(
        dtype=np.float32
    )
)


invalid_count = int(
    (
        ~np.isfinite(
            X_all
        )
    ).sum()
)


print(
    "Invalid feature cells:",
    invalid_count
)


if invalid_count > 0:

    raise ValueError(
        "Feature matrix contains NaN or infinite values."
    )


# ================================================================
# LABELS
# ================================================================

label_encoder = LabelEncoder()


y_all = label_encoder.fit_transform(
    df[
        target_col
    ].astype(str)
).astype(
    np.int64
)


class_names = list(
    label_encoder.classes_
)


NUMBER_CLASSES = len(
    class_names
)


# ================================================================
# PARTICIPANTS
# ================================================================

groups_all = (
    df[
        "PersonSrial"
    ]
    .to_numpy()
)


print(
    "Classes:",
    class_names
)

print(
    "Number of classes:",
    NUMBER_CLASSES
)

print(
    "Participants:",
    len(
        np.unique(
            groups_all
        )
    )
)


# ================================================================
# CLASS DISTRIBUTION
# ================================================================

class_distribution = []


print()

print(
    "CLASS DISTRIBUTION"
)


for class_index, class_name in enumerate(
    class_names
):

    count = int(
        np.sum(
            y_all
            == class_index
        )
    )


    percentage = (
        count
        / len(
            y_all
        )
        * 100.0
    )


    print(
        f"{class_name}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


    class_distribution.append(
        {
            "Class":
                class_name,

            "Count":
                count,

            "Percentage":
                percentage
        }
    )


pd.DataFrame(
    class_distribution
).to_csv(
    OUTPUT_DIR
    / "class_distribution.csv",
    index=False
)


print(
    "Loading time:",
    f"{time.time() - load_start:.2f}",
    "seconds"
)


# ================================================================
# MODALITY DETECTION
#
# The dataset is still EEG.
#
# These are engineered feature modality streams,
# not independent physical sensors.
# ================================================================

SPECTRAL_KEYWORDS = [
    "fft",
    "psd",
    "power",
    "spectral",
    "frequency",
    "freq",
    "delta",
    "theta",
    "alpha",
    "beta",
    "gamma",
    "bandpower",
    "band_power"
]


STATISTICAL_KEYWORDS = [
    "mean",
    "median",
    "std",
    "variance",
    "var",
    "rms",
    "skew",
    "kurt",
    "minimum",
    "maximum",
    "min",
    "max",
    "range",
    "iqr",
    "quartile",
    "percentile",
    "peak",
    "zero",
    "zcr",
    "hjorth_activity",
    "hjorth_mobility"
]


COMPLEXITY_KEYWORDS = [
    "entropy",
    "complexity",
    "fractal",
    "hurst",
    "dfa",
    "lyapunov",
    "sampen",
    "sample_entropy",
    "apen",
    "approximate_entropy",
    "permutation",
    "lzc",
    "lempel",
    "higuchi",
    "petrosian",
    "mobility",
    "hjorth_complexity"
]


def contains_keyword(
    feature_name,
    keywords
):

    normalized = str(
        feature_name
    ).lower()


    return any(
        keyword in normalized
        for keyword in keywords
    )


def build_feature_modalities(
    columns
):

    spectral = []

    statistical = []

    complexity = []

    unassigned = []


    for index, column in enumerate(
        columns
    ):

        if contains_keyword(
            column,
            COMPLEXITY_KEYWORDS
        ):

            complexity.append(
                index
            )


        elif contains_keyword(
            column,
            SPECTRAL_KEYWORDS
        ):

            spectral.append(
                index
            )


        elif contains_keyword(
            column,
            STATISTICAL_KEYWORDS
        ):

            statistical.append(
                index
            )


        else:

            unassigned.append(
                index
            )


    # ============================================================
    # Distribute features that could not be identified
    # among the currently smallest streams.
    # ============================================================

    groups = [
        spectral,
        statistical,
        complexity
    ]


    for index in unassigned:

        smallest_group = int(
            np.argmin(
                [
                    len(group)
                    for group in groups
                ]
            )
        )


        groups[
            smallest_group
        ].append(
            index
        )


    # ============================================================
    # If keyword based detection is extremely imbalanced,
    # use three balanced feature views instead.
    # ============================================================

    minimum_group_size = min(
        len(group)
        for group in groups
    )


    largest_group_size = max(
        len(group)
        for group in groups
    )


    semantic_grouping_ok = (
        minimum_group_size >= 8
        and
        largest_group_size
        <= (
            NUMBER_FEATURES
            * 0.70
        )
    )


    if semantic_grouping_ok:

        grouping_mode = (
            "Feature name based semantic grouping"
        )


        modality_indices = {
            "spectral_frequency":
                sorted(
                    groups[
                        0
                    ]
                ),

            "statistical_temporal":
                sorted(
                    groups[
                        1
                    ]
                ),

            "nonlinear_complexity":
                sorted(
                    groups[
                        2
                    ]
                )
        }


    else:

        grouping_mode = (
            "Balanced feature view fallback"
        )


        all_indices = np.arange(
            NUMBER_FEATURES
        )


        balanced_groups = np.array_split(
            all_indices,
            3
        )


        modality_indices = {
            "feature_view_1":
                balanced_groups[
                    0
                ].tolist(),

            "feature_view_2":
                balanced_groups[
                    1
                ].tolist(),

            "feature_view_3":
                balanced_groups[
                    2
                ].tolist()
        }


    return (
        modality_indices,
        grouping_mode
    )


MODALITY_INDICES, MODALITY_GROUPING_MODE = (
    build_feature_modalities(
        feature_cols
    )
)


MODALITY_NAMES = list(
    MODALITY_INDICES.keys()
)


NUMBER_MODALITIES = len(
    MODALITY_NAMES
)


print()

print("=" * 80)

print(
    "FEATURE MODALITY CONSTRUCTION"
)

print("=" * 80)

print(
    "Grouping mode:",
    MODALITY_GROUPING_MODE
)


for modality_name in MODALITY_NAMES:

    print(
        modality_name,
        ":",
        len(
            MODALITY_INDICES[
                modality_name
            ]
        ),
        "features"
    )


# ================================================================
# SAVE MODALITY INFORMATION
# ================================================================

modality_information = {
    "Grouping_Mode":
        MODALITY_GROUPING_MODE,

    "Important_Note":
        (
            "These streams represent EEG feature families "
            "or feature views, not independent physical sensors."
        ),

    "Modalities":
        {}
}


for modality_name in MODALITY_NAMES:

    indices = MODALITY_INDICES[
        modality_name
    ]


    modality_information[
        "Modalities"
    ][
        modality_name
    ] = {
        "Feature_Count":
            len(
                indices
            ),

        "Features":
            [
                feature_cols[
                    index
                ]
                for index in indices
            ]
    }


with open(
    OUTPUT_DIR
    / "modality_mapping.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        modality_information,
        file,
        indent=4
    )


# ================================================================
# RELEASE DATAFRAME
# ================================================================

del df

gc.collect()


# ================================================================
# PERSON WISE 70 / 20 / 10
# ================================================================

print()

print("=" * 80)

print(
    "PERSON WISE 70 / 20 / 10 SPLIT"
)

print("=" * 80)


persons = np.unique(
    groups_all
)


rng = np.random.RandomState(
    SEED
)


rng.shuffle(
    persons
)


number_persons = len(
    persons
)


number_test = max(
    1,
    int(
        round(
            0.20
            * number_persons
        )
    )
)


number_validation = max(
    1,
    int(
        round(
            0.10
            * number_persons
        )
    )
)


test_persons = persons[
    :number_test
]


validation_persons = persons[
    number_test:
    number_test
    + number_validation
]


train_persons = persons[
    number_test
    + number_validation:
]


assert (
    np.intersect1d(
        train_persons,
        validation_persons
    ).size == 0
)


assert (
    np.intersect1d(
        train_persons,
        test_persons
    ).size == 0
)


assert (
    np.intersect1d(
        validation_persons,
        test_persons
    ).size == 0
)


train_mask = np.isin(
    groups_all,
    train_persons
)


validation_mask = np.isin(
    groups_all,
    validation_persons
)


test_mask = np.isin(
    groups_all,
    test_persons
)


# ================================================================
# SPLIT ARRAYS
# ================================================================

X_train_raw = np.ascontiguousarray(
    X_all[
        train_mask
    ],
    dtype=np.float32
)


y_train = np.ascontiguousarray(
    y_all[
        train_mask
    ],
    dtype=np.int64
)


g_train = np.ascontiguousarray(
    groups_all[
        train_mask
    ]
)


X_validation_raw = np.ascontiguousarray(
    X_all[
        validation_mask
    ],
    dtype=np.float32
)


y_validation = np.ascontiguousarray(
    y_all[
        validation_mask
    ],
    dtype=np.int64
)


g_validation = np.ascontiguousarray(
    groups_all[
        validation_mask
    ]
)


X_test_raw = np.ascontiguousarray(
    X_all[
        test_mask
    ],
    dtype=np.float32
)


y_test = np.ascontiguousarray(
    y_all[
        test_mask
    ],
    dtype=np.int64
)


g_test = np.ascontiguousarray(
    groups_all[
        test_mask
    ]
)


TRAIN_SAMPLES = len(
    X_train_raw
)


VALIDATION_SAMPLES = len(
    X_validation_raw
)


TEST_SAMPLES = len(
    X_test_raw
)


print(
    "Training participants:",
    len(
        train_persons
    )
)

print(
    "Validation participants:",
    len(
        validation_persons
    )
)

print(
    "Testing participants:",
    len(
        test_persons
    )
)

print()

print(
    "Training:",
    X_train_raw.shape
)

print(
    "Validation:",
    X_validation_raw.shape
)

print(
    "Testing:",
    X_test_raw.shape
)


# ================================================================
# SAVE SPLIT
# ================================================================

pd.DataFrame(
    [
        {
            "Split":
                "Training",

            "Participants":
                len(
                    train_persons
                ),

            "Rows":
                TRAIN_SAMPLES
        },

        {
            "Split":
                "Validation",

            "Participants":
                len(
                    validation_persons
                ),

            "Rows":
                VALIDATION_SAMPLES
        },

        {
            "Split":
                "Testing",

            "Participants":
                len(
                    test_persons
                ),

            "Rows":
                TEST_SAMPLES
        }
    ]
).to_csv(
    OUTPUT_DIR
    / "split_summary.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "participant_split.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        {
            "training":
                [
                    str(x)
                    for x in train_persons
                ],

            "validation":
                [
                    str(x)
                    for x in validation_persons
                ],

            "testing":
                [
                    str(x)
                    for x in test_persons
                ]
        },
        file,
        indent=4
    )


# ================================================================
# RELEASE COMPLETE DATA
# ================================================================

del X_all
del y_all
del groups_all

del train_mask
del validation_mask
del test_mask

gc.collect()


# ================================================================
# CV INFORMATION
# ================================================================

gkf_splits = list(
    GroupKFold(
        n_splits=
            GROUP_K_FOLDS
    ).split(
        X_train_raw,
        y_train,
        g_train
    )
)


FULL_LOGO_FOLDS = len(
    np.unique(
        g_train
    )
)


print()

print(
    "GroupKFold folds:",
    len(
        gkf_splits
    )
)

print(
    "Full LOGO folds:",
    FULL_LOGO_FOLDS
)


# ================================================================
# FOLD SPECIFIC SCALING
# ================================================================

def prepare_fold(
    training_index,
    validation_index
):

    scaler = StandardScaler()


    X_fold_training = scaler.fit_transform(
        X_train_raw[
            training_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    X_fold_validation = scaler.transform(
        X_train_raw[
            validation_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    return (
        np.ascontiguousarray(
            X_fold_training
        ),
        np.ascontiguousarray(
            X_fold_validation
        )
    )


# ================================================================
# FINAL SCALER
#
# Training participants only.
# ================================================================

print()

print("=" * 80)

print(
    "FINAL TRAINING SCALER"
)

print("=" * 80)


final_scaler = StandardScaler()


X_train = final_scaler.fit_transform(
    X_train_raw
).astype(
    np.float32,
    copy=False
)


X_validation = final_scaler.transform(
    X_validation_raw
).astype(
    np.float32,
    copy=False
)


X_test = final_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


X_train = np.ascontiguousarray(
    X_train
)


X_validation = np.ascontiguousarray(
    X_validation
)


X_test = np.ascontiguousarray(
    X_test
)


joblib.dump(
    final_scaler,
    OUTPUT_DIR
    / "standard_scaler.joblib"
)


joblib.dump(
    label_encoder,
    OUTPUT_DIR
    / "label_encoder.joblib"
)


with open(
    OUTPUT_DIR
    / "feature_columns.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        feature_cols,
        file,
        indent=4
    )


# ================================================================
# CLASS WEIGHTS
# ================================================================

def get_class_weights(
    y
):

    present_classes = np.unique(
        y
    )


    values = compute_class_weight(
        class_weight="balanced",
        classes=
            present_classes,
        y=y
    )


    weights = np.ones(
        NUMBER_CLASSES,
        dtype=np.float32
    )


    weights[
        present_classes
    ] = values.astype(
        np.float32
    )


    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


# ================================================================
# POSITIONAL ENCODING
# ================================================================

class SinusoidalPositionEncoding(
    nn.Module
):

    def __init__(
        self,
        dimension,
        max_length=512
    ):

        super().__init__()


        position = torch.arange(
            max_length,
            dtype=torch.float32
        ).unsqueeze(
            1
        )


        divisor = torch.exp(
            torch.arange(
                0,
                dimension,
                2,
                dtype=torch.float32
            )
            * (
                -math.log(
                    10000.0
                )
                / dimension
            )
        )


        encoding = torch.zeros(
            max_length,
            dimension,
            dtype=torch.float32
        )


        encoding[
            :,
            0::2
        ] = torch.sin(
            position
            * divisor
        )


        encoding[
            :,
            1::2
        ] = torch.cos(
            position
            * divisor
        )


        self.register_buffer(
            "encoding",
            encoding.unsqueeze(
                0
            ),
            persistent=False
        )


    def forward(
        self,
        x
    ):

        return (
            x
            + self.encoding[
                :,
                :x.size(
                    1
                ),
                :
            ].to(
                dtype=x.dtype
            )
        )


# ================================================================
# CUSTOM MULTI HEAD ATTENTION
#
# Used instead of depending on a backend specific
# fused attention implementation.
# ================================================================

class MultiHeadAttention(
    nn.Module
):

    def __init__(
        self,
        dimension,
        number_heads,
        dropout
    ):

        super().__init__()


        if (
            dimension
            % number_heads
            != 0
        ):

            raise ValueError(
                "Embedding dimension must be divisible by attention heads."
            )


        self.dimension = (
            dimension
        )


        self.number_heads = (
            number_heads
        )


        self.head_dimension = (
            dimension
            // number_heads
        )


        self.scale = (
            self.head_dimension
            ** -0.5
        )


        self.query_projection = nn.Linear(
            dimension,
            dimension
        )


        self.key_projection = nn.Linear(
            dimension,
            dimension
        )


        self.value_projection = nn.Linear(
            dimension,
            dimension
        )


        self.output_projection = nn.Linear(
            dimension,
            dimension
        )


        self.attention_dropout = nn.Dropout(
            dropout
        )


        self.output_dropout = nn.Dropout(
            dropout
        )


    def split_heads(
        self,
        x
    ):

        batch_size = x.shape[
            0
        ]


        sequence_length = x.shape[
            1
        ]


        x = x.reshape(
            batch_size,
            sequence_length,
            self.number_heads,
            self.head_dimension
        )


        return x.transpose(
            1,
            2
        )


    def forward(
        self,
        query,
        key,
        value
    ):

        query = self.split_heads(
            self.query_projection(
                query
            )
        )


        key = self.split_heads(
            self.key_projection(
                key
            )
        )


        value = self.split_heads(
            self.value_projection(
                value
            )
        )


        scores = torch.matmul(
            query,
            key.transpose(
                -2,
                -1
            )
        )


        scores = (
            scores
            * self.scale
        )


        attention = torch.softmax(
            scores,
            dim=-1
        )


        attention = self.attention_dropout(
            attention
        )


        context = torch.matmul(
            attention,
            value
        )


        context = context.transpose(
            1,
            2
        ).contiguous()


        batch_size = context.shape[
            0
        ]


        sequence_length = context.shape[
            1
        ]


        context = context.reshape(
            batch_size,
            sequence_length,
            self.dimension
        )


        context = self.output_projection(
            context
        )


        return self.output_dropout(
            context
        )


# ================================================================
# TRANSFORMER FEED FORWARD
# ================================================================

class FeedForward(
    nn.Module
):

    def __init__(
        self,
        dimension,
        expansion,
        dropout
    ):

        super().__init__()


        hidden_dimension = (
            dimension
            * expansion
        )


        self.network = nn.Sequential(
            nn.Linear(
                dimension,
                hidden_dimension
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dimension,
                dimension
            ),

            nn.Dropout(
                dropout
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# ================================================================
# SELF ATTENTION TRANSFORMER BLOCK
# ================================================================

class TransformerBlock(
    nn.Module
):

    def __init__(
        self,
        dimension,
        number_heads,
        expansion,
        dropout
    ):

        super().__init__()


        self.attention_norm = nn.LayerNorm(
            dimension
        )


        self.attention = MultiHeadAttention(
            dimension,
            number_heads,
            dropout
        )


        self.ffn_norm = nn.LayerNorm(
            dimension
        )


        self.ffn = FeedForward(
            dimension,
            expansion,
            dropout
        )


    def forward(
        self,
        x
    ):

        normalized = self.attention_norm(
            x
        )


        x = (
            x
            + self.attention(
                normalized,
                normalized,
                normalized
            )
        )


        x = (
            x
            + self.ffn(
                self.ffn_norm(
                    x
                )
            )
        )


        return x


# ================================================================
# FEATURE TOKENIZER
#
# Converts one modality vector into Transformer tokens.
# ================================================================

class FeatureTokenizer(
    nn.Module
):

    def __init__(
        self,
        dimension,
        kernel_size,
        stride,
        dropout
    ):

        super().__init__()


        self.tokenizer = nn.Sequential(
            nn.Conv1d(
                1,
                dimension,
                kernel_size=
                    kernel_size,
                stride=
                    stride,
                padding=
                    kernel_size
                    // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                dimension
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )
        )


        self.position = SinusoidalPositionEncoding(
            dimension,
            max_length=512
        )


    def forward(
        self,
        x
    ):

        x = x.unsqueeze(
            1
        )


        x = self.tokenizer(
            x
        )


        x = x.transpose(
            1,
            2
        )


        return self.position(
            x
        )


# ================================================================
# CROSS MODAL ATTENTION LAYER
#
# Every modality queries the concatenation of
# the other two modality streams.
#
# All three updates are calculated from the same
# pre update representations.
# ================================================================

class CrossModalAttentionLayer(
    nn.Module
):

    def __init__(
        self,
        dimension,
        number_heads,
        expansion,
        dropout,
        number_modalities
    ):

        super().__init__()


        self.number_modalities = (
            number_modalities
        )


        self.query_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )
                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.context_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )
                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.cross_attentions = nn.ModuleList(
            [
                MultiHeadAttention(
                    dimension,
                    number_heads,
                    dropout
                )
                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.output_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )
                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.feedforward_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )
                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.feedforwards = nn.ModuleList(
            [
                FeedForward(
                    dimension,
                    expansion,
                    dropout
                )
                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        streams
    ):

        original_streams = streams


        updated_streams = []


        for modality_index in range(
            self.number_modalities
        ):

            query_stream = original_streams[
                modality_index
            ]


            other_streams = [
                original_streams[
                    other_index
                ]
                for other_index
                in range(
                    self.number_modalities
                )
                if other_index
                != modality_index
            ]


            context = torch.cat(
                other_streams,
                dim=1
            )


            normalized_query = self.query_norms[
                modality_index
            ](
                query_stream
            )


            normalized_context = self.context_norms[
                modality_index
            ](
                context
            )


            cross_output = self.cross_attentions[
                modality_index
            ](
                normalized_query,
                normalized_context,
                normalized_context
            )


            stream = self.output_norms[
                modality_index
            ](
                query_stream
                + self.dropout(
                    cross_output
                )
            )


            stream = (
                stream
                + self.feedforwards[
                    modality_index
                ](
                    self.feedforward_norms[
                        modality_index
                    ](
                        stream
                    )
                )
            )


            updated_streams.append(
                stream
            )


        return updated_streams


# ================================================================
# CROSS ATTENTION MULTIMODAL TRANSFORMER
# ================================================================

class CrossAttentionMultimodalTransformer(
    nn.Module
):

    def __init__(
        self,
        params,
        modality_indices
    ):

        super().__init__()


        self.modality_names = list(
            modality_indices.keys()
        )


        self.number_modalities = len(
            self.modality_names
        )


        dimension = int(
            params[
                "d_model"
            ]
        )


        number_heads = int(
            params[
                "num_heads"
            ]
        )


        number_self_layers = int(
            params[
                "self_layers"
            ]
        )


        number_cross_layers = int(
            params[
                "cross_layers"
            ]
        )


        expansion = int(
            params[
                "ffn_expansion"
            ]
        )


        tokenizer_kernel = int(
            params[
                "tokenizer_kernel"
            ]
        )


        tokenizer_stride = int(
            params[
                "tokenizer_stride"
            ]
        )


        dropout = float(
            params[
                "dropout"
            ]
        )


        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )


        # ========================================================
        # Store modality column indices as model buffers.
        # ========================================================

        self.modality_buffer_names = []


        for modality_number, modality_name in enumerate(
            self.modality_names
        ):

            buffer_name = (
                f"modality_indices_{modality_number}"
            )


            self.register_buffer(
                buffer_name,
                torch.tensor(
                    modality_indices[
                        modality_name
                    ],
                    dtype=torch.long
                ),
                persistent=True
            )


            self.modality_buffer_names.append(
                buffer_name
            )


        # ========================================================
        # Each modality has its own tokenizer.
        # ========================================================

        self.tokenizers = nn.ModuleList(
            [
                FeatureTokenizer(
                    dimension=
                        dimension,
                    kernel_size=
                        tokenizer_kernel,
                    stride=
                        tokenizer_stride,
                    dropout=
                        dropout
                )
                for _
                in range(
                    self.number_modalities
                )
            ]
        )


        # ========================================================
        # Modality embeddings
        # ========================================================

        self.modality_embeddings = nn.Parameter(
            torch.randn(
                self.number_modalities,
                1,
                dimension
            )
            * 0.02
        )


        # ========================================================
        # Independent self attention encoders
        # ========================================================

        self.self_encoders = nn.ModuleList()


        for _ in range(
            self.number_modalities
        ):

            modality_encoder = nn.ModuleList(
                [
                    TransformerBlock(
                        dimension=
                            dimension,
                        number_heads=
                            number_heads,
                        expansion=
                            expansion,
                        dropout=
                            dropout
                    )
                    for _
                    in range(
                        number_self_layers
                    )
                ]
            )


            self.self_encoders.append(
                modality_encoder
            )


        # ========================================================
        # Cross modal attention stages
        # ========================================================

        self.cross_layers = nn.ModuleList(
            [
                CrossModalAttentionLayer(
                    dimension=
                        dimension,
                    number_heads=
                        number_heads,
                    expansion=
                        expansion,
                    dropout=
                        dropout,
                    number_modalities=
                        self.number_modalities
                )
                for _
                in range(
                    number_cross_layers
                )
            ]
        )


        # ========================================================
        # Final fusion
        #
        # Three modality vectors
        # Global modality mean
        # Global modality standard deviation
        #
        # Total = 5 x d_model
        # ========================================================

        fusion_dimension = (
            dimension
            * (
                self.number_modalities
                + 2
            )
        )


        self.classifier = nn.Sequential(
            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        streams = []


        # ========================================================
        # Modality specific self attention
        # ========================================================

        for modality_index in range(
            self.number_modalities
        ):

            column_indices = getattr(
                self,
                self.modality_buffer_names[
                    modality_index
                ]
            )


            modality_input = torch.index_select(
                x,
                dim=1,
                index=
                    column_indices
            )


            stream = self.tokenizers[
                modality_index
            ](
                modality_input
            )


            modality_embedding = (
                self.modality_embeddings[
                    modality_index
                ]
                .unsqueeze(
                    0
                )
            )


            stream = (
                stream
                + modality_embedding
            )


            for encoder_block in self.self_encoders[
                modality_index
            ]:

                stream = encoder_block(
                    stream
                )


            streams.append(
                stream
            )


        # ========================================================
        # Cross modal attention
        # ========================================================

        for cross_layer in self.cross_layers:

            streams = cross_layer(
                streams
            )


        # ========================================================
        # Global pooling
        # ========================================================

        modality_vectors = [
            torch.mean(
                stream,
                dim=1
            )
            for stream in streams
        ]


        stacked_modalities = torch.stack(
            modality_vectors,
            dim=1
        )


        modality_mean = torch.mean(
            stacked_modalities,
            dim=1
        )


        modality_std = torch.std(
            stacked_modalities,
            dim=1,
            unbiased=False
        )


        fused = torch.cat(
            modality_vectors
            + [
                modality_mean,
                modality_std
            ],
            dim=1
        )


        return self.classifier(
            fused
        )


# ================================================================
# MODEL BUILDER
# ================================================================

def build_model(
    params,
    seed=SEED
):

    set_seed(
        seed
    )


    model = CrossAttentionMultimodalTransformer(
        params=
            params,
        modality_indices=
            MODALITY_INDICES
    )


    return model.to(
        DEVICE
    )


# ================================================================
# PARAMETER COUNT
# ================================================================

def trainable_parameter_count(
    model
):

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


# ================================================================
# DATA HOLDER
# ================================================================

class DeviceData:

    def __init__(
        self,
        X,
        y
    ):

        X_array = np.ascontiguousarray(
            X,
            dtype=np.float32
        )


        y_array = np.ascontiguousarray(
            y,
            dtype=np.int64
        )


        self.X = torch.from_numpy(
            X_array
        ).to(
            DEVICE
        )


        self.y = torch.from_numpy(
            y_array
        ).to(
            DEVICE
        )


    def __len__(
        self
    ):

        return int(
            self.y.shape[
                0
            ]
        )


# ================================================================
# BATCH GENERATOR
# ================================================================

def generate_batches(
    number_samples,
    shuffle=True
):

    if shuffle:

        indices = torch.randperm(
            number_samples,
            device=DEVICE
        )


    else:

        indices = torch.arange(
            number_samples,
            device=DEVICE
        )


    for start in range(
        0,
        number_samples,
        BATCH_SIZE
    ):

        yield indices[
            start:
            start
            + BATCH_SIZE
        ]


# ================================================================
# TRAIN ONE EPOCH
# ================================================================

def train_one_epoch(
    model,
    data,
    optimizer,
    criterion
):

    model.train()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    for indices in generate_batches(
        len(
            data
        ),
        shuffle=True
    ):

        X_batch = data.X[
            indices
        ]


        y_batch = data.y[
            indices
        ]


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            X_batch
        )


        loss = criterion(
            logits,
            y_batch
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )


        optimizer.step()


        current_batch_size = int(
            y_batch.shape[
                0
            ]
        )


        total_loss += (
            float(
                loss.detach().item()
            )
            * current_batch_size
        )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            current_batch_size
        )


    synchronize()


    return (
        total_loss
        / total_samples,

        total_correct
        / total_samples
    )


# ================================================================
# EVALUATE MODEL
# ================================================================

@torch.inference_mode()
def evaluate_model(
    model,
    data,
    criterion=None,
    return_probabilities=False
):

    model.eval()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    probability_parts = []


    for indices in generate_batches(
        len(
            data
        ),
        shuffle=False
    ):

        X_batch = data.X[
            indices
        ]


        y_batch = data.y[
            indices
        ]


        logits = model(
            X_batch
        )


        current_batch_size = int(
            y_batch.shape[
                0
            ]
        )


        if criterion is not None:

            loss = criterion(
                logits,
                y_batch
            )


            total_loss += (
                float(
                    loss.item()
                )
                * current_batch_size
            )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            current_batch_size
        )


        if return_probabilities:

            probability_parts.append(
                torch.softmax(
                    logits,
                    dim=1
                )
                .cpu()
            )


    synchronize()


    accuracy = (
        total_correct
        / total_samples
    )


    if criterion is None:

        mean_loss = np.nan


    else:

        mean_loss = (
            total_loss
            / total_samples
        )


    if return_probabilities:

        probabilities = (
            torch.cat(
                probability_parts,
                dim=0
            )
            .numpy()
        )


        return (
            mean_loss,
            accuracy,
            probabilities
        )


    return (
        mean_loss,
        accuracy
    )


# ================================================================
# FIT MODEL
# ================================================================

def fit_model(
    params,
    X_training,
    y_training,
    X_validation=None,
    y_validation=None,
    seed=SEED,
    epochs=MAX_EPOCHS,
    use_early_stopping=True,
    verbose=False
):

    clear_device()


    model = build_model(
        params,
        seed
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(
            params[
                "learning_rate"
            ]
        ),
        weight_decay=float(
            params[
                "weight_decay"
            ]
        )
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=
            LR_SCHEDULER_PATIENCE,
        min_lr=1e-6
    )


    class_weights = get_class_weights(
        y_training
    )


    criterion = nn.CrossEntropyLoss(
        weight=
            class_weights,
        label_smoothing=
            float(
                params[
                    "label_smoothing"
                ]
            )
    )


    training_data = DeviceData(
        X_training,
        y_training
    )


    if X_validation is not None:

        validation_data = DeviceData(
            X_validation,
            y_validation
        )


    else:

        validation_data = None


    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": []
    }


    best_loss = float(
        "inf"
    )


    best_state = None

    best_epoch = 1

    patience_counter = 0


    for epoch in range(
        1,
        epochs + 1
    ):

        training_loss, training_accuracy = train_one_epoch(
            model,
            training_data,
            optimizer,
            criterion
        )


        if validation_data is not None:

            validation_loss, validation_accuracy = evaluate_model(
                model,
                validation_data,
                criterion
            )


            monitored_loss = (
                validation_loss
            )


        else:

            validation_loss = np.nan

            validation_accuracy = np.nan

            monitored_loss = (
                training_loss
            )


        scheduler.step(
            monitored_loss
        )


        learning_rate = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )


        history[
            "loss"
        ].append(
            training_loss
        )


        history[
            "accuracy"
        ].append(
            training_accuracy
        )


        history[
            "val_loss"
        ].append(
            validation_loss
        )


        history[
            "val_accuracy"
        ].append(
            validation_accuracy
        )


        history[
            "learning_rate"
        ].append(
            learning_rate
        )


        if verbose:

            print(
                f"Epoch "
                f"{epoch:02d}/"
                f"{epochs} | "
                f"Train Loss "
                f"{training_loss:.4f} | "
                f"Train Acc "
                f"{training_accuracy:.4f} | "
                f"Val Loss "
                f"{validation_loss:.4f} | "
                f"Val Acc "
                f"{validation_accuracy:.4f} | "
                f"LR "
                f"{learning_rate:.2e}"
            )


        if use_early_stopping:

            if (
                monitored_loss
                < best_loss
                - 1e-5
            ):

                best_loss = (
                    monitored_loss
                )


                best_epoch = (
                    epoch
                )


                best_state = {
                    key:
                        value.detach()
                        .cpu()
                        .clone()

                    for key, value
                    in model.state_dict().items()
                }


                patience_counter = 0


            else:

                patience_counter += 1


                if (
                    patience_counter
                    >= EARLY_STOPPING_PATIENCE
                ):

                    if verbose:

                        print(
                            "Early stopping."
                        )


                    break


    if (
        use_early_stopping
        and
        best_state is not None
    ):

        model.load_state_dict(
            best_state
        )


    del optimizer
    del scheduler
    del criterion
    del class_weights
    del training_data


    if validation_data is not None:

        del validation_data


    if best_state is not None:

        del best_state


    clear_device()


    return (
        model,
        history,
        best_epoch
    )


# ================================================================
# PREDICT PROBABILITIES
# ================================================================

def predict_probabilities(
    model,
    X,
    y
):

    data = DeviceData(
        X,
        y
    )


    _, _, probabilities = evaluate_model(
        model,
        data,
        return_probabilities=True
    )


    del data

    clear_device()


    return probabilities


# ================================================================
# GPU SMOKE TEST
# ================================================================

print()

print("=" * 80)

print(
    "MODEL DEVICE SMOKE TEST"
)

print("=" * 80)


smoke_params = {
    "d_model":
        64,

    "num_heads":
        4,

    "self_layers":
        1,

    "cross_layers":
        1,

    "ffn_expansion":
        2,

    "tokenizer_kernel":
        5,

    "tokenizer_stride":
        2,

    "dropout":
        0.20,

    "classifier_hidden":
        128,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-4,

    "label_smoothing":
        0.05
}


smoke_model = build_model(
    smoke_params,
    SEED
)


smoke_input = torch.randn(
    8,
    NUMBER_FEATURES,
    dtype=torch.float32,
    device=DEVICE
)


with torch.inference_mode():

    smoke_output = smoke_model(
        smoke_input
    )


synchronize()


print(
    "Input:",
    smoke_input.shape
)

print(
    "Output:",
    smoke_output.shape
)

print(
    "Output device:",
    smoke_output.device
)

print(
    "Smoke test passed."
)


del smoke_model
del smoke_input
del smoke_output

clear_device()


# ================================================================
# OPTUNA OBJECTIVE
# ================================================================

def objective(
    trial
):

    params = {
        "d_model":

            trial.suggest_categorical(
                "d_model",
                [
                    64,
                    96
                ]
            ),

        "num_heads":

            trial.suggest_categorical(
                "num_heads",
                [
                    4,
                    8
                ]
            ),

        "self_layers":

            trial.suggest_int(
                "self_layers",
                1,
                2
            ),

        "cross_layers":

            trial.suggest_int(
                "cross_layers",
                1,
                2
            ),

        "ffn_expansion":

            trial.suggest_categorical(
                "ffn_expansion",
                [
                    2,
                    4
                ]
            ),

        "tokenizer_kernel":

            trial.suggest_categorical(
                "tokenizer_kernel",
                [
                    3,
                    5,
                    7
                ]
            ),

        "tokenizer_stride":

            trial.suggest_categorical(
                "tokenizer_stride",
                [
                    2,
                    4
                ]
            ),

        "dropout":

            trial.suggest_float(
                "dropout",
                0.10,
                0.35
            ),

        "classifier_hidden":

            trial.suggest_categorical(
                "classifier_hidden",
                [
                    128,
                    256
                ]
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.08
            )
    }


    print()

    print(
        f"OPTUNA TRIAL "
        f"{trial.number + 1}"
    )


    print(
        params
    )


    fold_scores = []


    for fold_number, (
        training_index,
        validation_index
    ) in enumerate(
        gkf_splits,
        start=1
    ):

        fold_start = time.time()


        (
            X_fold_training,
            X_fold_validation
        ) = prepare_fold(
            training_index,
            validation_index
        )


        model, history, best_epoch = fit_model(
            params,
            X_fold_training,
            y_train[
                training_index
            ],
            X_fold_validation,
            y_train[
                validation_index
            ],
            seed=
                SEED
                + fold_number,
            epochs=
                MAX_EPOCHS,
            use_early_stopping=
                True,
            verbose=
                False
        )


        probabilities = predict_probabilities(
            model,
            X_fold_validation,
            y_train[
                validation_index
            ]
        )


        predictions = np.argmax(
            probabilities,
            axis=1
        )


        fold_accuracy = accuracy_score(
            y_train[
                validation_index
            ],
            predictions
        )


        fold_scores.append(
            fold_accuracy
        )


        print(
            f"Trial "
            f"{trial.number + 1}, "
            f"Fold "
            f"{fold_number}/"
            f"{GROUP_K_FOLDS}, "
            f"Accuracy = "
            f"{fold_accuracy:.4f}, "
            f"Best Epoch = "
            f"{best_epoch}, "
            f"Time = "
            f"{(time.time() - fold_start) / 60:.2f} min, "
            f"RAM = "
            f"{memory_mb():.2f} MB"
        )


        del model
        del history
        del probabilities
        del predictions
        del X_fold_training
        del X_fold_validation


        clear_device()


    mean_accuracy = float(
        np.mean(
            fold_scores
        )
    )


    std_accuracy = float(
        np.std(
            fold_scores
        )
    )


    print(
        f"Trial "
        f"{trial.number + 1} "
        f"Mean = "
        f"{mean_accuracy:.4f} "
        f"plus or minus "
        f"{std_accuracy:.4f}"
    )


    return mean_accuracy


# ================================================================
# OPTUNA
# ================================================================

print()

print("=" * 80)

print(
    "OPTUNA HYPERPARAMETER OPTIMIZATION"
)

print("=" * 80)


optuna_start = time.time()


study = optuna.create_study(
    study_name=
        OPTUNA_STUDY_NAME,

    storage=
        OPTUNA_STORAGE,

    direction=
        "maximize",

    sampler=
        optuna.samplers.TPESampler(
            seed=SEED
        ),

    load_if_exists=
        True
)


completed_trials = sum(
    1
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
)


remaining_trials = max(
    0,
    N_TRIALS
    - completed_trials
)


print(
    "Previously completed trials:",
    completed_trials
)

print(
    "Remaining trials:",
    remaining_trials
)


if remaining_trials > 0:

    study.optimize(
        objective,
        n_trials=
            remaining_trials,
        show_progress_bar=
            False,
        gc_after_trial=
            True
    )


complete_trials = [
    trial
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
]


if len(
    complete_trials
) == 0:

    raise RuntimeError(
        "No Optuna trial completed successfully."
    )


optuna_time = (
    time.time()
    - optuna_start
)


best_params = dict(
    study.best_params
)


print()

print("=" * 80)

print(
    "OPTUNA COMPLETED"
)

print("=" * 80)

print(
    "Best GroupKFold Accuracy:",
    f"{study.best_value:.4f}"
)

print(
    "Best Parameters:"
)

print(
    best_params
)

print(
    "Optuna Time:",
    f"{optuna_time / 60:.2f}",
    "minutes"
)


study.trials_dataframe().to_csv(
    OUTPUT_DIR
    / "optuna_trials.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "best_parameters.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        best_params,
        file,
        indent=4
    )


# ================================================================
# CONFIGURATION ID
# ================================================================

configuration_text = json.dumps(
    {
        "best_params":
            best_params,

        "feature_count":
            NUMBER_FEATURES,

        "classes":
            class_names,

        "modalities":
            MODALITY_INDICES,

        "grouping_mode":
            MODALITY_GROUPING_MODE,

        "seed":
            SEED
    },
    sort_keys=True
)


CONFIG_ID = hashlib.sha256(
    configuration_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


print(
    "Configuration ID:",
    CONFIG_ID
)


# ================================================================
# MODEL ARCHITECTURE
# ================================================================

temporary_model = build_model(
    best_params,
    SEED
)


TRAINABLE_PARAMETERS = trainable_parameter_count(
    temporary_model
)


print(
    "Trainable parameters:",
    f"{TRAINABLE_PARAMETERS:,}"
)


with open(
    OUTPUT_DIR
    / "model_architecture.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        str(
            temporary_model
        )
    )


del temporary_model

clear_device()


# ================================================================
# FIVE FINAL RUNS
# ================================================================

print()

print("=" * 80)

print(
    "FIVE STOCHASTIC FINAL RUNS"
)

print("=" * 80)


run_records = []

completed_runs = set()


if FIVE_RUNS_PATH.exists():

    previous_runs = pd.read_csv(
        FIVE_RUNS_PATH
    )


    if (
        "Config_ID"
        in previous_runs.columns
    ):

        previous_runs = previous_runs[
            previous_runs[
                "Config_ID"
            ].astype(str)
            == CONFIG_ID
        ]


        for _, row in previous_runs.iterrows():

            run_number = int(
                row[
                    "Run"
                ]
            )


            checkpoint_path = (
                CHECKPOINT_DIR
                / f"run_{run_number:02d}.pt"
            )


            if checkpoint_path.exists():

                run_records.append(
                    row.to_dict()
                )

                completed_runs.add(
                    run_number
                )


print(
    "Previously completed runs:",
    len(
        completed_runs
    )
)


five_run_start = time.time()


for run_number in range(
    1,
    N_RUNS + 1
):

    if run_number in completed_runs:

        print(
            f"Run "
            f"{run_number}/"
            f"{N_RUNS} "
            f"already completed."
        )

        continue


    print()

    print(
        f"RUN "
        f"{run_number}/"
        f"{N_RUNS}"
    )


    run_start = time.time()


    run_seed = (
        SEED
        + run_number
        - 1
    )


    model, history, best_epoch = fit_model(
        best_params,
        X_train,
        y_train,
        X_validation,
        y_validation,
        seed=
            run_seed,
        epochs=
            MAX_EPOCHS,
        use_early_stopping=
            True,
        verbose=
            True
    )


    # ============================================================
    # TRAINING EVALUATION
    # ============================================================

    training_probabilities = predict_probabilities(
        model,
        X_train,
        y_train
    )


    training_predictions = np.argmax(
        training_probabilities,
        axis=1
    )


    training_accuracy = accuracy_score(
        y_train,
        training_predictions
    )


    # ============================================================
    # VALIDATION EVALUATION
    # ============================================================

    validation_probabilities = predict_probabilities(
        model,
        X_validation,
        y_validation
    )


    validation_predictions = np.argmax(
        validation_probabilities,
        axis=1
    )


    validation_accuracy = accuracy_score(
        y_validation,
        validation_predictions
    )


    run_time = (
        time.time()
        - run_start
    )


    record = {
        "Config_ID":
            CONFIG_ID,

        "Run":
            run_number,

        "Seed":
            run_seed,

        "Training_Accuracy":
            float(
                training_accuracy
            ),

        "Validation_Accuracy":
            float(
                validation_accuracy
            ),

        "Epochs_Run":
            int(
                len(
                    history[
                        "loss"
                    ]
                )
            ),

        "Best_Epoch":
            int(
                best_epoch
            ),

        "Execution_Time_Seconds":
            float(
                run_time
            )
    }


    model_state = {
        key:
            value.detach()
            .cpu()
            .clone()

        for key, value
        in model.state_dict().items()
    }


    checkpoint_path = (
        CHECKPOINT_DIR
        / f"run_{run_number:02d}.pt"
    )


    atomic_torch_save(
        {
            "Config_ID":
                CONFIG_ID,

            "record":
                record,

            "best_params":
                best_params,

            "model_state_dict":
                model_state,

            "history":
                history
        },
        checkpoint_path
    )


    run_records.append(
        record
    )


    temporary_run_table = pd.DataFrame(
        run_records
    )


    temporary_run_table = (
        temporary_run_table
        .sort_values(
            "Run"
        )
        .drop_duplicates(
            subset=[
                "Run"
            ],
            keep="last"
        )
    )


    atomic_csv(
        temporary_run_table,
        FIVE_RUNS_PATH
    )


    print()

    print(
        f"[RUN {run_number}] "
        f"Train = "
        f"{training_accuracy:.4f}, "
        f"Validation = "
        f"{validation_accuracy:.4f}, "
        f"Best Epoch = "
        f"{best_epoch}, "
        f"Time = "
        f"{run_time / 60:.2f} min"
    )


    del model
    del history
    del model_state
    del training_probabilities
    del training_predictions
    del validation_probabilities
    del validation_predictions


    clear_device()


five_run_time = (
    time.time()
    - five_run_start
)


# ================================================================
# COMPLETE RUN TABLE
# ================================================================

run_table = pd.read_csv(
    FIVE_RUNS_PATH
)


run_table = run_table[
    run_table[
        "Config_ID"
    ].astype(str)
    == CONFIG_ID
].copy()


run_table = (
    run_table
    .sort_values(
        "Run"
    )
    .drop_duplicates(
        subset=[
            "Run"
        ],
        keep="last"
    )
)


if len(
    run_table
) != N_RUNS:

    raise RuntimeError(
        "Five final runs are not complete."
    )


print()

print(
    run_table[
        [
            "Run",
            "Seed",
            "Training_Accuracy",
            "Validation_Accuracy",
            "Best_Epoch",
            "Execution_Time_Seconds"
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ================================================================
# BEST VALIDATION RUN
# ================================================================

selected_row = (
    run_table
    .sort_values(
        by=[
            "Validation_Accuracy",
            "Run"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[
        0
    ]
)


best_run_number = int(
    selected_row[
        "Run"
    ]
)


best_checkpoint_path = (
    CHECKPOINT_DIR
    / f"run_{best_run_number:02d}.pt"
)


best_bundle = torch.load(
    best_checkpoint_path,
    map_location="cpu",
    weights_only=False
)


best_record = best_bundle[
    "record"
]


best_history = best_bundle[
    "history"
]


print()

print(
    "Selected Run:",
    best_record[
        "Run"
    ]
)

print(
    "Selected Seed:",
    best_record[
        "Seed"
    ]
)

print(
    "Selected Validation Accuracy:",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Selected Best Epoch:",
    best_record[
        "Best_Epoch"
    ]
)


# ================================================================
# SAVE FINAL MODEL PACKAGE
# ================================================================

atomic_torch_save(
    {
        "Algorithm":
            "Cross Attention Multimodal Transformer",

        "Dataset":
            "Max Dataset",

        "Device_Type":
            DEVICE_TYPE,

        "Device_Name":
            DEVICE_NAME,

        "Config_ID":
            CONFIG_ID,

        "best_params":
            best_params,

        "selected_run":
            int(
                best_record[
                    "Run"
                ]
            ),

        "selected_seed":
            int(
                best_record[
                    "Seed"
                ]
            ),

        "best_epoch":
            int(
                best_record[
                    "Best_Epoch"
                ]
            ),

        "model_state_dict":
            best_bundle[
                "model_state_dict"
            ],

        "feature_columns":
            feature_cols,

        "modality_indices":
            MODALITY_INDICES,

        "modality_names":
            MODALITY_NAMES,

        "modality_grouping_mode":
            MODALITY_GROUPING_MODE,

        "class_names":
            class_names
    },
    BEST_MODEL_PATH
)


# ================================================================
# TRAINING HISTORY
# ================================================================

history_table = pd.DataFrame(
    {
        "Epoch":
            np.arange(
                1,
                len(
                    best_history[
                        "loss"
                    ]
                )
                + 1
            ),

        "Training_Loss":
            best_history[
                "loss"
            ],

        "Training_Accuracy":
            best_history[
                "accuracy"
            ],

        "Validation_Loss":
            best_history[
                "val_loss"
            ],

        "Validation_Accuracy":
            best_history[
                "val_accuracy"
            ],

        "Learning_Rate":
            best_history[
                "learning_rate"
            ]
    }
)


history_table.to_csv(
    OUTPUT_DIR
    / "training_history.csv",
    index=False
)


# ================================================================
# LOSS CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Loss"
    ],
    label=
        "Training Loss"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Loss"
    ],
    label=
        "Validation Loss"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Cross Entropy Loss"
)

ax.set_title(
    "Cross Attention Multimodal Transformer Loss"
)

ax.legend()

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# ACCURACY CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Accuracy"
    ],
    label=
        "Training Accuracy"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Accuracy"
    ],
    label=
        "Validation Accuracy"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Accuracy"
)

ax.set_title(
    "Cross Attention Multimodal Transformer Accuracy"
)

ax.legend()

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# LEARNING RATE CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Learning_Rate"
    ]
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Learning Rate"
)

ax.set_title(
    "Learning Rate Schedule"
)

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "learning_rate_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# FIVE RUN STATISTICS
# ================================================================

run_train_mean = float(
    run_table[
        "Training_Accuracy"
    ].mean()
)


run_train_std = float(
    run_table[
        "Training_Accuracy"
    ].std(
        ddof=0
    )
)


run_validation_mean = float(
    run_table[
        "Validation_Accuracy"
    ].mean()
)


run_validation_std = float(
    run_table[
        "Validation_Accuracy"
    ].std(
        ddof=0
    )
)


# ================================================================
# RELEASE FINAL TRAIN AND VALIDATION SCALED ARRAYS
#
# Raw training matrix is retained for LOGO.
# ================================================================

del X_train
del X_validation

del X_validation_raw
del X_test_raw

gc.collect()

clear_device()


# ================================================================
# FULL LOGO
#
# Fixed epoch count comes from independent validation participants.
#
# Held out LOGO participants never control early stopping.
# ================================================================

print()

print("=" * 80)

print(
    "FULL LEAVE ONE GROUP OUT"
)

print("=" * 80)


FIXED_LOGO_EPOCHS = max(
    1,
    int(
        best_record[
            "Best_Epoch"
        ]
    )
)


if LOGO_MAX_FOLDS is None:

    LOGO_FOLDS_TO_USE = (
        FULL_LOGO_FOLDS
    )


else:

    LOGO_FOLDS_TO_USE = min(
        int(
            LOGO_MAX_FOLDS
        ),
        FULL_LOGO_FOLDS
    )


print(
    "Available LOGO folds:",
    FULL_LOGO_FOLDS
)

print(
    "LOGO folds requested:",
    LOGO_FOLDS_TO_USE
)

print(
    "Fixed LOGO epochs:",
    FIXED_LOGO_EPOCHS
)


logo_config_text = json.dumps(
    {
        "Config_ID":
            CONFIG_ID,

        "Fixed_Epochs":
            FIXED_LOGO_EPOCHS,

        "Seed":
            int(
                best_record[
                    "Seed"
                ]
            )
    },
    sort_keys=True
)


LOGO_CONFIG_ID = hashlib.sha256(
    logo_config_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


# ================================================================
# LOAD LOGO CHECKPOINT
# ================================================================

logo_records = []

completed_logo_persons = set()


if LOGO_RESULT_PATH.exists():

    previous_logo = pd.read_csv(
        LOGO_RESULT_PATH
    )


    if (
        "Config_ID"
        in previous_logo.columns
    ):

        matching_logo = previous_logo[
            previous_logo[
                "Config_ID"
            ].astype(str)
            == LOGO_CONFIG_ID
        ].copy()


        if len(
            matching_logo
        ) > 0:

            logo_records = matching_logo.to_dict(
                orient="records"
            )


            completed_logo_persons = set(
                matching_logo[
                    "Held_Out_Person"
                ].astype(str)
            )


print(
    "Previously completed LOGO folds:",
    len(
        completed_logo_persons
    )
)


# ================================================================
# RUN LOGO
# ================================================================

logo_start = time.time()


logo_splitter = LeaveOneGroupOut()


for fold_number, (
    training_index,
    validation_index
) in enumerate(
    logo_splitter.split(
        X_train_raw,
        y_train,
        g_train
    ),
    start=1
):

    if fold_number > LOGO_FOLDS_TO_USE:

        break


    held_out_persons = np.unique(
        g_train[
            validation_index
        ]
    )


    if len(
        held_out_persons
    ) != 1:

        raise RuntimeError(
            "Unexpected LOGO participant grouping."
        )


    held_out_person = str(
        held_out_persons[
            0
        ]
    )


    if (
        held_out_person
        in completed_logo_persons
    ):

        print(
            f"LOGO "
            f"{fold_number}/"
            f"{LOGO_FOLDS_TO_USE}, "
            f"Participant "
            f"{held_out_person} "
            f"already completed."
        )

        continue


    fold_start = time.time()


    (
        X_logo_training,
        X_logo_validation
    ) = prepare_fold(
        training_index,
        validation_index
    )


    logo_model, _, _ = fit_model(
        best_params,
        X_logo_training,
        y_train[
            training_index
        ],
        X_validation=None,
        y_validation=None,
        seed=
            int(
                best_record[
                    "Seed"
                ]
            ),
        epochs=
            FIXED_LOGO_EPOCHS,
        use_early_stopping=
            False,
        verbose=
            False
    )


    logo_probabilities = predict_probabilities(
        logo_model,
        X_logo_validation,
        y_train[
            validation_index
        ]
    )


    logo_predictions = np.argmax(
        logo_probabilities,
        axis=1
    )


    fold_accuracy = accuracy_score(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_precision = precision_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_recall = recall_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_f1 = f1_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_mcc = matthews_corrcoef(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_time = (
        time.time()
        - fold_start
    )


    logo_record = {
        "Config_ID":
            LOGO_CONFIG_ID,

        "Fold":
            fold_number,

        "Held_Out_Person":
            held_out_person,

        "Rows":
            int(
                len(
                    validation_index
                )
            ),

        "Accuracy":
            float(
                fold_accuracy
            ),

        "Precision_Weighted":
            float(
                fold_precision
            ),

        "Recall_Weighted":
            float(
                fold_recall
            ),

        "F1_Weighted":
            float(
                fold_f1
            ),

        "MCC":
            float(
                fold_mcc
            ),

        "Fixed_Epochs":
            int(
                FIXED_LOGO_EPOCHS
            ),

        "Execution_Time_Seconds":
            float(
                fold_time
            )
    }


    logo_records.append(
        logo_record
    )


    atomic_csv(
        pd.DataFrame(
            logo_records
        ),
        LOGO_RESULT_PATH
    )


    completed_logo_persons.add(
        held_out_person
    )


    print(
        f"LOGO "
        f"{fold_number}/"
        f"{LOGO_FOLDS_TO_USE}, "
        f"Participant = "
        f"{held_out_person}, "
        f"Accuracy = "
        f"{fold_accuracy:.4f}, "
        f"F1 = "
        f"{fold_f1:.4f}, "
        f"MCC = "
        f"{fold_mcc:.4f}, "
        f"Time = "
        f"{fold_time / 60:.2f} min"
    )


    del logo_model
    del logo_probabilities
    del logo_predictions
    del X_logo_training
    del X_logo_validation


    clear_device()


logo_time = (
    time.time()
    - logo_start
)


# ================================================================
# LOGO SUMMARY
# ================================================================

logo_table = pd.read_csv(
    LOGO_RESULT_PATH
)


logo_table = logo_table[
    logo_table[
        "Config_ID"
    ].astype(str)
    == LOGO_CONFIG_ID
].copy()


logo_table = (
    logo_table
    .sort_values(
        "Fold"
    )
    .drop_duplicates(
        subset=[
            "Held_Out_Person"
        ],
        keep="last"
    )
)


logo_accuracies = (
    logo_table[
        "Accuracy"
    ]
    .astype(float)
    .to_numpy()
)


logo_mean = float(
    np.mean(
        logo_accuracies
    )
)


logo_std = float(
    np.std(
        logo_accuracies
    )
)


print()

print(
    "Full LOGO Accuracy:",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Completed LOGO folds:",
    len(
        logo_table
    )
)

print(
    "LOGO time this session:",
    f"{logo_time / 60:.2f}",
    "minutes"
)


# ================================================================
# RESTORE BEST MODEL
# ================================================================

best_model = build_model(
    best_params,
    seed=
        int(
            best_record[
                "Seed"
            ]
        )
)


best_model.load_state_dict(
    best_bundle[
        "model_state_dict"
    ]
)


best_model.eval()


# ================================================================
# FINAL UNTOUCHED TEST EVALUATION
# ================================================================

print()

print("=" * 80)

print(
    "FINAL UNTOUCHED TEST EVALUATION"
)

print("=" * 80)


test_start = time.time()


test_probabilities = predict_probabilities(
    best_model,
    X_test,
    y_test
)


test_predictions = np.argmax(
    test_probabilities,
    axis=1
)


test_time = (
    time.time()
    - test_start
)


# ================================================================
# TEST METRICS
# ================================================================

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)


test_precision = precision_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_recall = recall_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_weighted = f1_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_macro = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0
)


test_kappa = cohen_kappa_score(
    y_test,
    test_predictions
)


test_mcc = matthews_corrcoef(
    y_test,
    test_predictions
)


try:

    test_roc_auc = roc_auc_score(
        y_test,
        test_probabilities,
        multi_class="ovr",
        average="weighted"
    )


except Exception:

    test_roc_auc = np.nan


# ================================================================
# CLASSIFICATION REPORT
# ================================================================

report = classification_report(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    ),
    target_names=
        class_names,
    digits=4,
    zero_division=0
)


print()

print(
    "CLASSIFICATION REPORT"
)

print()

print(
    report
)


with open(
    OUTPUT_DIR
    / "classification_report.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        report
    )


report_table = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        labels=np.arange(
            NUMBER_CLASSES
        ),
        target_names=
            class_names,
        output_dict=True,
        zero_division=0
    )
).transpose()


report_table.to_csv(
    OUTPUT_DIR
    / "classification_report.csv"
)


# ================================================================
# CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    )
)


pd.DataFrame(
    cm,
    index=
        class_names,
    columns=
        class_names
).to_csv(
    OUTPUT_DIR
    / "confusion_matrix.csv"
)


fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "Cross Attention Multimodal Transformer Confusion Matrix"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            str(
                cm[
                    row,
                    column
                ]
            ),
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# NORMALIZED CONFUSION MATRIX
# ================================================================

row_sums = cm.sum(
    axis=1,
    keepdims=True
)


normalized_cm = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(
        cm,
        dtype=np.float64
    ),
    where=(
        row_sums != 0
    )
)


pd.DataFrame(
    normalized_cm,
    index=
        class_names,
    columns=
        class_names
).to_csv(
    OUTPUT_DIR
    / "normalized_confusion_matrix.csv"
)


fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    normalized_cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "Normalized Cross Attention Multimodal Transformer"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            f"{normalized_cm[row, column]:.2f}",
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "normalized_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# TEST PREDICTIONS
# ================================================================

prediction_table = pd.DataFrame(
    {
        "PersonSrial":
            g_test,

        "True_Class":
            label_encoder.inverse_transform(
                y_test
            ),

        "Predicted_Class":
            label_encoder.inverse_transform(
                test_predictions.astype(
                    np.int32
                )
            )
    }
)


for class_index, class_name in enumerate(
    class_names
):

    prediction_table[
        f"Probability_{class_name}"
    ] = test_probabilities[
        :,
        class_index
    ]


prediction_table.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False
)


# ================================================================
# TOTAL EXECUTION TIME
# ================================================================

total_execution_time = (
    time.time()
    - TOTAL_START
)


# ================================================================
# FINAL METRICS
# ================================================================

final_metrics = {
    "Algorithm":
        "Cross Attention Multimodal Transformer",

    "Dataset":
        "Max Dataset",

    "Framework":
        "PyTorch",

    "Input_File":
        str(
            FILE_PATH
        ),

    "Output_Directory":
        str(
            OUTPUT_DIR
        ),

    "Device_Type":
        DEVICE_TYPE,

    "Device_Name":
        DEVICE_NAME,

    "Feature_Count":
        int(
            NUMBER_FEATURES
        ),

    "Modality_Count":
        int(
            NUMBER_MODALITIES
        ),

    "Modality_Names":
        MODALITY_NAMES,

    "Modality_Grouping_Mode":
        MODALITY_GROUPING_MODE,

    "Trainable_Parameters":
        int(
            TRAINABLE_PARAMETERS
        ),

    "Classes":
        class_names,

    "Training_Participants":
        int(
            len(
                train_persons
            )
        ),

    "Validation_Participants":
        int(
            len(
                validation_persons
            )
        ),

    "Testing_Participants":
        int(
            len(
                test_persons
            )
        ),

    "Training_Samples":
        int(
            TRAIN_SAMPLES
        ),

    "Validation_Samples":
        int(
            VALIDATION_SAMPLES
        ),

    "Testing_Samples":
        int(
            TEST_SAMPLES
        ),

    "Optuna_Trials":
        int(
            N_TRIALS
        ),

    "Optuna_GroupKFold_Accuracy":
        float(
            study.best_value
        ),

    "Best_Parameters":
        best_params,

    "Selected_Run":
        int(
            best_record[
                "Run"
            ]
        ),

    "Selected_Seed":
        int(
            best_record[
                "Seed"
            ]
        ),

    "Selected_Best_Epoch":
        int(
            best_record[
                "Best_Epoch"
            ]
        ),

    "Training_Accuracy":
        float(
            best_record[
                "Training_Accuracy"
            ]
        ),

    "Validation_Accuracy":
        float(
            best_record[
                "Validation_Accuracy"
            ]
        ),

    "Testing_Accuracy":
        float(
            test_accuracy
        ),

    "Five_Run_Training_Mean":
        float(
            run_train_mean
        ),

    "Five_Run_Training_STD":
        float(
            run_train_std
        ),

    "Five_Run_Validation_Mean":
        float(
            run_validation_mean
        ),

    "Five_Run_Validation_STD":
        float(
            run_validation_std
        ),

    "Full_LOGO_Accuracy":
        float(
            logo_mean
        ),

    "Full_LOGO_STD":
        float(
            logo_std
        ),

    "Full_LOGO_Folds_Available":
        int(
            FULL_LOGO_FOLDS
        ),

    "Full_LOGO_Folds_Completed":
        int(
            len(
                logo_table
            )
        ),

    "Precision_Weighted":
        float(
            test_precision
        ),

    "Recall_Weighted":
        float(
            test_recall
        ),

    "F1_Weighted":
        float(
            test_f1_weighted
        ),

    "F1_Macro":
        float(
            test_f1_macro
        ),

    "Cohen_Kappa":
        float(
            test_kappa
        ),

    "MCC":
        float(
            test_mcc
        ),

    "ROC_AUC_Weighted_OVR":
        (
            None
            if np.isnan(
                test_roc_auc
            )
            else float(
                test_roc_auc
            )
        ),

    "Optuna_Time_Seconds":
        float(
            optuna_time
        ),

    "Five_Run_Time_This_Session_Seconds":
        float(
            five_run_time
        ),

    "LOGO_Time_This_Session_Seconds":
        float(
            logo_time
        ),

    "Test_Time_Seconds":
        float(
            test_time
        ),

    "Total_Execution_Time_Seconds":
        float(
            total_execution_time
        ),

    "Current_RAM_MB":
        float(
            memory_mb()
        )
}


with open(
    OUTPUT_DIR
    / "final_metrics.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        final_metrics,
        file,
        indent=4
    )


# ================================================================
# FINAL SUMMARY
# ================================================================

summary_lines = [
    "CROSS ATTENTION MULTIMODAL TRANSFORMER",
    "MAX DATASET",

    "",

    f"Input File: "
    f"{FILE_PATH}",

    f"Output Directory: "
    f"{OUTPUT_DIR}",

    "",

    f"Device: "
    f"{DEVICE_NAME}",

    f"Feature Count: "
    f"{NUMBER_FEATURES}",

    f"Feature Modalities: "
    f"{MODALITY_NAMES}",

    f"Grouping Mode: "
    f"{MODALITY_GROUPING_MODE}",

    f"Trainable Parameters: "
    f"{TRAINABLE_PARAMETERS}",

    "",

    f"Best Parameters: "
    f"{best_params}",

    "",

    f"Selected Run: "
    f"{best_record['Run']}",

    f"Selected Seed: "
    f"{best_record['Seed']}",

    f"Best Epoch: "
    f"{best_record['Best_Epoch']}",

    "",

    f"Optuna GroupKFold Accuracy: "
    f"{study.best_value:.4f}",

    f"Training Accuracy: "
    f"{best_record['Training_Accuracy']:.4f}",

    f"Validation Accuracy: "
    f"{best_record['Validation_Accuracy']:.4f}",

    f"Testing Accuracy: "
    f"{test_accuracy:.4f}",

    "",

    f"Five Run Training: "
    f"{run_train_mean:.4f} "
    f"plus or minus "
    f"{run_train_std:.4f}",

    f"Five Run Validation: "
    f"{run_validation_mean:.4f} "
    f"plus or minus "
    f"{run_validation_std:.4f}",

    "",

    f"Full LOGO Accuracy: "
    f"{logo_mean:.4f} "
    f"plus or minus "
    f"{logo_std:.4f}",

    f"Full LOGO Folds: "
    f"{len(logo_table)}",

    "",

    f"Precision Weighted: "
    f"{test_precision:.4f}",

    f"Recall Weighted: "
    f"{test_recall:.4f}",

    f"F1 Weighted: "
    f"{test_f1_weighted:.4f}",

    f"F1 Macro: "
    f"{test_f1_macro:.4f}",

    f"Cohen Kappa: "
    f"{test_kappa:.4f}",

    f"MCC: "
    f"{test_mcc:.4f}",

    (
        "ROC AUC Weighted OVR: N/A"
        if np.isnan(
            test_roc_auc
        )
        else
        f"ROC AUC Weighted OVR: "
        f"{test_roc_auc:.4f}"
    ),

    "",

    f"Optuna Time: "
    f"{optuna_time / 60:.2f} minutes",

    f"Five Run Time: "
    f"{five_run_time / 60:.2f} minutes",

    f"LOGO Time: "
    f"{logo_time / 60:.2f} minutes",

    f"Test Time: "
    f"{test_time:.2f} seconds",

    f"Total Execution Time: "
    f"{total_execution_time / 60:.2f} minutes"
]


with open(
    OUTPUT_DIR
    / "final_summary.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        "\n".join(
            summary_lines
        )
    )


# ================================================================
# FINAL SCREEN OUTPUT
# ================================================================

print()

print("#" * 80)

print(
    "FINAL CROSS ATTENTION MULTIMODAL TRANSFORMER RESULTS"
)

print(
    "MAX DATASET"
)

print("#" * 80)


print(
    "Device                    :",
    DEVICE_NAME
)

print(
    "Feature Count             :",
    NUMBER_FEATURES
)

print(
    "Grouping Mode             :",
    MODALITY_GROUPING_MODE
)


for modality_name in MODALITY_NAMES:

    print(
        f"{modality_name:<26}:",
        len(
            MODALITY_INDICES[
                modality_name
            ]
        ),
        "features"
    )


print(
    "Trainable Parameters      :",
    f"{TRAINABLE_PARAMETERS:,}"
)


print()

print(
    "Embedding Dimension       :",
    best_params[
        "d_model"
    ]
)

print(
    "Attention Heads           :",
    best_params[
        "num_heads"
    ]
)

print(
    "Self Attention Layers     :",
    best_params[
        "self_layers"
    ]
)

print(
    "Cross Attention Layers    :",
    best_params[
        "cross_layers"
    ]
)

print(
    "FFN Expansion             :",
    best_params[
        "ffn_expansion"
    ]
)

print(
    "Tokenizer Kernel          :",
    best_params[
        "tokenizer_kernel"
    ]
)

print(
    "Tokenizer Stride          :",
    best_params[
        "tokenizer_stride"
    ]
)

print(
    "Dropout                   :",
    f"{best_params['dropout']:.4f}"
)

print(
    "Learning Rate             :",
    f"{best_params['learning_rate']:.2e}"
)

print(
    "Weight Decay              :",
    f"{best_params['weight_decay']:.2e}"
)


print()

print(
    "Selected Run              :",
    best_record[
        "Run"
    ]
)

print(
    "Selected Seed             :",
    best_record[
        "Seed"
    ]
)

print(
    "Best Epoch                :",
    best_record[
        "Best_Epoch"
    ]
)


print()

print(
    "Optuna GroupKFold         :",
    f"{study.best_value:.4f}"
)

print(
    "Training Accuracy         :",
    f"{best_record['Training_Accuracy']:.4f}"
)

print(
    "Validation Accuracy       :",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Testing Accuracy          :",
    f"{test_accuracy:.4f}"
)


print()

print(
    "Five Run Training         :",
    f"{run_train_mean:.4f}",
    "plus or minus",
    f"{run_train_std:.4f}"
)

print(
    "Five Run Validation       :",
    f"{run_validation_mean:.4f}",
    "plus or minus",
    f"{run_validation_std:.4f}"
)


print()

print(
    "Full LOGO Accuracy        :",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Full LOGO Folds           :",
    len(
        logo_table
    ),
    "of",
    FULL_LOGO_FOLDS
)


print()

print(
    "Precision Weighted        :",
    f"{test_precision:.4f}"
)

print(
    "Recall Weighted           :",
    f"{test_recall:.4f}"
)

print(
    "F1 Weighted               :",
    f"{test_f1_weighted:.4f}"
)

print(
    "F1 Macro                  :",
    f"{test_f1_macro:.4f}"
)

print(
    "Cohen Kappa               :",
    f"{test_kappa:.4f}"
)

print(
    "MCC                       :",
    f"{test_mcc:.4f}"
)


if np.isnan(
    test_roc_auc
):

    print(
        "ROC AUC Weighted OVR      : N/A"
    )


else:

    print(
        "ROC AUC Weighted OVR      :",
        f"{test_roc_auc:.4f}"
    )


print()

print(
    "Optuna Time               :",
    f"{optuna_time / 60:.2f}",
    "minutes"
)

print(
    "Five Run Time             :",
    f"{five_run_time / 60:.2f}",
    "minutes"
)

print(
    "Full LOGO Time            :",
    f"{logo_time / 60:.2f}",
    "minutes"
)

print(
    "Test Prediction Time      :",
    f"{test_time:.2f}",
    "seconds"
)

print(
    "Total Execution Time      :",
    f"{total_execution_time / 60:.2f}",
    "minutes"
)

print(
    "Current RAM               :",
    f"{memory_mb():.2f}",
    "MB"
)


print()

print(
    "No dataset rows were subsampled."
)

print(
    "All five GroupKFold folds were used."
)

print(
    "Full training participant LOGO was requested."
)

print(
    "Every validation fold used its own training fitted scaler."
)

print(
    "Cross attention was performed among all three feature streams."
)

print(
    "Best run was selected only with validation participants."
)

print(
    "Test participants were not used for tuning or model selection."
)

print(
    "Held out LOGO participants were not used for early stopping."
)

print(
    "Optuna, final runs, and LOGO support resume after interruption."
)


print()

print(
    "ALL RESULTS SAVED IN:"
)

print(
    OUTPUT_DIR
)

print("#" * 80)

/home/rakib/Desktop/ml_apply/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE CONFIGURATION
PyTorch version: 2.14.1+xpu
Selected device: xpu:0
Device type: xpu
Device name: Intel(R) Arc(TM) A750 Graphics
CPU threads: 8

CROSS ATTENTION MULTIMODAL TRANSFORMER
MAX DATASET
Input:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/Cleaned_oN_Selected_Max.csv

Output:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/Cross_Attention_Multimodal_Transformer_Results

Optuna trials: 10
GroupKFold folds: 5
Final runs: 5
Maximum epochs: 40
Batch size: 512
Full LOGO: True
Raw dataset shape: (149350, 238)
RAM after loading: 984.09 MB
Feature count: 232
Invalid feature cells: 0
Classes: ['class_lecture', 'music', 'normal', 'reading']
Number of classes: 4
Participants: 60

CLASS DISTRIBUTION
class_lecture: 39857 (26.69%)
music: 32276 (21.61%)
normal: 39327 (26.33%)
reading: 37890 (25.37%)
Loading time: 8.42 seconds

FEATURE MODALITY CONSTRUCTION
Grouping mode: Balanced feature view fallback
feature_view_1 : 78 features
feature_view_2 : 77 features
feature_view_3 : 77 fea

**Apply (Multimodal CNN-LSTM) on Max Dataset**

In [3]:
# ================================================================
# MULTIMODAL CNN LSTM
# MAX DATASET
#
# UPDATED INTEL XPU TRAINABLE VERSION
#
# IMPORTANT
#
# Native torch.nn.LSTM is NOT used.
#
# PyTorch 2.14.1 XPU currently reports:
#
# oneDNN LSTM on XPU only supports inference mode
#
# Therefore this implementation uses a manual standard LSTM
# constructed from Linear, sigmoid, tanh and elementwise
# operations, allowing training on Intel Arc XPU.
#
# ================================================================
#
# DATASET
#
# Complete Max dataset
# 232 engineered EEG features
#
# PERSON INDEPENDENT DESIGN
#
# 70 percent training participants
# 10 percent validation participants
# 20 percent testing participants
#
# 5 fold GroupKFold
# Full Leave One Group Out
#
# ================================================================
#
# ANTI OVERFITTING DESIGN
#
# Participant independent splitting
# Fold specific StandardScaler
# Input Gaussian noise
# Input dropout
# CNN dropout
# Batch normalization
# LSTM dropout
# Dense dropout
# AdamW weight decay
# Label smoothing
# Balanced class weights
# Gradient clipping
# Validation loss early stopping
# ReduceLROnPlateau
# Controlled architecture size
# Five stochastic final runs
#
# ================================================================
#
# GPU
#
# Intel XPU preferred
# CUDA fallback
# CPU fallback
#
# ================================================================
#
# RESUME SUPPORT
#
# Persistent Optuna database
# Five final run checkpoints
# Full LOGO checkpoint
#
# ================================================================


# ================================================================
# IMPORTS
# ================================================================

import os
import gc
import json
import time
import random
import hashlib
import warnings

from pathlib import Path

import joblib
import psutil
import optuna

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.model_selection import (
    GroupKFold,
    LeaveOneGroupOut
)

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.utils.class_weight import (
    compute_class_weight
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ================================================================
# WARNINGS
# ================================================================

warnings.filterwarnings(
    "ignore"
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# ================================================================
# ARCHITECTURE VERSION
#
# This prevents checkpoints from the old native nn.LSTM
# implementation from being reused.
# ================================================================

ARCHITECTURE_VERSION = (
    "manual_xpu_lstm_v2"
)


# ================================================================
# GLOBAL CONFIGURATION
# ================================================================

SEED = 42

N_TRIALS = 10

N_RUNS = 5

GROUP_K_FOLDS = 5


# ================================================================
# TRAINING CONFIGURATION
# ================================================================

MAX_EPOCHS = 50

BATCH_SIZE = 1024

EARLY_STOPPING_PATIENCE = 6

LR_SCHEDULER_PATIENCE = 2

GRADIENT_CLIP = 1.0

EARLY_STOPPING_MIN_DELTA = 1e-4


# ================================================================
# FULL LOGO
#
# None means all 42 training participants.
# ================================================================

LOGO_MAX_FOLDS = None


# ================================================================
# CPU THREADS
# ================================================================

CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)


torch.set_num_threads(
    CPU_THREADS
)


# ================================================================
# INPUT PATH
# ================================================================

FILE_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset/"
    "Cleaned_oN_Selected_Max.csv"
)


if not FILE_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found: {FILE_PATH}"
    )


# ================================================================
# OUTPUT
# ================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/max_dataset"
)


OUTPUT_DIR = (
    OUTPUT_ROOT
    / "Multimodal_CNN_LSTM_Results"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHECKPOINT_DIR = (
    OUTPUT_DIR
    / "checkpoints"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# IMPORTANT PATHS
# ================================================================

OPTUNA_DATABASE = (
    OUTPUT_DIR
    / "multimodal_cnn_lstm_optuna.db"
)


OPTUNA_STORAGE = (
    f"sqlite:///{OPTUNA_DATABASE}"
)


OPTUNA_STUDY_NAME = (
    "Max_Multimodal_CNN_LSTM_ManualXPU_v2"
)


FIVE_RUNS_PATH = (
    OUTPUT_DIR
    / "five_runs.csv"
)


LOGO_RESULT_PATH = (
    OUTPUT_DIR
    / "full_logo_results.csv"
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    / "best_multimodal_cnn_lstm.pt"
)


# ================================================================
# MEMORY MONITOR
# ================================================================

PROCESS = psutil.Process(
    os.getpid()
)


def memory_mb():

    return (
        PROCESS.memory_info().rss
        / 1024
        / 1024
    )


# ================================================================
# ATOMIC CSV SAVE
# ================================================================

def atomic_csv(
    dataframe,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name + ".tmp"
    )


    dataframe.to_csv(
        temporary,
        index=False
    )


    os.replace(
        temporary,
        path
    )


# ================================================================
# ATOMIC TORCH SAVE
# ================================================================

def atomic_torch_save(
    obj,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name + ".tmp"
    )


    torch.save(
        obj,
        temporary
    )


    os.replace(
        temporary,
        path
    )


# ================================================================
# RANDOM SEED
# ================================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if (
        hasattr(
            torch,
            "xpu"
        )
        and
        torch.xpu.is_available()
    ):

        try:

            torch.xpu.manual_seed_all(
                seed
            )

        except Exception:

            pass


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    SEED
)


# ================================================================
# DEVICE DETECTION
#
# Intel XPU
# CUDA
# CPU
# ================================================================

if (
    hasattr(
        torch,
        "xpu"
    )
    and
    torch.xpu.is_available()
):

    DEVICE = torch.device(
        "xpu:0"
    )

    DEVICE_TYPE = "xpu"

    DEVICE_NAME = torch.xpu.get_device_name(
        0
    )


elif torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda:0"
    )

    DEVICE_TYPE = "cuda"

    DEVICE_NAME = torch.cuda.get_device_name(
        0
    )


else:

    DEVICE = torch.device(
        "cpu"
    )

    DEVICE_TYPE = "cpu"

    DEVICE_NAME = "CPU"


# ================================================================
# DEVICE HELPERS
# ================================================================

def synchronize():

    if DEVICE_TYPE == "xpu":

        torch.xpu.synchronize()


    elif DEVICE_TYPE == "cuda":

        torch.cuda.synchronize()


def clear_device():

    gc.collect()


    if DEVICE_TYPE == "xpu":

        try:

            torch.xpu.empty_cache()

        except Exception:

            pass


    elif DEVICE_TYPE == "cuda":

        torch.cuda.empty_cache()


# ================================================================
# DEVICE INFORMATION
# ================================================================

print("=" * 80)

print(
    "DEVICE CONFIGURATION"
)

print("=" * 80)

print(
    "PyTorch version:",
    torch.__version__
)

print(
    "Selected device:",
    DEVICE
)

print(
    "Device type:",
    DEVICE_TYPE
)

print(
    "Device name:",
    DEVICE_NAME
)

print(
    "CPU threads:",
    CPU_THREADS
)

print(
    "Architecture version:",
    ARCHITECTURE_VERSION
)

print("=" * 80)


# ================================================================
# TOTAL TIMER
# ================================================================

TOTAL_START = time.time()


# ================================================================
# EXPERIMENT INFORMATION
# ================================================================

print()

print("=" * 80)

print(
    "MULTIMODAL CNN LSTM"
)

print(
    "MAX DATASET"
)

print(
    "MANUAL XPU TRAINABLE LSTM"
)

print("=" * 80)


print(
    "Input dataset:"
)

print(
    FILE_PATH
)

print()

print(
    "Result folder:"
)

print(
    OUTPUT_DIR
)

print()

print(
    "Optuna trials:",
    N_TRIALS
)

print(
    "GroupKFold folds:",
    GROUP_K_FOLDS
)

print(
    "Final runs:",
    N_RUNS
)

print(
    "Maximum epochs:",
    MAX_EPOCHS
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Early stopping patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "LOGO cap:",
    LOGO_MAX_FOLDS
)

print(
    "Data subsampling: NONE"
)

print("=" * 80)


# ================================================================
# LOAD DATASET
# ================================================================

print()

print("=" * 80)

print(
    "LOADING COMPLETE MAX DATASET"
)

print("=" * 80)


load_start = time.time()


df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)


print(
    "Raw dataset shape:",
    df.shape
)

print(
    "RAM after loading:",
    f"{memory_mb():.2f} MB"
)


# ================================================================
# COLUMN DEFINITIONS
# ================================================================

metadata_cols = [
    "PersonSrial",
    "Window",
    "Window_Start",
    "Window_End",
    "Window_Samples"
]


target_col = "Class"


required_columns = [
    "PersonSrial",
    target_col
]


for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"Required column missing: {column}"
        )


# ================================================================
# FEATURE COLUMNS
# ================================================================

feature_cols = [
    column
    for column in df.columns
    if column not in metadata_cols
    and column != target_col
]


NUMBER_FEATURES = len(
    feature_cols
)


if NUMBER_FEATURES == 0:

    raise ValueError(
        "No feature columns were found."
    )


print(
    "Feature count:",
    NUMBER_FEATURES
)


# ================================================================
# FEATURE MATRIX
# ================================================================

X_all = (
    df[
        feature_cols
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(
        dtype=np.float32
    )
)


# ================================================================
# INVALID FEATURE CHECK
# ================================================================

invalid_count = int(
    (
        ~np.isfinite(
            X_all
        )
    ).sum()
)


print(
    "Invalid feature cells:",
    invalid_count
)


if invalid_count > 0:

    raise ValueError(
        "Dataset contains NaN or infinite feature values."
    )


# ================================================================
# LABEL ENCODING
# ================================================================

label_encoder = LabelEncoder()


y_all = label_encoder.fit_transform(
    df[
        target_col
    ].astype(str)
).astype(
    np.int64
)


class_names = list(
    label_encoder.classes_
)


NUMBER_CLASSES = len(
    class_names
)


# ================================================================
# PARTICIPANTS
# ================================================================

groups_all = (
    df[
        "PersonSrial"
    ]
    .to_numpy()
)


print()

print(
    "Feature matrix:",
    X_all.shape
)

print(
    "Classes:",
    class_names
)

print(
    "Number of classes:",
    NUMBER_CLASSES
)

print(
    "Participants:",
    len(
        np.unique(
            groups_all
        )
    )
)

print(
    "Loading time:",
    f"{time.time() - load_start:.2f}",
    "seconds"
)


# ================================================================
# CLASS DISTRIBUTION
# ================================================================

print()

print(
    "CLASS DISTRIBUTION"
)


class_distribution_records = []


for class_index, class_name in enumerate(
    class_names
):

    count = int(
        np.sum(
            y_all == class_index
        )
    )


    percentage = (
        count
        / len(
            y_all
        )
        * 100.0
    )


    print(
        f"{class_name}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


    class_distribution_records.append(
        {
            "Class":
                class_name,

            "Count":
                count,

            "Percentage":
                percentage
        }
    )


pd.DataFrame(
    class_distribution_records
).to_csv(
    OUTPUT_DIR
    / "class_distribution.csv",
    index=False
)


# ================================================================
# FEATURE MODALITY CONSTRUCTION
#
# These represent EEG feature families or feature views.
#
# They are NOT claimed to be independent physical sensors.
# ================================================================

SPECTRAL_KEYWORDS = [
    "fft",
    "psd",
    "power",
    "bandpower",
    "band_power",
    "spectral",
    "frequency",
    "freq",
    "delta",
    "theta",
    "alpha",
    "beta",
    "gamma"
]


STATISTICAL_KEYWORDS = [
    "mean",
    "median",
    "std",
    "variance",
    "var",
    "rms",
    "skew",
    "kurt",
    "minimum",
    "maximum",
    "min",
    "max",
    "range",
    "iqr",
    "quartile",
    "percentile",
    "peak",
    "zero",
    "zcr",
    "activity"
]


COMPLEXITY_KEYWORDS = [
    "entropy",
    "complexity",
    "fractal",
    "hurst",
    "dfa",
    "lyapunov",
    "sampen",
    "sample_entropy",
    "apen",
    "approximate_entropy",
    "permutation",
    "lzc",
    "lempel",
    "higuchi",
    "petrosian",
    "mobility"
]


def contains_keyword(
    feature_name,
    keywords
):

    feature_name = str(
        feature_name
    ).lower()


    return any(
        keyword in feature_name
        for keyword in keywords
    )


def construct_modalities(
    columns
):

    spectral = []

    statistical = []

    complexity = []

    unassigned = []


    for index, column in enumerate(
        columns
    ):

        if contains_keyword(
            column,
            COMPLEXITY_KEYWORDS
        ):

            complexity.append(
                index
            )


        elif contains_keyword(
            column,
            SPECTRAL_KEYWORDS
        ):

            spectral.append(
                index
            )


        elif contains_keyword(
            column,
            STATISTICAL_KEYWORDS
        ):

            statistical.append(
                index
            )


        else:

            unassigned.append(
                index
            )


    groups = [
        spectral,
        statistical,
        complexity
    ]


    # ============================================================
    # Assign unrecognized features to the smallest group
    # ============================================================

    for index in unassigned:

        smallest_group = int(
            np.argmin(
                [
                    len(group)
                    for group in groups
                ]
            )
        )


        groups[
            smallest_group
        ].append(
            index
        )


    minimum_group_size = min(
        len(group)
        for group in groups
    )


    maximum_group_size = max(
        len(group)
        for group in groups
    )


    semantic_grouping_valid = (
        minimum_group_size >= 16
        and
        maximum_group_size
        <= NUMBER_FEATURES * 0.70
    )


    if semantic_grouping_valid:

        grouping_mode = (
            "Feature name based semantic grouping"
        )


        modality_indices = {
            "spectral_frequency":
                sorted(
                    spectral
                ),

            "statistical_temporal":
                sorted(
                    statistical
                ),

            "nonlinear_complexity":
                sorted(
                    complexity
                )
        }


    else:

        grouping_mode = (
            "Balanced feature view fallback"
        )


        all_indices = np.arange(
            NUMBER_FEATURES
        )


        balanced_groups = np.array_split(
            all_indices,
            3
        )


        modality_indices = {
            "feature_view_1":
                balanced_groups[
                    0
                ].tolist(),

            "feature_view_2":
                balanced_groups[
                    1
                ].tolist(),

            "feature_view_3":
                balanced_groups[
                    2
                ].tolist()
        }


    return (
        modality_indices,
        grouping_mode
    )


(
    MODALITY_INDICES,
    MODALITY_GROUPING_MODE

) = construct_modalities(
    feature_cols
)


MODALITY_NAMES = list(
    MODALITY_INDICES.keys()
)


NUMBER_MODALITIES = len(
    MODALITY_NAMES
)


print()

print("=" * 80)

print(
    "EEG FEATURE MODALITIES"
)

print("=" * 80)


print(
    "Grouping mode:",
    MODALITY_GROUPING_MODE
)


for modality_name in MODALITY_NAMES:

    print(
        modality_name,
        ":",
        len(
            MODALITY_INDICES[
                modality_name
            ]
        ),
        "features"
    )


# ================================================================
# SAVE MODALITY MAPPING
# ================================================================

modality_mapping = {
    "Grouping_Mode":
        MODALITY_GROUPING_MODE,

    "Scientific_Note":
        (
            "Streams represent EEG engineered feature families "
            "or feature views, not independent physical sensors."
        ),

    "Modalities":
        {}
}


for modality_name in MODALITY_NAMES:

    indices = MODALITY_INDICES[
        modality_name
    ]


    modality_mapping[
        "Modalities"
    ][
        modality_name
    ] = {
        "Feature_Count":
            len(
                indices
            ),

        "Features":
            [
                feature_cols[
                    index
                ]
                for index in indices
            ]
    }


with open(
    OUTPUT_DIR
    / "modality_mapping.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        modality_mapping,
        file,
        indent=4
    )


# ================================================================
# RELEASE DATAFRAME
# ================================================================

del df

gc.collect()


# ================================================================
# PERSON WISE 70 / 20 / 10 SPLIT
# ================================================================

print()

print("=" * 80)

print(
    "PERSON WISE 70 / 20 / 10 SPLIT"
)

print("=" * 80)


persons = np.unique(
    groups_all
)


rng = np.random.RandomState(
    SEED
)


rng.shuffle(
    persons
)


number_persons = len(
    persons
)


number_test = max(
    1,
    int(
        round(
            0.20
            * number_persons
        )
    )
)


number_validation = max(
    1,
    int(
        round(
            0.10
            * number_persons
        )
    )
)


test_persons = persons[
    :number_test
]


validation_persons = persons[
    number_test:
    number_test
    + number_validation
]


train_persons = persons[
    number_test
    + number_validation:
]


# ================================================================
# VERIFY NO PARTICIPANT OVERLAP
# ================================================================

assert (
    np.intersect1d(
        train_persons,
        validation_persons
    ).size == 0
)


assert (
    np.intersect1d(
        train_persons,
        test_persons
    ).size == 0
)


assert (
    np.intersect1d(
        validation_persons,
        test_persons
    ).size == 0
)


# ================================================================
# MASKS
# ================================================================

train_mask = np.isin(
    groups_all,
    train_persons
)


validation_mask = np.isin(
    groups_all,
    validation_persons
)


test_mask = np.isin(
    groups_all,
    test_persons
)


# ================================================================
# SPLIT MATRICES
# ================================================================

X_train_raw = np.ascontiguousarray(
    X_all[
        train_mask
    ],
    dtype=np.float32
)


y_train = np.ascontiguousarray(
    y_all[
        train_mask
    ],
    dtype=np.int64
)


g_train = np.ascontiguousarray(
    groups_all[
        train_mask
    ]
)


X_validation_raw = np.ascontiguousarray(
    X_all[
        validation_mask
    ],
    dtype=np.float32
)


y_validation = np.ascontiguousarray(
    y_all[
        validation_mask
    ],
    dtype=np.int64
)


g_validation = np.ascontiguousarray(
    groups_all[
        validation_mask
    ]
)


X_test_raw = np.ascontiguousarray(
    X_all[
        test_mask
    ],
    dtype=np.float32
)


y_test = np.ascontiguousarray(
    y_all[
        test_mask
    ],
    dtype=np.int64
)


g_test = np.ascontiguousarray(
    groups_all[
        test_mask
    ]
)


TRAIN_SAMPLES = len(
    X_train_raw
)


VALIDATION_SAMPLES = len(
    X_validation_raw
)


TEST_SAMPLES = len(
    X_test_raw
)


print(
    "Training participants:",
    len(
        train_persons
    )
)

print(
    "Validation participants:",
    len(
        validation_persons
    )
)

print(
    "Testing participants:",
    len(
        test_persons
    )
)

print()

print(
    "Training:",
    X_train_raw.shape
)

print(
    "Validation:",
    X_validation_raw.shape
)

print(
    "Testing:",
    X_test_raw.shape
)


# ================================================================
# SAVE SPLIT INFORMATION
# ================================================================

pd.DataFrame(
    [
        {
            "Split":
                "Training",

            "Participants":
                len(
                    train_persons
                ),

            "Rows":
                TRAIN_SAMPLES
        },

        {
            "Split":
                "Validation",

            "Participants":
                len(
                    validation_persons
                ),

            "Rows":
                VALIDATION_SAMPLES
        },

        {
            "Split":
                "Testing",

            "Participants":
                len(
                    test_persons
                ),

            "Rows":
                TEST_SAMPLES
        }
    ]
).to_csv(
    OUTPUT_DIR
    / "split_summary.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "participant_split.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        {
            "training":
                [
                    str(x)
                    for x in train_persons
                ],

            "validation":
                [
                    str(x)
                    for x in validation_persons
                ],

            "testing":
                [
                    str(x)
                    for x in test_persons
                ]
        },
        file,
        indent=4
    )


# ================================================================
# RELEASE COMPLETE ARRAYS
# ================================================================

del X_all
del y_all
del groups_all

del train_mask
del validation_mask
del test_mask

gc.collect()


print(
    "RAM after split:",
    f"{memory_mb():.2f} MB"
)


# ================================================================
# GROUPKFOLD
# ================================================================

gkf_splits = list(
    GroupKFold(
        n_splits=
            GROUP_K_FOLDS
    ).split(
        X_train_raw,
        y_train,
        g_train
    )
)


FULL_LOGO_FOLDS = len(
    np.unique(
        g_train
    )
)


print()

print(
    "GroupKFold folds:",
    len(
        gkf_splits
    )
)

print(
    "Full LOGO folds:",
    FULL_LOGO_FOLDS
)


# ================================================================
# FOLD SPECIFIC STANDARDIZATION
# ================================================================

def prepare_fold(
    training_index,
    validation_index
):

    scaler = StandardScaler()


    X_fold_training = scaler.fit_transform(
        X_train_raw[
            training_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    X_fold_validation = scaler.transform(
        X_train_raw[
            validation_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    return (
        np.ascontiguousarray(
            X_fold_training
        ),
        np.ascontiguousarray(
            X_fold_validation
        )
    )


# ================================================================
# CLASS WEIGHTS
# ================================================================

def get_class_weights(
    y
):

    present_classes = np.unique(
        y
    )


    calculated_weights = compute_class_weight(
        class_weight="balanced",
        classes=
            present_classes,
        y=y
    )


    weights = np.ones(
        NUMBER_CLASSES,
        dtype=np.float32
    )


    weights[
        present_classes
    ] = calculated_weights.astype(
        np.float32
    )


    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


# ================================================================
# GAUSSIAN INPUT NOISE
# ================================================================

class GaussianNoise(
    nn.Module
):

    def __init__(
        self,
        standard_deviation
    ):

        super().__init__()


        self.standard_deviation = float(
            standard_deviation
        )


    def forward(
        self,
        x
    ):

        if (
            self.training
            and
            self.standard_deviation > 0
        ):

            return (
                x
                + torch.randn_like(
                    x
                )
                * self.standard_deviation
            )


        return x


# ================================================================
# MANUAL STANDARD LSTM
#
# XPU TRAINABLE
#
# Standard equations:
#
# i_t = sigmoid(...)
# f_t = sigmoid(...)
# g_t = tanh(...)
# o_t = sigmoid(...)
#
# c_t = f_t * c_prev + i_t * g_t
#
# h_t = o_t * tanh(c_t)
#
# Native nn.LSTM is intentionally NOT used.
# ================================================================

class XPUManualLSTM(
    nn.Module
):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers=1,
        dropout=0.0
    ):

        super().__init__()


        self.input_size = int(
            input_size
        )


        self.hidden_size = int(
            hidden_size
        )


        self.num_layers = int(
            num_layers
        )


        self.dropout_probability = float(
            dropout
        )


        self.input_linears = nn.ModuleList()

        self.hidden_linears = nn.ModuleList()


        for layer_index in range(
            self.num_layers
        ):

            current_input_size = (
                self.input_size
                if layer_index == 0
                else self.hidden_size
            )


            self.input_linears.append(
                nn.Linear(
                    current_input_size,
                    4
                    * self.hidden_size,
                    bias=True
                )
            )


            self.hidden_linears.append(
                nn.Linear(
                    self.hidden_size,
                    4
                    * self.hidden_size,
                    bias=False
                )
            )


        self.inter_layer_dropout = nn.Dropout(
            self.dropout_probability
        )


        self.reset_parameters()


    # ============================================================
    # INITIALIZATION
    # ============================================================

    def reset_parameters(
        self
    ):

        for input_linear in self.input_linears:

            nn.init.xavier_uniform_(
                input_linear.weight
            )


            nn.init.zeros_(
                input_linear.bias
            )


            # ====================================================
            # Forget gate bias initialization
            # ====================================================

            with torch.no_grad():

                input_linear.bias[
                    self.hidden_size:
                    2 * self.hidden_size
                ].fill_(
                    1.0
                )


        for hidden_linear in self.hidden_linears:

            nn.init.orthogonal_(
                hidden_linear.weight
            )


    # ============================================================
    # FORWARD
    # ============================================================

    def forward(
        self,
        x
    ):

        # ========================================================
        # x shape:
        #
        # batch x sequence x features
        # ========================================================

        batch_size = x.shape[
            0
        ]


        sequence_length = x.shape[
            1
        ]


        layer_input = x


        final_hidden_states = []

        final_cell_states = []


        # ========================================================
        # STACKED LSTM
        # ========================================================

        for layer_index in range(
            self.num_layers
        ):

            hidden_state = torch.zeros(
                batch_size,
                self.hidden_size,
                dtype=
                    x.dtype,
                device=
                    x.device
            )


            cell_state = torch.zeros(
                batch_size,
                self.hidden_size,
                dtype=
                    x.dtype,
                device=
                    x.device
            )


            # ====================================================
            # Precompute input transformation for all timesteps.
            #
            # This reduces repeated Linear calls and improves
            # performance relative to performing both matrix
            # multiplications inside the Python timestep loop.
            # ====================================================

            input_projection = self.input_linears[
                layer_index
            ](
                layer_input
            )


            timestep_outputs = []


            for timestep in range(
                sequence_length
            ):

                gates = (
                    input_projection[
                        :,
                        timestep,
                        :
                    ]
                    +
                    self.hidden_linears[
                        layer_index
                    ](
                        hidden_state
                    )
                )


                (
                    input_gate,
                    forget_gate,
                    candidate_gate,
                    output_gate

                ) = gates.chunk(
                    4,
                    dim=1
                )


                input_gate = torch.sigmoid(
                    input_gate
                )


                forget_gate = torch.sigmoid(
                    forget_gate
                )


                candidate_gate = torch.tanh(
                    candidate_gate
                )


                output_gate = torch.sigmoid(
                    output_gate
                )


                cell_state = (
                    forget_gate
                    * cell_state
                    +
                    input_gate
                    * candidate_gate
                )


                hidden_state = (
                    output_gate
                    * torch.tanh(
                        cell_state
                    )
                )


                timestep_outputs.append(
                    hidden_state
                )


            layer_output = torch.stack(
                timestep_outputs,
                dim=1
            )


            if (
                layer_index
                < self.num_layers - 1
            ):

                layer_output = self.inter_layer_dropout(
                    layer_output
                )


            layer_input = (
                layer_output
            )


            final_hidden_states.append(
                hidden_state
            )


            final_cell_states.append(
                cell_state
            )


        hidden_stack = torch.stack(
            final_hidden_states,
            dim=0
        )


        cell_stack = torch.stack(
            final_cell_states,
            dim=0
        )


        return (
            layer_input,
            (
                hidden_stack,
                cell_stack
            )
        )


# ================================================================
# MODALITY CNN LSTM ENCODER
# ================================================================

class ModalityCNNLSTMEncoder(
    nn.Module
):

    def __init__(
        self,
        params
    ):

        super().__init__()


        base_channels = int(
            params[
                "base_channels"
            ]
        )


        kernel_size = int(
            params[
                "kernel_size"
            ]
        )


        lstm_hidden = int(
            params[
                "lstm_hidden"
            ]
        )


        lstm_layers = int(
            params[
                "lstm_layers"
            ]
        )


        conv_dropout = float(
            params[
                "conv_dropout"
            ]
        )


        lstm_dropout = float(
            params[
                "lstm_dropout"
            ]
        )


        padding = (
            kernel_size
            // 2
        )


        # ========================================================
        # CNN BLOCKS
        # ========================================================

        self.cnn = nn.Sequential(

            nn.Conv1d(
                1,
                base_channels,
                kernel_size=
                    kernel_size,
                padding=
                    padding,
                bias=False
            ),

            nn.BatchNorm1d(
                base_channels
            ),

            nn.GELU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Dropout1d(
                conv_dropout
            ),

            nn.Conv1d(
                base_channels,
                base_channels * 2,
                kernel_size=
                    kernel_size,
                padding=
                    padding,
                bias=False
            ),

            nn.BatchNorm1d(
                base_channels * 2
            ),

            nn.GELU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Dropout1d(
                conv_dropout
            )
        )


        # ========================================================
        # MANUAL XPU LSTM
        # ========================================================

        self.lstm = XPUManualLSTM(
            input_size=
                base_channels * 2,

            hidden_size=
                lstm_hidden,

            num_layers=
                lstm_layers,

            dropout=
                lstm_dropout
        )


        # ========================================================
        # Last state plus average state
        # ========================================================

        self.output_dimension = (
            lstm_hidden
            * 2
        )


        self.output_norm = nn.LayerNorm(
            self.output_dimension
        )


        self.output_dropout = nn.Dropout(
            lstm_dropout
        )


    def forward(
        self,
        x
    ):

        # ========================================================
        # batch x modality features
        # ========================================================

        x = x.unsqueeze(
            1
        )


        # ========================================================
        # CNN
        #
        # batch x channels x reduced sequence
        # ========================================================

        x = self.cnn(
            x
        )


        # ========================================================
        # LSTM format
        #
        # batch x reduced sequence x channels
        # ========================================================

        x = x.transpose(
            1,
            2
        ).contiguous()


        sequence_output, _ = self.lstm(
            x
        )


        last_representation = sequence_output[
            :,
            -1,
            :
        ]


        mean_representation = torch.mean(
            sequence_output,
            dim=1
        )


        representation = torch.cat(
            [
                last_representation,
                mean_representation
            ],
            dim=1
        )


        representation = self.output_norm(
            representation
        )


        representation = self.output_dropout(
            representation
        )


        return representation


# ================================================================
# MULTIMODAL CNN LSTM
# ================================================================

class MultimodalCNNLSTM(
    nn.Module
):

    def __init__(
        self,
        params,
        modality_indices
    ):

        super().__init__()


        self.modality_names = list(
            modality_indices.keys()
        )


        self.number_modalities = len(
            self.modality_names
        )


        self.modality_buffer_names = []


        # ========================================================
        # STORE FEATURE INDICES AS MODEL BUFFERS
        # ========================================================

        for modality_number, modality_name in enumerate(
            self.modality_names
        ):

            buffer_name = (
                f"modality_indices_{modality_number}"
            )


            self.register_buffer(
                buffer_name,
                torch.tensor(
                    modality_indices[
                        modality_name
                    ],
                    dtype=torch.long
                ),
                persistent=True
            )


            self.modality_buffer_names.append(
                buffer_name
            )


        # ========================================================
        # INPUT REGULARIZATION
        # ========================================================

        self.input_noise = GaussianNoise(
            params[
                "input_noise"
            ]
        )


        self.input_dropout = nn.Dropout(
            float(
                params[
                    "input_dropout"
                ]
            )
        )


        # ========================================================
        # THREE INDEPENDENT CNN LSTM ENCODERS
        # ========================================================

        self.encoders = nn.ModuleList(
            [
                ModalityCNNLSTMEncoder(
                    params
                )
                for _
                in range(
                    self.number_modalities
                )
            ]
        )


        stream_dimension = (
            self.encoders[
                0
            ].output_dimension
        )


        # ========================================================
        # MODALITY ATTENTION GATE
        # ========================================================

        gate_hidden = max(
            16,
            stream_dimension
            // 2
        )


        self.modality_gate = nn.Sequential(

            nn.Linear(
                stream_dimension,
                gate_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                float(
                    params[
                        "dense_dropout"
                    ]
                )
            ),

            nn.Linear(
                gate_hidden,
                1
            )
        )


        # ========================================================
        # FUSION
        #
        # modality 1
        # modality 2
        # modality 3
        # weighted fusion
        # modality standard deviation
        # ========================================================

        fusion_dimension = (
            stream_dimension
            * (
                self.number_modalities
                + 2
            )
        )


        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )


        dense_dropout = float(
            params[
                "dense_dropout"
            ]
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        # ========================================================
        # TRAINING ONLY REGULARIZATION
        # ========================================================

        x = self.input_noise(
            x
        )


        x = self.input_dropout(
            x
        )


        modality_vectors = []


        # ========================================================
        # PROCESS EACH MODALITY
        # ========================================================

        for modality_index in range(
            self.number_modalities
        ):

            column_indices = getattr(
                self,
                self.modality_buffer_names[
                    modality_index
                ]
            )


            modality_input = torch.index_select(
                x,
                dim=1,
                index=
                    column_indices
            )


            modality_vector = self.encoders[
                modality_index
            ](
                modality_input
            )


            modality_vectors.append(
                modality_vector
            )


        # ========================================================
        # batch x modalities x representation dimension
        # ========================================================

        stacked_modalities = torch.stack(
            modality_vectors,
            dim=1
        )


        # ========================================================
        # LEARNED MODALITY WEIGHTS
        # ========================================================

        gate_logits = torch.cat(
            [
                self.modality_gate(
                    vector
                )
                for vector in modality_vectors
            ],
            dim=1
        )


        modality_weights = torch.softmax(
            gate_logits,
            dim=1
        )


        weighted_representation = torch.sum(
            stacked_modalities
            * modality_weights.unsqueeze(
                -1
            ),
            dim=1
        )


        # ========================================================
        # CROSS MODAL VARIATION
        # ========================================================

        modality_std = torch.std(
            stacked_modalities,
            dim=1,
            unbiased=False
        )


        # ========================================================
        # FINAL FUSION
        # ========================================================

        fused = torch.cat(
            modality_vectors
            + [
                weighted_representation,
                modality_std
            ],
            dim=1
        )


        return self.classifier(
            fused
        )


# ================================================================
# BUILD MODEL
# ================================================================

def build_model(
    params,
    seed=SEED
):

    set_seed(
        seed
    )


    model = MultimodalCNNLSTM(
        params=
            params,
        modality_indices=
            MODALITY_INDICES
    )


    return model.to(
        DEVICE
    )


# ================================================================
# TRAINABLE PARAMETERS
# ================================================================

def count_trainable_parameters(
    model
):

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


# ================================================================
# BATCH GENERATOR
#
# Dataset remains in system RAM.
#
# Only one minibatch is moved to XPU at a time.
# ================================================================

def batch_generator(
    X,
    y,
    shuffle=True
):

    number_samples = len(
        y
    )


    indices = np.arange(
        number_samples
    )


    if shuffle:

        np.random.shuffle(
            indices
        )


    for start in range(
        0,
        number_samples,
        BATCH_SIZE
    ):

        batch_indices = indices[
            start:
            start
            + BATCH_SIZE
        ]


        X_batch_numpy = np.ascontiguousarray(
            X[
                batch_indices
            ],
            dtype=np.float32
        )


        y_batch_numpy = np.ascontiguousarray(
            y[
                batch_indices
            ],
            dtype=np.int64
        )


        X_batch = torch.from_numpy(
            X_batch_numpy
        ).to(
            DEVICE
        )


        y_batch = torch.from_numpy(
            y_batch_numpy
        ).to(
            DEVICE
        )


        yield (
            X_batch,
            y_batch
        )


# ================================================================
# TRAIN ONE EPOCH
# ================================================================

def train_one_epoch(
    model,
    X,
    y,
    optimizer,
    criterion
):

    model.train()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    for X_batch, y_batch in batch_generator(
        X,
        y,
        shuffle=True
    ):

        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            X_batch
        )


        loss = criterion(
            logits,
            y_batch
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )


        optimizer.step()


        batch_size = int(
            y_batch.shape[
                0
            ]
        )


        total_loss += (
            float(
                loss.detach().item()
            )
            * batch_size
        )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


        del X_batch
        del y_batch
        del logits
        del loss


    synchronize()


    return (
        total_loss
        / total_samples,

        total_correct
        / total_samples
    )


# ================================================================
# EVALUATION
# ================================================================

@torch.inference_mode()
def evaluate_model(
    model,
    X,
    y,
    criterion=None,
    return_probabilities=False
):

    model.eval()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    probability_parts = []


    for X_batch, y_batch in batch_generator(
        X,
        y,
        shuffle=False
    ):

        logits = model(
            X_batch
        )


        batch_size = int(
            y_batch.shape[
                0
            ]
        )


        if criterion is not None:

            loss = criterion(
                logits,
                y_batch
            )


            total_loss += (
                float(
                    loss.item()
                )
                * batch_size
            )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


        if return_probabilities:

            probability_parts.append(
                torch.softmax(
                    logits,
                    dim=1
                )
                .cpu()
            )


        del X_batch
        del y_batch
        del logits


    synchronize()


    accuracy = (
        total_correct
        / total_samples
    )


    if criterion is None:

        mean_loss = np.nan


    else:

        mean_loss = (
            total_loss
            / total_samples
        )


    if return_probabilities:

        probabilities = (
            torch.cat(
                probability_parts,
                dim=0
            )
            .numpy()
        )


        return (
            mean_loss,
            accuracy,
            probabilities
        )


    return (
        mean_loss,
        accuracy
    )


# ================================================================
# FIT MODEL
# ================================================================

def fit_model(
    params,
    X_training,
    y_training,
    X_validation=None,
    y_validation=None,
    seed=SEED,
    epochs=MAX_EPOCHS,
    use_early_stopping=True,
    verbose=False
):

    clear_device()

    set_seed(
        seed
    )


    model = build_model(
        params,
        seed
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),

        lr=float(
            params[
                "learning_rate"
            ]
        ),

        weight_decay=float(
            params[
                "weight_decay"
            ]
        )
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=
            LR_SCHEDULER_PATIENCE,
        min_lr=1e-6
    )


    class_weights = get_class_weights(
        y_training
    )


    criterion = nn.CrossEntropyLoss(
        weight=
            class_weights,

        label_smoothing=
            float(
                params[
                    "label_smoothing"
                ]
            )
    )


    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": []
    }


    best_loss = float(
        "inf"
    )


    best_state = None

    best_epoch = 1

    patience_counter = 0


    for epoch in range(
        1,
        epochs + 1
    ):

        training_loss, training_accuracy = train_one_epoch(
            model,
            X_training,
            y_training,
            optimizer,
            criterion
        )


        if X_validation is not None:

            (
                validation_loss,
                validation_accuracy

            ) = evaluate_model(
                model,
                X_validation,
                y_validation,
                criterion=
                    criterion,
                return_probabilities=
                    False
            )


            monitored_loss = (
                validation_loss
            )


        else:

            validation_loss = np.nan

            validation_accuracy = np.nan

            monitored_loss = (
                training_loss
            )


        scheduler.step(
            monitored_loss
        )


        learning_rate = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )


        history[
            "loss"
        ].append(
            training_loss
        )


        history[
            "accuracy"
        ].append(
            training_accuracy
        )


        history[
            "val_loss"
        ].append(
            validation_loss
        )


        history[
            "val_accuracy"
        ].append(
            validation_accuracy
        )


        history[
            "learning_rate"
        ].append(
            learning_rate
        )


        if verbose:

            print(
                f"Epoch "
                f"{epoch:02d}/"
                f"{epochs} | "
                f"Train Loss "
                f"{training_loss:.4f} | "
                f"Train Acc "
                f"{training_accuracy:.4f} | "
                f"Val Loss "
                f"{validation_loss:.4f} | "
                f"Val Acc "
                f"{validation_accuracy:.4f} | "
                f"LR "
                f"{learning_rate:.2e}"
            )


        # ========================================================
        # EARLY STOPPING
        # ========================================================

        if use_early_stopping:

            if (
                monitored_loss
                < best_loss
                - EARLY_STOPPING_MIN_DELTA
            ):

                best_loss = (
                    monitored_loss
                )


                best_epoch = (
                    epoch
                )


                best_state = {
                    key:
                        value.detach()
                        .cpu()
                        .clone()

                    for key, value
                    in model.state_dict().items()
                }


                patience_counter = 0


            else:

                patience_counter += 1


                if (
                    patience_counter
                    >= EARLY_STOPPING_PATIENCE
                ):

                    if verbose:

                        print(
                            "Early stopping."
                        )


                    break


    # ============================================================
    # RESTORE BEST VALIDATION LOSS STATE
    # ============================================================

    if (
        use_early_stopping
        and
        best_state is not None
    ):

        model.load_state_dict(
            best_state
        )


    del optimizer
    del scheduler
    del criterion
    del class_weights


    if best_state is not None:

        del best_state


    clear_device()


    return (
        model,
        history,
        best_epoch
    )


# ================================================================
# PREDICT PROBABILITIES
# ================================================================

def predict_probabilities(
    model,
    X,
    y
):

    (
        _,
        _,
        probabilities

    ) = evaluate_model(
        model,
        X,
        y,
        criterion=None,
        return_probabilities=True
    )


    clear_device()


    return probabilities


# ================================================================
# REAL XPU TRAINING SMOKE TEST
#
# Unlike the previous smoke test, this tests:
#
# forward
# loss
# backward
# gradients
# optimizer step
#
# Therefore actual training support is verified.
# ================================================================

print()

print("=" * 80)

print(
    "REAL MODEL TRAINING SMOKE TEST"
)

print("=" * 80)


smoke_params = {
    "base_channels":
        32,

    "kernel_size":
        5,

    "lstm_hidden":
        64,

    "lstm_layers":
        1,

    "conv_dropout":
        0.25,

    "lstm_dropout":
        0.25,

    "input_dropout":
        0.10,

    "input_noise":
        0.02,

    "classifier_hidden":
        128,

    "dense_dropout":
        0.35,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-4,

    "label_smoothing":
        0.05
}


smoke_model = build_model(
    smoke_params,
    SEED
)


smoke_model.train()


smoke_optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)


smoke_input = torch.randn(
    16,
    NUMBER_FEATURES,
    dtype=torch.float32,
    device=DEVICE
)


smoke_target = torch.randint(
    low=0,
    high=NUMBER_CLASSES,
    size=(
        16,
    ),
    dtype=torch.long,
    device=DEVICE
)


smoke_optimizer.zero_grad(
    set_to_none=True
)


smoke_output = smoke_model(
    smoke_input
)


smoke_loss = nn.CrossEntropyLoss()(
    smoke_output,
    smoke_target
)


smoke_loss.backward()


gradient_norm = torch.nn.utils.clip_grad_norm_(
    smoke_model.parameters(),
    GRADIENT_CLIP
)


smoke_optimizer.step()


synchronize()


print(
    "Input shape:",
    smoke_input.shape
)

print(
    "Output shape:",
    smoke_output.shape
)

print(
    "Output device:",
    smoke_output.device
)

print(
    "Loss:",
    float(
        smoke_loss.detach().item()
    )
)

print(
    "Gradient norm:",
    float(
        gradient_norm.detach().cpu().item()
        if torch.is_tensor(
            gradient_norm
        )
        else gradient_norm
    )
)

print(
    "Backward propagation: SUCCESS"
)

print(
    "Optimizer step: SUCCESS"
)

print(
    "Training smoke test: PASSED"
)


del smoke_model
del smoke_optimizer
del smoke_input
del smoke_target
del smoke_output
del smoke_loss
del gradient_norm

clear_device()


# ================================================================
# OPTUNA OBJECTIVE
#
# Controlled search space to limit unnecessary model capacity.
# ================================================================

def objective(
    trial
):

    params = {
        "base_channels":

            trial.suggest_categorical(
                "base_channels",
                [
                    32,
                    48,
                    64
                ]
            ),

        "kernel_size":

            trial.suggest_categorical(
                "kernel_size",
                [
                    3,
                    5,
                    7
                ]
            ),

        "lstm_hidden":

            trial.suggest_categorical(
                "lstm_hidden",
                [
                    32,
                    64,
                    96
                ]
            ),

        "lstm_layers":

            trial.suggest_int(
                "lstm_layers",
                1,
                2
            ),

        "conv_dropout":

            trial.suggest_float(
                "conv_dropout",
                0.15,
                0.40
            ),

        "lstm_dropout":

            trial.suggest_float(
                "lstm_dropout",
                0.15,
                0.40
            ),

        "input_dropout":

            trial.suggest_float(
                "input_dropout",
                0.05,
                0.20
            ),

        "input_noise":

            trial.suggest_float(
                "input_noise",
                0.0,
                0.05
            ),

        "classifier_hidden":

            trial.suggest_categorical(
                "classifier_hidden",
                [
                    64,
                    128
                ]
            ),

        "dense_dropout":

            trial.suggest_float(
                "dense_dropout",
                0.25,
                0.50
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                2e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.10
            )
    }


    print()

    print(
        f"OPTUNA TRIAL "
        f"{trial.number + 1}"
    )


    print(
        params
    )


    fold_scores = []


    for fold_number, (
        training_index,
        validation_index

    ) in enumerate(
        gkf_splits,
        start=1
    ):

        fold_start = time.time()


        # ========================================================
        # FOLD SPECIFIC SCALING
        # ========================================================

        (
            X_fold_training,
            X_fold_validation

        ) = prepare_fold(
            training_index,
            validation_index
        )


        # ========================================================
        # TRAIN
        # ========================================================

        (
            model,
            history,
            best_epoch

        ) = fit_model(
            params,
            X_fold_training,
            y_train[
                training_index
            ],
            X_fold_validation,
            y_train[
                validation_index
            ],
            seed=
                SEED
                + fold_number,
            epochs=
                MAX_EPOCHS,
            use_early_stopping=
                True,
            verbose=
                False
        )


        probabilities = predict_probabilities(
            model,
            X_fold_validation,
            y_train[
                validation_index
            ]
        )


        predictions = np.argmax(
            probabilities,
            axis=1
        )


        fold_accuracy = accuracy_score(
            y_train[
                validation_index
            ],
            predictions
        )


        fold_scores.append(
            fold_accuracy
        )


        print(
            f"Trial "
            f"{trial.number + 1}, "
            f"Fold "
            f"{fold_number}/"
            f"{GROUP_K_FOLDS}, "
            f"Accuracy = "
            f"{fold_accuracy:.4f}, "
            f"Best Epoch = "
            f"{best_epoch}, "
            f"Time = "
            f"{(time.time() - fold_start) / 60:.2f} min, "
            f"RAM = "
            f"{memory_mb():.2f} MB"
        )


        del model
        del history
        del probabilities
        del predictions
        del X_fold_training
        del X_fold_validation


        clear_device()


    mean_accuracy = float(
        np.mean(
            fold_scores
        )
    )


    std_accuracy = float(
        np.std(
            fold_scores
        )
    )


    print(
        f"Trial "
        f"{trial.number + 1} "
        f"Mean Accuracy = "
        f"{mean_accuracy:.4f} "
        f"plus or minus "
        f"{std_accuracy:.4f}"
    )


    return mean_accuracy


# ================================================================
# OPTUNA
# ================================================================

print()

print("=" * 80)

print(
    "OPTUNA HYPERPARAMETER OPTIMIZATION"
)

print("=" * 80)


optuna_start = time.time()


study = optuna.create_study(
    study_name=
        OPTUNA_STUDY_NAME,

    storage=
        OPTUNA_STORAGE,

    direction=
        "maximize",

    sampler=
        optuna.samplers.TPESampler(
            seed=SEED
        ),

    load_if_exists=
        True
)


completed_trials = sum(
    1
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
)


remaining_trials = max(
    0,
    N_TRIALS
    - completed_trials
)


print(
    "Previously completed trials:",
    completed_trials
)

print(
    "Remaining trials:",
    remaining_trials
)


if remaining_trials > 0:

    study.optimize(
        objective,
        n_trials=
            remaining_trials,
        show_progress_bar=
            False,
        gc_after_trial=
            True
    )


complete_trials = [
    trial
    for trial in study.trials
    if trial.state
    == optuna.trial.TrialState.COMPLETE
]


if len(
    complete_trials
) == 0:

    raise RuntimeError(
        "No Optuna trial completed successfully."
    )


optuna_time = (
    time.time()
    - optuna_start
)


best_params = dict(
    study.best_params
)


print()

print("=" * 80)

print(
    "OPTUNA COMPLETED"
)

print("=" * 80)


print(
    "Best GroupKFold Accuracy:",
    f"{study.best_value:.4f}"
)

print(
    "Best Parameters:"
)

print(
    best_params
)

print(
    "Optuna Time:",
    f"{optuna_time / 60:.2f}",
    "minutes"
)


study.trials_dataframe().to_csv(
    OUTPUT_DIR
    / "optuna_trials.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "best_parameters.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        best_params,
        file,
        indent=4
    )


# ================================================================
# CONFIGURATION ID
# ================================================================

configuration_text = json.dumps(
    {
        "Architecture_Version":
            ARCHITECTURE_VERSION,

        "Best_Params":
            best_params,

        "Features":
            NUMBER_FEATURES,

        "Classes":
            class_names,

        "Modalities":
            MODALITY_INDICES,

        "Grouping_Mode":
            MODALITY_GROUPING_MODE,

        "Seed":
            SEED
    },
    sort_keys=True
)


CONFIG_ID = hashlib.sha256(
    configuration_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


print(
    "Configuration ID:",
    CONFIG_ID
)


# ================================================================
# MODEL ARCHITECTURE
# ================================================================

temporary_model = build_model(
    best_params,
    SEED
)


TRAINABLE_PARAMETERS = count_trainable_parameters(
    temporary_model
)


print(
    "Trainable parameters:",
    f"{TRAINABLE_PARAMETERS:,}"
)


with open(
    OUTPUT_DIR
    / "model_architecture.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        str(
            temporary_model
        )
    )


del temporary_model

clear_device()


# ================================================================
# FINAL STANDARDIZATION
#
# Delayed until AFTER Optuna to reduce memory during CV.
#
# Scaler is fitted only on training participants.
# ================================================================

print()

print("=" * 80)

print(
    "FINAL TRAINING SCALER"
)

print("=" * 80)


final_scaler = StandardScaler()


X_train = final_scaler.fit_transform(
    X_train_raw
).astype(
    np.float32,
    copy=False
)


X_validation = final_scaler.transform(
    X_validation_raw
).astype(
    np.float32,
    copy=False
)


X_test = final_scaler.transform(
    X_test_raw
).astype(
    np.float32,
    copy=False
)


X_train = np.ascontiguousarray(
    X_train
)


X_validation = np.ascontiguousarray(
    X_validation
)


X_test = np.ascontiguousarray(
    X_test
)


print(
    "Training:",
    X_train.shape
)

print(
    "Validation:",
    X_validation.shape
)

print(
    "Testing:",
    X_test.shape
)

print(
    "RAM:",
    f"{memory_mb():.2f} MB"
)


# ================================================================
# SAVE PREPROCESSING
# ================================================================

joblib.dump(
    final_scaler,
    OUTPUT_DIR
    / "standard_scaler.joblib"
)


joblib.dump(
    label_encoder,
    OUTPUT_DIR
    / "label_encoder.joblib"
)


with open(
    OUTPUT_DIR
    / "feature_columns.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        feature_cols,
        file,
        indent=4
    )


# ================================================================
# FIVE FINAL RUNS
# ================================================================

print()

print("=" * 80)

print(
    "FIVE FINAL STOCHASTIC RUNS"
)

print("=" * 80)


run_records = []

completed_runs = set()


# ================================================================
# RESUME PREVIOUS RUNS
# ================================================================

if FIVE_RUNS_PATH.exists():

    previous_runs = pd.read_csv(
        FIVE_RUNS_PATH
    )


    if (
        "Config_ID"
        in previous_runs.columns
    ):

        matching_runs = previous_runs[
            previous_runs[
                "Config_ID"
            ].astype(str)
            == CONFIG_ID
        ]


        for _, row in matching_runs.iterrows():

            run_number = int(
                row[
                    "Run"
                ]
            )


            checkpoint_path = (
                CHECKPOINT_DIR
                / f"manual_lstm_run_{run_number:02d}.pt"
            )


            if checkpoint_path.exists():

                run_records.append(
                    row.to_dict()
                )


                completed_runs.add(
                    run_number
                )


print(
    "Previously completed final runs:",
    len(
        completed_runs
    )
)


five_run_start = time.time()


for run_number in range(
    1,
    N_RUNS + 1
):

    if run_number in completed_runs:

        print(
            f"Run "
            f"{run_number}/"
            f"{N_RUNS} "
            f"already completed."
        )

        continue


    print()

    print(
        f"RUN "
        f"{run_number}/"
        f"{N_RUNS}"
    )


    run_start = time.time()


    run_seed = (
        SEED
        + run_number
        - 1
    )


    (
        model,
        history,
        best_epoch

    ) = fit_model(
        best_params,
        X_train,
        y_train,
        X_validation,
        y_validation,
        seed=
            run_seed,
        epochs=
            MAX_EPOCHS,
        use_early_stopping=
            True,
        verbose=
            True
    )


    # ============================================================
    # TRAINING PERFORMANCE
    # ============================================================

    training_probabilities = predict_probabilities(
        model,
        X_train,
        y_train
    )


    training_predictions = np.argmax(
        training_probabilities,
        axis=1
    )


    training_accuracy = accuracy_score(
        y_train,
        training_predictions
    )


    # ============================================================
    # VALIDATION PERFORMANCE
    # ============================================================

    validation_probabilities = predict_probabilities(
        model,
        X_validation,
        y_validation
    )


    validation_predictions = np.argmax(
        validation_probabilities,
        axis=1
    )


    validation_accuracy = accuracy_score(
        y_validation,
        validation_predictions
    )


    generalization_gap = (
        training_accuracy
        - validation_accuracy
    )


    absolute_generalization_gap = abs(
        generalization_gap
    )


    run_time = (
        time.time()
        - run_start
    )


    record = {
        "Config_ID":
            CONFIG_ID,

        "Architecture_Version":
            ARCHITECTURE_VERSION,

        "Run":
            run_number,

        "Seed":
            run_seed,

        "Training_Accuracy":
            float(
                training_accuracy
            ),

        "Validation_Accuracy":
            float(
                validation_accuracy
            ),

        "Generalization_Gap":
            float(
                generalization_gap
            ),

        "Absolute_Generalization_Gap":
            float(
                absolute_generalization_gap
            ),

        "Epochs_Run":
            int(
                len(
                    history[
                        "loss"
                    ]
                )
            ),

        "Best_Epoch":
            int(
                best_epoch
            ),

        "Execution_Time_Seconds":
            float(
                run_time
            )
    }


    # ============================================================
    # MODEL STATE TO CPU
    # ============================================================

    model_state = {
        key:
            value.detach()
            .cpu()
            .clone()

        for key, value
        in model.state_dict().items()
    }


    checkpoint_path = (
        CHECKPOINT_DIR
        / f"manual_lstm_run_{run_number:02d}.pt"
    )


    atomic_torch_save(
        {
            "Config_ID":
                CONFIG_ID,

            "Architecture_Version":
                ARCHITECTURE_VERSION,

            "record":
                record,

            "best_params":
                best_params,

            "model_state_dict":
                model_state,

            "history":
                history
        },
        checkpoint_path
    )


    run_records.append(
        record
    )


    run_table_temp = pd.DataFrame(
        run_records
    )


    run_table_temp = (
        run_table_temp
        .sort_values(
            "Run"
        )
        .drop_duplicates(
            subset=[
                "Run"
            ],
            keep="last"
        )
    )


    atomic_csv(
        run_table_temp,
        FIVE_RUNS_PATH
    )


    print()

    print(
        f"[RUN {run_number}] "
        f"Train = "
        f"{training_accuracy:.4f}, "
        f"Validation = "
        f"{validation_accuracy:.4f}, "
        f"Gap = "
        f"{generalization_gap:.4f}, "
        f"Best Epoch = "
        f"{best_epoch}, "
        f"Time = "
        f"{run_time / 60:.2f} min"
    )


    del model
    del history
    del model_state
    del training_probabilities
    del training_predictions
    del validation_probabilities
    del validation_predictions


    clear_device()


five_run_time = (
    time.time()
    - five_run_start
)


# ================================================================
# LOAD COMPLETE FIVE RUN TABLE
# ================================================================

run_table = pd.read_csv(
    FIVE_RUNS_PATH
)


run_table = run_table[
    run_table[
        "Config_ID"
    ].astype(str)
    == CONFIG_ID
].copy()


run_table = (
    run_table
    .sort_values(
        "Run"
    )
    .drop_duplicates(
        subset=[
            "Run"
        ],
        keep="last"
    )
)


if len(
    run_table
) != N_RUNS:

    raise RuntimeError(
        "Five final runs are not complete."
    )


print()

print(
    run_table[
        [
            "Run",
            "Seed",
            "Training_Accuracy",
            "Validation_Accuracy",
            "Generalization_Gap",
            "Absolute_Generalization_Gap",
            "Best_Epoch",
            "Execution_Time_Seconds"
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ================================================================
# SELECT BEST RUN
#
# Primary:
# highest validation accuracy
#
# Tie breaker:
# lowest absolute generalization gap
# ================================================================

selected_row = (
    run_table
    .sort_values(
        by=[
            "Validation_Accuracy",
            "Absolute_Generalization_Gap",
            "Run"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
    .iloc[
        0
    ]
)


best_run_number = int(
    selected_row[
        "Run"
    ]
)


best_checkpoint_path = (
    CHECKPOINT_DIR
    / f"manual_lstm_run_{best_run_number:02d}.pt"
)


best_bundle = torch.load(
    best_checkpoint_path,
    map_location="cpu",
    weights_only=False
)


best_record = best_bundle[
    "record"
]


best_history = best_bundle[
    "history"
]


print()

print(
    "Selected Run:",
    best_record[
        "Run"
    ]
)

print(
    "Selected Seed:",
    best_record[
        "Seed"
    ]
)

print(
    "Selected Validation Accuracy:",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Selected Generalization Gap:",
    f"{best_record['Generalization_Gap']:.4f}"
)

print(
    "Selected Best Epoch:",
    best_record[
        "Best_Epoch"
    ]
)


# ================================================================
# SAVE SELECTED MODEL PACKAGE
# ================================================================

atomic_torch_save(
    {
        "Algorithm":
            "Multimodal CNN LSTM",

        "Architecture_Version":
            ARCHITECTURE_VERSION,

        "Recurrent_Implementation":
            "Manual standard LSTM for XPU training",

        "Dataset":
            "Max Dataset",

        "Config_ID":
            CONFIG_ID,

        "Device_Type":
            DEVICE_TYPE,

        "Device_Name":
            DEVICE_NAME,

        "best_params":
            best_params,

        "selected_run":
            int(
                best_record[
                    "Run"
                ]
            ),

        "selected_seed":
            int(
                best_record[
                    "Seed"
                ]
            ),

        "best_epoch":
            int(
                best_record[
                    "Best_Epoch"
                ]
            ),

        "model_state_dict":
            best_bundle[
                "model_state_dict"
            ],

        "feature_columns":
            feature_cols,

        "modality_indices":
            MODALITY_INDICES,

        "modality_names":
            MODALITY_NAMES,

        "modality_grouping_mode":
            MODALITY_GROUPING_MODE,

        "class_names":
            class_names,

        "feature_count":
            NUMBER_FEATURES
    },
    BEST_MODEL_PATH
)


# ================================================================
# TRAINING HISTORY
# ================================================================

history_table = pd.DataFrame(
    {
        "Epoch":
            np.arange(
                1,
                len(
                    best_history[
                        "loss"
                    ]
                )
                + 1
            ),

        "Training_Loss":
            best_history[
                "loss"
            ],

        "Training_Accuracy":
            best_history[
                "accuracy"
            ],

        "Validation_Loss":
            best_history[
                "val_loss"
            ],

        "Validation_Accuracy":
            best_history[
                "val_accuracy"
            ],

        "Learning_Rate":
            best_history[
                "learning_rate"
            ]
    }
)


history_table[
    "Generalization_Gap"
] = (
    history_table[
        "Training_Accuracy"
    ]
    -
    history_table[
        "Validation_Accuracy"
    ]
)


history_table.to_csv(
    OUTPUT_DIR
    / "training_history.csv",
    index=False
)


# ================================================================
# LOSS CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Loss"
    ],
    label=
        "Training Loss"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Loss"
    ],
    label=
        "Validation Loss"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Cross Entropy Loss"
)

ax.set_title(
    "Multimodal CNN LSTM Loss Curve"
)

ax.legend()

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# ACCURACY CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Training_Accuracy"
    ],
    label=
        "Training Accuracy"
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Validation_Accuracy"
    ],
    label=
        "Validation Accuracy"
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Accuracy"
)

ax.set_title(
    "Multimodal CNN LSTM Accuracy Curve"
)

ax.legend()

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# GENERALIZATION GAP CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Generalization_Gap"
    ]
)


ax.axhline(
    0.0,
    linewidth=1
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Training Accuracy minus Validation Accuracy"
)

ax.set_title(
    "Generalization Gap"
)

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "generalization_gap_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# LEARNING RATE CURVE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.plot(
    history_table[
        "Epoch"
    ],
    history_table[
        "Learning_Rate"
    ]
)


ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Learning Rate"
)

ax.set_title(
    "Learning Rate Schedule"
)

ax.grid(
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "learning_rate_curve.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# FIVE RUN STATISTICS
# ================================================================

run_train_mean = float(
    run_table[
        "Training_Accuracy"
    ].mean()
)


run_train_std = float(
    run_table[
        "Training_Accuracy"
    ].std(
        ddof=0
    )
)


run_validation_mean = float(
    run_table[
        "Validation_Accuracy"
    ].mean()
)


run_validation_std = float(
    run_table[
        "Validation_Accuracy"
    ].std(
        ddof=0
    )
)


run_gap_mean = float(
    run_table[
        "Generalization_Gap"
    ].mean()
)


run_gap_std = float(
    run_table[
        "Generalization_Gap"
    ].std(
        ddof=0
    )
)


# ================================================================
# RELEASE FINAL TRAIN AND VALIDATION SCALED MATRICES
#
# Raw training matrix remains for LOGO.
#
# Scaled test matrix remains for final test evaluation.
# ================================================================

del X_train
del X_validation

del X_validation_raw
del X_test_raw

gc.collect()

clear_device()


# ================================================================
# FULL LEAVE ONE GROUP OUT
# ================================================================

print()

print("=" * 80)

print(
    "FULL LEAVE ONE GROUP OUT"
)

print("=" * 80)


FIXED_LOGO_EPOCHS = max(
    1,
    int(
        best_record[
            "Best_Epoch"
        ]
    )
)


if LOGO_MAX_FOLDS is None:

    LOGO_FOLDS_TO_USE = (
        FULL_LOGO_FOLDS
    )


else:

    LOGO_FOLDS_TO_USE = min(
        int(
            LOGO_MAX_FOLDS
        ),
        FULL_LOGO_FOLDS
    )


print(
    "Available LOGO folds:",
    FULL_LOGO_FOLDS
)

print(
    "LOGO folds requested:",
    LOGO_FOLDS_TO_USE
)

print(
    "Fixed LOGO epochs:",
    FIXED_LOGO_EPOCHS
)


# ================================================================
# LOGO CONFIGURATION ID
# ================================================================

logo_configuration_text = json.dumps(
    {
        "Architecture_Version":
            ARCHITECTURE_VERSION,

        "Config_ID":
            CONFIG_ID,

        "Fixed_Epochs":
            FIXED_LOGO_EPOCHS,

        "Selected_Seed":
            int(
                best_record[
                    "Seed"
                ]
            )
    },
    sort_keys=True
)


LOGO_CONFIG_ID = hashlib.sha256(
    logo_configuration_text.encode(
        "utf8"
    )
).hexdigest()[
    :16
]


# ================================================================
# RESUME LOGO
# ================================================================

logo_records = []

completed_logo_persons = set()


if LOGO_RESULT_PATH.exists():

    previous_logo = pd.read_csv(
        LOGO_RESULT_PATH
    )


    if (
        "Config_ID"
        in previous_logo.columns
    ):

        matching_logo = previous_logo[
            previous_logo[
                "Config_ID"
            ].astype(str)
            == LOGO_CONFIG_ID
        ].copy()


        if len(
            matching_logo
        ) > 0:

            logo_records = matching_logo.to_dict(
                orient="records"
            )


            completed_logo_persons = set(
                matching_logo[
                    "Held_Out_Person"
                ].astype(str)
            )


print(
    "Previously completed LOGO folds:",
    len(
        completed_logo_persons
    )
)


# ================================================================
# RUN LOGO
# ================================================================

logo_start = time.time()


logo_splitter = LeaveOneGroupOut()


for fold_number, (
    training_index,
    validation_index

) in enumerate(
    logo_splitter.split(
        X_train_raw,
        y_train,
        g_train
    ),
    start=1
):

    if fold_number > LOGO_FOLDS_TO_USE:

        break


    held_out_persons = np.unique(
        g_train[
            validation_index
        ]
    )


    if len(
        held_out_persons
    ) != 1:

        raise RuntimeError(
            "Unexpected LOGO participant grouping."
        )


    held_out_person = str(
        held_out_persons[
            0
        ]
    )


    if (
        held_out_person
        in completed_logo_persons
    ):

        print(
            f"LOGO "
            f"{fold_number}/"
            f"{LOGO_FOLDS_TO_USE}, "
            f"Participant "
            f"{held_out_person} "
            f"already completed."
        )

        continue


    fold_start = time.time()


    # ============================================================
    # FOLD SPECIFIC STANDARDIZATION
    # ============================================================

    (
        X_logo_training,
        X_logo_validation

    ) = prepare_fold(
        training_index,
        validation_index
    )


    # ============================================================
    # FIXED EPOCH TRAINING
    #
    # Held out participant is not used for early stopping.
    # ============================================================

    (
        logo_model,
        _,
        _

    ) = fit_model(
        best_params,
        X_logo_training,
        y_train[
            training_index
        ],
        X_validation=None,
        y_validation=None,
        seed=
            int(
                best_record[
                    "Seed"
                ]
            ),
        epochs=
            FIXED_LOGO_EPOCHS,
        use_early_stopping=
            False,
        verbose=
            False
    )


    logo_probabilities = predict_probabilities(
        logo_model,
        X_logo_validation,
        y_train[
            validation_index
        ]
    )


    logo_predictions = np.argmax(
        logo_probabilities,
        axis=1
    )


    fold_accuracy = accuracy_score(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_precision = precision_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_recall = recall_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_f1 = f1_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="weighted",
        zero_division=0
    )


    fold_macro_f1 = f1_score(
        y_train[
            validation_index
        ],
        logo_predictions,
        average="macro",
        zero_division=0
    )


    fold_mcc = matthews_corrcoef(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_kappa = cohen_kappa_score(
        y_train[
            validation_index
        ],
        logo_predictions
    )


    fold_time = (
        time.time()
        - fold_start
    )


    logo_record = {
        "Config_ID":
            LOGO_CONFIG_ID,

        "Architecture_Version":
            ARCHITECTURE_VERSION,

        "Fold":
            fold_number,

        "Held_Out_Person":
            held_out_person,

        "Rows":
            int(
                len(
                    validation_index
                )
            ),

        "Accuracy":
            float(
                fold_accuracy
            ),

        "Precision_Weighted":
            float(
                fold_precision
            ),

        "Recall_Weighted":
            float(
                fold_recall
            ),

        "F1_Weighted":
            float(
                fold_f1
            ),

        "F1_Macro":
            float(
                fold_macro_f1
            ),

        "Cohen_Kappa":
            float(
                fold_kappa
            ),

        "MCC":
            float(
                fold_mcc
            ),

        "Fixed_Epochs":
            int(
                FIXED_LOGO_EPOCHS
            ),

        "Execution_Time_Seconds":
            float(
                fold_time
            )
    }


    logo_records.append(
        logo_record
    )


    atomic_csv(
        pd.DataFrame(
            logo_records
        ),
        LOGO_RESULT_PATH
    )


    completed_logo_persons.add(
        held_out_person
    )


    print(
        f"LOGO "
        f"{fold_number}/"
        f"{LOGO_FOLDS_TO_USE}, "
        f"Participant = "
        f"{held_out_person}, "
        f"Accuracy = "
        f"{fold_accuracy:.4f}, "
        f"F1 = "
        f"{fold_f1:.4f}, "
        f"MCC = "
        f"{fold_mcc:.4f}, "
        f"Time = "
        f"{fold_time / 60:.2f} min"
    )


    del logo_model
    del logo_probabilities
    del logo_predictions
    del X_logo_training
    del X_logo_validation


    clear_device()


logo_time = (
    time.time()
    - logo_start
)


# ================================================================
# LOGO SUMMARY
# ================================================================

logo_table = pd.read_csv(
    LOGO_RESULT_PATH
)


logo_table = logo_table[
    logo_table[
        "Config_ID"
    ].astype(str)
    == LOGO_CONFIG_ID
].copy()


logo_table = (
    logo_table
    .sort_values(
        "Fold"
    )
    .drop_duplicates(
        subset=[
            "Held_Out_Person"
        ],
        keep="last"
    )
)


logo_scores = (
    logo_table[
        "Accuracy"
    ]
    .astype(float)
    .to_numpy()
)


logo_mean = float(
    np.mean(
        logo_scores
    )
)


logo_std = float(
    np.std(
        logo_scores
    )
)


print()

print(
    "Full LOGO Accuracy:",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Completed LOGO folds:",
    len(
        logo_table
    )
)

print(
    "LOGO time this session:",
    f"{logo_time / 60:.2f}",
    "minutes"
)


# ================================================================
# RESTORE BEST MODEL
# ================================================================

best_model = build_model(
    best_params,
    seed=
        int(
            best_record[
                "Seed"
            ]
        )
)


best_model.load_state_dict(
    best_bundle[
        "model_state_dict"
    ]
)


best_model.eval()


# ================================================================
# FINAL UNTOUCHED TEST EVALUATION
# ================================================================

print()

print("=" * 80)

print(
    "FINAL UNTOUCHED TEST EVALUATION"
)

print("=" * 80)


test_start = time.time()


test_probabilities = predict_probabilities(
    best_model,
    X_test,
    y_test
)


test_predictions = np.argmax(
    test_probabilities,
    axis=1
)


test_time = (
    time.time()
    - test_start
)


# ================================================================
# TEST METRICS
# ================================================================

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)


test_precision = precision_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_recall = recall_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_weighted = f1_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)


test_f1_macro = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0
)


test_kappa = cohen_kappa_score(
    y_test,
    test_predictions
)


test_mcc = matthews_corrcoef(
    y_test,
    test_predictions
)


try:

    test_roc_auc = roc_auc_score(
        y_test,
        test_probabilities,
        multi_class="ovr",
        average="weighted"
    )


except Exception:

    test_roc_auc = np.nan


# ================================================================
# CLASSIFICATION REPORT
# ================================================================

report = classification_report(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    ),
    target_names=
        class_names,
    digits=4,
    zero_division=0
)


print()

print(
    "CLASSIFICATION REPORT"
)

print()

print(
    report
)


with open(
    OUTPUT_DIR
    / "classification_report.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        report
    )


report_table = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        labels=np.arange(
            NUMBER_CLASSES
        ),
        target_names=
            class_names,
        output_dict=True,
        zero_division=0
    )
).transpose()


report_table.to_csv(
    OUTPUT_DIR
    / "classification_report.csv"
)


# ================================================================
# CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=np.arange(
        NUMBER_CLASSES
    )
)


pd.DataFrame(
    cm,
    index=
        class_names,
    columns=
        class_names
).to_csv(
    OUTPUT_DIR
    / "confusion_matrix.csv"
)


# ================================================================
# CONFUSION MATRIX FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "Multimodal CNN LSTM Confusion Matrix"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            str(
                cm[
                    row,
                    column
                ]
            ),
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# NORMALIZED CONFUSION MATRIX
# ================================================================

row_sums = cm.sum(
    axis=1,
    keepdims=True
)


normalized_cm = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(
        cm,
        dtype=np.float64
    ),
    where=(
        row_sums != 0
    )
)


pd.DataFrame(
    normalized_cm,
    index=
        class_names,
    columns=
        class_names
).to_csv(
    OUTPUT_DIR
    / "normalized_confusion_matrix.csv"
)


# ================================================================
# NORMALIZED CONFUSION MATRIX FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


image = ax.imshow(
    normalized_cm
)


ax.set_xticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUMBER_CLASSES
    )
)


ax.set_xticklabels(
    class_names,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    class_names
)


ax.set_xlabel(
    "Predicted Class"
)


ax.set_ylabel(
    "True Class"
)


ax.set_title(
    "Normalized Multimodal CNN LSTM Confusion Matrix"
)


for row in range(
    NUMBER_CLASSES
):

    for column in range(
        NUMBER_CLASSES
    ):

        ax.text(
            column,
            row,
            f"{normalized_cm[row, column]:.2f}",
            ha="center",
            va="center"
        )


fig.colorbar(
    image,
    ax=ax
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "normalized_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ================================================================
# SAVE TEST PREDICTIONS
# ================================================================

prediction_table = pd.DataFrame(
    {
        "PersonSrial":
            g_test,

        "True_Class":
            label_encoder.inverse_transform(
                y_test
            ),

        "Predicted_Class":
            label_encoder.inverse_transform(
                test_predictions.astype(
                    np.int32
                )
            )
    }
)


for class_index, class_name in enumerate(
    class_names
):

    prediction_table[
        f"Probability_{class_name}"
    ] = test_probabilities[
        :,
        class_index
    ]


prediction_table.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False
)


# ================================================================
# TOTAL EXECUTION TIME
# ================================================================

total_execution_time = (
    time.time()
    - TOTAL_START
)


# ================================================================
# FINAL METRICS
# ================================================================

final_metrics = {
    "Algorithm":
        "Multimodal CNN LSTM",

    "Architecture_Version":
        ARCHITECTURE_VERSION,

    "Recurrent_Implementation":
        "Manual standard LSTM",

    "Dataset":
        "Max Dataset",

    "Framework":
        "PyTorch",

    "Input_File":
        str(
            FILE_PATH
        ),

    "Output_Directory":
        str(
            OUTPUT_DIR
        ),

    "Device_Type":
        DEVICE_TYPE,

    "Device_Name":
        DEVICE_NAME,

    "Feature_Count":
        int(
            NUMBER_FEATURES
        ),

    "Number_of_Classes":
        int(
            NUMBER_CLASSES
        ),

    "Classes":
        class_names,

    "Modality_Count":
        int(
            NUMBER_MODALITIES
        ),

    "Modality_Names":
        MODALITY_NAMES,

    "Modality_Grouping_Mode":
        MODALITY_GROUPING_MODE,

    "Trainable_Parameters":
        int(
            TRAINABLE_PARAMETERS
        ),

    "Training_Participants":
        int(
            len(
                train_persons
            )
        ),

    "Validation_Participants":
        int(
            len(
                validation_persons
            )
        ),

    "Testing_Participants":
        int(
            len(
                test_persons
            )
        ),

    "Training_Samples":
        int(
            TRAIN_SAMPLES
        ),

    "Validation_Samples":
        int(
            VALIDATION_SAMPLES
        ),

    "Testing_Samples":
        int(
            TEST_SAMPLES
        ),

    "Optuna_Trials":
        int(
            N_TRIALS
        ),

    "Optuna_GroupKFold_Accuracy":
        float(
            study.best_value
        ),

    "Best_Parameters":
        best_params,

    "Selected_Run":
        int(
            best_record[
                "Run"
            ]
        ),

    "Selected_Seed":
        int(
            best_record[
                "Seed"
            ]
        ),

    "Selected_Best_Epoch":
        int(
            best_record[
                "Best_Epoch"
            ]
        ),

    "Training_Accuracy":
        float(
            best_record[
                "Training_Accuracy"
            ]
        ),

    "Validation_Accuracy":
        float(
            best_record[
                "Validation_Accuracy"
            ]
        ),

    "Generalization_Gap":
        float(
            best_record[
                "Generalization_Gap"
            ]
        ),

    "Testing_Accuracy":
        float(
            test_accuracy
        ),

    "Five_Run_Training_Mean":
        float(
            run_train_mean
        ),

    "Five_Run_Training_STD":
        float(
            run_train_std
        ),

    "Five_Run_Validation_Mean":
        float(
            run_validation_mean
        ),

    "Five_Run_Validation_STD":
        float(
            run_validation_std
        ),

    "Five_Run_Generalization_Gap_Mean":
        float(
            run_gap_mean
        ),

    "Five_Run_Generalization_Gap_STD":
        float(
            run_gap_std
        ),

    "Full_LOGO_Accuracy":
        float(
            logo_mean
        ),

    "Full_LOGO_STD":
        float(
            logo_std
        ),

    "Full_LOGO_Folds_Available":
        int(
            FULL_LOGO_FOLDS
        ),

    "Full_LOGO_Folds_Completed":
        int(
            len(
                logo_table
            )
        ),

    "Precision_Weighted":
        float(
            test_precision
        ),

    "Recall_Weighted":
        float(
            test_recall
        ),

    "F1_Weighted":
        float(
            test_f1_weighted
        ),

    "F1_Macro":
        float(
            test_f1_macro
        ),

    "Cohen_Kappa":
        float(
            test_kappa
        ),

    "MCC":
        float(
            test_mcc
        ),

    "ROC_AUC_Weighted_OVR":
        (
            None
            if np.isnan(
                test_roc_auc
            )
            else float(
                test_roc_auc
            )
        ),

    "Optuna_Time_Seconds":
        float(
            optuna_time
        ),

    "Five_Run_Time_This_Session_Seconds":
        float(
            five_run_time
        ),

    "LOGO_Time_This_Session_Seconds":
        float(
            logo_time
        ),

    "Test_Prediction_Time_Seconds":
        float(
            test_time
        ),

    "Total_Execution_Time_Seconds":
        float(
            total_execution_time
        ),

    "Current_RAM_MB":
        float(
            memory_mb()
        )
}


with open(
    OUTPUT_DIR
    / "final_metrics.json",
    "w",
    encoding="utf8"
) as file:

    json.dump(
        final_metrics,
        file,
        indent=4
    )


# ================================================================
# FINAL SUMMARY TEXT
# ================================================================

summary_lines = [
    "MULTIMODAL CNN LSTM",
    "MAX DATASET",
    "MANUAL XPU TRAINABLE LSTM",

    "",

    f"Architecture Version: "
    f"{ARCHITECTURE_VERSION}",

    f"Input File: "
    f"{FILE_PATH}",

    f"Output Directory: "
    f"{OUTPUT_DIR}",

    "",

    f"Device Type: "
    f"{DEVICE_TYPE}",

    f"Device Name: "
    f"{DEVICE_NAME}",

    "",

    f"Feature Count: "
    f"{NUMBER_FEATURES}",

    f"Classes: "
    f"{class_names}",

    f"Modality Grouping: "
    f"{MODALITY_GROUPING_MODE}",

    f"Modalities: "
    f"{MODALITY_NAMES}",

    f"Trainable Parameters: "
    f"{TRAINABLE_PARAMETERS}",

    "",

    f"Best Parameters: "
    f"{best_params}",

    "",

    f"Selected Run: "
    f"{best_record['Run']}",

    f"Selected Seed: "
    f"{best_record['Seed']}",

    f"Best Epoch: "
    f"{best_record['Best_Epoch']}",

    "",

    f"Optuna GroupKFold Accuracy: "
    f"{study.best_value:.4f}",

    f"Training Accuracy: "
    f"{best_record['Training_Accuracy']:.4f}",

    f"Validation Accuracy: "
    f"{best_record['Validation_Accuracy']:.4f}",

    f"Generalization Gap: "
    f"{best_record['Generalization_Gap']:.4f}",

    f"Testing Accuracy: "
    f"{test_accuracy:.4f}",

    "",

    f"Five Run Training Accuracy: "
    f"{run_train_mean:.4f} "
    f"plus or minus "
    f"{run_train_std:.4f}",

    f"Five Run Validation Accuracy: "
    f"{run_validation_mean:.4f} "
    f"plus or minus "
    f"{run_validation_std:.4f}",

    f"Five Run Generalization Gap: "
    f"{run_gap_mean:.4f} "
    f"plus or minus "
    f"{run_gap_std:.4f}",

    "",

    f"Full LOGO Accuracy: "
    f"{logo_mean:.4f} "
    f"plus or minus "
    f"{logo_std:.4f}",

    f"Full LOGO Folds: "
    f"{len(logo_table)}",

    "",

    f"Precision Weighted: "
    f"{test_precision:.4f}",

    f"Recall Weighted: "
    f"{test_recall:.4f}",

    f"F1 Weighted: "
    f"{test_f1_weighted:.4f}",

    f"F1 Macro: "
    f"{test_f1_macro:.4f}",

    f"Cohen Kappa: "
    f"{test_kappa:.4f}",

    f"MCC: "
    f"{test_mcc:.4f}",

    (
        "ROC AUC Weighted OVR: N/A"
        if np.isnan(
            test_roc_auc
        )
        else
        f"ROC AUC Weighted OVR: "
        f"{test_roc_auc:.4f}"
    ),

    "",

    f"Optuna Time: "
    f"{optuna_time / 60:.2f} minutes",

    f"Five Run Time: "
    f"{five_run_time / 60:.2f} minutes",

    f"Full LOGO Time: "
    f"{logo_time / 60:.2f} minutes",

    f"Test Prediction Time: "
    f"{test_time:.2f} seconds",

    f"Total Execution Time: "
    f"{total_execution_time / 60:.2f} minutes"
]


with open(
    OUTPUT_DIR
    / "final_summary.txt",
    "w",
    encoding="utf8"
) as file:

    file.write(
        "\n".join(
            summary_lines
        )
    )


# ================================================================
# FINAL SCREEN OUTPUT
# ================================================================

print()

print("#" * 80)

print(
    "FINAL MULTIMODAL CNN LSTM RESULTS"
)

print(
    "MAX DATASET"
)

print("#" * 80)


print(
    "Architecture Version      :",
    ARCHITECTURE_VERSION
)

print(
    "Recurrent Layer           : Manual Standard LSTM"
)

print(
    "Device                    :",
    DEVICE_NAME
)

print(
    "Feature Count             :",
    NUMBER_FEATURES
)

print(
    "Modality Grouping         :",
    MODALITY_GROUPING_MODE
)


for modality_name in MODALITY_NAMES:

    print(
        f"{modality_name:<26}:",
        len(
            MODALITY_INDICES[
                modality_name
            ]
        ),
        "features"
    )


print(
    "Trainable Parameters      :",
    f"{TRAINABLE_PARAMETERS:,}"
)


print()

print(
    "Base CNN Channels         :",
    best_params[
        "base_channels"
    ]
)

print(
    "CNN Kernel Size           :",
    best_params[
        "kernel_size"
    ]
)

print(
    "LSTM Hidden Units         :",
    best_params[
        "lstm_hidden"
    ]
)

print(
    "LSTM Layers               :",
    best_params[
        "lstm_layers"
    ]
)

print(
    "CNN Dropout               :",
    f"{best_params['conv_dropout']:.4f}"
)

print(
    "LSTM Dropout              :",
    f"{best_params['lstm_dropout']:.4f}"
)

print(
    "Input Dropout             :",
    f"{best_params['input_dropout']:.4f}"
)

print(
    "Input Noise               :",
    f"{best_params['input_noise']:.4f}"
)

print(
    "Dense Dropout             :",
    f"{best_params['dense_dropout']:.4f}"
)

print(
    "Learning Rate             :",
    f"{best_params['learning_rate']:.2e}"
)

print(
    "Weight Decay              :",
    f"{best_params['weight_decay']:.2e}"
)

print(
    "Label Smoothing           :",
    f"{best_params['label_smoothing']:.4f}"
)


print()

print(
    "Selected Run              :",
    best_record[
        "Run"
    ]
)

print(
    "Selected Seed             :",
    best_record[
        "Seed"
    ]
)

print(
    "Best Epoch                :",
    best_record[
        "Best_Epoch"
    ]
)


print()

print(
    "Optuna GroupKFold         :",
    f"{study.best_value:.4f}"
)

print(
    "Training Accuracy         :",
    f"{best_record['Training_Accuracy']:.4f}"
)

print(
    "Validation Accuracy       :",
    f"{best_record['Validation_Accuracy']:.4f}"
)

print(
    "Generalization Gap        :",
    f"{best_record['Generalization_Gap']:.4f}"
)

print(
    "Testing Accuracy          :",
    f"{test_accuracy:.4f}"
)


print()

print(
    "Five Run Training         :",
    f"{run_train_mean:.4f}",
    "plus or minus",
    f"{run_train_std:.4f}"
)

print(
    "Five Run Validation       :",
    f"{run_validation_mean:.4f}",
    "plus or minus",
    f"{run_validation_std:.4f}"
)

print(
    "Five Run Gap              :",
    f"{run_gap_mean:.4f}",
    "plus or minus",
    f"{run_gap_std:.4f}"
)


print()

print(
    "Full LOGO Accuracy        :",
    f"{logo_mean:.4f}",
    "plus or minus",
    f"{logo_std:.4f}"
)

print(
    "Full LOGO Folds           :",
    len(
        logo_table
    ),
    "of",
    FULL_LOGO_FOLDS
)


print()

print(
    "Precision Weighted        :",
    f"{test_precision:.4f}"
)

print(
    "Recall Weighted           :",
    f"{test_recall:.4f}"
)

print(
    "F1 Weighted               :",
    f"{test_f1_weighted:.4f}"
)

print(
    "F1 Macro                  :",
    f"{test_f1_macro:.4f}"
)

print(
    "Cohen Kappa               :",
    f"{test_kappa:.4f}"
)

print(
    "MCC                       :",
    f"{test_mcc:.4f}"
)


if np.isnan(
    test_roc_auc
):

    print(
        "ROC AUC Weighted OVR      : N/A"
    )


else:

    print(
        "ROC AUC Weighted OVR      :",
        f"{test_roc_auc:.4f}"
    )


print()

print(
    "Optuna Time               :",
    f"{optuna_time / 60:.2f}",
    "minutes"
)

print(
    "Five Run Time             :",
    f"{five_run_time / 60:.2f}",
    "minutes"
)

print(
    "Full LOGO Time            :",
    f"{logo_time / 60:.2f}",
    "minutes"
)

print(
    "Test Prediction Time      :",
    f"{test_time:.2f}",
    "seconds"
)

print(
    "Total Execution Time      :",
    f"{total_execution_time / 60:.2f}",
    "minutes"
)

print(
    "Current RAM               :",
    f"{memory_mb():.2f}",
    "MB"
)


print()

print(
    "Native torch.nn.LSTM was NOT used."
)

print(
    "The recurrent network used explicit standard LSTM gates."
)

print(
    "XPU backward propagation was verified before Optuna."
)

print(
    "No dataset rows were subsampled."
)

print(
    "All five GroupKFold folds were used."
)

print(
    "Full training participant LOGO was requested."
)

print(
    "Every CV fold used an independently fitted scaler."
)

print(
    "Input noise, dropout, weight decay and label smoothing "
    "were used to reduce overfitting."
)

print(
    "Validation loss controlled early stopping."
)

print(
    "Final model selection used validation participants only."
)

print(
    "Test participants were not used for tuning or model selection."
)

print(
    "Held out LOGO participants were not used for early stopping."
)

print(
    "Optuna, final runs and LOGO can resume after interruption."
)


print()

print(
    "ALL RESULTS SAVED IN:"
)

print(
    OUTPUT_DIR
)

print("#" * 80)

DEVICE CONFIGURATION
PyTorch version: 2.14.1+xpu
Selected device: xpu:0
Device type: xpu
Device name: Intel(R) Arc(TM) A750 Graphics
CPU threads: 8
Architecture version: manual_xpu_lstm_v2

MULTIMODAL CNN LSTM
MAX DATASET
MANUAL XPU TRAINABLE LSTM
Input dataset:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/Cleaned_oN_Selected_Max.csv

Result folder:
/home/rakib/Desktop/ml_apply/DL_Apply/max_dataset/Multimodal_CNN_LSTM_Results

Optuna trials: 10
GroupKFold folds: 5
Final runs: 5
Maximum epochs: 50
Batch size: 1024
Early stopping patience: 6
LOGO cap: None
Data subsampling: NONE

LOADING COMPLETE MAX DATASET
Raw dataset shape: (149350, 238)
RAM after loading: 4725.47 MB
Feature count: 232
Invalid feature cells: 0

Feature matrix: (149350, 232)
Classes: ['class_lecture', 'music', 'normal', 'reading']
Number of classes: 4
Participants: 60
Loading time: 8.18 seconds

CLASS DISTRIBUTION
class_lecture: 39857 (26.69%)
music: 32276 (21.61%)
normal: 39327 (26.33%)
reading: 37890 (25.37%)

E

**Apply DL On Med Dataset**
**Apply 4 DL Algorithm on Med Dataset**

*DL Models:*

*1. CNN*

*2. EEG-Conformer + 1D-TCN (Dual-Stream Cross-Attention Network)*

*3. Cross-Attention Multimodal Transformer*

*4. Multimodal CNN-LSTM*

In [1]:
# =============================================================================
# FOUR DEEP LEARNING MODELS
# MED DATASET
#
# 1. CNN
# 2. EEG Conformer + 1D TCN Dual Stream Cross Attention
# 3. Cross Attention Multimodal Transformer
# 4. Multimodal CNN LSTM
#
# INPUT
# /home/rakib/Desktop/ml_apply/DL_Apply/med_dataset/
# Cleaned_oN_Selected_Med.csv
#
# OUTPUT ROOT
# /home/rakib/Desktop/ml_apply/DL_Apply/med_dataset
#
# EACH ALGORITHM GETS ITS OWN RESULT FOLDER
#
# GPU PRIORITY
# Intel XPU
# CUDA
# CPU
#
# RESTART SUPPORT
# Optuna SQLite
# Final run checkpoints
# Full LOGO checkpoints
#
# TEST DATA ARE USED ONLY FOR FINAL EVALUATION
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import gc
import json
import time
import math
import random
import hashlib
import traceback
import warnings

from pathlib import Path

import joblib
import psutil
import optuna

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)


# =============================================================================
# WARNINGS
# =============================================================================

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


# =============================================================================
# GLOBAL EXPERIMENT CONFIGURATION
# =============================================================================

SEED = 42

N_TRIALS = 10
N_RUNS = 5
GROUP_K_FOLDS = 5

LOGO_MAX_FOLDS = None

GRADIENT_CLIP = 1.0
LR_SCHEDULER_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4

CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)

torch.set_num_threads(CPU_THREADS)


# =============================================================================
# ALGORITHMS TO RUN
# =============================================================================

ALGORITHMS_TO_RUN = [
    "CNN",
    "EEG_CONFORMER_TCN",
    "CROSS_ATTENTION_MULTIMODAL_TRANSFORMER",
    "MULTIMODAL_CNN_LSTM",
]


# =============================================================================
# INPUT AND OUTPUT
# =============================================================================

FILE_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/med_dataset/"
    "Cleaned_oN_Selected_Med.csv"
)

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/med_dataset"
)

if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Dataset was not found: {FILE_PATH}"
    )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# PROCESS MEMORY
# =============================================================================

PROCESS = psutil.Process(os.getpid())


def memory_mb():
    return (
        PROCESS.memory_info().rss
        / 1024
        / 1024
    )


# =============================================================================
# ATOMIC FILE SAVING
# =============================================================================

def atomic_csv(dataframe, path):
    path = Path(path)

    temporary = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary,
        index=False
    )

    os.replace(
        temporary,
        path
    )


def atomic_torch_save(obj, path):
    path = Path(path)

    temporary = path.with_name(
        path.name + ".tmp"
    )

    torch.save(
        obj,
        temporary
    )

    os.replace(
        temporary,
        path
    )


def save_json(obj, path):
    path = Path(path)

    temporary = path.with_name(
        path.name + ".tmp"
    )

    with open(
        temporary,
        "w",
        encoding="utf8"
    ) as file:
        json.dump(
            obj,
            file,
            indent=4
        )

    os.replace(
        temporary,
        path
    )


# =============================================================================
# RANDOM SEED
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if (
        hasattr(torch, "xpu")
        and torch.xpu.is_available()
    ):
        try:
            torch.xpu.manual_seed_all(seed)
        except Exception:
            pass

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# =============================================================================
# DEVICE DETECTION
# =============================================================================

if (
    hasattr(torch, "xpu")
    and torch.xpu.is_available()
):
    DEVICE = torch.device("xpu:0")
    DEVICE_TYPE = "xpu"
    DEVICE_NAME = torch.xpu.get_device_name(0)

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    DEVICE_TYPE = "cuda"
    DEVICE_NAME = torch.cuda.get_device_name(0)

else:
    DEVICE = torch.device("cpu")
    DEVICE_TYPE = "cpu"
    DEVICE_NAME = "CPU"


def synchronize():
    if DEVICE_TYPE == "xpu":
        torch.xpu.synchronize()

    elif DEVICE_TYPE == "cuda":
        torch.cuda.synchronize()


def clear_device():
    gc.collect()

    if DEVICE_TYPE == "xpu":
        try:
            torch.xpu.empty_cache()
        except Exception:
            pass

    elif DEVICE_TYPE == "cuda":
        torch.cuda.empty_cache()


print("=" * 90)
print("DEVICE CONFIGURATION")
print("=" * 90)
print("PyTorch version :", torch.__version__)
print("Device          :", DEVICE)
print("Device type     :", DEVICE_TYPE)
print("Device name     :", DEVICE_NAME)
print("CPU threads     :", CPU_THREADS)
print("=" * 90)


# =============================================================================
# TOTAL PROGRAM TIMER
# =============================================================================

PROGRAM_START = time.time()


# =============================================================================
# LOAD MED DATASET
# =============================================================================

print()
print("=" * 90)
print("LOADING MED DATASET")
print("=" * 90)

load_start = time.time()

df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)

print(
    "Raw dataset shape:",
    df.shape
)

print(
    "RAM after loading:",
    f"{memory_mb():.2f} MB"
)


# =============================================================================
# COLUMNS
# =============================================================================

METADATA_COLUMNS = [
    "PersonSrial",
    "Window",
    "Window_Start",
    "Window_End",
    "Window_Samples",
]

TARGET_COLUMN = "Class"

for required_column in [
    "PersonSrial",
    TARGET_COLUMN,
]:
    if required_column not in df.columns:
        raise ValueError(
            f"Missing required column: "
            f"{required_column}"
        )


FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in METADATA_COLUMNS
    and column != TARGET_COLUMN
]

NUMBER_FEATURES = len(
    FEATURE_COLUMNS
)

if NUMBER_FEATURES == 0:
    raise ValueError(
        "No feature columns were found."
    )

print(
    "Feature count:",
    NUMBER_FEATURES
)


# =============================================================================
# FEATURE MATRIX
# =============================================================================

X_ALL = (
    df[
        FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(
        dtype=np.float32
    )
)

invalid_count = int(
    (
        ~np.isfinite(X_ALL)
    ).sum()
)

print(
    "Invalid feature cells:",
    invalid_count
)

if invalid_count > 0:
    raise ValueError(
        "Dataset contains NaN or infinite feature values."
    )


# =============================================================================
# LABELS
# =============================================================================

LABEL_ENCODER = LabelEncoder()

Y_ALL = LABEL_ENCODER.fit_transform(
    df[
        TARGET_COLUMN
    ].astype(str)
).astype(
    np.int64
)

CLASS_NAMES = list(
    LABEL_ENCODER.classes_
)

NUMBER_CLASSES = len(
    CLASS_NAMES
)


# =============================================================================
# PARTICIPANTS
# =============================================================================

GROUPS_ALL = (
    df[
        "PersonSrial"
    ]
    .to_numpy()
)

print(
    "Feature matrix:",
    X_ALL.shape
)

print(
    "Classes:",
    CLASS_NAMES
)

print(
    "Number of classes:",
    NUMBER_CLASSES
)

print(
    "Participants:",
    len(
        np.unique(
            GROUPS_ALL
        )
    )
)


# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

CLASS_DISTRIBUTION_RECORDS = []

print()
print("CLASS DISTRIBUTION")

for class_index, class_name in enumerate(
    CLASS_NAMES
):
    count = int(
        np.sum(
            Y_ALL == class_index
        )
    )

    percentage = (
        count
        / len(Y_ALL)
        * 100.0
    )

    print(
        f"{class_name}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )

    CLASS_DISTRIBUTION_RECORDS.append(
        {
            "Class": class_name,
            "Count": count,
            "Percentage": percentage,
        }
    )

CLASS_DISTRIBUTION_TABLE = pd.DataFrame(
    CLASS_DISTRIBUTION_RECORDS
)


# =============================================================================
# DATASET SIGNATURE
#
# Used to prevent stale experiment results from another dataset revision
# from being reused.
# =============================================================================

file_stat = FILE_PATH.stat()

dataset_signature_payload = {
    "path": str(FILE_PATH),
    "size": int(file_stat.st_size),
    "mtime_ns": int(file_stat.st_mtime_ns),
    "rows": int(len(Y_ALL)),
    "features": FEATURE_COLUMNS,
    "classes": CLASS_NAMES,
}

DATASET_SIGNATURE = hashlib.sha256(
    json.dumps(
        dataset_signature_payload,
        sort_keys=True
    ).encode("utf8")
).hexdigest()[:16]

print(
    "Dataset signature:",
    DATASET_SIGNATURE
)

print(
    "Loading time:",
    f"{time.time() - load_start:.2f} seconds"
)


# =============================================================================
# FEATURE MODALITY DETECTION
#
# These are feature families or feature views from the EEG feature table.
# They are not claimed to be separate physical sensors.
# =============================================================================

SPECTRAL_KEYWORDS = [
    "fft",
    "psd",
    "power",
    "bandpower",
    "band_power",
    "spectral",
    "frequency",
    "freq",
    "delta",
    "theta",
    "alpha",
    "beta",
    "gamma",
]

STATISTICAL_KEYWORDS = [
    "mean",
    "median",
    "std",
    "variance",
    "var",
    "rms",
    "skew",
    "kurt",
    "minimum",
    "maximum",
    "min",
    "max",
    "range",
    "iqr",
    "quartile",
    "percentile",
    "peak",
    "zero",
    "zcr",
    "activity",
]

COMPLEXITY_KEYWORDS = [
    "entropy",
    "complexity",
    "fractal",
    "hurst",
    "dfa",
    "lyapunov",
    "sampen",
    "sample_entropy",
    "apen",
    "approximate_entropy",
    "permutation",
    "lzc",
    "lempel",
    "higuchi",
    "petrosian",
    "mobility",
]


def contains_keyword(
    feature_name,
    keywords
):
    feature_name = str(
        feature_name
    ).lower()

    return any(
        keyword in feature_name
        for keyword in keywords
    )


def construct_modalities(columns):
    spectral = []
    statistical = []
    complexity = []
    unassigned = []

    for index, column in enumerate(columns):

        if contains_keyword(
            column,
            COMPLEXITY_KEYWORDS
        ):
            complexity.append(index)

        elif contains_keyword(
            column,
            SPECTRAL_KEYWORDS
        ):
            spectral.append(index)

        elif contains_keyword(
            column,
            STATISTICAL_KEYWORDS
        ):
            statistical.append(index)

        else:
            unassigned.append(index)

    groups = [
        spectral,
        statistical,
        complexity,
    ]

    for index in unassigned:
        smallest_group = int(
            np.argmin(
                [
                    len(group)
                    for group in groups
                ]
            )
        )

        groups[
            smallest_group
        ].append(index)

    minimum_size = min(
        len(group)
        for group in groups
    )

    maximum_size = max(
        len(group)
        for group in groups
    )

    semantic_grouping_valid = (
        minimum_size >= 12
        and
        maximum_size
        <= NUMBER_FEATURES * 0.70
    )

    if semantic_grouping_valid:
        grouping_mode = (
            "Feature name based semantic grouping"
        )

        modality_indices = {
            "spectral_frequency":
                sorted(
                    spectral
                ),

            "statistical_temporal":
                sorted(
                    statistical
                ),

            "nonlinear_complexity":
                sorted(
                    complexity
                ),
        }

    else:
        grouping_mode = (
            "Balanced feature view fallback"
        )

        balanced_groups = np.array_split(
            np.arange(
                NUMBER_FEATURES
            ),
            3
        )

        modality_indices = {
            "feature_view_1":
                balanced_groups[
                    0
                ].tolist(),

            "feature_view_2":
                balanced_groups[
                    1
                ].tolist(),

            "feature_view_3":
                balanced_groups[
                    2
                ].tolist(),
        }

    return (
        modality_indices,
        grouping_mode
    )


(
    MODALITY_INDICES,
    MODALITY_GROUPING_MODE,

) = construct_modalities(
    FEATURE_COLUMNS
)

MODALITY_NAMES = list(
    MODALITY_INDICES.keys()
)

NUMBER_MODALITIES = len(
    MODALITY_NAMES
)

print()
print("=" * 90)
print("FEATURE MODALITY CONSTRUCTION")
print("=" * 90)

print(
    "Grouping mode:",
    MODALITY_GROUPING_MODE
)

for modality_name in MODALITY_NAMES:
    print(
        modality_name,
        ":",
        len(
            MODALITY_INDICES[
                modality_name
            ]
        ),
        "features"
    )


# =============================================================================
# RELEASE DATAFRAME
# =============================================================================

del df
gc.collect()


# =============================================================================
# PARTICIPANT WISE 70 / 20 / 10 SPLIT
# =============================================================================

print()
print("=" * 90)
print("PERSON WISE 70 / 20 / 10 SPLIT")
print("=" * 90)

persons = np.unique(
    GROUPS_ALL
)

rng = np.random.RandomState(
    SEED
)

rng.shuffle(
    persons
)

number_persons = len(
    persons
)

number_test = max(
    1,
    int(
        round(
            0.20
            * number_persons
        )
    )
)

number_validation = max(
    1,
    int(
        round(
            0.10
            * number_persons
        )
    )
)

TEST_PERSONS = persons[
    :number_test
]

VALIDATION_PERSONS = persons[
    number_test:
    number_test
    + number_validation
]

TRAIN_PERSONS = persons[
    number_test
    + number_validation:
]


# =============================================================================
# VERIFY PARTICIPANT ISOLATION
# =============================================================================

assert (
    np.intersect1d(
        TRAIN_PERSONS,
        VALIDATION_PERSONS
    ).size == 0
)

assert (
    np.intersect1d(
        TRAIN_PERSONS,
        TEST_PERSONS
    ).size == 0
)

assert (
    np.intersect1d(
        VALIDATION_PERSONS,
        TEST_PERSONS
    ).size == 0
)


# =============================================================================
# SPLIT MASKS
# =============================================================================

train_mask = np.isin(
    GROUPS_ALL,
    TRAIN_PERSONS
)

validation_mask = np.isin(
    GROUPS_ALL,
    VALIDATION_PERSONS
)

test_mask = np.isin(
    GROUPS_ALL,
    TEST_PERSONS
)


# =============================================================================
# RAW SPLIT MATRICES
# =============================================================================

X_TRAIN_RAW = np.ascontiguousarray(
    X_ALL[
        train_mask
    ],
    dtype=np.float32
)

Y_TRAIN = np.ascontiguousarray(
    Y_ALL[
        train_mask
    ],
    dtype=np.int64
)

G_TRAIN = np.asarray(
    GROUPS_ALL[
        train_mask
    ]
)

X_VALIDATION_RAW = np.ascontiguousarray(
    X_ALL[
        validation_mask
    ],
    dtype=np.float32
)

Y_VALIDATION = np.ascontiguousarray(
    Y_ALL[
        validation_mask
    ],
    dtype=np.int64
)

G_VALIDATION = np.asarray(
    GROUPS_ALL[
        validation_mask
    ]
)

X_TEST_RAW = np.ascontiguousarray(
    X_ALL[
        test_mask
    ],
    dtype=np.float32
)

Y_TEST = np.ascontiguousarray(
    Y_ALL[
        test_mask
    ],
    dtype=np.int64
)

G_TEST = np.asarray(
    GROUPS_ALL[
        test_mask
    ]
)


# =============================================================================
# SPLIT COUNTS
# =============================================================================

TRAIN_SAMPLES = len(
    X_TRAIN_RAW
)

VALIDATION_SAMPLES = len(
    X_VALIDATION_RAW
)

TEST_SAMPLES = len(
    X_TEST_RAW
)

FULL_LOGO_FOLDS = len(
    np.unique(
        G_TRAIN
    )
)

print(
    "Training participants:",
    len(TRAIN_PERSONS)
)

print(
    "Validation participants:",
    len(VALIDATION_PERSONS)
)

print(
    "Testing participants:",
    len(TEST_PERSONS)
)

print()

print(
    "Training:",
    X_TRAIN_RAW.shape
)

print(
    "Validation:",
    X_VALIDATION_RAW.shape
)

print(
    "Testing:",
    X_TEST_RAW.shape
)

print(
    "GroupKFold folds:",
    GROUP_K_FOLDS
)

print(
    "Full LOGO folds:",
    FULL_LOGO_FOLDS
)

print(
    "RAM after split:",
    f"{memory_mb():.2f} MB"
)


# =============================================================================
# SPLIT INFORMATION
# =============================================================================

SPLIT_SUMMARY_TABLE = pd.DataFrame(
    [
        {
            "Split":
                "Training",

            "Participants":
                len(TRAIN_PERSONS),

            "Rows":
                TRAIN_SAMPLES,
        },

        {
            "Split":
                "Validation",

            "Participants":
                len(VALIDATION_PERSONS),

            "Rows":
                VALIDATION_SAMPLES,
        },

        {
            "Split":
                "Testing",

            "Participants":
                len(TEST_PERSONS),

            "Rows":
                TEST_SAMPLES,
        },
    ]
)

PARTICIPANT_SPLIT = {
    "training":
        [
            str(x)
            for x in TRAIN_PERSONS
        ],

    "validation":
        [
            str(x)
            for x in VALIDATION_PERSONS
        ],

    "testing":
        [
            str(x)
            for x in TEST_PERSONS
        ],
}


# =============================================================================
# RELEASE COMPLETE DATASET MATRICES
# =============================================================================

del X_ALL
del Y_ALL
del GROUPS_ALL

del train_mask
del validation_mask
del test_mask

gc.collect()


# =============================================================================
# FINAL TRAINING SCALER
#
# This is used only for the five final runs and final test.
#
# Cross validation and LOGO always fit a new scaler inside each fold.
# =============================================================================

FINAL_SCALER = StandardScaler()

X_TRAIN = FINAL_SCALER.fit_transform(
    X_TRAIN_RAW
).astype(
    np.float32,
    copy=False
)

X_VALIDATION = FINAL_SCALER.transform(
    X_VALIDATION_RAW
).astype(
    np.float32,
    copy=False
)

X_TEST = FINAL_SCALER.transform(
    X_TEST_RAW
).astype(
    np.float32,
    copy=False
)

X_TRAIN = np.ascontiguousarray(
    X_TRAIN
)

X_VALIDATION = np.ascontiguousarray(
    X_VALIDATION
)

X_TEST = np.ascontiguousarray(
    X_TEST
)


# =============================================================================
# FRESH GROUPKFOLD ITERATOR
# =============================================================================

def group_kfold_iterator():
    splitter = GroupKFold(
        n_splits=
            GROUP_K_FOLDS
    )

    return splitter.split(
        X_TRAIN_RAW,
        Y_TRAIN,
        G_TRAIN
    )


# =============================================================================
# FOLD SPECIFIC STANDARDIZATION
# =============================================================================

def prepare_fold(
    training_index,
    validation_index
):
    scaler = StandardScaler()

    X_fold_training = scaler.fit_transform(
        X_TRAIN_RAW[
            training_index
        ]
    ).astype(
        np.float32,
        copy=False
    )

    X_fold_validation = scaler.transform(
        X_TRAIN_RAW[
            validation_index
        ]
    ).astype(
        np.float32,
        copy=False
    )

    return (
        np.ascontiguousarray(
            X_fold_training
        ),

        np.ascontiguousarray(
            X_fold_validation
        ),
    )


# =============================================================================
# CLASS WEIGHTS
# =============================================================================

def get_class_weights(y):
    present_classes = np.unique(
        y
    )

    calculated = compute_class_weight(
        class_weight="balanced",
        classes=
            present_classes,
        y=y
    )

    weights = np.ones(
        NUMBER_CLASSES,
        dtype=np.float32
    )

    weights[
        present_classes
    ] = calculated.astype(
        np.float32
    )

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


# =============================================================================
# COMMON MINI BATCH GENERATOR
#
# Complete datasets remain in system memory.
# Only the current batch is copied to the GPU.
# =============================================================================

def batch_generator(
    X,
    y,
    batch_size,
    shuffle=False,
    rng=None
):
    number_samples = len(
        y
    )

    indices = np.arange(
        number_samples
    )

    if shuffle:
        if rng is None:
            np.random.shuffle(
                indices
            )
        else:
            rng.shuffle(
                indices
            )

    for start in range(
        0,
        number_samples,
        batch_size
    ):
        batch_indices = indices[
            start:
            start
            + batch_size
        ]

        X_numpy = np.ascontiguousarray(
            X[
                batch_indices
            ],
            dtype=np.float32
        )

        y_numpy = np.ascontiguousarray(
            y[
                batch_indices
            ],
            dtype=np.int64
        )

        X_batch = torch.from_numpy(
            X_numpy
        ).to(
            DEVICE
        )

        y_batch = torch.from_numpy(
            y_numpy
        ).to(
            DEVICE
        )

        yield (
            X_batch,
            y_batch
        )


# =============================================================================
# MODEL 1
# CNN
# =============================================================================

class CNN1D(nn.Module):

    def __init__(
        self,
        params
    ):
        super().__init__()

        n_blocks = int(
            params[
                "n_conv_blocks"
            ]
        )

        base_filters = int(
            params[
                "base_filters"
            ]
        )

        kernel_size = int(
            params[
                "kernel_size"
            ]
        )

        conv_dropout = float(
            params[
                "conv_dropout"
            ]
        )

        padding = (
            kernel_size
            // 2
        )

        layers = []

        input_channels = 1

        for block_index in range(
            n_blocks
        ):
            output_channels = (
                base_filters
                * (
                    2
                    ** block_index
                )
            )

            layers.extend(
                [
                    nn.Conv1d(
                        input_channels,
                        output_channels,
                        kernel_size=
                            kernel_size,
                        padding=
                            padding,
                        bias=False
                    ),

                    nn.BatchNorm1d(
                        output_channels
                    ),

                    nn.GELU(),

                    nn.Conv1d(
                        output_channels,
                        output_channels,
                        kernel_size=
                            kernel_size,
                        padding=
                            padding,
                        bias=False
                    ),

                    nn.BatchNorm1d(
                        output_channels
                    ),

                    nn.GELU(),

                    nn.MaxPool1d(
                        kernel_size=2
                    ),

                    nn.Dropout(
                        conv_dropout
                    ),
                ]
            )

            input_channels = (
                output_channels
            )

        self.features = nn.Sequential(
            *layers
        )

        self.global_pool = nn.AdaptiveAvgPool1d(
            1
        )

        dense_units = int(
            params[
                "dense_units"
            ]
        )

        dense_dropout = float(
            params[
                "dense_dropout"
            ]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                input_channels,
                dense_units
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                dense_units,
                NUMBER_CLASSES
            ),
        )

    def forward(
        self,
        x
    ):
        x = x.unsqueeze(1)

        x = self.features(
            x
        )

        x = self.global_pool(
            x
        )

        return self.classifier(
            x
        )


# =============================================================================
# POSITION ENCODING
# =============================================================================

class SinusoidalPositionEncoding(nn.Module):

    def __init__(
        self,
        dimension,
        max_length=512
    ):
        super().__init__()

        position = torch.arange(
            max_length,
            dtype=torch.float32
        ).unsqueeze(1)

        divisor = torch.exp(
            torch.arange(
                0,
                dimension,
                2,
                dtype=torch.float32
            )
            * (
                -math.log(
                    10000.0
                )
                / dimension
            )
        )

        encoding = torch.zeros(
            max_length,
            dimension,
            dtype=torch.float32
        )

        encoding[
            :,
            0::2
        ] = torch.sin(
            position
            * divisor
        )

        encoding[
            :,
            1::2
        ] = torch.cos(
            position
            * divisor
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
            persistent=False
        )

    def forward(
        self,
        x
    ):
        return (
            x
            + self.encoding[
                :,
                :x.size(1),
                :
            ].to(
                dtype=x.dtype
            )
        )


# =============================================================================
# CUSTOM MULTI HEAD ATTENTION
#
# Basic matrix operations are used for broad Intel XPU compatibility.
# =============================================================================

class CustomMultiHeadAttention(nn.Module):

    def __init__(
        self,
        dimension,
        number_heads,
        dropout
    ):
        super().__init__()

        if (
            dimension
            % number_heads
            != 0
        ):
            raise ValueError(
                "dimension must be divisible by number_heads"
            )

        self.dimension = (
            dimension
        )

        self.number_heads = (
            number_heads
        )

        self.head_dimension = (
            dimension
            // number_heads
        )

        self.scale = (
            self.head_dimension
            ** -0.5
        )

        self.query_projection = nn.Linear(
            dimension,
            dimension
        )

        self.key_projection = nn.Linear(
            dimension,
            dimension
        )

        self.value_projection = nn.Linear(
            dimension,
            dimension
        )

        self.output_projection = nn.Linear(
            dimension,
            dimension
        )

        self.attention_dropout = nn.Dropout(
            dropout
        )

        self.output_dropout = nn.Dropout(
            dropout
        )

    def split_heads(
        self,
        x
    ):
        batch_size = x.shape[0]
        sequence_length = x.shape[1]

        x = x.reshape(
            batch_size,
            sequence_length,
            self.number_heads,
            self.head_dimension
        )

        return x.transpose(
            1,
            2
        )

    def forward(
        self,
        query,
        key,
        value
    ):
        query = self.split_heads(
            self.query_projection(
                query
            )
        )

        key = self.split_heads(
            self.key_projection(
                key
            )
        )

        value = self.split_heads(
            self.value_projection(
                value
            )
        )

        scores = torch.matmul(
            query,
            key.transpose(
                -2,
                -1
            )
        )

        scores = (
            scores
            * self.scale
        )

        attention = torch.softmax(
            scores,
            dim=-1
        )

        attention = self.attention_dropout(
            attention
        )

        context = torch.matmul(
            attention,
            value
        )

        context = context.transpose(
            1,
            2
        ).contiguous()

        batch_size = context.shape[0]
        sequence_length = context.shape[1]

        context = context.reshape(
            batch_size,
            sequence_length,
            self.dimension
        )

        context = self.output_projection(
            context
        )

        return self.output_dropout(
            context
        )


# =============================================================================
# FEED FORWARD BLOCK
# =============================================================================

class FeedForward(nn.Module):

    def __init__(
        self,
        dimension,
        expansion,
        dropout
    ):
        super().__init__()

        hidden_dimension = (
            dimension
            * expansion
        )

        self.network = nn.Sequential(
            nn.Linear(
                dimension,
                hidden_dimension
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dimension,
                dimension
            ),

            nn.Dropout(
                dropout
            ),
        )

    def forward(
        self,
        x
    ):
        return self.network(
            x
        )


# =============================================================================
# CONFORMER FEED FORWARD
# =============================================================================

class ConformerFeedForward(nn.Module):

    def __init__(
        self,
        dimension,
        dropout
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                dimension,
                dimension * 4
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                dimension * 4,
                dimension
            ),

            nn.Dropout(
                dropout
            ),
        )

    def forward(
        self,
        x
    ):
        return self.network(
            x
        )


# =============================================================================
# CONFORMER CONVOLUTION MODULE
# =============================================================================

class ConformerConvModule(nn.Module):

    def __init__(
        self,
        dimension,
        kernel_size,
        dropout
    ):
        super().__init__()

        self.pointwise_in = nn.Conv1d(
            dimension,
            dimension * 2,
            kernel_size=1
        )

        self.depthwise = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=
                kernel_size,
            padding=
                kernel_size // 2,
            groups=
                dimension,
            bias=False
        )

        self.batch_norm = nn.BatchNorm1d(
            dimension
        )

        self.pointwise_out = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=1
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x
    ):
        x = x.transpose(
            1,
            2
        )

        x = self.pointwise_in(
            x
        )

        x = F.glu(
            x,
            dim=1
        )

        x = self.depthwise(
            x
        )

        x = self.batch_norm(
            x
        )

        x = F.silu(
            x
        )

        x = self.pointwise_out(
            x
        )

        x = self.dropout(
            x
        )

        return x.transpose(
            1,
            2
        )


# =============================================================================
# CONFORMER BLOCK
# =============================================================================

class ConformerBlock(nn.Module):

    def __init__(
        self,
        dimension,
        number_heads,
        conv_kernel,
        dropout
    ):
        super().__init__()

        self.ff1_norm = nn.LayerNorm(
            dimension
        )

        self.ff1 = ConformerFeedForward(
            dimension,
            dropout
        )

        self.attention_norm = nn.LayerNorm(
            dimension
        )

        self.self_attention = CustomMultiHeadAttention(
            dimension,
            number_heads,
            dropout
        )

        self.conv_norm = nn.LayerNorm(
            dimension
        )

        self.conv_module = ConformerConvModule(
            dimension,
            conv_kernel,
            dropout
        )

        self.ff2_norm = nn.LayerNorm(
            dimension
        )

        self.ff2 = ConformerFeedForward(
            dimension,
            dropout
        )

        self.final_norm = nn.LayerNorm(
            dimension
        )

    def forward(
        self,
        x
    ):
        x = (
            x
            + 0.5
            * self.ff1(
                self.ff1_norm(
                    x
                )
            )
        )

        normalized = self.attention_norm(
            x
        )

        x = (
            x
            + self.self_attention(
                normalized,
                normalized,
                normalized
            )
        )

        x = (
            x
            + self.conv_module(
                self.conv_norm(
                    x
                )
            )
        )

        x = (
            x
            + 0.5
            * self.ff2(
                self.ff2_norm(
                    x
                )
            )
        )

        return self.final_norm(
            x
        )


# =============================================================================
# TCN RESIDUAL BLOCK
# =============================================================================

class TCNResidualBlock(nn.Module):

    def __init__(
        self,
        channels,
        kernel_size,
        dilation,
        dropout
    ):
        super().__init__()

        padding = (
            dilation
            * (
                kernel_size
                - 1
            )
            // 2
        )

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=
                kernel_size,
            dilation=
                dilation,
            padding=
                padding,
            bias=False
        )

        self.bn1 = nn.BatchNorm1d(
            channels
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=
                kernel_size,
            dilation=
                dilation,
            padding=
                padding,
            bias=False
        )

        self.bn2 = nn.BatchNorm1d(
            channels
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x
    ):
        residual = x

        x = self.conv1(
            x
        )

        x = self.bn1(
            x
        )

        x = F.gelu(
            x
        )

        x = self.dropout(
            x
        )

        x = self.conv2(
            x
        )

        x = self.bn2(
            x
        )

        x = F.gelu(
            x
        )

        x = self.dropout(
            x
        )

        return (
            x
            + residual
        )


# =============================================================================
# MODEL 2
# EEG CONFORMER + 1D TCN
# DUAL STREAM CROSS ATTENTION
# =============================================================================

class EEGConformerTCNCrossAttention(nn.Module):

    PATCH_KERNEL_SIZE = 7
    PATCH_STRIDE = 4

    def __init__(
        self,
        params
    ):
        super().__init__()

        dimension = int(
            params[
                "d_model"
            ]
        )

        number_heads = int(
            params[
                "num_heads"
            ]
        )

        n_conformer_blocks = int(
            params[
                "n_conformer_blocks"
            ]
        )

        n_tcn_blocks = int(
            params[
                "n_tcn_blocks"
            ]
        )

        conformer_kernel = int(
            params[
                "conformer_kernel"
            ]
        )

        tcn_kernel = int(
            params[
                "tcn_kernel"
            ]
        )

        dropout = float(
            params[
                "dropout"
            ]
        )

        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )

        self.conformer_stem = nn.Sequential(
            nn.Conv1d(
                1,
                dimension,
                kernel_size=
                    self.PATCH_KERNEL_SIZE,
                stride=
                    self.PATCH_STRIDE,
                padding=
                    self.PATCH_KERNEL_SIZE // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                dimension
            ),

            nn.GELU(),
        )

        self.position_encoding = (
            SinusoidalPositionEncoding(
                dimension,
                max_length=512
            )
        )

        self.conformer_blocks = nn.ModuleList(
            [
                ConformerBlock(
                    dimension=
                        dimension,
                    number_heads=
                        number_heads,
                    conv_kernel=
                        conformer_kernel,
                    dropout=
                        dropout
                )

                for _ in range(
                    n_conformer_blocks
                )
            ]
        )

        self.tcn_stem = nn.Sequential(
            nn.Conv1d(
                1,
                dimension,
                kernel_size=
                    self.PATCH_KERNEL_SIZE,
                stride=
                    self.PATCH_STRIDE,
                padding=
                    self.PATCH_KERNEL_SIZE // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                dimension
            ),

            nn.GELU(),
        )

        self.tcn_blocks = nn.ModuleList(
            [
                TCNResidualBlock(
                    channels=
                        dimension,
                    kernel_size=
                        tcn_kernel,
                    dilation=
                        2 ** block_index,
                    dropout=
                        dropout
                )

                for block_index in range(
                    n_tcn_blocks
                )
            ]
        )

        self.conformer_query_norm = nn.LayerNorm(
            dimension
        )

        self.tcn_context_norm = nn.LayerNorm(
            dimension
        )

        self.tcn_query_norm = nn.LayerNorm(
            dimension
        )

        self.conformer_context_norm = nn.LayerNorm(
            dimension
        )

        self.conformer_queries_tcn = (
            CustomMultiHeadAttention(
                dimension,
                number_heads,
                dropout
            )
        )

        self.tcn_queries_conformer = (
            CustomMultiHeadAttention(
                dimension,
                number_heads,
                dropout
            )
        )

        self.conformer_fusion_norm = nn.LayerNorm(
            dimension
        )

        self.tcn_fusion_norm = nn.LayerNorm(
            dimension
        )

        self.cross_dropout = nn.Dropout(
            dropout
        )

        fusion_dimension = (
            dimension * 4
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            ),
        )

    def forward(
        self,
        x
    ):
        x = x.unsqueeze(1)

        conformer = self.conformer_stem(
            x
        )

        conformer = conformer.transpose(
            1,
            2
        )

        conformer = self.position_encoding(
            conformer
        )

        for block in self.conformer_blocks:
            conformer = block(
                conformer
            )

        tcn = self.tcn_stem(
            x
        )

        for block in self.tcn_blocks:
            tcn = block(
                tcn
            )

        tcn = tcn.transpose(
            1,
            2
        )

        conformer_query = (
            self.conformer_query_norm(
                conformer
            )
        )

        tcn_context = (
            self.tcn_context_norm(
                tcn
            )
        )

        conformer_cross = (
            self.conformer_queries_tcn(
                conformer_query,
                tcn_context,
                tcn_context
            )
        )

        conformer = self.conformer_fusion_norm(
            conformer
            + self.cross_dropout(
                conformer_cross
            )
        )

        tcn_query = self.tcn_query_norm(
            tcn
        )

        conformer_context = (
            self.conformer_context_norm(
                conformer
            )
        )

        tcn_cross = (
            self.tcn_queries_conformer(
                tcn_query,
                conformer_context,
                conformer_context
            )
        )

        tcn = self.tcn_fusion_norm(
            tcn
            + self.cross_dropout(
                tcn_cross
            )
        )

        conformer_vector = torch.mean(
            conformer,
            dim=1
        )

        tcn_vector = torch.mean(
            tcn,
            dim=1
        )

        difference = torch.abs(
            conformer_vector
            - tcn_vector
        )

        interaction = (
            conformer_vector
            * tcn_vector
        )

        fused = torch.cat(
            [
                conformer_vector,
                tcn_vector,
                difference,
                interaction,
            ],
            dim=1
        )

        return self.classifier(
            fused
        )


# =============================================================================
# TRANSFORMER BLOCK
# =============================================================================

class TransformerBlock(nn.Module):

    def __init__(
        self,
        dimension,
        number_heads,
        expansion,
        dropout
    ):
        super().__init__()

        self.attention_norm = nn.LayerNorm(
            dimension
        )

        self.attention = CustomMultiHeadAttention(
            dimension,
            number_heads,
            dropout
        )

        self.ffn_norm = nn.LayerNorm(
            dimension
        )

        self.ffn = FeedForward(
            dimension,
            expansion,
            dropout
        )

    def forward(
        self,
        x
    ):
        normalized = self.attention_norm(
            x
        )

        x = (
            x
            + self.attention(
                normalized,
                normalized,
                normalized
            )
        )

        x = (
            x
            + self.ffn(
                self.ffn_norm(
                    x
                )
            )
        )

        return x


# =============================================================================
# FEATURE TOKENIZER
# =============================================================================

class FeatureTokenizer(nn.Module):

    def __init__(
        self,
        dimension,
        kernel_size,
        stride,
        dropout
    ):
        super().__init__()

        self.tokenizer = nn.Sequential(
            nn.Conv1d(
                1,
                dimension,
                kernel_size=
                    kernel_size,
                stride=
                    stride,
                padding=
                    kernel_size // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                dimension
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),
        )

        self.position = SinusoidalPositionEncoding(
            dimension,
            max_length=512
        )

    def forward(
        self,
        x
    ):
        x = x.unsqueeze(
            1
        )

        x = self.tokenizer(
            x
        )

        x = x.transpose(
            1,
            2
        )

        return self.position(
            x
        )


# =============================================================================
# CROSS MODAL ATTENTION BLOCK
# =============================================================================

class CrossModalAttentionLayer(nn.Module):

    def __init__(
        self,
        dimension,
        number_heads,
        expansion,
        dropout,
        number_modalities
    ):
        super().__init__()

        self.number_modalities = (
            number_modalities
        )

        self.query_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )

                for _ in range(
                    number_modalities
                )
            ]
        )

        self.context_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )

                for _ in range(
                    number_modalities
                )
            ]
        )

        self.attentions = nn.ModuleList(
            [
                CustomMultiHeadAttention(
                    dimension,
                    number_heads,
                    dropout
                )

                for _ in range(
                    number_modalities
                )
            ]
        )

        self.output_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )

                for _ in range(
                    number_modalities
                )
            ]
        )

        self.ffn_norms = nn.ModuleList(
            [
                nn.LayerNorm(
                    dimension
                )

                for _ in range(
                    number_modalities
                )
            ]
        )

        self.ffns = nn.ModuleList(
            [
                FeedForward(
                    dimension,
                    expansion,
                    dropout
                )

                for _ in range(
                    number_modalities
                )
            ]
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        streams
    ):
        original_streams = streams

        updated_streams = []

        for modality_index in range(
            self.number_modalities
        ):
            query_stream = original_streams[
                modality_index
            ]

            other_streams = [
                original_streams[
                    other_index
                ]

                for other_index in range(
                    self.number_modalities
                )

                if other_index
                != modality_index
            ]

            context = torch.cat(
                other_streams,
                dim=1
            )

            normalized_query = (
                self.query_norms[
                    modality_index
                ](
                    query_stream
                )
            )

            normalized_context = (
                self.context_norms[
                    modality_index
                ](
                    context
                )
            )

            cross_output = (
                self.attentions[
                    modality_index
                ](
                    normalized_query,
                    normalized_context,
                    normalized_context
                )
            )

            stream = (
                self.output_norms[
                    modality_index
                ](
                    query_stream
                    + self.dropout(
                        cross_output
                    )
                )
            )

            stream = (
                stream
                + self.ffns[
                    modality_index
                ](
                    self.ffn_norms[
                        modality_index
                    ](
                        stream
                    )
                )
            )

            updated_streams.append(
                stream
            )

        return updated_streams


# =============================================================================
# BASIC MODALITY STANDARD DEVIATION
#
# Implemented with basic operations for GPU compatibility.
# =============================================================================

def modality_standard_deviation(
    stacked_modalities
):
    mean = torch.mean(
        stacked_modalities,
        dim=1,
        keepdim=True
    )

    variance = torch.mean(
        (
            stacked_modalities
            - mean
        ) ** 2,
        dim=1
    )

    return torch.sqrt(
        variance
        + 1e-6
    )


# =============================================================================
# MODEL 3
# CROSS ATTENTION MULTIMODAL TRANSFORMER
# =============================================================================

class CrossAttentionMultimodalTransformer(
    nn.Module
):

    def __init__(
        self,
        params
    ):
        super().__init__()

        self.modality_names = list(
            MODALITY_INDICES.keys()
        )

        self.number_modalities = len(
            self.modality_names
        )

        dimension = int(
            params[
                "d_model"
            ]
        )

        number_heads = int(
            params[
                "num_heads"
            ]
        )

        self_layers = int(
            params[
                "self_layers"
            ]
        )

        cross_layers = int(
            params[
                "cross_layers"
            ]
        )

        expansion = int(
            params[
                "ffn_expansion"
            ]
        )

        tokenizer_kernel = int(
            params[
                "tokenizer_kernel"
            ]
        )

        tokenizer_stride = int(
            params[
                "tokenizer_stride"
            ]
        )

        dropout = float(
            params[
                "dropout"
            ]
        )

        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )

        self.modality_buffer_names = []

        for modality_number, modality_name in enumerate(
            self.modality_names
        ):
            buffer_name = (
                f"modality_indices_{modality_number}"
            )

            self.register_buffer(
                buffer_name,
                torch.tensor(
                    MODALITY_INDICES[
                        modality_name
                    ],
                    dtype=torch.long
                ),
                persistent=True
            )

            self.modality_buffer_names.append(
                buffer_name
            )

        self.tokenizers = nn.ModuleList(
            [
                FeatureTokenizer(
                    dimension=
                        dimension,
                    kernel_size=
                        tokenizer_kernel,
                    stride=
                        tokenizer_stride,
                    dropout=
                        dropout
                )

                for _ in range(
                    self.number_modalities
                )
            ]
        )

        self.modality_embeddings = nn.Parameter(
            torch.randn(
                self.number_modalities,
                1,
                dimension
            )
            * 0.02
        )

        self.self_encoders = nn.ModuleList()

        for _ in range(
            self.number_modalities
        ):
            encoder = nn.ModuleList(
                [
                    TransformerBlock(
                        dimension=
                            dimension,
                        number_heads=
                            number_heads,
                        expansion=
                            expansion,
                        dropout=
                            dropout
                    )

                    for _ in range(
                        self_layers
                    )
                ]
            )

            self.self_encoders.append(
                encoder
            )

        self.cross_attention_layers = (
            nn.ModuleList(
                [
                    CrossModalAttentionLayer(
                        dimension=
                            dimension,
                        number_heads=
                            number_heads,
                        expansion=
                            expansion,
                        dropout=
                            dropout,
                        number_modalities=
                            self.number_modalities
                    )

                    for _ in range(
                        cross_layers
                    )
                ]
            )
        )

        fusion_dimension = (
            dimension
            * (
                self.number_modalities
                + 2
            )
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            ),
        )

    def forward(
        self,
        x
    ):
        streams = []

        for modality_index in range(
            self.number_modalities
        ):
            column_indices = getattr(
                self,
                self.modality_buffer_names[
                    modality_index
                ]
            )

            modality_input = torch.index_select(
                x,
                dim=1,
                index=
                    column_indices
            )

            stream = self.tokenizers[
                modality_index
            ](
                modality_input
            )

            modality_embedding = (
                self.modality_embeddings[
                    modality_index
                ].unsqueeze(0)
            )

            stream = (
                stream
                + modality_embedding
            )

            for block in self.self_encoders[
                modality_index
            ]:
                stream = block(
                    stream
                )

            streams.append(
                stream
            )

        for cross_layer in self.cross_attention_layers:
            streams = cross_layer(
                streams
            )

        modality_vectors = [
            torch.mean(
                stream,
                dim=1
            )

            for stream in streams
        ]

        stacked = torch.stack(
            modality_vectors,
            dim=1
        )

        modality_mean = torch.mean(
            stacked,
            dim=1
        )

        modality_std = modality_standard_deviation(
            stacked
        )

        fused = torch.cat(
            modality_vectors
            + [
                modality_mean,
                modality_std,
            ],
            dim=1
        )

        return self.classifier(
            fused
        )


# =============================================================================
# GAUSSIAN INPUT NOISE
# =============================================================================

class GaussianNoise(nn.Module):

    def __init__(
        self,
        standard_deviation
    ):
        super().__init__()

        self.standard_deviation = float(
            standard_deviation
        )

    def forward(
        self,
        x
    ):
        if (
            self.training
            and
            self.standard_deviation > 0
        ):
            return (
                x
                + torch.randn_like(
                    x
                )
                * self.standard_deviation
            )

        return x


# =============================================================================
# MANUAL STANDARD LSTM
#
# Native nn.LSTM is intentionally not used because the current Intel XPU
# backend on this system does not support the training path.
# =============================================================================

class XPUManualLSTM(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers=1,
        dropout=0.0
    ):
        super().__init__()

        self.input_size = int(
            input_size
        )

        self.hidden_size = int(
            hidden_size
        )

        self.num_layers = int(
            num_layers
        )

        self.dropout_probability = float(
            dropout
        )

        self.input_linears = nn.ModuleList()
        self.hidden_linears = nn.ModuleList()

        for layer_index in range(
            self.num_layers
        ):
            current_input_size = (
                self.input_size
                if layer_index == 0
                else self.hidden_size
            )

            self.input_linears.append(
                nn.Linear(
                    current_input_size,
                    4
                    * self.hidden_size,
                    bias=True
                )
            )

            self.hidden_linears.append(
                nn.Linear(
                    self.hidden_size,
                    4
                    * self.hidden_size,
                    bias=False
                )
            )

        self.inter_layer_dropout = nn.Dropout(
            self.dropout_probability
        )

        self.reset_parameters()

    def reset_parameters(
        self
    ):
        for input_linear in self.input_linears:
            nn.init.xavier_uniform_(
                input_linear.weight
            )

            nn.init.zeros_(
                input_linear.bias
            )

            with torch.no_grad():
                input_linear.bias[
                    self.hidden_size:
                    2 * self.hidden_size
                ].fill_(
                    1.0
                )

        for hidden_linear in self.hidden_linears:
            nn.init.orthogonal_(
                hidden_linear.weight
            )

    def forward(
        self,
        x
    ):
        batch_size = x.shape[0]
        sequence_length = x.shape[1]

        layer_input = x

        final_hidden_states = []
        final_cell_states = []

        for layer_index in range(
            self.num_layers
        ):
            hidden_state = torch.zeros(
                batch_size,
                self.hidden_size,
                dtype=x.dtype,
                device=x.device
            )

            cell_state = torch.zeros(
                batch_size,
                self.hidden_size,
                dtype=x.dtype,
                device=x.device
            )

            input_projection = (
                self.input_linears[
                    layer_index
                ](
                    layer_input
                )
            )

            timestep_outputs = []

            for timestep in range(
                sequence_length
            ):
                gates = (
                    input_projection[
                        :,
                        timestep,
                        :
                    ]
                    +
                    self.hidden_linears[
                        layer_index
                    ](
                        hidden_state
                    )
                )

                (
                    input_gate,
                    forget_gate,
                    candidate_gate,
                    output_gate,
                ) = gates.chunk(
                    4,
                    dim=1
                )

                input_gate = torch.sigmoid(
                    input_gate
                )

                forget_gate = torch.sigmoid(
                    forget_gate
                )

                candidate_gate = torch.tanh(
                    candidate_gate
                )

                output_gate = torch.sigmoid(
                    output_gate
                )

                cell_state = (
                    forget_gate
                    * cell_state
                    +
                    input_gate
                    * candidate_gate
                )

                hidden_state = (
                    output_gate
                    * torch.tanh(
                        cell_state
                    )
                )

                timestep_outputs.append(
                    hidden_state
                )

            layer_output = torch.stack(
                timestep_outputs,
                dim=1
            )

            if (
                layer_index
                < self.num_layers - 1
            ):
                layer_output = (
                    self.inter_layer_dropout(
                        layer_output
                    )
                )

            layer_input = (
                layer_output
            )

            final_hidden_states.append(
                hidden_state
            )

            final_cell_states.append(
                cell_state
            )

        hidden_stack = torch.stack(
            final_hidden_states,
            dim=0
        )

        cell_stack = torch.stack(
            final_cell_states,
            dim=0
        )

        return (
            layer_input,
            (
                hidden_stack,
                cell_stack,
            )
        )


# =============================================================================
# MODALITY CNN LSTM ENCODER
# =============================================================================

class ModalityCNNLSTMEncoder(nn.Module):

    def __init__(
        self,
        params
    ):
        super().__init__()

        base_channels = int(
            params[
                "base_channels"
            ]
        )

        kernel_size = int(
            params[
                "kernel_size"
            ]
        )

        lstm_hidden = int(
            params[
                "lstm_hidden"
            ]
        )

        lstm_layers = int(
            params[
                "lstm_layers"
            ]
        )

        conv_dropout = float(
            params[
                "conv_dropout"
            ]
        )

        lstm_dropout = float(
            params[
                "lstm_dropout"
            ]
        )

        padding = (
            kernel_size
            // 2
        )

        self.cnn = nn.Sequential(
            nn.Conv1d(
                1,
                base_channels,
                kernel_size=
                    kernel_size,
                padding=
                    padding,
                bias=False
            ),

            nn.BatchNorm1d(
                base_channels
            ),

            nn.GELU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Dropout1d(
                conv_dropout
            ),

            nn.Conv1d(
                base_channels,
                base_channels * 2,
                kernel_size=
                    kernel_size,
                padding=
                    padding,
                bias=False
            ),

            nn.BatchNorm1d(
                base_channels * 2
            ),

            nn.GELU(),

            nn.MaxPool1d(
                kernel_size=2
            ),

            nn.Dropout1d(
                conv_dropout
            ),
        )

        self.lstm = XPUManualLSTM(
            input_size=
                base_channels * 2,

            hidden_size=
                lstm_hidden,

            num_layers=
                lstm_layers,

            dropout=
                lstm_dropout
        )

        self.output_dimension = (
            lstm_hidden * 2
        )

        self.output_norm = nn.LayerNorm(
            self.output_dimension
        )

        self.output_dropout = nn.Dropout(
            lstm_dropout
        )

    def forward(
        self,
        x
    ):
        x = x.unsqueeze(
            1
        )

        x = self.cnn(
            x
        )

        x = x.transpose(
            1,
            2
        ).contiguous()

        sequence_output, _ = self.lstm(
            x
        )

        final_state = sequence_output[
            :,
            -1,
            :
        ]

        mean_state = torch.mean(
            sequence_output,
            dim=1
        )

        representation = torch.cat(
            [
                final_state,
                mean_state,
            ],
            dim=1
        )

        representation = self.output_norm(
            representation
        )

        return self.output_dropout(
            representation
        )


# =============================================================================
# MODEL 4
# MULTIMODAL CNN LSTM
# =============================================================================

class MultimodalCNNLSTM(nn.Module):

    def __init__(
        self,
        params
    ):
        super().__init__()

        self.modality_names = list(
            MODALITY_INDICES.keys()
        )

        self.number_modalities = len(
            self.modality_names
        )

        self.modality_buffer_names = []

        for modality_number, modality_name in enumerate(
            self.modality_names
        ):
            buffer_name = (
                f"modality_indices_{modality_number}"
            )

            self.register_buffer(
                buffer_name,
                torch.tensor(
                    MODALITY_INDICES[
                        modality_name
                    ],
                    dtype=torch.long
                ),
                persistent=True
            )

            self.modality_buffer_names.append(
                buffer_name
            )

        self.input_noise = GaussianNoise(
            params[
                "input_noise"
            ]
        )

        self.input_dropout = nn.Dropout(
            float(
                params[
                    "input_dropout"
                ]
            )
        )

        self.encoders = nn.ModuleList(
            [
                ModalityCNNLSTMEncoder(
                    params
                )

                for _ in range(
                    self.number_modalities
                )
            ]
        )

        stream_dimension = (
            self.encoders[
                0
            ].output_dimension
        )

        gate_hidden = max(
            16,
            stream_dimension // 2
        )

        dense_dropout = float(
            params[
                "dense_dropout"
            ]
        )

        self.modality_gate = nn.Sequential(
            nn.Linear(
                stream_dimension,
                gate_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                gate_hidden,
                1
            ),
        )

        fusion_dimension = (
            stream_dimension
            * (
                self.number_modalities
                + 2
            )
        )

        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            ),
        )

    def forward(
        self,
        x
    ):
        x = self.input_noise(
            x
        )

        x = self.input_dropout(
            x
        )

        modality_vectors = []

        for modality_index in range(
            self.number_modalities
        ):
            column_indices = getattr(
                self,
                self.modality_buffer_names[
                    modality_index
                ]
            )

            modality_input = torch.index_select(
                x,
                dim=1,
                index=
                    column_indices
            )

            modality_vector = self.encoders[
                modality_index
            ](
                modality_input
            )

            modality_vectors.append(
                modality_vector
            )

        stacked = torch.stack(
            modality_vectors,
            dim=1
        )

        gate_logits = torch.cat(
            [
                self.modality_gate(
                    vector
                )

                for vector in modality_vectors
            ],
            dim=1
        )

        modality_weights = torch.softmax(
            gate_logits,
            dim=1
        )

        weighted_representation = torch.sum(
            stacked
            * modality_weights.unsqueeze(
                -1
            ),
            dim=1
        )

        modality_std = modality_standard_deviation(
            stacked
        )

        fused = torch.cat(
            modality_vectors
            + [
                weighted_representation,
                modality_std,
            ],
            dim=1
        )

        return self.classifier(
            fused
        )


# =============================================================================
# MODEL BUILDERS
# =============================================================================

def build_cnn(params):
    return CNN1D(
        params
    )


def build_conformer_tcn(params):
    return EEGConformerTCNCrossAttention(
        params
    )


def build_multimodal_transformer(params):
    return CrossAttentionMultimodalTransformer(
        params
    )


def build_multimodal_cnn_lstm(params):
    return MultimodalCNNLSTM(
        params
    )


# =============================================================================
# OPTUNA SEARCH SPACES
# =============================================================================

def search_cnn(trial):
    return {
        "n_conv_blocks":
            trial.suggest_int(
                "n_conv_blocks",
                2,
                3
            ),

        "base_filters":
            trial.suggest_categorical(
                "base_filters",
                [
                    32,
                    64,
                ]
            ),

        "kernel_size":
            trial.suggest_categorical(
                "kernel_size",
                [
                    3,
                    5,
                    7,
                ]
            ),

        "conv_dropout":
            trial.suggest_float(
                "conv_dropout",
                0.10,
                0.35
            ),

        "dense_units":
            trial.suggest_categorical(
                "dense_units",
                [
                    64,
                    128,
                    256,
                ]
            ),

        "dense_dropout":
            trial.suggest_float(
                "dense_dropout",
                0.15,
                0.50
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                1e-4,
                3e-3,
                log=True
            ),

        "weight_decay":
            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":
            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.10
            ),
    }


def search_conformer_tcn(trial):
    return {
        "d_model":
            trial.suggest_categorical(
                "d_model",
                [
                    64,
                    96,
                ]
            ),

        "num_heads":
            trial.suggest_categorical(
                "num_heads",
                [
                    4,
                    8,
                ]
            ),

        "n_conformer_blocks":
            trial.suggest_int(
                "n_conformer_blocks",
                1,
                2
            ),

        "n_tcn_blocks":
            trial.suggest_int(
                "n_tcn_blocks",
                2,
                3
            ),

        "conformer_kernel":
            trial.suggest_categorical(
                "conformer_kernel",
                [
                    7,
                    15,
                    31,
                ]
            ),

        "tcn_kernel":
            trial.suggest_categorical(
                "tcn_kernel",
                [
                    3,
                    5,
                ]
            ),

        "dropout":
            trial.suggest_float(
                "dropout",
                0.10,
                0.35
            ),

        "classifier_hidden":
            trial.suggest_categorical(
                "classifier_hidden",
                [
                    128,
                    256,
                ]
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":
            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":
            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.08
            ),
    }


def search_multimodal_transformer(trial):
    return {
        "d_model":
            trial.suggest_categorical(
                "d_model",
                [
                    64,
                    96,
                ]
            ),

        "num_heads":
            trial.suggest_categorical(
                "num_heads",
                [
                    4,
                    8,
                ]
            ),

        "self_layers":
            trial.suggest_int(
                "self_layers",
                1,
                2
            ),

        "cross_layers":
            trial.suggest_int(
                "cross_layers",
                1,
                2
            ),

        "ffn_expansion":
            trial.suggest_categorical(
                "ffn_expansion",
                [
                    2,
                    4,
                ]
            ),

        "tokenizer_kernel":
            trial.suggest_categorical(
                "tokenizer_kernel",
                [
                    3,
                    5,
                    7,
                ]
            ),

        "tokenizer_stride":
            trial.suggest_categorical(
                "tokenizer_stride",
                [
                    2,
                    4,
                ]
            ),

        "dropout":
            trial.suggest_float(
                "dropout",
                0.10,
                0.35
            ),

        "classifier_hidden":
            trial.suggest_categorical(
                "classifier_hidden",
                [
                    128,
                    256,
                ]
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":
            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":
            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.08
            ),
    }


def search_multimodal_cnn_lstm(trial):
    return {
        "base_channels":
            trial.suggest_categorical(
                "base_channels",
                [
                    32,
                    48,
                    64,
                ]
            ),

        "kernel_size":
            trial.suggest_categorical(
                "kernel_size",
                [
                    3,
                    5,
                    7,
                ]
            ),

        "lstm_hidden":
            trial.suggest_categorical(
                "lstm_hidden",
                [
                    32,
                    64,
                    96,
                ]
            ),

        "lstm_layers":
            trial.suggest_int(
                "lstm_layers",
                1,
                2
            ),

        "conv_dropout":
            trial.suggest_float(
                "conv_dropout",
                0.15,
                0.40
            ),

        "lstm_dropout":
            trial.suggest_float(
                "lstm_dropout",
                0.15,
                0.40
            ),

        "input_dropout":
            trial.suggest_float(
                "input_dropout",
                0.05,
                0.20
            ),

        "input_noise":
            trial.suggest_float(
                "input_noise",
                0.0,
                0.05
            ),

        "classifier_hidden":
            trial.suggest_categorical(
                "classifier_hidden",
                [
                    64,
                    128,
                ]
            ),

        "dense_dropout":
            trial.suggest_float(
                "dense_dropout",
                0.25,
                0.50
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":
            trial.suggest_float(
                "weight_decay",
                1e-6,
                2e-3,
                log=True
            ),

        "label_smoothing":
            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.10
            ),
    }


# =============================================================================
# SMOKE TEST PARAMETERS
# =============================================================================

CNN_SMOKE_PARAMS = {
    "n_conv_blocks": 2,
    "base_filters": 32,
    "kernel_size": 5,
    "conv_dropout": 0.20,
    "dense_units": 64,
    "dense_dropout": 0.30,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
}

CONFORMER_SMOKE_PARAMS = {
    "d_model": 64,
    "num_heads": 4,
    "n_conformer_blocks": 1,
    "n_tcn_blocks": 2,
    "conformer_kernel": 15,
    "tcn_kernel": 3,
    "dropout": 0.20,
    "classifier_hidden": 128,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
}

TRANSFORMER_SMOKE_PARAMS = {
    "d_model": 64,
    "num_heads": 4,
    "self_layers": 1,
    "cross_layers": 1,
    "ffn_expansion": 2,
    "tokenizer_kernel": 5,
    "tokenizer_stride": 2,
    "dropout": 0.20,
    "classifier_hidden": 128,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
}

CNN_LSTM_SMOKE_PARAMS = {
    "base_channels": 32,
    "kernel_size": 5,
    "lstm_hidden": 64,
    "lstm_layers": 1,
    "conv_dropout": 0.25,
    "lstm_dropout": 0.25,
    "input_dropout": 0.10,
    "input_noise": 0.02,
    "classifier_hidden": 128,
    "dense_dropout": 0.35,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
}


# =============================================================================
# ALGORITHM SPECIFICATIONS
# =============================================================================

ALGORITHM_SPECS = {
    "CNN": {
        "display_name":
            "CNN",

        "folder":
            "CNN_Results",

        "version":
            "med_cnn_v2",

        "builder":
            build_cnn,

        "search":
            search_cnn,

        "smoke_params":
            CNN_SMOKE_PARAMS,

        "batch_size":
            2048,

        "max_epochs":
            50,

        "patience":
            6,

        "uses_modalities":
            False,
    },

    "EEG_CONFORMER_TCN": {
        "display_name":
            "EEG Conformer + 1D TCN Dual Stream Cross Attention",

        "folder":
            "EEG_Conformer_TCN_CrossAttention_Results",

        "version":
            "med_conformer_tcn_v2",

        "builder":
            build_conformer_tcn,

        "search":
            search_conformer_tcn,

        "smoke_params":
            CONFORMER_SMOKE_PARAMS,

        "batch_size":
            512,

        "max_epochs":
            40,

        "patience":
            5,

        "uses_modalities":
            False,
    },

    "CROSS_ATTENTION_MULTIMODAL_TRANSFORMER": {
        "display_name":
            "Cross Attention Multimodal Transformer",

        "folder":
            "Cross_Attention_Multimodal_Transformer_Results",

        "version":
            "med_multimodal_transformer_v2",

        "builder":
            build_multimodal_transformer,

        "search":
            search_multimodal_transformer,

        "smoke_params":
            TRANSFORMER_SMOKE_PARAMS,

        "batch_size":
            512,

        "max_epochs":
            40,

        "patience":
            5,

        "uses_modalities":
            True,
    },

    "MULTIMODAL_CNN_LSTM": {
        "display_name":
            "Multimodal CNN LSTM",

        "folder":
            "Multimodal_CNN_LSTM_Results",

        "version":
            "med_multimodal_cnn_manual_lstm_v3",

        "builder":
            build_multimodal_cnn_lstm,

        "search":
            search_multimodal_cnn_lstm,

        "smoke_params":
            CNN_LSTM_SMOKE_PARAMS,

        "batch_size":
            1024,

        "max_epochs":
            50,

        "patience":
            6,

        "uses_modalities":
            True,
    },
}


# =============================================================================
# INSTANTIATE MODEL
# =============================================================================

def instantiate_model(
    spec,
    params,
    seed
):
    set_seed(seed)

    model = spec[
        "builder"
    ](
        params
    )

    return model.to(
        DEVICE
    )


# =============================================================================
# PARAMETER COUNT
# =============================================================================

def count_trainable_parameters(
    model
):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


# =============================================================================
# TRAIN ONE EPOCH
# =============================================================================

def train_one_epoch(
    model,
    X,
    y,
    optimizer,
    criterion,
    batch_size,
    rng
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for X_batch, y_batch in batch_generator(
        X,
        y,
        batch_size=
            batch_size,
        shuffle=
            True,
        rng=
            rng
    ):
        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            X_batch
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )

        optimizer.step()

        current_batch_size = int(
            y_batch.shape[0]
        )

        total_loss += (
            float(
                loss.detach().item()
            )
            * current_batch_size
        )

        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )

        total_samples += (
            current_batch_size
        )

        del X_batch
        del y_batch
        del logits
        del loss

    synchronize()

    return (
        total_loss
        / total_samples,

        total_correct
        / total_samples,
    )


# =============================================================================
# MODEL EVALUATION
# =============================================================================

@torch.inference_mode()
def evaluate_model(
    model,
    X,
    y,
    batch_size,
    criterion=None,
    return_probabilities=False
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    probability_parts = []

    for X_batch, y_batch in batch_generator(
        X,
        y,
        batch_size=
            batch_size,
        shuffle=
            False
    ):
        logits = model(
            X_batch
        )

        current_batch_size = int(
            y_batch.shape[0]
        )

        if criterion is not None:
            loss = criterion(
                logits,
                y_batch
            )

            total_loss += (
                float(
                    loss.item()
                )
                * current_batch_size
            )

        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )

        total_samples += (
            current_batch_size
        )

        if return_probabilities:
            probability_parts.append(
                torch.softmax(
                    logits,
                    dim=1
                )
                .cpu()
            )

        del X_batch
        del y_batch
        del logits

    synchronize()

    accuracy = (
        total_correct
        / total_samples
    )

    if criterion is None:
        mean_loss = np.nan
    else:
        mean_loss = (
            total_loss
            / total_samples
        )

    if return_probabilities:
        probabilities = (
            torch.cat(
                probability_parts,
                dim=0
            )
            .numpy()
        )

        return (
            mean_loss,
            accuracy,
            probabilities,
        )

    return (
        mean_loss,
        accuracy,
    )


# =============================================================================
# FIT MODEL
# =============================================================================

def fit_model(
    spec,
    params,
    X_training,
    y_training,
    X_validation_data=None,
    y_validation_data=None,
    seed=SEED,
    epochs=None,
    use_early_stopping=True,
    verbose=False
):
    clear_device()

    if epochs is None:
        epochs = spec[
            "max_epochs"
        ]

    batch_size = spec[
        "batch_size"
    ]

    patience = spec[
        "patience"
    ]

    set_seed(seed)

    training_rng = np.random.default_rng(
        seed
    )

    model = instantiate_model(
        spec,
        params,
        seed
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),

        lr=float(
            params[
                "learning_rate"
            ]
        ),

        weight_decay=float(
            params[
                "weight_decay"
            ]
        )
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=
                LR_SCHEDULER_PATIENCE,
            min_lr=1e-6
        )
    )

    class_weights = get_class_weights(
        y_training
    )

    criterion = nn.CrossEntropyLoss(
        weight=
            class_weights,

        label_smoothing=
            float(
                params[
                    "label_smoothing"
                ]
            )
    )

    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": [],
    }

    best_loss = float("inf")
    best_state = None
    best_epoch = 1
    patience_counter = 0

    for epoch in range(
        1,
        epochs + 1
    ):
        (
            training_loss,
            training_accuracy,

        ) = train_one_epoch(
            model,
            X_training,
            y_training,
            optimizer,
            criterion,
            batch_size,
            training_rng
        )

        if X_validation_data is not None:
            (
                validation_loss,
                validation_accuracy,

            ) = evaluate_model(
                model,
                X_validation_data,
                y_validation_data,
                batch_size,
                criterion=
                    criterion,
                return_probabilities=
                    False
            )

            monitored_loss = (
                validation_loss
            )

        else:
            validation_loss = np.nan
            validation_accuracy = np.nan

            monitored_loss = (
                training_loss
            )

        scheduler.step(
            monitored_loss
        )

        current_lr = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )

        history[
            "loss"
        ].append(
            training_loss
        )

        history[
            "accuracy"
        ].append(
            training_accuracy
        )

        history[
            "val_loss"
        ].append(
            validation_loss
        )

        history[
            "val_accuracy"
        ].append(
            validation_accuracy
        )

        history[
            "learning_rate"
        ].append(
            current_lr
        )

        if verbose:
            print(
                f"Epoch "
                f"{epoch:02d}/"
                f"{epochs} | "
                f"Train Loss "
                f"{training_loss:.4f} | "
                f"Train Acc "
                f"{training_accuracy:.4f} | "
                f"Val Loss "
                f"{validation_loss:.4f} | "
                f"Val Acc "
                f"{validation_accuracy:.4f} | "
                f"LR "
                f"{current_lr:.2e}"
            )

        if use_early_stopping:
            if (
                monitored_loss
                < best_loss
                - EARLY_STOPPING_MIN_DELTA
            ):
                best_loss = (
                    monitored_loss
                )

                best_epoch = (
                    epoch
                )

                best_state = {
                    key:
                        value.detach()
                        .cpu()
                        .clone()

                    for key, value
                    in model.state_dict().items()
                }

                patience_counter = 0

            else:
                patience_counter += 1

                if (
                    patience_counter
                    >= patience
                ):
                    if verbose:
                        print(
                            "Early stopping."
                        )

                    break

    if (
        use_early_stopping
        and best_state is not None
    ):
        model.load_state_dict(
            best_state
        )

    if not use_early_stopping:
        best_epoch = len(
            history[
                "loss"
            ]
        )

    del optimizer
    del scheduler
    del criterion
    del class_weights

    if best_state is not None:
        del best_state

    clear_device()

    return (
        model,
        history,
        best_epoch,
    )


# =============================================================================
# PREDICT PROBABILITIES
# =============================================================================

def predict_probabilities(
    spec,
    model,
    X,
    y
):
    (
        _,
        _,
        probabilities,

    ) = evaluate_model(
        model,
        X,
        y,
        spec[
            "batch_size"
        ],
        criterion=None,
        return_probabilities=True
    )

    return probabilities


# =============================================================================
# REAL GPU TRAINING SMOKE TEST
#
# This checks forward propagation, backward propagation and optimizer step.
# =============================================================================

def run_smoke_test(
    spec
):
    print()
    print("=" * 90)
    print(
        spec[
            "display_name"
        ]
    )
    print("REAL GPU TRAINING SMOKE TEST")
    print("=" * 90)

    params = spec[
        "smoke_params"
    ]

    model = instantiate_model(
        spec,
        params,
        SEED
    )

    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-4,
        weight_decay=1e-4
    )

    X_smoke = torch.randn(
        16,
        NUMBER_FEATURES,
        dtype=torch.float32,
        device=DEVICE
    )

    y_smoke = torch.randint(
        low=0,
        high=
            NUMBER_CLASSES,
        size=(16,),
        dtype=torch.long,
        device=DEVICE
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    output = model(
        X_smoke
    )

    loss = nn.CrossEntropyLoss()(
        output,
        y_smoke
    )

    loss.backward()

    gradient_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )
    )

    optimizer.step()

    synchronize()

    print(
        "Input shape  :",
        X_smoke.shape
    )

    print(
        "Output shape :",
        output.shape
    )

    print(
        "Output device:",
        output.device
    )

    print(
        "Loss         :",
        float(
            loss.detach().item()
        )
    )

    if torch.is_tensor(
        gradient_norm
    ):
        gradient_value = float(
            gradient_norm.detach()
            .cpu()
            .item()
        )
    else:
        gradient_value = float(
            gradient_norm
        )

    print(
        "Gradient norm:",
        gradient_value
    )

    print(
        "Backward     : SUCCESS"
    )

    print(
        "Optimizer    : SUCCESS"
    )

    print(
        "Smoke test   : PASSED"
    )

    del model
    del optimizer
    del X_smoke
    del y_smoke
    del output
    del loss
    del gradient_norm

    clear_device()


# =============================================================================
# SAVE COMMON EXPERIMENT INFORMATION
# =============================================================================

def save_common_files(
    output_dir,
    uses_modalities
):
    CLASS_DISTRIBUTION_TABLE.to_csv(
        output_dir
        / "class_distribution.csv",
        index=False
    )

    SPLIT_SUMMARY_TABLE.to_csv(
        output_dir
        / "split_summary.csv",
        index=False
    )

    save_json(
        PARTICIPANT_SPLIT,
        output_dir
        / "participant_split.json"
    )

    save_json(
        FEATURE_COLUMNS,
        output_dir
        / "feature_columns.json"
    )

    joblib.dump(
        FINAL_SCALER,
        output_dir
        / "standard_scaler.joblib"
    )

    joblib.dump(
        LABEL_ENCODER,
        output_dir
        / "label_encoder.joblib"
    )

    if uses_modalities:
        modality_mapping = {
            "Grouping_Mode":
                MODALITY_GROUPING_MODE,

            "Scientific_Note":
                (
                    "These streams are EEG feature families "
                    "or feature views, not separate physical sensors."
                ),

            "Modalities":
                {},
        }

        for modality_name in MODALITY_NAMES:
            indices = MODALITY_INDICES[
                modality_name
            ]

            modality_mapping[
                "Modalities"
            ][
                modality_name
            ] = {
                "Feature_Count":
                    len(indices),

                "Features":
                    [
                        FEATURE_COLUMNS[
                            index
                        ]
                        for index in indices
                    ],
            }

        save_json(
            modality_mapping,
            output_dir
            / "modality_mapping.json"
        )


# =============================================================================
# PLOT TRAINING HISTORY
# =============================================================================

def save_training_curves(
    history,
    output_dir,
    algorithm_name
):
    epochs = np.arange(
        1,
        len(
            history[
                "loss"
            ]
        ) + 1
    )

    history_table = pd.DataFrame(
        {
            "Epoch":
                epochs,

            "Training_Loss":
                history[
                    "loss"
                ],

            "Training_Accuracy":
                history[
                    "accuracy"
                ],

            "Validation_Loss":
                history[
                    "val_loss"
                ],

            "Validation_Accuracy":
                history[
                    "val_accuracy"
                ],

            "Learning_Rate":
                history[
                    "learning_rate"
                ],
        }
    )

    history_table[
        "Generalization_Gap"
    ] = (
        history_table[
            "Training_Accuracy"
        ]
        -
        history_table[
            "Validation_Accuracy"
        ]
    )

    history_table.to_csv(
        output_dir
        / "training_history.csv",
        index=False
    )

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.plot(
        epochs,
        history[
            "loss"
        ],
        label=
            "Training Loss"
    )

    ax.plot(
        epochs,
        history[
            "val_loss"
        ],
        label=
            "Validation Loss"
    )

    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Cross Entropy Loss"
    )

    ax.set_title(
        f"{algorithm_name} Loss Curve"
    )

    ax.legend()
    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        output_dir
        / "loss_curve.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.plot(
        epochs,
        history[
            "accuracy"
        ],
        label=
            "Training Accuracy"
    )

    ax.plot(
        epochs,
        history[
            "val_accuracy"
        ],
        label=
            "Validation Accuracy"
    )

    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Accuracy"
    )

    ax.set_title(
        f"{algorithm_name} Accuracy Curve"
    )

    ax.legend()
    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        output_dir
        / "accuracy_curve.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.plot(
        epochs,
        history_table[
            "Generalization_Gap"
        ]
    )

    ax.axhline(
        0.0,
        linewidth=1
    )

    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Training Accuracy minus Validation Accuracy"
    )

    ax.set_title(
        f"{algorithm_name} Generalization Gap"
    )

    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        output_dir
        / "generalization_gap_curve.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.plot(
        epochs,
        history[
            "learning_rate"
        ]
    )

    ax.set_xlabel(
        "Epoch"
    )

    ax.set_ylabel(
        "Learning Rate"
    )

    ax.set_title(
        f"{algorithm_name} Learning Rate"
    )

    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        output_dir
        / "learning_rate_curve.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


# =============================================================================
# CONFUSION MATRIX OUTPUT
# =============================================================================

def save_confusion_matrices(
    y_true,
    y_prediction,
    output_dir,
    algorithm_name
):
    cm = confusion_matrix(
        y_true,
        y_prediction,
        labels=np.arange(
            NUMBER_CLASSES
        )
    )

    pd.DataFrame(
        cm,
        index=
            CLASS_NAMES,
        columns=
            CLASS_NAMES
    ).to_csv(
        output_dir
        / "confusion_matrix.csv"
    )

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    image = ax.imshow(
        cm
    )

    ax.set_xticks(
        np.arange(
            NUMBER_CLASSES
        )
    )

    ax.set_yticks(
        np.arange(
            NUMBER_CLASSES
        )
    )

    ax.set_xticklabels(
        CLASS_NAMES,
        rotation=45,
        ha="right"
    )

    ax.set_yticklabels(
        CLASS_NAMES
    )

    ax.set_xlabel(
        "Predicted Class"
    )

    ax.set_ylabel(
        "True Class"
    )

    ax.set_title(
        f"{algorithm_name} Confusion Matrix"
    )

    for row in range(
        NUMBER_CLASSES
    ):
        for column in range(
            NUMBER_CLASSES
        ):
            ax.text(
                column,
                row,
                str(
                    cm[
                        row,
                        column
                    ]
                ),
                ha="center",
                va="center"
            )

    fig.colorbar(
        image,
        ax=ax
    )

    fig.tight_layout()

    fig.savefig(
        output_dir
        / "confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    row_sums = cm.sum(
        axis=1,
        keepdims=True
    )

    normalized_cm = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(
            cm,
            dtype=np.float64
        ),
        where=(
            row_sums != 0
        )
    )

    pd.DataFrame(
        normalized_cm,
        index=
            CLASS_NAMES,
        columns=
            CLASS_NAMES
    ).to_csv(
        output_dir
        / "normalized_confusion_matrix.csv"
    )

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    image = ax.imshow(
        normalized_cm
    )

    ax.set_xticks(
        np.arange(
            NUMBER_CLASSES
        )
    )

    ax.set_yticks(
        np.arange(
            NUMBER_CLASSES
        )
    )

    ax.set_xticklabels(
        CLASS_NAMES,
        rotation=45,
        ha="right"
    )

    ax.set_yticklabels(
        CLASS_NAMES
    )

    ax.set_xlabel(
        "Predicted Class"
    )

    ax.set_ylabel(
        "True Class"
    )

    ax.set_title(
        f"Normalized {algorithm_name} Confusion Matrix"
    )

    for row in range(
        NUMBER_CLASSES
    ):
        for column in range(
            NUMBER_CLASSES
        ):
            ax.text(
                column,
                row,
                f"{normalized_cm[row, column]:.2f}",
                ha="center",
                va="center"
            )

    fig.colorbar(
        image,
        ax=ax
    )

    fig.tight_layout()

    fig.savefig(
        output_dir
        / "normalized_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    return cm


# =============================================================================
# OPTUNA RUNNER
# =============================================================================

def run_optuna(
    spec,
    output_dir
):
    algorithm_name = spec[
        "display_name"
    ]

    study_name = (
        spec[
            "version"
        ]
        + "_"
        + DATASET_SIGNATURE
    )

    database_path = (
        output_dir
        / "optuna.db"
    )

    storage = (
        f"sqlite:///{database_path}"
    )

    def objective(
        trial
    ):
        params = spec[
            "search"
        ](
            trial
        )

        print()
        print(
            f"{algorithm_name}"
        )

        print(
            f"Optuna Trial "
            f"{trial.number + 1}"
        )

        print(
            params
        )

        fold_scores = []

        for fold_number, (
            training_index,
            validation_index,

        ) in enumerate(
            group_kfold_iterator(),
            start=1
        ):
            fold_start = time.time()

            (
                X_fold_training,
                X_fold_validation,

            ) = prepare_fold(
                training_index,
                validation_index
            )

            (
                model,
                history,
                best_epoch,

            ) = fit_model(
                spec,
                params,
                X_fold_training,
                Y_TRAIN[
                    training_index
                ],
                X_fold_validation,
                Y_TRAIN[
                    validation_index
                ],
                seed=
                    SEED
                    + fold_number,
                epochs=
                    spec[
                        "max_epochs"
                    ],
                use_early_stopping=
                    True,
                verbose=
                    False
            )

            probabilities = predict_probabilities(
                spec,
                model,
                X_fold_validation,
                Y_TRAIN[
                    validation_index
                ]
            )

            predictions = np.argmax(
                probabilities,
                axis=1
            )

            fold_accuracy = accuracy_score(
                Y_TRAIN[
                    validation_index
                ],
                predictions
            )

            fold_scores.append(
                float(
                    fold_accuracy
                )
            )

            print(
                f"Trial "
                f"{trial.number + 1}, "
                f"Fold "
                f"{fold_number}/"
                f"{GROUP_K_FOLDS}, "
                f"Accuracy = "
                f"{fold_accuracy:.4f}, "
                f"Best Epoch = "
                f"{best_epoch}, "
                f"Time = "
                f"{(time.time() - fold_start) / 60:.2f} min, "
                f"RAM = "
                f"{memory_mb():.2f} MB"
            )

            del model
            del history
            del probabilities
            del predictions
            del X_fold_training
            del X_fold_validation

            clear_device()

        mean_accuracy = float(
            np.mean(
                fold_scores
            )
        )

        std_accuracy = float(
            np.std(
                fold_scores
            )
        )

        trial.set_user_attr(
            "fold_scores",
            json.dumps(
                fold_scores
            )
        )

        trial.set_user_attr(
            "fold_std",
            std_accuracy
        )

        print(
            f"Trial "
            f"{trial.number + 1} "
            f"Mean Accuracy = "
            f"{mean_accuracy:.4f} "
            f"plus or minus "
            f"{std_accuracy:.4f}"
        )

        return mean_accuracy

    study = optuna.create_study(
        study_name=
            study_name,
        storage=
            storage,
        direction=
            "maximize",
        sampler=
            optuna.samplers.TPESampler(
                seed=SEED
            ),
        load_if_exists=
            True
    )

    def completed_count():
        return sum(
            1
            for trial in study.trials
            if trial.state
            == optuna.trial.TrialState.COMPLETE
        )

    print()
    print("=" * 90)
    print(
        algorithm_name
    )
    print("OPTUNA")
    print("=" * 90)

    print(
        "Previously completed trials:",
        completed_count()
    )

    # =========================================================================
    # Run one trial at a time.
    #
    # This makes restart behavior straightforward.
    # A power cut inside one trial only requires that unfinished trial
    # to be repeated after restart.
    # =========================================================================

    while completed_count() < N_TRIALS:
        print(
            "Completed:",
            completed_count(),
            "of",
            N_TRIALS
        )

        study.optimize(
            objective,
            n_trials=1,
            show_progress_bar=False,
            gc_after_trial=True
        )

    study.trials_dataframe().to_csv(
        output_dir
        / "optuna_trials.csv",
        index=False
    )

    best_params = dict(
        study.best_params
    )

    save_json(
        best_params,
        output_dir
        / "best_parameters.json"
    )

    print()
    print(
        "Best GroupKFold Accuracy:",
        f"{study.best_value:.4f}"
    )

    print(
        "Best Parameters:"
    )

    print(
        best_params
    )

    return (
        study,
        best_params,
    )


# =============================================================================
# CONFIGURATION ID
# =============================================================================

def build_config_id(
    spec,
    best_params
):
    payload = {
        "algorithm":
            spec[
                "display_name"
            ],

        "version":
            spec[
                "version"
            ],

        "dataset_signature":
            DATASET_SIGNATURE,

        "best_params":
            best_params,

        "seed":
            SEED,

        "features":
            NUMBER_FEATURES,

        "classes":
            CLASS_NAMES,

        "modalities":
            (
                MODALITY_INDICES
                if spec[
                    "uses_modalities"
                ]
                else None
            ),
    }

    return hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True
        ).encode(
            "utf8"
        )
    ).hexdigest()[:16]


# =============================================================================
# FIVE FINAL RUNS
# =============================================================================

def run_final_runs(
    spec,
    best_params,
    config_id,
    output_dir
):
    algorithm_name = spec[
        "display_name"
    ]

    checkpoint_dir = (
        output_dir
        / "checkpoints"
    )

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    run_records = []

    print()
    print("=" * 90)
    print(
        algorithm_name
    )
    print("FIVE FINAL RUNS")
    print("=" * 90)

    for run_number in range(
        1,
        N_RUNS + 1
    ):
        checkpoint_path = (
            checkpoint_dir
            / f"run_{run_number:02d}.pt"
        )

        valid_checkpoint = False
        bundle = None

        if checkpoint_path.exists():
            try:
                bundle = torch.load(
                    checkpoint_path,
                    map_location="cpu",
                    weights_only=False
                )

                valid_checkpoint = (
                    str(
                        bundle.get(
                            "Config_ID",
                            ""
                        )
                    )
                    == config_id
                )

            except Exception:
                valid_checkpoint = False

        if valid_checkpoint:
            record = bundle[
                "record"
            ]

            run_records.append(
                record
            )

            print(
                f"Run "
                f"{run_number}/"
                f"{N_RUNS} "
                f"already completed."
            )

            continue

        print()
        print(
            f"RUN "
            f"{run_number}/"
            f"{N_RUNS}"
        )

        run_start = time.time()

        run_seed = (
            SEED
            + run_number
            - 1
        )

        (
            model,
            history,
            best_epoch,

        ) = fit_model(
            spec,
            best_params,
            X_TRAIN,
            Y_TRAIN,
            X_VALIDATION,
            Y_VALIDATION,
            seed=
                run_seed,
            epochs=
                spec[
                    "max_epochs"
                ],
            use_early_stopping=
                True,
            verbose=
                True
        )

        train_probabilities = predict_probabilities(
            spec,
            model,
            X_TRAIN,
            Y_TRAIN
        )

        train_predictions = np.argmax(
            train_probabilities,
            axis=1
        )

        training_accuracy = accuracy_score(
            Y_TRAIN,
            train_predictions
        )

        validation_probabilities = (
            predict_probabilities(
                spec,
                model,
                X_VALIDATION,
                Y_VALIDATION
            )
        )

        validation_predictions = np.argmax(
            validation_probabilities,
            axis=1
        )

        validation_accuracy = accuracy_score(
            Y_VALIDATION,
            validation_predictions
        )

        generalization_gap = (
            training_accuracy
            - validation_accuracy
        )

        run_time = (
            time.time()
            - run_start
        )

        record = {
            "Config_ID":
                config_id,

            "Run":
                run_number,

            "Seed":
                run_seed,

            "Training_Accuracy":
                float(
                    training_accuracy
                ),

            "Validation_Accuracy":
                float(
                    validation_accuracy
                ),

            "Generalization_Gap":
                float(
                    generalization_gap
                ),

            "Absolute_Generalization_Gap":
                float(
                    abs(
                        generalization_gap
                    )
                ),

            "Best_Epoch":
                int(
                    best_epoch
                ),

            "Epochs_Run":
                int(
                    len(
                        history[
                            "loss"
                        ]
                    )
                ),

            "Execution_Time_Seconds":
                float(
                    run_time
                ),
        }

        model_state = {
            key:
                value.detach()
                .cpu()
                .clone()

            for key, value
            in model.state_dict().items()
        }

        atomic_torch_save(
            {
                "Config_ID":
                    config_id,

                "record":
                    record,

                "best_params":
                    best_params,

                "model_state_dict":
                    model_state,

                "history":
                    history,
            },
            checkpoint_path
        )

        run_records.append(
            record
        )

        atomic_csv(
            pd.DataFrame(
                run_records
            )
            .sort_values(
                "Run"
            )
            .drop_duplicates(
                subset=[
                    "Run"
                ],
                keep="last"
            ),
            output_dir
            / "five_runs.csv"
        )

        print(
            f"Train = "
            f"{training_accuracy:.4f}, "
            f"Validation = "
            f"{validation_accuracy:.4f}, "
            f"Gap = "
            f"{generalization_gap:.4f}, "
            f"Best Epoch = "
            f"{best_epoch}, "
            f"Time = "
            f"{run_time / 60:.2f} min"
        )

        del model
        del history
        del model_state
        del train_probabilities
        del train_predictions
        del validation_probabilities
        del validation_predictions

        clear_device()

    run_table = pd.DataFrame(
        run_records
    )

    run_table = (
        run_table
        .sort_values(
            "Run"
        )
        .drop_duplicates(
            subset=[
                "Run"
            ],
            keep="last"
        )
    )

    if len(
        run_table
    ) != N_RUNS:
        raise RuntimeError(
            "Five final runs are not complete."
        )

    atomic_csv(
        run_table,
        output_dir
        / "five_runs.csv"
    )

    selected_row = (
        run_table
        .sort_values(
            by=[
                "Validation_Accuracy",
                "Absolute_Generalization_Gap",
                "Run",
            ],
            ascending=[
                False,
                True,
                True,
            ]
        )
        .iloc[0]
    )

    best_run_number = int(
        selected_row[
            "Run"
        ]
    )

    best_checkpoint_path = (
        checkpoint_dir
        / f"run_{best_run_number:02d}.pt"
    )

    best_bundle = torch.load(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print()
    print(
        "Selected Run:",
        best_run_number
    )

    print(
        "Validation Accuracy:",
        f"{best_bundle['record']['Validation_Accuracy']:.4f}"
    )

    print(
        "Generalization Gap:",
        f"{best_bundle['record']['Generalization_Gap']:.4f}"
    )

    return (
        run_table,
        best_bundle,
    )


# =============================================================================
# FULL LOGO
# =============================================================================

def run_full_logo(
    spec,
    best_params,
    config_id,
    best_record,
    output_dir
):
    algorithm_name = spec[
        "display_name"
    ]

    fixed_epochs = max(
        1,
        int(
            best_record[
                "Best_Epoch"
            ]
        )
    )

    if LOGO_MAX_FOLDS is None:
        logo_folds_to_use = (
            FULL_LOGO_FOLDS
        )

    else:
        logo_folds_to_use = min(
            int(
                LOGO_MAX_FOLDS
            ),
            FULL_LOGO_FOLDS
        )

    logo_configuration = {
        "Config_ID":
            config_id,

        "Fixed_Epochs":
            fixed_epochs,

        "Selected_Seed":
            int(
                best_record[
                    "Seed"
                ]
            ),
    }

    logo_config_id = hashlib.sha256(
        json.dumps(
            logo_configuration,
            sort_keys=True
        ).encode(
            "utf8"
        )
    ).hexdigest()[:16]

    logo_path = (
        output_dir
        / "full_logo_results.csv"
    )

    logo_records = []
    completed_persons = set()

    if logo_path.exists():
        try:
            previous_logo = pd.read_csv(
                logo_path,
                dtype={
                    "Config_ID": str,
                    "Held_Out_Person": str,
                }
            )

            if (
                "Config_ID"
                in previous_logo.columns
            ):
                matching_logo = previous_logo[
                    previous_logo[
                        "Config_ID"
                    ].astype(str)
                    == logo_config_id
                ].copy()

                if len(
                    matching_logo
                ) > 0:
                    logo_records = matching_logo.to_dict(
                        orient="records"
                    )

                    completed_persons = set(
                        matching_logo[
                            "Held_Out_Person"
                        ].astype(str)
                    )

        except Exception:
            logo_records = []
            completed_persons = set()

    print()
    print("=" * 90)
    print(
        algorithm_name
    )
    print("FULL LEAVE ONE GROUP OUT")
    print("=" * 90)

    print(
        "Available folds:",
        FULL_LOGO_FOLDS
    )

    print(
        "Requested folds:",
        logo_folds_to_use
    )

    print(
        "Fixed epochs:",
        fixed_epochs
    )

    print(
        "Previously completed:",
        len(
            completed_persons
        )
    )

    logo_start = time.time()

    splitter = LeaveOneGroupOut()

    for fold_number, (
        training_index,
        validation_index,

    ) in enumerate(
        splitter.split(
            X_TRAIN_RAW,
            Y_TRAIN,
            G_TRAIN
        ),
        start=1
    ):
        if (
            fold_number
            > logo_folds_to_use
        ):
            break

        held_out_values = np.unique(
            G_TRAIN[
                validation_index
            ]
        )

        if len(
            held_out_values
        ) != 1:
            raise RuntimeError(
                "Unexpected LOGO group structure."
            )

        held_out_person = str(
            held_out_values[0]
        )

        if (
            held_out_person
            in completed_persons
        ):
            print(
                f"LOGO "
                f"{fold_number}/"
                f"{logo_folds_to_use}, "
                f"Participant "
                f"{held_out_person}, "
                f"already completed."
            )

            continue

        fold_start = time.time()

        (
            X_logo_training,
            X_logo_validation,

        ) = prepare_fold(
            training_index,
            validation_index
        )

        (
            logo_model,
            _,
            _,

        ) = fit_model(
            spec,
            best_params,
            X_logo_training,
            Y_TRAIN[
                training_index
            ],
            X_validation_data=None,
            y_validation_data=None,
            seed=
                int(
                    best_record[
                        "Seed"
                    ]
                ),
            epochs=
                fixed_epochs,
            use_early_stopping=
                False,
            verbose=
                False
        )

        probabilities = predict_probabilities(
            spec,
            logo_model,
            X_logo_validation,
            Y_TRAIN[
                validation_index
            ]
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        y_true = Y_TRAIN[
            validation_index
        ]

        fold_accuracy = accuracy_score(
            y_true,
            predictions
        )

        fold_precision = precision_score(
            y_true,
            predictions,
            average="weighted",
            zero_division=0
        )

        fold_recall = recall_score(
            y_true,
            predictions,
            average="weighted",
            zero_division=0
        )

        fold_f1 = f1_score(
            y_true,
            predictions,
            average="weighted",
            zero_division=0
        )

        fold_macro_f1 = f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        )

        fold_mcc = matthews_corrcoef(
            y_true,
            predictions
        )

        fold_kappa = cohen_kappa_score(
            y_true,
            predictions
        )

        fold_time = (
            time.time()
            - fold_start
        )

        record = {
            "Config_ID":
                logo_config_id,

            "Fold":
                fold_number,

            "Held_Out_Person":
                held_out_person,

            "Rows":
                int(
                    len(
                        validation_index
                    )
                ),

            "Accuracy":
                float(
                    fold_accuracy
                ),

            "Precision_Weighted":
                float(
                    fold_precision
                ),

            "Recall_Weighted":
                float(
                    fold_recall
                ),

            "F1_Weighted":
                float(
                    fold_f1
                ),

            "F1_Macro":
                float(
                    fold_macro_f1
                ),

            "Cohen_Kappa":
                float(
                    fold_kappa
                ),

            "MCC":
                float(
                    fold_mcc
                ),

            "Fixed_Epochs":
                int(
                    fixed_epochs
                ),

            "Execution_Time_Seconds":
                float(
                    fold_time
                ),
        }

        logo_records.append(
            record
        )

        atomic_csv(
            pd.DataFrame(
                logo_records
            ),
            logo_path
        )

        completed_persons.add(
            held_out_person
        )

        print(
            f"LOGO "
            f"{fold_number}/"
            f"{logo_folds_to_use}, "
            f"Participant = "
            f"{held_out_person}, "
            f"Accuracy = "
            f"{fold_accuracy:.4f}, "
            f"F1 = "
            f"{fold_f1:.4f}, "
            f"MCC = "
            f"{fold_mcc:.4f}, "
            f"Time = "
            f"{fold_time / 60:.2f} min"
        )

        del logo_model
        del probabilities
        del predictions
        del X_logo_training
        del X_logo_validation

        clear_device()

    logo_table = pd.read_csv(
        logo_path,
        dtype={
            "Config_ID": str,
            "Held_Out_Person": str,
        }
    )

    logo_table = logo_table[
        logo_table[
            "Config_ID"
        ].astype(str)
        == logo_config_id
    ].copy()

    logo_table = (
        logo_table
        .sort_values(
            "Fold"
        )
        .drop_duplicates(
            subset=[
                "Held_Out_Person"
            ],
            keep="last"
        )
    )

    logo_accuracy = (
        logo_table[
            "Accuracy"
        ]
        .astype(float)
        .to_numpy()
    )

    logo_mean = float(
        np.mean(
            logo_accuracy
        )
    )

    logo_std = float(
        np.std(
            logo_accuracy
        )
    )

    logo_time = (
        time.time()
        - logo_start
    )

    print()
    print(
        "Full LOGO Accuracy:",
        f"{logo_mean:.4f}",
        "plus or minus",
        f"{logo_std:.4f}"
    )

    print(
        "Completed LOGO folds:",
        len(
            logo_table
        )
    )

    return (
        logo_table,
        logo_mean,
        logo_std,
        logo_time,
    )


# =============================================================================
# FINAL TEST EVALUATION
# =============================================================================

def final_test_evaluation(
    spec,
    best_params,
    best_bundle,
    output_dir
):
    model = instantiate_model(
        spec,
        best_params,
        seed=
            int(
                best_bundle[
                    "record"
                ][
                    "Seed"
                ]
            )
    )

    model.load_state_dict(
        best_bundle[
            "model_state_dict"
        ]
    )

    model.eval()

    test_start = time.time()

    probabilities = predict_probabilities(
        spec,
        model,
        X_TEST,
        Y_TEST
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    test_time = (
        time.time()
        - test_start
    )

    accuracy = accuracy_score(
        Y_TEST,
        predictions
    )

    precision = precision_score(
        Y_TEST,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        Y_TEST,
        predictions,
        average="weighted",
        zero_division=0
    )

    weighted_f1 = f1_score(
        Y_TEST,
        predictions,
        average="weighted",
        zero_division=0
    )

    macro_f1 = f1_score(
        Y_TEST,
        predictions,
        average="macro",
        zero_division=0
    )

    kappa = cohen_kappa_score(
        Y_TEST,
        predictions
    )

    mcc = matthews_corrcoef(
        Y_TEST,
        predictions
    )

    try:
        roc_auc = roc_auc_score(
            Y_TEST,
            probabilities,
            multi_class="ovr",
            average="weighted"
        )

    except Exception:
        roc_auc = np.nan

    report_text = classification_report(
        Y_TEST,
        predictions,
        labels=np.arange(
            NUMBER_CLASSES
        ),
        target_names=
            CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    with open(
        output_dir
        / "classification_report.txt",
        "w",
        encoding="utf8"
    ) as file:
        file.write(
            report_text
        )

    report_table = pd.DataFrame(
        classification_report(
            Y_TEST,
            predictions,
            labels=np.arange(
                NUMBER_CLASSES
            ),
            target_names=
                CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    ).transpose()

    report_table.to_csv(
        output_dir
        / "classification_report.csv"
    )

    save_confusion_matrices(
        Y_TEST,
        predictions,
        output_dir,
        spec[
            "display_name"
        ]
    )

    prediction_table = pd.DataFrame(
        {
            "PersonSrial":
                G_TEST,

            "True_Class":
                LABEL_ENCODER.inverse_transform(
                    Y_TEST
                ),

            "Predicted_Class":
                LABEL_ENCODER.inverse_transform(
                    predictions.astype(
                        np.int32
                    )
                ),
        }
    )

    for class_index, class_name in enumerate(
        CLASS_NAMES
    ):
        prediction_table[
            f"Probability_{class_name}"
        ] = probabilities[
            :,
            class_index
        ]

    prediction_table.to_csv(
        output_dir
        / "test_predictions.csv",
        index=False
    )

    print()
    print("CLASSIFICATION REPORT")
    print()
    print(report_text)

    del model

    clear_device()

    return {
        "Testing_Accuracy":
            float(
                accuracy
            ),

        "Precision_Weighted":
            float(
                precision
            ),

        "Recall_Weighted":
            float(
                recall
            ),

        "F1_Weighted":
            float(
                weighted_f1
            ),

        "F1_Macro":
            float(
                macro_f1
            ),

        "Cohen_Kappa":
            float(
                kappa
            ),

        "MCC":
            float(
                mcc
            ),

        "ROC_AUC_Weighted_OVR":
            (
                None
                if np.isnan(
                    roc_auc
                )
                else float(
                    roc_auc
                )
            ),

        "Test_Time_Seconds":
            float(
                test_time
            ),
    }


# =============================================================================
# COMPLETE ALGORITHM RUNNER
# =============================================================================

def run_algorithm(
    algorithm_key
):
    spec = ALGORITHM_SPECS[
        algorithm_key
    ]

    algorithm_name = spec[
        "display_name"
    ]

    output_dir = (
        OUTPUT_ROOT
        / spec[
            "folder"
        ]
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    (
        output_dir
        / "checkpoints"
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    algorithm_start = time.time()

    print()
    print("#" * 90)
    print(
        algorithm_name
    )
    print("MED DATASET")
    print("#" * 90)

    print(
        "Output folder:"
    )

    print(
        output_dir
    )

    print(
        "Batch size:",
        spec[
            "batch_size"
        ]
    )

    print(
        "Maximum epochs:",
        spec[
            "max_epochs"
        ]
    )

    print(
        "Optuna trials:",
        N_TRIALS
    )

    print(
        "Final runs:",
        N_RUNS
    )

    print(
        "GroupKFold:",
        GROUP_K_FOLDS
    )

    print(
        "Full LOGO:",
        FULL_LOGO_FOLDS
    )

    save_common_files(
        output_dir,
        spec[
            "uses_modalities"
        ]
    )

    # =========================================================================
    # GPU TRAINING TEST
    # =========================================================================

    run_smoke_test(
        spec
    )

    # =========================================================================
    # OPTUNA
    # =========================================================================

    optuna_start = time.time()

    (
        study,
        best_params,

    ) = run_optuna(
        spec,
        output_dir
    )

    optuna_time = (
        time.time()
        - optuna_start
    )

    config_id = build_config_id(
        spec,
        best_params
    )

    print(
        "Configuration ID:",
        config_id
    )

    # =========================================================================
    # SAVE ARCHITECTURE
    # =========================================================================

    temporary_model = instantiate_model(
        spec,
        best_params,
        SEED
    )

    trainable_parameters = (
        count_trainable_parameters(
            temporary_model
        )
    )

    with open(
        output_dir
        / "model_architecture.txt",
        "w",
        encoding="utf8"
    ) as file:
        file.write(
            str(
                temporary_model
            )
        )

    del temporary_model

    clear_device()

    print(
        "Trainable parameters:",
        f"{trainable_parameters:,}"
    )

    # =========================================================================
    # FIVE RUNS
    # =========================================================================

    final_run_start = time.time()

    (
        run_table,
        best_bundle,

    ) = run_final_runs(
        spec,
        best_params,
        config_id,
        output_dir
    )

    final_run_time = (
        time.time()
        - final_run_start
    )

    best_record = best_bundle[
        "record"
    ]

    best_history = best_bundle[
        "history"
    ]

    save_training_curves(
        best_history,
        output_dir,
        algorithm_name
    )

    # =========================================================================
    # SAVE BEST MODEL
    # =========================================================================

    atomic_torch_save(
        {
            "Algorithm":
                algorithm_name,

            "Algorithm_Key":
                algorithm_key,

            "Version":
                spec[
                    "version"
                ],

            "Dataset":
                "Med Dataset",

            "Dataset_Signature":
                DATASET_SIGNATURE,

            "Config_ID":
                config_id,

            "Device_Type":
                DEVICE_TYPE,

            "Device_Name":
                DEVICE_NAME,

            "Best_Params":
                best_params,

            "Selected_Run":
                int(
                    best_record[
                        "Run"
                    ]
                ),

            "Selected_Seed":
                int(
                    best_record[
                        "Seed"
                    ]
                ),

            "Best_Epoch":
                int(
                    best_record[
                        "Best_Epoch"
                    ]
                ),

            "Model_State_Dict":
                best_bundle[
                    "model_state_dict"
                ],

            "Feature_Columns":
                FEATURE_COLUMNS,

            "Class_Names":
                CLASS_NAMES,

            "Modality_Indices":
                (
                    MODALITY_INDICES
                    if spec[
                        "uses_modalities"
                    ]
                    else None
                ),
        },
        output_dir
        / "best_model.pt"
    )

    # =========================================================================
    # FULL LOGO
    # =========================================================================

    (
        logo_table,
        logo_mean,
        logo_std,
        logo_time,

    ) = run_full_logo(
        spec,
        best_params,
        config_id,
        best_record,
        output_dir
    )

    # =========================================================================
    # FINAL TEST
    # =========================================================================

    print()
    print("=" * 90)
    print(
        algorithm_name
    )
    print("FINAL UNTOUCHED TEST EVALUATION")
    print("=" * 90)

    test_metrics = final_test_evaluation(
        spec,
        best_params,
        best_bundle,
        output_dir
    )

    # =========================================================================
    # FIVE RUN STATISTICS
    # =========================================================================

    train_mean = float(
        run_table[
            "Training_Accuracy"
        ].mean()
    )

    train_std = float(
        run_table[
            "Training_Accuracy"
        ].std(
            ddof=0
        )
    )

    validation_mean = float(
        run_table[
            "Validation_Accuracy"
        ].mean()
    )

    validation_std = float(
        run_table[
            "Validation_Accuracy"
        ].std(
            ddof=0
        )
    )

    gap_mean = float(
        run_table[
            "Generalization_Gap"
        ].mean()
    )

    gap_std = float(
        run_table[
            "Generalization_Gap"
        ].std(
            ddof=0
        )
    )

    algorithm_time = (
        time.time()
        - algorithm_start
    )

    # =========================================================================
    # FINAL METRICS
    # =========================================================================

    final_metrics = {
        "Algorithm":
            algorithm_name,

        "Algorithm_Key":
            algorithm_key,

        "Version":
            spec[
                "version"
            ],

        "Dataset":
            "Med Dataset",

        "Dataset_Signature":
            DATASET_SIGNATURE,

        "Config_ID":
            config_id,

        "Input_File":
            str(
                FILE_PATH
            ),

        "Output_Directory":
            str(
                output_dir
            ),

        "Framework":
            "PyTorch",

        "Device_Type":
            DEVICE_TYPE,

        "Device_Name":
            DEVICE_NAME,

        "Feature_Count":
            int(
                NUMBER_FEATURES
            ),

        "Classes":
            CLASS_NAMES,

        "Training_Participants":
            int(
                len(
                    TRAIN_PERSONS
                )
            ),

        "Validation_Participants":
            int(
                len(
                    VALIDATION_PERSONS
                )
            ),

        "Testing_Participants":
            int(
                len(
                    TEST_PERSONS
                )
            ),

        "Training_Samples":
            int(
                TRAIN_SAMPLES
            ),

        "Validation_Samples":
            int(
                VALIDATION_SAMPLES
            ),

        "Testing_Samples":
            int(
                TEST_SAMPLES
            ),

        "Trainable_Parameters":
            int(
                trainable_parameters
            ),

        "Optuna_Trials":
            int(
                N_TRIALS
            ),

        "Optuna_GroupKFold_Accuracy":
            float(
                study.best_value
            ),

        "Best_Parameters":
            best_params,

        "Selected_Run":
            int(
                best_record[
                    "Run"
                ]
            ),

        "Selected_Seed":
            int(
                best_record[
                    "Seed"
                ]
            ),

        "Selected_Best_Epoch":
            int(
                best_record[
                    "Best_Epoch"
                ]
            ),

        "Training_Accuracy":
            float(
                best_record[
                    "Training_Accuracy"
                ]
            ),

        "Validation_Accuracy":
            float(
                best_record[
                    "Validation_Accuracy"
                ]
            ),

        "Generalization_Gap":
            float(
                best_record[
                    "Generalization_Gap"
                ]
            ),

        "Five_Run_Training_Mean":
            train_mean,

        "Five_Run_Training_STD":
            train_std,

        "Five_Run_Validation_Mean":
            validation_mean,

        "Five_Run_Validation_STD":
            validation_std,

        "Five_Run_Gap_Mean":
            gap_mean,

        "Five_Run_Gap_STD":
            gap_std,

        "Full_LOGO_Accuracy":
            float(
                logo_mean
            ),

        "Full_LOGO_STD":
            float(
                logo_std
            ),

        "Full_LOGO_Folds_Available":
            int(
                FULL_LOGO_FOLDS
            ),

        "Full_LOGO_Folds_Completed":
            int(
                len(
                    logo_table
                )
            ),

        **test_metrics,

        "Optuna_Time_Seconds":
            float(
                optuna_time
            ),

        "Five_Run_Time_Seconds":
            float(
                final_run_time
            ),

        "LOGO_Time_Seconds":
            float(
                logo_time
            ),

        "Algorithm_Total_Time_Seconds":
            float(
                algorithm_time
            ),

        "Current_RAM_MB":
            float(
                memory_mb()
            ),
    }

    if spec[
        "uses_modalities"
    ]:
        final_metrics[
            "Modality_Grouping_Mode"
        ] = (
            MODALITY_GROUPING_MODE
        )

        final_metrics[
            "Modality_Names"
        ] = (
            MODALITY_NAMES
        )

    save_json(
        final_metrics,
        output_dir
        / "final_metrics.json"
    )

    # =========================================================================
    # SUMMARY TEXT
    # =========================================================================

    summary_lines = [
        algorithm_name,
        "MED DATASET",
        "",
        f"Input File: {FILE_PATH}",
        f"Output Directory: {output_dir}",
        "",
        f"Device: {DEVICE_NAME}",
        f"Features: {NUMBER_FEATURES}",
        f"Classes: {CLASS_NAMES}",
        f"Trainable Parameters: {trainable_parameters}",
        "",
        f"Training Participants: {len(TRAIN_PERSONS)}",
        f"Validation Participants: {len(VALIDATION_PERSONS)}",
        f"Testing Participants: {len(TEST_PERSONS)}",
        "",
        f"Best Parameters: {best_params}",
        "",
        f"Selected Run: {best_record['Run']}",
        f"Selected Seed: {best_record['Seed']}",
        f"Best Epoch: {best_record['Best_Epoch']}",
        "",
        f"Optuna GroupKFold Accuracy: {study.best_value:.4f}",
        f"Training Accuracy: {best_record['Training_Accuracy']:.4f}",
        f"Validation Accuracy: {best_record['Validation_Accuracy']:.4f}",
        f"Generalization Gap: {best_record['Generalization_Gap']:.4f}",
        f"Testing Accuracy: {test_metrics['Testing_Accuracy']:.4f}",
        "",
        f"Five Run Training: {train_mean:.4f} plus or minus {train_std:.4f}",
        f"Five Run Validation: {validation_mean:.4f} plus or minus {validation_std:.4f}",
        f"Full LOGO Accuracy: {logo_mean:.4f} plus or minus {logo_std:.4f}",
        f"Full LOGO Folds: {len(logo_table)}",
        "",
        f"Precision Weighted: {test_metrics['Precision_Weighted']:.4f}",
        f"Recall Weighted: {test_metrics['Recall_Weighted']:.4f}",
        f"F1 Weighted: {test_metrics['F1_Weighted']:.4f}",
        f"F1 Macro: {test_metrics['F1_Macro']:.4f}",
        f"Cohen Kappa: {test_metrics['Cohen_Kappa']:.4f}",
        f"MCC: {test_metrics['MCC']:.4f}",
        (
            "ROC AUC Weighted OVR: N/A"
            if test_metrics[
                "ROC_AUC_Weighted_OVR"
            ] is None
            else
            f"ROC AUC Weighted OVR: "
            f"{test_metrics['ROC_AUC_Weighted_OVR']:.4f}"
        ),
        "",
        f"Optuna Time: {optuna_time / 60:.2f} minutes",
        f"Five Run Time: {final_run_time / 60:.2f} minutes",
        f"LOGO Time: {logo_time / 60:.2f} minutes",
        f"Algorithm Total Time: {algorithm_time / 60:.2f} minutes",
    ]

    if spec[
        "uses_modalities"
    ]:
        summary_lines.extend(
            [
                "",
                f"Modality Grouping: {MODALITY_GROUPING_MODE}",
                f"Modalities: {MODALITY_NAMES}",
            ]
        )

    with open(
        output_dir
        / "final_summary.txt",
        "w",
        encoding="utf8"
    ) as file:
        file.write(
            "\n".join(
                summary_lines
            )
        )

    # =========================================================================
    # FINAL SCREEN SUMMARY
    # =========================================================================

    print()
    print("#" * 90)
    print(
        algorithm_name
    )
    print("FINAL RESULTS")
    print("#" * 90)

    print(
        "Optuna GroupKFold         :",
        f"{study.best_value:.4f}"
    )

    print(
        "Training Accuracy         :",
        f"{best_record['Training_Accuracy']:.4f}"
    )

    print(
        "Validation Accuracy       :",
        f"{best_record['Validation_Accuracy']:.4f}"
    )

    print(
        "Testing Accuracy          :",
        f"{test_metrics['Testing_Accuracy']:.4f}"
    )

    print(
        "Generalization Gap        :",
        f"{best_record['Generalization_Gap']:.4f}"
    )

    print(
        "Full LOGO Accuracy        :",
        f"{logo_mean:.4f}",
        "plus or minus",
        f"{logo_std:.4f}"
    )

    print(
        "Full LOGO Folds           :",
        len(
            logo_table
        ),
        "of",
        FULL_LOGO_FOLDS
    )

    print(
        "Precision Weighted        :",
        f"{test_metrics['Precision_Weighted']:.4f}"
    )

    print(
        "Recall Weighted           :",
        f"{test_metrics['Recall_Weighted']:.4f}"
    )

    print(
        "F1 Weighted               :",
        f"{test_metrics['F1_Weighted']:.4f}"
    )

    print(
        "F1 Macro                  :",
        f"{test_metrics['F1_Macro']:.4f}"
    )

    print(
        "Cohen Kappa               :",
        f"{test_metrics['Cohen_Kappa']:.4f}"
    )

    print(
        "MCC                       :",
        f"{test_metrics['MCC']:.4f}"
    )

    if (
        test_metrics[
            "ROC_AUC_Weighted_OVR"
        ]
        is None
    ):
        print(
            "ROC AUC Weighted OVR      : N/A"
        )

    else:
        print(
            "ROC AUC Weighted OVR      :",
            f"{test_metrics['ROC_AUC_Weighted_OVR']:.4f}"
        )

    print(
        "Algorithm Time            :",
        f"{algorithm_time / 60:.2f}",
        "minutes"
    )

    print()
    print(
        "RESULTS SAVED IN:"
    )

    print(
        output_dir
    )

    print("#" * 90)

    del study
    del best_bundle

    clear_device()

    return final_metrics


# =============================================================================
# RUN ALL FOUR ALGORITHMS
# =============================================================================

ALL_RESULTS = []


for algorithm_key in ALGORITHMS_TO_RUN:

    try:

        result = run_algorithm(
            algorithm_key
        )

        ALL_RESULTS.append(
            result
        )

    except KeyboardInterrupt:
        print()
        print(
            "Execution was manually interrupted."
        )

        print(
            "Completed checkpoints remain saved."
        )

        raise

    except Exception as error:
        spec = ALGORITHM_SPECS[
            algorithm_key
        ]

        error_output = (
            OUTPUT_ROOT
            / spec[
                "folder"
            ]
        )

        error_output.mkdir(
            parents=True,
            exist_ok=True
        )

        error_text = traceback.format_exc()

        with open(
            error_output
            / "last_error.txt",
            "w",
            encoding="utf8"
        ) as file:
            file.write(
                error_text
            )

        print()
        print("=" * 90)
        print(
            "ERROR IN:"
        )

        print(
            spec[
                "display_name"
            ]
        )

        print("=" * 90)

        print(
            error_text
        )

        print(
            "Error log saved in:"
        )

        print(
            error_output
            / "last_error.txt"
        )

        raise


# =============================================================================
# PROGRAM FINISHED
# =============================================================================

program_time = (
    time.time()
    - PROGRAM_START
)

print()
print("=" * 90)
print("ALL FOUR MED DATASET DL EXPERIMENTS COMPLETED")
print("=" * 90)

print(
    "Total program time:",
    f"{program_time / 60:.2f}",
    "minutes"
)

print(
    "Final RAM:",
    f"{memory_mb():.2f} MB"
)

print()
print("RESULT FOLDERS")

for algorithm_key in ALGORITHMS_TO_RUN:
    spec = ALGORITHM_SPECS[
        algorithm_key
    ]

    print(
        OUTPUT_ROOT
        / spec[
            "folder"
        ]
    )

print("=" * 90)

/home/rakib/Desktop/ml_apply/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE CONFIGURATION
PyTorch version : 2.14.1+xpu
Device          : xpu:0
Device type     : xpu
Device name     : Intel(R) Arc(TM) A750 Graphics
CPU threads     : 8

LOADING MED DATASET
Raw dataset shape: (156741, 102)
RAM after loading: 836.69 MB
Feature count: 96
Invalid feature cells: 0
Feature matrix: (156741, 96)
Classes: ['class_lecture', 'music', 'normal', 'reading']
Number of classes: 4
Participants: 60

CLASS DISTRIBUTION
class_lecture: 41000 (26.16%)
music: 33781 (21.55%)
normal: 42471 (27.10%)
reading: 39489 (25.19%)
Dataset signature: bd8d92dde88694cb
Loading time: 3.56 seconds

FEATURE MODALITY CONSTRUCTION
Grouping mode: Balanced feature view fallback
feature_view_1 : 32 features
feature_view_2 : 32 features
feature_view_3 : 32 features

PERSON WISE 70 / 20 / 10 SPLIT
Training participants: 42
Validation participants: 6
Testing participants: 12

Training: (109891, 96)
Validation: (16018, 96)
Testing: (30832, 96)
GroupKFold folds: 5
Full LOGO folds: 42
RAM after split: 899

**Apply DL On Min Dataset**
**Apply 4 DL Algorithm on Min Dataset**

*DL Models:*

*1. CNN*

*2. EEG-Conformer + 1D-TCN (Dual-Stream Cross-Attention Network)*

*3. Cross-Attention Multimodal Transformer*

*4. Multimodal CNN-LSTM*

In [1]:
# =============================================================================
# FOUR DEEP LEARNING MODELS
# MIN DATASET
#
# 1. CNN
# 2. EEG Conformer + 1D TCN Dual Stream Cross Attention
# 3. Cross Attention Multimodal Transformer
# 4. Multimodal CNN LSTM
#
# INPUT
# /home/rakib/Desktop/ml_apply/DL_Apply/min_dataset/
# Cleaned_oN_Selected_Min.csv
#
# OUTPUT ROOT
# /home/rakib/Desktop/ml_apply/DL_Apply/min_dataset
#
# EACH ALGORITHM GETS ITS OWN RESULT FOLDER
#
# GPU PRIORITY
# Intel XPU
# CUDA
# CPU
#
# RESTART SUPPORT
# Persistent Optuna SQLite
# Final run checkpoints
# Full LOGO checkpoints
#
# TEST DATA ARE USED ONLY FOR FINAL EVALUATION
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import gc
import json
import time
import math
import random
import hashlib
import traceback
import warnings

from pathlib import Path

import joblib
import psutil
import optuna

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import (
    GroupKFold,
    LeaveOneGroupOut
)

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.utils.class_weight import (
    compute_class_weight
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# =============================================================================
# WARNINGS
# =============================================================================

warnings.filterwarnings(
    "ignore"
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# =============================================================================
# GLOBAL EXPERIMENT CONFIGURATION
# =============================================================================

SEED = 42

N_TRIALS = 10

N_RUNS = 5

GROUP_K_FOLDS = 5

LOGO_MAX_FOLDS = None

GRADIENT_CLIP = 1.0

LR_SCHEDULER_PATIENCE = 2

EARLY_STOPPING_MIN_DELTA = 1e-4


# =============================================================================
# CPU THREAD CONTROL
# =============================================================================

CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 1
    )
)


torch.set_num_threads(
    CPU_THREADS
)


# =============================================================================
# ALGORITHMS TO RUN
# =============================================================================

ALGORITHMS_TO_RUN = [
    "CNN",
    "EEG_CONFORMER_TCN",
    "CROSS_ATTENTION_MULTIMODAL_TRANSFORMER",
    "MULTIMODAL_CNN_LSTM"
]


# =============================================================================
# INPUT PATH
# =============================================================================

FILE_PATH = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/min_dataset/"
    "Cleaned_oN_Selected_Min.csv"
)


# =============================================================================
# OUTPUT ROOT
# =============================================================================

OUTPUT_ROOT = Path(
    "/home/rakib/Desktop/ml_apply/"
    "DL_Apply/min_dataset"
)


if not FILE_PATH.exists():

    raise FileNotFoundError(
        f"Dataset was not found: {FILE_PATH}"
    )


OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# PROCESS MEMORY
# =============================================================================

PROCESS = psutil.Process(
    os.getpid()
)


def memory_mb():

    return (
        PROCESS.memory_info().rss
        / 1024
        / 1024
    )


# =============================================================================
# ATOMIC CSV SAVE
# =============================================================================

def atomic_csv(
    dataframe,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name
        + ".tmp"
    )


    dataframe.to_csv(
        temporary,
        index=False
    )


    os.replace(
        temporary,
        path
    )


# =============================================================================
# ATOMIC TORCH SAVE
# =============================================================================

def atomic_torch_save(
    obj,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name
        + ".tmp"
    )


    torch.save(
        obj,
        temporary
    )


    os.replace(
        temporary,
        path
    )


# =============================================================================
# ATOMIC JSON SAVE
# =============================================================================

def save_json(
    obj,
    path
):

    path = Path(
        path
    )


    temporary = path.with_name(
        path.name
        + ".tmp"
    )


    with open(
        temporary,
        "w",
        encoding="utf8"
    ) as file:

        json.dump(
            obj,
            file,
            indent=4
        )


    os.replace(
        temporary,
        path
    )


# =============================================================================
# RANDOM SEED
# =============================================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )


    np.random.seed(
        seed
    )


    torch.manual_seed(
        seed
    )


    if (
        hasattr(
            torch,
            "xpu"
        )
        and
        torch.xpu.is_available()
    ):

        try:

            torch.xpu.manual_seed_all(
                seed
            )

        except Exception:

            pass


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    SEED
)


# =============================================================================
# DEVICE DETECTION
#
# Priority:
#
# Intel XPU
# CUDA
# CPU
# =============================================================================

if (
    hasattr(
        torch,
        "xpu"
    )
    and
    torch.xpu.is_available()
):

    DEVICE = torch.device(
        "xpu:0"
    )

    DEVICE_TYPE = "xpu"

    DEVICE_NAME = torch.xpu.get_device_name(
        0
    )


elif torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda:0"
    )

    DEVICE_TYPE = "cuda"

    DEVICE_NAME = torch.cuda.get_device_name(
        0
    )


else:

    DEVICE = torch.device(
        "cpu"
    )

    DEVICE_TYPE = "cpu"

    DEVICE_NAME = "CPU"


# =============================================================================
# DEVICE HELPERS
# =============================================================================

def synchronize():

    if DEVICE_TYPE == "xpu":

        torch.xpu.synchronize()


    elif DEVICE_TYPE == "cuda":

        torch.cuda.synchronize()


def clear_device():

    gc.collect()


    if DEVICE_TYPE == "xpu":

        try:

            torch.xpu.empty_cache()

        except Exception:

            pass


    elif DEVICE_TYPE == "cuda":

        torch.cuda.empty_cache()


# =============================================================================
# DEVICE INFORMATION
# =============================================================================

print(
    "=" * 90
)

print(
    "DEVICE CONFIGURATION"
)

print(
    "=" * 90
)

print(
    "PyTorch version :",
    torch.__version__
)

print(
    "Selected device :",
    DEVICE
)

print(
    "Device type     :",
    DEVICE_TYPE
)

print(
    "Device name     :",
    DEVICE_NAME
)

print(
    "CPU threads     :",
    CPU_THREADS
)

print(
    "=" * 90
)


# =============================================================================
# TOTAL PROGRAM TIMER
# =============================================================================

PROGRAM_START = time.time()


# =============================================================================
# LOAD MIN DATASET
# =============================================================================

print()

print(
    "=" * 90
)

print(
    "LOADING MIN DATASET"
)

print(
    "=" * 90
)


load_start = time.time()


df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)


print(
    "Raw dataset shape:",
    df.shape
)


print(
    "RAM after loading:",
    f"{memory_mb():.2f} MB"
)


# =============================================================================
# COLUMN DEFINITIONS
# =============================================================================

METADATA_COLUMNS = [
    "PersonSrial",
    "Window",
    "Window_Start",
    "Window_End",
    "Window_Samples"
]


TARGET_COLUMN = "Class"


for required_column in [
    "PersonSrial",
    TARGET_COLUMN
]:

    if required_column not in df.columns:

        raise ValueError(
            f"Missing required column: "
            f"{required_column}"
        )


# =============================================================================
# FEATURE COLUMNS
# =============================================================================

FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in METADATA_COLUMNS
    and column != TARGET_COLUMN
]


NUMBER_FEATURES = len(
    FEATURE_COLUMNS
)


if NUMBER_FEATURES == 0:

    raise ValueError(
        "No feature columns were found."
    )


if NUMBER_FEATURES < 3:

    raise ValueError(
        "At least three features are required "
        "for the multimodal feature view models."
    )


print(
    "Feature count:",
    NUMBER_FEATURES
)


# =============================================================================
# FEATURE MATRIX
# =============================================================================

X_ALL = (
    df[
        FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(
        dtype=np.float32
    )
)


# =============================================================================
# INVALID DATA CHECK
# =============================================================================

invalid_count = int(
    (
        ~np.isfinite(
            X_ALL
        )
    ).sum()
)


print(
    "Invalid feature cells:",
    invalid_count
)


if invalid_count > 0:

    raise ValueError(
        "Dataset contains NaN or infinite feature values."
    )


# =============================================================================
# LABEL ENCODING
# =============================================================================

LABEL_ENCODER = LabelEncoder()


Y_ALL = LABEL_ENCODER.fit_transform(
    df[
        TARGET_COLUMN
    ].astype(str)
).astype(
    np.int64
)


CLASS_NAMES = list(
    LABEL_ENCODER.classes_
)


NUMBER_CLASSES = len(
    CLASS_NAMES
)


# =============================================================================
# PARTICIPANTS
# =============================================================================

GROUPS_ALL = (
    df[
        "PersonSrial"
    ]
    .to_numpy()
)


print()

print(
    "Feature matrix:",
    X_ALL.shape
)

print(
    "Classes:",
    CLASS_NAMES
)

print(
    "Number of classes:",
    NUMBER_CLASSES
)

print(
    "Participants:",
    len(
        np.unique(
            GROUPS_ALL
        )
    )
)


# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

CLASS_DISTRIBUTION_RECORDS = []


print()

print(
    "CLASS DISTRIBUTION"
)


for class_index, class_name in enumerate(
    CLASS_NAMES
):

    count = int(
        np.sum(
            Y_ALL
            == class_index
        )
    )


    percentage = (
        count
        / len(
            Y_ALL
        )
        * 100.0
    )


    print(
        f"{class_name}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )


    CLASS_DISTRIBUTION_RECORDS.append(
        {
            "Class":
                class_name,

            "Count":
                count,

            "Percentage":
                percentage
        }
    )


CLASS_DISTRIBUTION_TABLE = pd.DataFrame(
    CLASS_DISTRIBUTION_RECORDS
)


# =============================================================================
# DATASET SIGNATURE
#
# Used so that results from a different revision of the dataset
# are not accidentally reused.
# =============================================================================

file_stat = FILE_PATH.stat()


dataset_signature_payload = {

    "path":
        str(
            FILE_PATH
        ),

    "size":
        int(
            file_stat.st_size
        ),

    "mtime_ns":
        int(
            file_stat.st_mtime_ns
        ),

    "rows":
        int(
            len(
                Y_ALL
            )
        ),

    "features":
        FEATURE_COLUMNS,

    "classes":
        CLASS_NAMES
}


DATASET_SIGNATURE = hashlib.sha256(
    json.dumps(
        dataset_signature_payload,
        sort_keys=True
    ).encode(
        "utf8"
    )
).hexdigest()[
    :16
]


print(
    "Dataset signature:",
    DATASET_SIGNATURE
)


print(
    "Loading time:",
    f"{time.time() - load_start:.2f}",
    "seconds"
)


# =============================================================================
# FEATURE MODALITY DETECTION
#
# These are feature families or feature views.
#
# They are not automatically claimed to represent different physical sensors.
# =============================================================================

SPECTRAL_KEYWORDS = [
    "fft",
    "psd",
    "power",
    "bandpower",
    "band_power",
    "spectral",
    "frequency",
    "freq",
    "delta",
    "theta",
    "alpha",
    "beta",
    "gamma"
]


STATISTICAL_KEYWORDS = [
    "mean",
    "median",
    "std",
    "variance",
    "var",
    "rms",
    "skew",
    "kurt",
    "minimum",
    "maximum",
    "min",
    "max",
    "range",
    "iqr",
    "quartile",
    "percentile",
    "peak",
    "zero",
    "zcr",
    "activity"
]


COMPLEXITY_KEYWORDS = [
    "entropy",
    "complexity",
    "fractal",
    "hurst",
    "dfa",
    "lyapunov",
    "sampen",
    "sample_entropy",
    "apen",
    "approximate_entropy",
    "permutation",
    "lzc",
    "lempel",
    "higuchi",
    "petrosian",
    "mobility"
]


def contains_keyword(
    feature_name,
    keywords
):

    feature_name = str(
        feature_name
    ).lower()


    return any(
        keyword
        in feature_name

        for keyword
        in keywords
    )


def construct_modalities(
    columns
):

    spectral = []

    statistical = []

    complexity = []

    unassigned = []


    for index, column in enumerate(
        columns
    ):

        if contains_keyword(
            column,
            COMPLEXITY_KEYWORDS
        ):

            complexity.append(
                index
            )


        elif contains_keyword(
            column,
            SPECTRAL_KEYWORDS
        ):

            spectral.append(
                index
            )


        elif contains_keyword(
            column,
            STATISTICAL_KEYWORDS
        ):

            statistical.append(
                index
            )


        else:

            unassigned.append(
                index
            )


    groups = [
        spectral,
        statistical,
        complexity
    ]


    # =========================================================================
    # Distribute unrecognized features among the smallest streams.
    # =========================================================================

    for index in unassigned:

        smallest_group = int(
            np.argmin(
                [
                    len(
                        group
                    )
                    for group
                    in groups
                ]
            )
        )


        groups[
            smallest_group
        ].append(
            index
        )


    minimum_size = min(
        len(
            group
        )
        for group
        in groups
    )


    maximum_size = max(
        len(
            group
        )
        for group
        in groups
    )


    # =========================================================================
    # Min dataset may contain fewer selected features.
    #
    # Semantic grouping is retained only when all streams have
    # enough features to form meaningful model inputs.
    # =========================================================================

    semantic_grouping_valid = (
        minimum_size >= 8
        and
        maximum_size
        <= NUMBER_FEATURES * 0.70
    )


    if semantic_grouping_valid:

        grouping_mode = (
            "Feature name based semantic grouping"
        )


        modality_indices = {

            "spectral_frequency":
                sorted(
                    spectral
                ),

            "statistical_temporal":
                sorted(
                    statistical
                ),

            "nonlinear_complexity":
                sorted(
                    complexity
                )
        }


    else:

        grouping_mode = (
            "Balanced feature view fallback"
        )


        balanced_groups = np.array_split(
            np.arange(
                NUMBER_FEATURES
            ),
            3
        )


        modality_indices = {

            "feature_view_1":
                balanced_groups[
                    0
                ].tolist(),

            "feature_view_2":
                balanced_groups[
                    1
                ].tolist(),

            "feature_view_3":
                balanced_groups[
                    2
                ].tolist()
        }


    for name, indices in modality_indices.items():

        if len(
            indices
        ) == 0:

            raise ValueError(
                f"Empty modality detected: {name}"
            )


    return (
        modality_indices,
        grouping_mode
    )


(
    MODALITY_INDICES,
    MODALITY_GROUPING_MODE

) = construct_modalities(
    FEATURE_COLUMNS
)


MODALITY_NAMES = list(
    MODALITY_INDICES.keys()
)


NUMBER_MODALITIES = len(
    MODALITY_NAMES
)


print()

print(
    "=" * 90
)

print(
    "FEATURE MODALITY CONSTRUCTION"
)

print(
    "=" * 90
)


print(
    "Grouping mode:",
    MODALITY_GROUPING_MODE
)


for modality_name in MODALITY_NAMES:

    print(
        modality_name,
        ":",
        len(
            MODALITY_INDICES[
                modality_name
            ]
        ),
        "features"
    )


# =============================================================================
# RELEASE DATAFRAME
# =============================================================================

del df

gc.collect()


# =============================================================================
# PERSON WISE 70 / 20 / 10 SPLIT
# =============================================================================

print()

print(
    "=" * 90
)

print(
    "PERSON WISE 70 / 20 / 10 SPLIT"
)

print(
    "=" * 90
)


persons = np.unique(
    GROUPS_ALL
)


rng = np.random.RandomState(
    SEED
)


rng.shuffle(
    persons
)


number_persons = len(
    persons
)


number_test = max(
    1,
    int(
        round(
            0.20
            * number_persons
        )
    )
)


number_validation = max(
    1,
    int(
        round(
            0.10
            * number_persons
        )
    )
)


TEST_PERSONS = persons[
    :number_test
]


VALIDATION_PERSONS = persons[
    number_test:
    number_test
    + number_validation
]


TRAIN_PERSONS = persons[
    number_test
    + number_validation:
]


# =============================================================================
# VERIFY PARTICIPANT ISOLATION
# =============================================================================

assert (
    np.intersect1d(
        TRAIN_PERSONS,
        VALIDATION_PERSONS
    ).size == 0
)


assert (
    np.intersect1d(
        TRAIN_PERSONS,
        TEST_PERSONS
    ).size == 0
)


assert (
    np.intersect1d(
        VALIDATION_PERSONS,
        TEST_PERSONS
    ).size == 0
)


# =============================================================================
# MASKS
# =============================================================================

train_mask = np.isin(
    GROUPS_ALL,
    TRAIN_PERSONS
)


validation_mask = np.isin(
    GROUPS_ALL,
    VALIDATION_PERSONS
)


test_mask = np.isin(
    GROUPS_ALL,
    TEST_PERSONS
)


# =============================================================================
# RAW SPLITS
# =============================================================================

X_TRAIN_RAW = np.ascontiguousarray(
    X_ALL[
        train_mask
    ],
    dtype=np.float32
)


Y_TRAIN = np.ascontiguousarray(
    Y_ALL[
        train_mask
    ],
    dtype=np.int64
)


G_TRAIN = np.asarray(
    GROUPS_ALL[
        train_mask
    ]
)


X_VALIDATION_RAW = np.ascontiguousarray(
    X_ALL[
        validation_mask
    ],
    dtype=np.float32
)


Y_VALIDATION = np.ascontiguousarray(
    Y_ALL[
        validation_mask
    ],
    dtype=np.int64
)


G_VALIDATION = np.asarray(
    GROUPS_ALL[
        validation_mask
    ]
)


X_TEST_RAW = np.ascontiguousarray(
    X_ALL[
        test_mask
    ],
    dtype=np.float32
)


Y_TEST = np.ascontiguousarray(
    Y_ALL[
        test_mask
    ],
    dtype=np.int64
)


G_TEST = np.asarray(
    GROUPS_ALL[
        test_mask
    ]
)


# =============================================================================
# COUNTS
# =============================================================================

TRAIN_SAMPLES = len(
    X_TRAIN_RAW
)


VALIDATION_SAMPLES = len(
    X_VALIDATION_RAW
)


TEST_SAMPLES = len(
    X_TEST_RAW
)


FULL_LOGO_FOLDS = len(
    np.unique(
        G_TRAIN
    )
)


print(
    "Training participants:",
    len(
        TRAIN_PERSONS
    )
)

print(
    "Validation participants:",
    len(
        VALIDATION_PERSONS
    )
)

print(
    "Testing participants:",
    len(
        TEST_PERSONS
    )
)

print()

print(
    "Training:",
    X_TRAIN_RAW.shape
)

print(
    "Validation:",
    X_VALIDATION_RAW.shape
)

print(
    "Testing:",
    X_TEST_RAW.shape
)

print()

print(
    "GroupKFold folds:",
    GROUP_K_FOLDS
)

print(
    "Full LOGO folds:",
    FULL_LOGO_FOLDS
)

print(
    "RAM after split:",
    f"{memory_mb():.2f} MB"
)


# =============================================================================
# SPLIT SUMMARY
# =============================================================================

SPLIT_SUMMARY_TABLE = pd.DataFrame(
    [

        {
            "Split":
                "Training",

            "Participants":
                len(
                    TRAIN_PERSONS
                ),

            "Rows":
                TRAIN_SAMPLES
        },

        {
            "Split":
                "Validation",

            "Participants":
                len(
                    VALIDATION_PERSONS
                ),

            "Rows":
                VALIDATION_SAMPLES
        },

        {
            "Split":
                "Testing",

            "Participants":
                len(
                    TEST_PERSONS
                ),

            "Rows":
                TEST_SAMPLES
        }
    ]
)


PARTICIPANT_SPLIT = {

    "training":
        [
            str(
                x
            )
            for x
            in TRAIN_PERSONS
        ],

    "validation":
        [
            str(
                x
            )
            for x
            in VALIDATION_PERSONS
        ],

    "testing":
        [
            str(
                x
            )
            for x
            in TEST_PERSONS
        ]
}


# =============================================================================
# RELEASE FULL ARRAYS
# =============================================================================

del X_ALL

del Y_ALL

del GROUPS_ALL

del train_mask

del validation_mask

del test_mask

gc.collect()


# =============================================================================
# FINAL TRAINING SCALER
#
# Used only for final five runs and final test.
#
# Every GroupKFold and LOGO fold fits its own independent scaler.
# =============================================================================

FINAL_SCALER = StandardScaler()


X_TRAIN = FINAL_SCALER.fit_transform(
    X_TRAIN_RAW
).astype(
    np.float32,
    copy=False
)


X_VALIDATION = FINAL_SCALER.transform(
    X_VALIDATION_RAW
).astype(
    np.float32,
    copy=False
)


X_TEST = FINAL_SCALER.transform(
    X_TEST_RAW
).astype(
    np.float32,
    copy=False
)


X_TRAIN = np.ascontiguousarray(
    X_TRAIN
)


X_VALIDATION = np.ascontiguousarray(
    X_VALIDATION
)


X_TEST = np.ascontiguousarray(
    X_TEST
)


# =============================================================================
# FRESH GROUPKFOLD ITERATOR
# =============================================================================

def group_kfold_iterator():

    splitter = GroupKFold(
        n_splits=
            GROUP_K_FOLDS
    )


    return splitter.split(
        X_TRAIN_RAW,
        Y_TRAIN,
        G_TRAIN
    )


# =============================================================================
# FOLD SPECIFIC STANDARDIZATION
# =============================================================================

def prepare_fold(
    training_index,
    validation_index
):

    scaler = StandardScaler()


    X_fold_training = scaler.fit_transform(
        X_TRAIN_RAW[
            training_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    X_fold_validation = scaler.transform(
        X_TRAIN_RAW[
            validation_index
        ]
    ).astype(
        np.float32,
        copy=False
    )


    return (

        np.ascontiguousarray(
            X_fold_training
        ),

        np.ascontiguousarray(
            X_fold_validation
        )
    )


# =============================================================================
# CLASS WEIGHTS
# =============================================================================

def get_class_weights(
    y
):

    present_classes = np.unique(
        y
    )


    calculated = compute_class_weight(
        class_weight="balanced",
        classes=
            present_classes,
        y=y
    )


    weights = np.ones(
        NUMBER_CLASSES,
        dtype=np.float32
    )


    weights[
        present_classes
    ] = calculated.astype(
        np.float32
    )


    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


# =============================================================================
# BATCH GENERATOR
#
# Dataset remains in system RAM.
#
# Only the current mini batch is moved to the GPU.
# =============================================================================

def batch_generator(
    X,
    y,
    batch_size,
    shuffle=False,
    rng=None
):

    number_samples = len(
        y
    )


    indices = np.arange(
        number_samples
    )


    if shuffle:

        if rng is None:

            np.random.shuffle(
                indices
            )

        else:

            rng.shuffle(
                indices
            )


    for start in range(
        0,
        number_samples,
        batch_size
    ):

        batch_indices = indices[
            start:
            start
            + batch_size
        ]


        X_numpy = np.ascontiguousarray(
            X[
                batch_indices
            ],
            dtype=np.float32
        )


        y_numpy = np.ascontiguousarray(
            y[
                batch_indices
            ],
            dtype=np.int64
        )


        X_batch = torch.from_numpy(
            X_numpy
        ).to(
            DEVICE
        )


        y_batch = torch.from_numpy(
            y_numpy
        ).to(
            DEVICE
        )


        yield (
            X_batch,
            y_batch
        )


# =============================================================================
# MODEL 1
# CNN
# =============================================================================

class CNN1D(
    nn.Module
):

    def __init__(
        self,
        params
    ):

        super().__init__()


        n_blocks = int(
            params[
                "n_conv_blocks"
            ]
        )


        base_filters = int(
            params[
                "base_filters"
            ]
        )


        kernel_size = int(
            params[
                "kernel_size"
            ]
        )


        conv_dropout = float(
            params[
                "conv_dropout"
            ]
        )


        padding = (
            kernel_size
            // 2
        )


        layers = []

        input_channels = 1


        for block_index in range(
            n_blocks
        ):

            output_channels = (
                base_filters
                * (
                    2
                    ** block_index
                )
            )


            layers.extend(
                [

                    nn.Conv1d(
                        input_channels,
                        output_channels,
                        kernel_size=
                            kernel_size,
                        padding=
                            padding,
                        bias=False
                    ),

                    nn.BatchNorm1d(
                        output_channels
                    ),

                    nn.GELU(),

                    nn.Conv1d(
                        output_channels,
                        output_channels,
                        kernel_size=
                            kernel_size,
                        padding=
                            padding,
                        bias=False
                    ),

                    nn.BatchNorm1d(
                        output_channels
                    ),

                    nn.GELU(),

                    nn.MaxPool1d(
                        kernel_size=2,
                        ceil_mode=True
                    ),

                    nn.Dropout(
                        conv_dropout
                    )
                ]
            )


            input_channels = (
                output_channels
            )


        self.features = nn.Sequential(
            *layers
        )


        self.global_pool = nn.AdaptiveAvgPool1d(
            1
        )


        dense_units = int(
            params[
                "dense_units"
            ]
        )


        dense_dropout = float(
            params[
                "dense_dropout"
            ]
        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                input_channels,
                dense_units
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                dense_units,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        x = x.unsqueeze(
            1
        )


        x = self.features(
            x
        )


        x = self.global_pool(
            x
        )


        return self.classifier(
            x
        )


# =============================================================================
# SINUSOIDAL POSITION ENCODING
# =============================================================================

class SinusoidalPositionEncoding(
    nn.Module
):

    def __init__(
        self,
        dimension,
        max_length=512
    ):

        super().__init__()


        position = torch.arange(
            max_length,
            dtype=torch.float32
        ).unsqueeze(
            1
        )


        divisor = torch.exp(
            torch.arange(
                0,
                dimension,
                2,
                dtype=torch.float32
            )
            * (
                -math.log(
                    10000.0
                )
                / dimension
            )
        )


        encoding = torch.zeros(
            max_length,
            dimension,
            dtype=torch.float32
        )


        encoding[
            :,
            0::2
        ] = torch.sin(
            position
            * divisor
        )


        encoding[
            :,
            1::2
        ] = torch.cos(
            position
            * divisor
        )


        self.register_buffer(
            "encoding",
            encoding.unsqueeze(
                0
            ),
            persistent=False
        )


    def forward(
        self,
        x
    ):

        return (
            x
            + self.encoding[
                :,
                :x.size(
                    1
                ),
                :
            ].to(
                dtype=x.dtype
            )
        )


# =============================================================================
# CUSTOM MULTI HEAD ATTENTION
#
# Uses basic matrix operations for broad Intel XPU compatibility.
# =============================================================================

class CustomMultiHeadAttention(
    nn.Module
):

    def __init__(
        self,
        dimension,
        number_heads,
        dropout
    ):

        super().__init__()


        if (
            dimension
            % number_heads
            != 0
        ):

            raise ValueError(
                "dimension must be divisible "
                "by number_heads"
            )


        self.dimension = (
            dimension
        )


        self.number_heads = (
            number_heads
        )


        self.head_dimension = (
            dimension
            // number_heads
        )


        self.scale = (
            self.head_dimension
            ** -0.5
        )


        self.query_projection = nn.Linear(
            dimension,
            dimension
        )


        self.key_projection = nn.Linear(
            dimension,
            dimension
        )


        self.value_projection = nn.Linear(
            dimension,
            dimension
        )


        self.output_projection = nn.Linear(
            dimension,
            dimension
        )


        self.attention_dropout = nn.Dropout(
            dropout
        )


        self.output_dropout = nn.Dropout(
            dropout
        )


    def split_heads(
        self,
        x
    ):

        batch_size = x.shape[
            0
        ]


        sequence_length = x.shape[
            1
        ]


        x = x.reshape(
            batch_size,
            sequence_length,
            self.number_heads,
            self.head_dimension
        )


        return x.transpose(
            1,
            2
        )


    def forward(
        self,
        query,
        key,
        value
    ):

        query = self.split_heads(
            self.query_projection(
                query
            )
        )


        key = self.split_heads(
            self.key_projection(
                key
            )
        )


        value = self.split_heads(
            self.value_projection(
                value
            )
        )


        scores = torch.matmul(
            query,
            key.transpose(
                -2,
                -1
            )
        )


        scores = (
            scores
            * self.scale
        )


        attention = torch.softmax(
            scores,
            dim=-1
        )


        attention = self.attention_dropout(
            attention
        )


        context = torch.matmul(
            attention,
            value
        )


        context = context.transpose(
            1,
            2
        ).contiguous()


        batch_size = context.shape[
            0
        ]


        sequence_length = context.shape[
            1
        ]


        context = context.reshape(
            batch_size,
            sequence_length,
            self.dimension
        )


        context = self.output_projection(
            context
        )


        return self.output_dropout(
            context
        )


# =============================================================================
# GENERIC FEED FORWARD
# =============================================================================

class FeedForward(
    nn.Module
):

    def __init__(
        self,
        dimension,
        expansion,
        dropout
    ):

        super().__init__()


        hidden_dimension = (
            dimension
            * expansion
        )


        self.network = nn.Sequential(

            nn.Linear(
                dimension,
                hidden_dimension
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dimension,
                dimension
            ),

            nn.Dropout(
                dropout
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# =============================================================================
# CONFORMER FEED FORWARD
# =============================================================================

class ConformerFeedForward(
    nn.Module
):

    def __init__(
        self,
        dimension,
        dropout
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                dimension,
                dimension * 4
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                dimension * 4,
                dimension
            ),

            nn.Dropout(
                dropout
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# =============================================================================
# CONFORMER CONVOLUTION MODULE
# =============================================================================

class ConformerConvModule(
    nn.Module
):

    def __init__(
        self,
        dimension,
        kernel_size,
        dropout
    ):

        super().__init__()


        self.pointwise_in = nn.Conv1d(
            dimension,
            dimension * 2,
            kernel_size=1
        )


        self.depthwise = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=
                kernel_size,
            padding=
                kernel_size
                // 2,
            groups=
                dimension,
            bias=False
        )


        self.batch_norm = nn.BatchNorm1d(
            dimension
        )


        self.pointwise_out = nn.Conv1d(
            dimension,
            dimension,
            kernel_size=1
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x
    ):

        x = x.transpose(
            1,
            2
        )


        x = self.pointwise_in(
            x
        )


        x = F.glu(
            x,
            dim=1
        )


        x = self.depthwise(
            x
        )


        x = self.batch_norm(
            x
        )


        x = F.silu(
            x
        )


        x = self.pointwise_out(
            x
        )


        x = self.dropout(
            x
        )


        return x.transpose(
            1,
            2
        )


# =============================================================================
# CONFORMER BLOCK
# =============================================================================

class ConformerBlock(
    nn.Module
):

    def __init__(
        self,
        dimension,
        number_heads,
        conv_kernel,
        dropout
    ):

        super().__init__()


        self.ff1_norm = nn.LayerNorm(
            dimension
        )


        self.ff1 = ConformerFeedForward(
            dimension,
            dropout
        )


        self.attention_norm = nn.LayerNorm(
            dimension
        )


        self.self_attention = CustomMultiHeadAttention(
            dimension,
            number_heads,
            dropout
        )


        self.conv_norm = nn.LayerNorm(
            dimension
        )


        self.conv_module = ConformerConvModule(
            dimension,
            conv_kernel,
            dropout
        )


        self.ff2_norm = nn.LayerNorm(
            dimension
        )


        self.ff2 = ConformerFeedForward(
            dimension,
            dropout
        )


        self.final_norm = nn.LayerNorm(
            dimension
        )


    def forward(
        self,
        x
    ):

        x = (
            x
            + 0.5
            * self.ff1(
                self.ff1_norm(
                    x
                )
            )
        )


        normalized = self.attention_norm(
            x
        )


        x = (
            x
            + self.self_attention(
                normalized,
                normalized,
                normalized
            )
        )


        x = (
            x
            + self.conv_module(
                self.conv_norm(
                    x
                )
            )
        )


        x = (
            x
            + 0.5
            * self.ff2(
                self.ff2_norm(
                    x
                )
            )
        )


        return self.final_norm(
            x
        )


# =============================================================================
# TCN RESIDUAL BLOCK
# =============================================================================

class TCNResidualBlock(
    nn.Module
):

    def __init__(
        self,
        channels,
        kernel_size,
        dilation,
        dropout
    ):

        super().__init__()


        padding = (
            dilation
            * (
                kernel_size
                - 1
            )
            // 2
        )


        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=
                kernel_size,
            dilation=
                dilation,
            padding=
                padding,
            bias=False
        )


        self.bn1 = nn.BatchNorm1d(
            channels
        )


        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=
                kernel_size,
            dilation=
                dilation,
            padding=
                padding,
            bias=False
        )


        self.bn2 = nn.BatchNorm1d(
            channels
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x
    ):

        residual = x


        x = self.conv1(
            x
        )


        x = self.bn1(
            x
        )


        x = F.gelu(
            x
        )


        x = self.dropout(
            x
        )


        x = self.conv2(
            x
        )


        x = self.bn2(
            x
        )


        x = F.gelu(
            x
        )


        x = self.dropout(
            x
        )


        return (
            x
            + residual
        )


# =============================================================================
# MODEL 2
# EEG CONFORMER + 1D TCN
# DUAL STREAM CROSS ATTENTION
# =============================================================================

class EEGConformerTCNCrossAttention(
    nn.Module
):

    PATCH_KERNEL_SIZE = 7

    PATCH_STRIDE = 4


    def __init__(
        self,
        params
    ):

        super().__init__()


        dimension = int(
            params[
                "d_model"
            ]
        )


        number_heads = int(
            params[
                "num_heads"
            ]
        )


        n_conformer_blocks = int(
            params[
                "n_conformer_blocks"
            ]
        )


        n_tcn_blocks = int(
            params[
                "n_tcn_blocks"
            ]
        )


        conformer_kernel = int(
            params[
                "conformer_kernel"
            ]
        )


        tcn_kernel = int(
            params[
                "tcn_kernel"
            ]
        )


        dropout = float(
            params[
                "dropout"
            ]
        )


        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )


        self.conformer_stem = nn.Sequential(

            nn.Conv1d(
                1,
                dimension,
                kernel_size=
                    self.PATCH_KERNEL_SIZE,
                stride=
                    self.PATCH_STRIDE,
                padding=
                    self.PATCH_KERNEL_SIZE
                    // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                dimension
            ),

            nn.GELU()
        )


        self.position_encoding = (
            SinusoidalPositionEncoding(
                dimension,
                max_length=512
            )
        )


        self.conformer_blocks = nn.ModuleList(
            [

                ConformerBlock(
                    dimension=
                        dimension,
                    number_heads=
                        number_heads,
                    conv_kernel=
                        conformer_kernel,
                    dropout=
                        dropout
                )

                for _
                in range(
                    n_conformer_blocks
                )
            ]
        )


        self.tcn_stem = nn.Sequential(

            nn.Conv1d(
                1,
                dimension,
                kernel_size=
                    self.PATCH_KERNEL_SIZE,
                stride=
                    self.PATCH_STRIDE,
                padding=
                    self.PATCH_KERNEL_SIZE
                    // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                dimension
            ),

            nn.GELU()
        )


        self.tcn_blocks = nn.ModuleList(
            [

                TCNResidualBlock(
                    channels=
                        dimension,
                    kernel_size=
                        tcn_kernel,
                    dilation=
                        2
                        ** block_index,
                    dropout=
                        dropout
                )

                for block_index
                in range(
                    n_tcn_blocks
                )
            ]
        )


        self.conformer_query_norm = nn.LayerNorm(
            dimension
        )


        self.tcn_context_norm = nn.LayerNorm(
            dimension
        )


        self.tcn_query_norm = nn.LayerNorm(
            dimension
        )


        self.conformer_context_norm = nn.LayerNorm(
            dimension
        )


        self.conformer_queries_tcn = (
            CustomMultiHeadAttention(
                dimension,
                number_heads,
                dropout
            )
        )


        self.tcn_queries_conformer = (
            CustomMultiHeadAttention(
                dimension,
                number_heads,
                dropout
            )
        )


        self.conformer_fusion_norm = nn.LayerNorm(
            dimension
        )


        self.tcn_fusion_norm = nn.LayerNorm(
            dimension
        )


        self.cross_dropout = nn.Dropout(
            dropout
        )


        fusion_dimension = (
            dimension
            * 4
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        x = x.unsqueeze(
            1
        )


        # =========================================================================
        # CONFORMER STREAM
        # =========================================================================

        conformer = self.conformer_stem(
            x
        )


        conformer = conformer.transpose(
            1,
            2
        )


        conformer = self.position_encoding(
            conformer
        )


        for block in self.conformer_blocks:

            conformer = block(
                conformer
            )


        # =========================================================================
        # TCN STREAM
        # =========================================================================

        tcn = self.tcn_stem(
            x
        )


        for block in self.tcn_blocks:

            tcn = block(
                tcn
            )


        tcn = tcn.transpose(
            1,
            2
        )


        # =========================================================================
        # CONFORMER QUERIES TCN
        # =========================================================================

        conformer_query = (
            self.conformer_query_norm(
                conformer
            )
        )


        tcn_context = (
            self.tcn_context_norm(
                tcn
            )
        )


        conformer_cross = (
            self.conformer_queries_tcn(
                conformer_query,
                tcn_context,
                tcn_context
            )
        )


        conformer = self.conformer_fusion_norm(
            conformer
            + self.cross_dropout(
                conformer_cross
            )
        )


        # =========================================================================
        # TCN QUERIES CONFORMER
        # =========================================================================

        tcn_query = self.tcn_query_norm(
            tcn
        )


        conformer_context = (
            self.conformer_context_norm(
                conformer
            )
        )


        tcn_cross = (
            self.tcn_queries_conformer(
                tcn_query,
                conformer_context,
                conformer_context
            )
        )


        tcn = self.tcn_fusion_norm(
            tcn
            + self.cross_dropout(
                tcn_cross
            )
        )


        # =========================================================================
        # FUSION
        # =========================================================================

        conformer_vector = torch.mean(
            conformer,
            dim=1
        )


        tcn_vector = torch.mean(
            tcn,
            dim=1
        )


        difference = torch.abs(
            conformer_vector
            - tcn_vector
        )


        interaction = (
            conformer_vector
            * tcn_vector
        )


        fused = torch.cat(
            [
                conformer_vector,
                tcn_vector,
                difference,
                interaction
            ],
            dim=1
        )


        return self.classifier(
            fused
        )


# =============================================================================
# TRANSFORMER BLOCK
# =============================================================================

class TransformerBlock(
    nn.Module
):

    def __init__(
        self,
        dimension,
        number_heads,
        expansion,
        dropout
    ):

        super().__init__()


        self.attention_norm = nn.LayerNorm(
            dimension
        )


        self.attention = CustomMultiHeadAttention(
            dimension,
            number_heads,
            dropout
        )


        self.ffn_norm = nn.LayerNorm(
            dimension
        )


        self.ffn = FeedForward(
            dimension,
            expansion,
            dropout
        )


    def forward(
        self,
        x
    ):

        normalized = self.attention_norm(
            x
        )


        x = (
            x
            + self.attention(
                normalized,
                normalized,
                normalized
            )
        )


        x = (
            x
            + self.ffn(
                self.ffn_norm(
                    x
                )
            )
        )


        return x


# =============================================================================
# FEATURE TOKENIZER
# =============================================================================

class FeatureTokenizer(
    nn.Module
):

    def __init__(
        self,
        dimension,
        kernel_size,
        stride,
        dropout
    ):

        super().__init__()


        self.tokenizer = nn.Sequential(

            nn.Conv1d(
                1,
                dimension,
                kernel_size=
                    kernel_size,
                stride=
                    stride,
                padding=
                    kernel_size
                    // 2,
                bias=False
            ),

            nn.BatchNorm1d(
                dimension
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )
        )


        self.position = SinusoidalPositionEncoding(
            dimension,
            max_length=512
        )


    def forward(
        self,
        x
    ):

        x = x.unsqueeze(
            1
        )


        x = self.tokenizer(
            x
        )


        x = x.transpose(
            1,
            2
        )


        return self.position(
            x
        )


# =============================================================================
# CROSS MODAL ATTENTION BLOCK
# =============================================================================

class CrossModalAttentionLayer(
    nn.Module
):

    def __init__(
        self,
        dimension,
        number_heads,
        expansion,
        dropout,
        number_modalities
    ):

        super().__init__()


        self.number_modalities = (
            number_modalities
        )


        self.query_norms = nn.ModuleList(
            [

                nn.LayerNorm(
                    dimension
                )

                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.context_norms = nn.ModuleList(
            [

                nn.LayerNorm(
                    dimension
                )

                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.attentions = nn.ModuleList(
            [

                CustomMultiHeadAttention(
                    dimension,
                    number_heads,
                    dropout
                )

                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.output_norms = nn.ModuleList(
            [

                nn.LayerNorm(
                    dimension
                )

                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.ffn_norms = nn.ModuleList(
            [

                nn.LayerNorm(
                    dimension
                )

                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.ffns = nn.ModuleList(
            [

                FeedForward(
                    dimension,
                    expansion,
                    dropout
                )

                for _
                in range(
                    number_modalities
                )
            ]
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        streams
    ):

        original_streams = streams

        updated_streams = []


        for modality_index in range(
            self.number_modalities
        ):

            query_stream = original_streams[
                modality_index
            ]


            other_streams = [

                original_streams[
                    other_index
                ]

                for other_index
                in range(
                    self.number_modalities
                )

                if other_index
                != modality_index
            ]


            context = torch.cat(
                other_streams,
                dim=1
            )


            normalized_query = (
                self.query_norms[
                    modality_index
                ](
                    query_stream
                )
            )


            normalized_context = (
                self.context_norms[
                    modality_index
                ](
                    context
                )
            )


            cross_output = (
                self.attentions[
                    modality_index
                ](
                    normalized_query,
                    normalized_context,
                    normalized_context
                )
            )


            stream = (
                self.output_norms[
                    modality_index
                ](
                    query_stream
                    + self.dropout(
                        cross_output
                    )
                )
            )


            stream = (
                stream
                + self.ffns[
                    modality_index
                ](
                    self.ffn_norms[
                        modality_index
                    ](
                        stream
                    )
                )
            )


            updated_streams.append(
                stream
            )


        return updated_streams


# =============================================================================
# MODALITY STANDARD DEVIATION
#
# Uses simple operations for GPU compatibility.
# =============================================================================

def modality_standard_deviation(
    stacked_modalities
):

    mean = torch.mean(
        stacked_modalities,
        dim=1,
        keepdim=True
    )


    variance = torch.mean(
        (
            stacked_modalities
            - mean
        ) ** 2,
        dim=1
    )


    return torch.sqrt(
        variance
        + 1e-6
    )


# =============================================================================
# MODEL 3
# CROSS ATTENTION MULTIMODAL TRANSFORMER
# =============================================================================

class CrossAttentionMultimodalTransformer(
    nn.Module
):

    def __init__(
        self,
        params
    ):

        super().__init__()


        self.modality_names = list(
            MODALITY_INDICES.keys()
        )


        self.number_modalities = len(
            self.modality_names
        )


        dimension = int(
            params[
                "d_model"
            ]
        )


        number_heads = int(
            params[
                "num_heads"
            ]
        )


        self_layers = int(
            params[
                "self_layers"
            ]
        )


        cross_layers = int(
            params[
                "cross_layers"
            ]
        )


        expansion = int(
            params[
                "ffn_expansion"
            ]
        )


        tokenizer_kernel = int(
            params[
                "tokenizer_kernel"
            ]
        )


        tokenizer_stride = int(
            params[
                "tokenizer_stride"
            ]
        )


        dropout = float(
            params[
                "dropout"
            ]
        )


        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )


        self.modality_buffer_names = []


        for modality_number, modality_name in enumerate(
            self.modality_names
        ):

            buffer_name = (
                f"modality_indices_"
                f"{modality_number}"
            )


            self.register_buffer(
                buffer_name,
                torch.tensor(
                    MODALITY_INDICES[
                        modality_name
                    ],
                    dtype=torch.long
                ),
                persistent=True
            )


            self.modality_buffer_names.append(
                buffer_name
            )


        self.tokenizers = nn.ModuleList(
            [

                FeatureTokenizer(
                    dimension=
                        dimension,
                    kernel_size=
                        tokenizer_kernel,
                    stride=
                        tokenizer_stride,
                    dropout=
                        dropout
                )

                for _
                in range(
                    self.number_modalities
                )
            ]
        )


        self.modality_embeddings = nn.Parameter(
            torch.randn(
                self.number_modalities,
                1,
                dimension
            )
            * 0.02
        )


        self.self_encoders = nn.ModuleList()


        for _ in range(
            self.number_modalities
        ):

            encoder = nn.ModuleList(
                [

                    TransformerBlock(
                        dimension=
                            dimension,
                        number_heads=
                            number_heads,
                        expansion=
                            expansion,
                        dropout=
                            dropout
                    )

                    for _
                    in range(
                        self_layers
                    )
                ]
            )


            self.self_encoders.append(
                encoder
            )


        self.cross_attention_layers = nn.ModuleList(
            [

                CrossModalAttentionLayer(
                    dimension=
                        dimension,
                    number_heads=
                        number_heads,
                    expansion=
                        expansion,
                    dropout=
                        dropout,
                    number_modalities=
                        self.number_modalities
                )

                for _
                in range(
                    cross_layers
                )
            ]
        )


        fusion_dimension = (
            dimension
            * (
                self.number_modalities
                + 2
            )
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        streams = []


        # =========================================================================
        # MODALITY SPECIFIC TRANSFORMERS
        # =========================================================================

        for modality_index in range(
            self.number_modalities
        ):

            column_indices = getattr(
                self,
                self.modality_buffer_names[
                    modality_index
                ]
            )


            modality_input = torch.index_select(
                x,
                dim=1,
                index=
                    column_indices
            )


            stream = self.tokenizers[
                modality_index
            ](
                modality_input
            )


            modality_embedding = (
                self.modality_embeddings[
                    modality_index
                ]
                .unsqueeze(
                    0
                )
            )


            stream = (
                stream
                + modality_embedding
            )


            for block in self.self_encoders[
                modality_index
            ]:

                stream = block(
                    stream
                )


            streams.append(
                stream
            )


        # =========================================================================
        # CROSS ATTENTION
        # =========================================================================

        for cross_layer in self.cross_attention_layers:

            streams = cross_layer(
                streams
            )


        # =========================================================================
        # FUSION
        # =========================================================================

        modality_vectors = [

            torch.mean(
                stream,
                dim=1
            )

            for stream
            in streams
        ]


        stacked = torch.stack(
            modality_vectors,
            dim=1
        )


        modality_mean = torch.mean(
            stacked,
            dim=1
        )


        modality_std = modality_standard_deviation(
            stacked
        )


        fused = torch.cat(
            modality_vectors
            + [
                modality_mean,
                modality_std
            ],
            dim=1
        )


        return self.classifier(
            fused
        )


# =============================================================================
# GAUSSIAN INPUT NOISE
# =============================================================================

class GaussianNoise(
    nn.Module
):

    def __init__(
        self,
        standard_deviation
    ):

        super().__init__()


        self.standard_deviation = float(
            standard_deviation
        )


    def forward(
        self,
        x
    ):

        if (
            self.training
            and
            self.standard_deviation > 0
        ):

            return (
                x
                + torch.randn_like(
                    x
                )
                * self.standard_deviation
            )


        return x


# =============================================================================
# MANUAL STANDARD LSTM
#
# Native nn.LSTM is intentionally not used.
#
# This implementation supports training with the Intel XPU setup
# that previously produced a oneDNN LSTM training error.
# =============================================================================

class XPUManualLSTM(
    nn.Module
):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers=1,
        dropout=0.0
    ):

        super().__init__()


        self.input_size = int(
            input_size
        )


        self.hidden_size = int(
            hidden_size
        )


        self.num_layers = int(
            num_layers
        )


        self.dropout_probability = float(
            dropout
        )


        self.input_linears = nn.ModuleList()

        self.hidden_linears = nn.ModuleList()


        for layer_index in range(
            self.num_layers
        ):

            current_input_size = (
                self.input_size

                if layer_index == 0

                else self.hidden_size
            )


            self.input_linears.append(
                nn.Linear(
                    current_input_size,
                    4
                    * self.hidden_size,
                    bias=True
                )
            )


            self.hidden_linears.append(
                nn.Linear(
                    self.hidden_size,
                    4
                    * self.hidden_size,
                    bias=False
                )
            )


        self.inter_layer_dropout = nn.Dropout(
            self.dropout_probability
        )


        self.reset_parameters()


    def reset_parameters(
        self
    ):

        for input_linear in self.input_linears:

            nn.init.xavier_uniform_(
                input_linear.weight
            )


            nn.init.zeros_(
                input_linear.bias
            )


            with torch.no_grad():

                input_linear.bias[
                    self.hidden_size:
                    2
                    * self.hidden_size
                ].fill_(
                    1.0
                )


        for hidden_linear in self.hidden_linears:

            nn.init.orthogonal_(
                hidden_linear.weight
            )


    def forward(
        self,
        x
    ):

        batch_size = x.shape[
            0
        ]


        sequence_length = x.shape[
            1
        ]


        layer_input = x


        final_hidden_states = []

        final_cell_states = []


        for layer_index in range(
            self.num_layers
        ):

            hidden_state = torch.zeros(
                batch_size,
                self.hidden_size,
                dtype=
                    x.dtype,
                device=
                    x.device
            )


            cell_state = torch.zeros(
                batch_size,
                self.hidden_size,
                dtype=
                    x.dtype,
                device=
                    x.device
            )


            input_projection = (
                self.input_linears[
                    layer_index
                ](
                    layer_input
                )
            )


            timestep_outputs = []


            for timestep in range(
                sequence_length
            ):

                gates = (
                    input_projection[
                        :,
                        timestep,
                        :
                    ]
                    +
                    self.hidden_linears[
                        layer_index
                    ](
                        hidden_state
                    )
                )


                (
                    input_gate,
                    forget_gate,
                    candidate_gate,
                    output_gate

                ) = gates.chunk(
                    4,
                    dim=1
                )


                input_gate = torch.sigmoid(
                    input_gate
                )


                forget_gate = torch.sigmoid(
                    forget_gate
                )


                candidate_gate = torch.tanh(
                    candidate_gate
                )


                output_gate = torch.sigmoid(
                    output_gate
                )


                cell_state = (
                    forget_gate
                    * cell_state
                    +
                    input_gate
                    * candidate_gate
                )


                hidden_state = (
                    output_gate
                    * torch.tanh(
                        cell_state
                    )
                )


                timestep_outputs.append(
                    hidden_state
                )


            layer_output = torch.stack(
                timestep_outputs,
                dim=1
            )


            if (
                layer_index
                < self.num_layers
                - 1
            ):

                layer_output = (
                    self.inter_layer_dropout(
                        layer_output
                    )
                )


            layer_input = (
                layer_output
            )


            final_hidden_states.append(
                hidden_state
            )


            final_cell_states.append(
                cell_state
            )


        hidden_stack = torch.stack(
            final_hidden_states,
            dim=0
        )


        cell_stack = torch.stack(
            final_cell_states,
            dim=0
        )


        return (
            layer_input,
            (
                hidden_stack,
                cell_stack
            )
        )


# =============================================================================
# CNN LSTM MODALITY ENCODER
# =============================================================================

class ModalityCNNLSTMEncoder(
    nn.Module
):

    def __init__(
        self,
        params
    ):

        super().__init__()


        base_channels = int(
            params[
                "base_channels"
            ]
        )


        kernel_size = int(
            params[
                "kernel_size"
            ]
        )


        lstm_hidden = int(
            params[
                "lstm_hidden"
            ]
        )


        lstm_layers = int(
            params[
                "lstm_layers"
            ]
        )


        conv_dropout = float(
            params[
                "conv_dropout"
            ]
        )


        lstm_dropout = float(
            params[
                "lstm_dropout"
            ]
        )


        padding = (
            kernel_size
            // 2
        )


        self.cnn = nn.Sequential(

            nn.Conv1d(
                1,
                base_channels,
                kernel_size=
                    kernel_size,
                padding=
                    padding,
                bias=False
            ),

            nn.BatchNorm1d(
                base_channels
            ),

            nn.GELU(),

            nn.MaxPool1d(
                kernel_size=2,
                ceil_mode=True
            ),

            nn.Dropout(
                conv_dropout
            ),

            nn.Conv1d(
                base_channels,
                base_channels * 2,
                kernel_size=
                    kernel_size,
                padding=
                    padding,
                bias=False
            ),

            nn.BatchNorm1d(
                base_channels * 2
            ),

            nn.GELU(),

            nn.MaxPool1d(
                kernel_size=2,
                ceil_mode=True
            ),

            nn.Dropout(
                conv_dropout
            )
        )


        self.lstm = XPUManualLSTM(

            input_size=
                base_channels
                * 2,

            hidden_size=
                lstm_hidden,

            num_layers=
                lstm_layers,

            dropout=
                lstm_dropout
        )


        self.output_dimension = (
            lstm_hidden
            * 2
        )


        self.output_norm = nn.LayerNorm(
            self.output_dimension
        )


        self.output_dropout = nn.Dropout(
            lstm_dropout
        )


    def forward(
        self,
        x
    ):

        x = x.unsqueeze(
            1
        )


        x = self.cnn(
            x
        )


        x = x.transpose(
            1,
            2
        ).contiguous()


        sequence_output, _ = self.lstm(
            x
        )


        final_state = sequence_output[
            :,
            -1,
            :
        ]


        mean_state = torch.mean(
            sequence_output,
            dim=1
        )


        representation = torch.cat(
            [
                final_state,
                mean_state
            ],
            dim=1
        )


        representation = self.output_norm(
            representation
        )


        return self.output_dropout(
            representation
        )


# =============================================================================
# MODEL 4
# MULTIMODAL CNN LSTM
# =============================================================================

class MultimodalCNNLSTM(
    nn.Module
):

    def __init__(
        self,
        params
    ):

        super().__init__()


        self.modality_names = list(
            MODALITY_INDICES.keys()
        )


        self.number_modalities = len(
            self.modality_names
        )


        self.modality_buffer_names = []


        for modality_number, modality_name in enumerate(
            self.modality_names
        ):

            buffer_name = (
                f"modality_indices_"
                f"{modality_number}"
            )


            self.register_buffer(
                buffer_name,
                torch.tensor(
                    MODALITY_INDICES[
                        modality_name
                    ],
                    dtype=torch.long
                ),
                persistent=True
            )


            self.modality_buffer_names.append(
                buffer_name
            )


        self.input_noise = GaussianNoise(
            params[
                "input_noise"
            ]
        )


        self.input_dropout = nn.Dropout(
            float(
                params[
                    "input_dropout"
                ]
            )
        )


        self.encoders = nn.ModuleList(
            [

                ModalityCNNLSTMEncoder(
                    params
                )

                for _
                in range(
                    self.number_modalities
                )
            ]
        )


        stream_dimension = (
            self.encoders[
                0
            ].output_dimension
        )


        gate_hidden = max(
            16,
            stream_dimension
            // 2
        )


        dense_dropout = float(
            params[
                "dense_dropout"
            ]
        )


        self.modality_gate = nn.Sequential(

            nn.Linear(
                stream_dimension,
                gate_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                gate_hidden,
                1
            )
        )


        fusion_dimension = (
            stream_dimension
            * (
                self.number_modalities
                + 2
            )
        )


        classifier_hidden = int(
            params[
                "classifier_hidden"
            ]
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(
                fusion_dimension
            ),

            nn.Linear(
                fusion_dimension,
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dense_dropout
            ),

            nn.Linear(
                classifier_hidden,
                NUMBER_CLASSES
            )
        )


    def forward(
        self,
        x
    ):

        x = self.input_noise(
            x
        )


        x = self.input_dropout(
            x
        )


        modality_vectors = []


        for modality_index in range(
            self.number_modalities
        ):

            column_indices = getattr(
                self,
                self.modality_buffer_names[
                    modality_index
                ]
            )


            modality_input = torch.index_select(
                x,
                dim=1,
                index=
                    column_indices
            )


            modality_vector = self.encoders[
                modality_index
            ](
                modality_input
            )


            modality_vectors.append(
                modality_vector
            )


        stacked = torch.stack(
            modality_vectors,
            dim=1
        )


        gate_logits = torch.cat(
            [

                self.modality_gate(
                    vector
                )

                for vector
                in modality_vectors
            ],
            dim=1
        )


        modality_weights = torch.softmax(
            gate_logits,
            dim=1
        )


        weighted_representation = torch.sum(
            stacked
            * modality_weights.unsqueeze(
                -1
            ),
            dim=1
        )


        modality_std = modality_standard_deviation(
            stacked
        )


        fused = torch.cat(
            modality_vectors
            + [
                weighted_representation,
                modality_std
            ],
            dim=1
        )


        return self.classifier(
            fused
        )


# =============================================================================
# MODEL BUILDERS
# =============================================================================

def build_cnn(
    params
):

    return CNN1D(
        params
    )


def build_conformer_tcn(
    params
):

    return EEGConformerTCNCrossAttention(
        params
    )


def build_multimodal_transformer(
    params
):

    return CrossAttentionMultimodalTransformer(
        params
    )


def build_multimodal_cnn_lstm(
    params
):

    return MultimodalCNNLSTM(
        params
    )


# =============================================================================
# OPTUNA SEARCH SPACE
# CNN
# =============================================================================

def search_cnn(
    trial
):

    return {

        "n_conv_blocks":

            trial.suggest_int(
                "n_conv_blocks",
                2,
                3
            ),

        "base_filters":

            trial.suggest_categorical(
                "base_filters",
                [
                    32,
                    64
                ]
            ),

        "kernel_size":

            trial.suggest_categorical(
                "kernel_size",
                [
                    3,
                    5,
                    7
                ]
            ),

        "conv_dropout":

            trial.suggest_float(
                "conv_dropout",
                0.10,
                0.35
            ),

        "dense_units":

            trial.suggest_categorical(
                "dense_units",
                [
                    64,
                    128,
                    256
                ]
            ),

        "dense_dropout":

            trial.suggest_float(
                "dense_dropout",
                0.15,
                0.50
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                3e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.10
            )
    }


# =============================================================================
# OPTUNA SEARCH SPACE
# CONFORMER TCN
# =============================================================================

def search_conformer_tcn(
    trial
):

    return {

        "d_model":

            trial.suggest_categorical(
                "d_model",
                [
                    64,
                    96
                ]
            ),

        "num_heads":

            trial.suggest_categorical(
                "num_heads",
                [
                    4,
                    8
                ]
            ),

        "n_conformer_blocks":

            trial.suggest_int(
                "n_conformer_blocks",
                1,
                2
            ),

        "n_tcn_blocks":

            trial.suggest_int(
                "n_tcn_blocks",
                2,
                3
            ),

        "conformer_kernel":

            trial.suggest_categorical(
                "conformer_kernel",
                [
                    7,
                    15,
                    31
                ]
            ),

        "tcn_kernel":

            trial.suggest_categorical(
                "tcn_kernel",
                [
                    3,
                    5
                ]
            ),

        "dropout":

            trial.suggest_float(
                "dropout",
                0.10,
                0.35
            ),

        "classifier_hidden":

            trial.suggest_categorical(
                "classifier_hidden",
                [
                    128,
                    256
                ]
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.08
            )
    }


# =============================================================================
# OPTUNA SEARCH SPACE
# CROSS ATTENTION MULTIMODAL TRANSFORMER
# =============================================================================

def search_multimodal_transformer(
    trial
):

    return {

        "d_model":

            trial.suggest_categorical(
                "d_model",
                [
                    64,
                    96
                ]
            ),

        "num_heads":

            trial.suggest_categorical(
                "num_heads",
                [
                    4,
                    8
                ]
            ),

        "self_layers":

            trial.suggest_int(
                "self_layers",
                1,
                2
            ),

        "cross_layers":

            trial.suggest_int(
                "cross_layers",
                1,
                2
            ),

        "ffn_expansion":

            trial.suggest_categorical(
                "ffn_expansion",
                [
                    2,
                    4
                ]
            ),

        "tokenizer_kernel":

            trial.suggest_categorical(
                "tokenizer_kernel",
                [
                    3,
                    5,
                    7
                ]
            ),

        "tokenizer_stride":

            trial.suggest_categorical(
                "tokenizer_stride",
                [
                    2,
                    4
                ]
            ),

        "dropout":

            trial.suggest_float(
                "dropout",
                0.10,
                0.35
            ),

        "classifier_hidden":

            trial.suggest_categorical(
                "classifier_hidden",
                [
                    128,
                    256
                ]
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.08
            )
    }


# =============================================================================
# OPTUNA SEARCH SPACE
# MULTIMODAL CNN LSTM
# =============================================================================

def search_multimodal_cnn_lstm(
    trial
):

    return {

        "base_channels":

            trial.suggest_categorical(
                "base_channels",
                [
                    32,
                    48,
                    64
                ]
            ),

        "kernel_size":

            trial.suggest_categorical(
                "kernel_size",
                [
                    3,
                    5,
                    7
                ]
            ),

        "lstm_hidden":

            trial.suggest_categorical(
                "lstm_hidden",
                [
                    32,
                    64,
                    96
                ]
            ),

        "lstm_layers":

            trial.suggest_int(
                "lstm_layers",
                1,
                2
            ),

        "conv_dropout":

            trial.suggest_float(
                "conv_dropout",
                0.15,
                0.40
            ),

        "lstm_dropout":

            trial.suggest_float(
                "lstm_dropout",
                0.15,
                0.40
            ),

        "input_dropout":

            trial.suggest_float(
                "input_dropout",
                0.05,
                0.20
            ),

        "input_noise":

            trial.suggest_float(
                "input_noise",
                0.0,
                0.05
            ),

        "classifier_hidden":

            trial.suggest_categorical(
                "classifier_hidden",
                [
                    64,
                    128
                ]
            ),

        "dense_dropout":

            trial.suggest_float(
                "dense_dropout",
                0.25,
                0.50
            ),

        "learning_rate":

            trial.suggest_float(
                "learning_rate",
                1e-4,
                2e-3,
                log=True
            ),

        "weight_decay":

            trial.suggest_float(
                "weight_decay",
                1e-6,
                2e-3,
                log=True
            ),

        "label_smoothing":

            trial.suggest_float(
                "label_smoothing",
                0.0,
                0.10
            )
    }


# =============================================================================
# SMOKE TEST PARAMETERS
# =============================================================================

CNN_SMOKE_PARAMS = {

    "n_conv_blocks":
        2,

    "base_filters":
        32,

    "kernel_size":
        5,

    "conv_dropout":
        0.20,

    "dense_units":
        64,

    "dense_dropout":
        0.30,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-4,

    "label_smoothing":
        0.05
}


CONFORMER_SMOKE_PARAMS = {

    "d_model":
        64,

    "num_heads":
        4,

    "n_conformer_blocks":
        1,

    "n_tcn_blocks":
        2,

    "conformer_kernel":
        15,

    "tcn_kernel":
        3,

    "dropout":
        0.20,

    "classifier_hidden":
        128,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-4,

    "label_smoothing":
        0.05
}


TRANSFORMER_SMOKE_PARAMS = {

    "d_model":
        64,

    "num_heads":
        4,

    "self_layers":
        1,

    "cross_layers":
        1,

    "ffn_expansion":
        2,

    "tokenizer_kernel":
        5,

    "tokenizer_stride":
        2,

    "dropout":
        0.20,

    "classifier_hidden":
        128,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-4,

    "label_smoothing":
        0.05
}


CNN_LSTM_SMOKE_PARAMS = {

    "base_channels":
        32,

    "kernel_size":
        5,

    "lstm_hidden":
        64,

    "lstm_layers":
        1,

    "conv_dropout":
        0.25,

    "lstm_dropout":
        0.25,

    "input_dropout":
        0.10,

    "input_noise":
        0.02,

    "classifier_hidden":
        128,

    "dense_dropout":
        0.35,

    "learning_rate":
        5e-4,

    "weight_decay":
        1e-4,

    "label_smoothing":
        0.05
}


# =============================================================================
# ALGORITHM SPECIFICATIONS
# =============================================================================

ALGORITHM_SPECS = {

    "CNN": {

        "display_name":
            "CNN",

        "folder":
            "CNN_Results",

        "version":
            "min_cnn_v3",

        "builder":
            build_cnn,

        "search":
            search_cnn,

        "smoke_params":
            CNN_SMOKE_PARAMS,

        "batch_size":
            2048,

        "max_epochs":
            50,

        "patience":
            6,

        "uses_modalities":
            False
    },


    "EEG_CONFORMER_TCN": {

        "display_name":
            (
                "EEG Conformer + 1D TCN "
                "Dual Stream Cross Attention"
            ),

        "folder":
            (
                "EEG_Conformer_TCN_"
                "CrossAttention_Results"
            ),

        "version":
            "min_conformer_tcn_v3",

        "builder":
            build_conformer_tcn,

        "search":
            search_conformer_tcn,

        "smoke_params":
            CONFORMER_SMOKE_PARAMS,

        "batch_size":
            512,

        "max_epochs":
            40,

        "patience":
            5,

        "uses_modalities":
            False
    },


    "CROSS_ATTENTION_MULTIMODAL_TRANSFORMER": {

        "display_name":
            (
                "Cross Attention "
                "Multimodal Transformer"
            ),

        "folder":
            (
                "Cross_Attention_Multimodal_"
                "Transformer_Results"
            ),

        "version":
            "min_multimodal_transformer_v3",

        "builder":
            build_multimodal_transformer,

        "search":
            search_multimodal_transformer,

        "smoke_params":
            TRANSFORMER_SMOKE_PARAMS,

        "batch_size":
            512,

        "max_epochs":
            40,

        "patience":
            5,

        "uses_modalities":
            True
    },


    "MULTIMODAL_CNN_LSTM": {

        "display_name":
            "Multimodal CNN LSTM",

        "folder":
            "Multimodal_CNN_LSTM_Results",

        "version":
            (
                "min_multimodal_cnn_"
                "manual_lstm_v4"
            ),

        "builder":
            build_multimodal_cnn_lstm,

        "search":
            search_multimodal_cnn_lstm,

        "smoke_params":
            CNN_LSTM_SMOKE_PARAMS,

        "batch_size":
            1024,

        "max_epochs":
            50,

        "patience":
            6,

        "uses_modalities":
            True
    }
}


# =============================================================================
# MODEL INSTANTIATION
# =============================================================================

def instantiate_model(
    spec,
    params,
    seed
):

    set_seed(
        seed
    )


    model = spec[
        "builder"
    ](
        params
    )


    return model.to(
        DEVICE
    )


# =============================================================================
# PARAMETER COUNT
# =============================================================================

def count_trainable_parameters(
    model
):

    return sum(
        parameter.numel()

        for parameter
        in model.parameters()

        if parameter.requires_grad
    )


# =============================================================================
# TRAIN ONE EPOCH
# =============================================================================

def train_one_epoch(
    model,
    X,
    y,
    optimizer,
    criterion,
    batch_size,
    rng
):

    model.train()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    for X_batch, y_batch in batch_generator(
        X,
        y,
        batch_size=
            batch_size,
        shuffle=
            True,
        rng=
            rng
    ):

        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            X_batch
        )


        loss = criterion(
            logits,
            y_batch
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )


        optimizer.step()


        current_batch_size = int(
            y_batch.shape[
                0
            ]
        )


        total_loss += (
            float(
                loss.detach()
                .item()
            )
            * current_batch_size
        )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            current_batch_size
        )


        del X_batch

        del y_batch

        del logits

        del loss


    synchronize()


    return (
        total_loss
        / total_samples,

        total_correct
        / total_samples
    )


# =============================================================================
# EVALUATE MODEL
# =============================================================================

@torch.inference_mode()
def evaluate_model(
    model,
    X,
    y,
    batch_size,
    criterion=None,
    return_probabilities=False
):

    model.eval()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    probability_parts = []


    for X_batch, y_batch in batch_generator(
        X,
        y,
        batch_size=
            batch_size,
        shuffle=
            False
    ):

        logits = model(
            X_batch
        )


        current_batch_size = int(
            y_batch.shape[
                0
            ]
        )


        if criterion is not None:

            loss = criterion(
                logits,
                y_batch
            )


            total_loss += (
                float(
                    loss.item()
                )
                * current_batch_size
            )


        total_correct += int(
            (
                logits.argmax(
                    dim=1
                )
                == y_batch
            )
            .sum()
            .item()
        )


        total_samples += (
            current_batch_size
        )


        if return_probabilities:

            probability_parts.append(
                torch.softmax(
                    logits,
                    dim=1
                )
                .cpu()
            )


        del X_batch

        del y_batch

        del logits


    synchronize()


    accuracy = (
        total_correct
        / total_samples
    )


    if criterion is None:

        mean_loss = np.nan


    else:

        mean_loss = (
            total_loss
            / total_samples
        )


    if return_probabilities:

        probabilities = (
            torch.cat(
                probability_parts,
                dim=0
            )
            .numpy()
        )


        return (
            mean_loss,
            accuracy,
            probabilities
        )


    return (
        mean_loss,
        accuracy
    )


# =============================================================================
# TRAIN MODEL
# =============================================================================

def fit_model(
    spec,
    params,
    X_training,
    y_training,
    X_validation_data=None,
    y_validation_data=None,
    seed=SEED,
    epochs=None,
    use_early_stopping=True,
    verbose=False
):

    clear_device()


    if epochs is None:

        epochs = spec[
            "max_epochs"
        ]


    batch_size = spec[
        "batch_size"
    ]


    patience = spec[
        "patience"
    ]


    set_seed(
        seed
    )


    training_rng = np.random.default_rng(
        seed
    )


    model = instantiate_model(
        spec,
        params,
        seed
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),

        lr=float(
            params[
                "learning_rate"
            ]
        ),

        weight_decay=float(
            params[
                "weight_decay"
            ]
        )
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=
            LR_SCHEDULER_PATIENCE,
        min_lr=1e-6
    )


    class_weights = get_class_weights(
        y_training
    )


    criterion = nn.CrossEntropyLoss(
        weight=
            class_weights,

        label_smoothing=
            float(
                params[
                    "label_smoothing"
                ]
            )
    )


    history = {

        "loss":
            [],

        "accuracy":
            [],

        "val_loss":
            [],

        "val_accuracy":
            [],

        "learning_rate":
            []
    }


    best_loss = float(
        "inf"
    )


    best_state = None

    best_epoch = 1

    patience_counter = 0


    for epoch in range(
        1,
        epochs
        + 1
    ):

        (
            training_loss,
            training_accuracy

        ) = train_one_epoch(
            model,
            X_training,
            y_training,
            optimizer,
            criterion,
            batch_size,
            training_rng
        )


        if X_validation_data is not None:

            (
                validation_loss,
                validation_accuracy

            ) = evaluate_model(
                model,
                X_validation_data,
                y_validation_data,
                batch_size,
                criterion=
                    criterion,
                return_probabilities=
                    False
            )


            monitored_loss = (
                validation_loss
            )


        else:

            validation_loss = np.nan

            validation_accuracy = np.nan


            monitored_loss = (
                training_loss
            )


        scheduler.step(
            monitored_loss
        )


        current_lr = float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        )


        history[
            "loss"
        ].append(
            training_loss
        )


        history[
            "accuracy"
        ].append(
            training_accuracy
        )


        history[
            "val_loss"
        ].append(
            validation_loss
        )


        history[
            "val_accuracy"
        ].append(
            validation_accuracy
        )


        history[
            "learning_rate"
        ].append(
            current_lr
        )


        if verbose:

            print(
                f"Epoch "
                f"{epoch:02d}/"
                f"{epochs} | "
                f"Train Loss "
                f"{training_loss:.4f} | "
                f"Train Acc "
                f"{training_accuracy:.4f} | "
                f"Val Loss "
                f"{validation_loss:.4f} | "
                f"Val Acc "
                f"{validation_accuracy:.4f} | "
                f"LR "
                f"{current_lr:.2e}"
            )


        if use_early_stopping:

            if (
                monitored_loss
                < best_loss
                - EARLY_STOPPING_MIN_DELTA
            ):

                best_loss = (
                    monitored_loss
                )


                best_epoch = (
                    epoch
                )


                best_state = {

                    key:
                        value.detach()
                        .cpu()
                        .clone()

                    for key, value
                    in model.state_dict().items()
                }


                patience_counter = 0


            else:

                patience_counter += 1


                if (
                    patience_counter
                    >= patience
                ):

                    if verbose:

                        print(
                            "Early stopping."
                        )


                    break


    if (
        use_early_stopping
        and
        best_state is not None
    ):

        model.load_state_dict(
            best_state
        )


    if not use_early_stopping:

        best_epoch = len(
            history[
                "loss"
            ]
        )


    del optimizer

    del scheduler

    del criterion

    del class_weights


    if best_state is not None:

        del best_state


    clear_device()


    return (
        model,
        history,
        best_epoch
    )


# =============================================================================
# PREDICT PROBABILITIES
# =============================================================================

def predict_probabilities(
    spec,
    model,
    X,
    y
):

    (
        _,
        _,
        probabilities

    ) = evaluate_model(
        model,
        X,
        y,
        spec[
            "batch_size"
        ],
        criterion=None,
        return_probabilities=True
    )


    return probabilities


# =============================================================================
# REAL GPU TRAINING SMOKE TEST
#
# Tests:
#
# forward
# loss
# backward
# gradient calculation
# optimizer step
# =============================================================================

def run_smoke_test(
    spec
):

    print()

    print(
        "=" * 90
    )

    print(
        spec[
            "display_name"
        ]
    )

    print(
        "REAL GPU TRAINING SMOKE TEST"
    )

    print(
        "=" * 90
    )


    params = spec[
        "smoke_params"
    ]


    model = instantiate_model(
        spec,
        params,
        SEED
    )


    model.train()


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-4,
        weight_decay=1e-4
    )


    X_smoke = torch.randn(
        16,
        NUMBER_FEATURES,
        dtype=torch.float32,
        device=DEVICE
    )


    y_smoke = torch.randint(
        low=0,
        high=
            NUMBER_CLASSES,
        size=(
            16,
        ),
        dtype=torch.long,
        device=DEVICE
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    output = model(
        X_smoke
    )


    loss = nn.CrossEntropyLoss()(
        output,
        y_smoke
    )


    loss.backward()


    gradient_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRADIENT_CLIP
    )


    optimizer.step()


    synchronize()


    print(
        "Input shape  :",
        X_smoke.shape
    )

    print(
        "Output shape :",
        output.shape
    )

    print(
        "Output device:",
        output.device
    )

    print(
        "Loss         :",
        float(
            loss.detach()
            .item()
        )
    )


    if torch.is_tensor(
        gradient_norm
    ):

        gradient_value = float(
            gradient_norm.detach()
            .cpu()
            .item()
        )


    else:

        gradient_value = float(
            gradient_norm
        )


    print(
        "Gradient norm:",
        gradient_value
    )

    print(
        "Backward     : SUCCESS"
    )

    print(
        "Optimizer    : SUCCESS"
    )

    print(
        "Smoke test   : PASSED"
    )


    del model

    del optimizer

    del X_smoke

    del y_smoke

    del output

    del loss

    del gradient_norm


    clear_device()


# =============================================================================
# SAVE COMMON FILES
# =============================================================================

def save_common_files(
    output_dir,
    uses_modalities
):

    CLASS_DISTRIBUTION_TABLE.to_csv(
        output_dir
        / "class_distribution.csv",
        index=False
    )


    SPLIT_SUMMARY_TABLE.to_csv(
        output_dir
        / "split_summary.csv",
        index=False
    )


    save_json(
        PARTICIPANT_SPLIT,
        output_dir
        / "participant_split.json"
    )


    save_json(
        FEATURE_COLUMNS,
        output_dir
        / "feature_columns.json"
    )


    joblib.dump(
        FINAL_SCALER,
        output_dir
        / "standard_scaler.joblib"
    )


    joblib.dump(
        LABEL_ENCODER,
        output_dir
        / "label_encoder.joblib"
    )


    if uses_modalities:

        modality_mapping = {

            "Grouping_Mode":
                MODALITY_GROUPING_MODE,

            "Scientific_Note":
                (
                    "Streams represent EEG feature families "
                    "or feature views, not automatically "
                    "different physical sensors."
                ),

            "Modalities":
                {}
        }


        for modality_name in MODALITY_NAMES:

            indices = MODALITY_INDICES[
                modality_name
            ]


            modality_mapping[
                "Modalities"
            ][
                modality_name
            ] = {

                "Feature_Count":
                    len(
                        indices
                    ),

                "Features":
                    [
                        FEATURE_COLUMNS[
                            index
                        ]

                        for index
                        in indices
                    ]
            }


        save_json(
            modality_mapping,
            output_dir
            / "modality_mapping.json"
        )


# =============================================================================
# SAVE TRAINING CURVES
# =============================================================================

def save_training_curves(
    history,
    output_dir,
    algorithm_name
):

    epochs = np.arange(
        1,
        len(
            history[
                "loss"
            ]
        )
        + 1
    )


    history_table = pd.DataFrame(
        {

            "Epoch":
                epochs,

            "Training_Loss":
                history[
                    "loss"
                ],

            "Training_Accuracy":
                history[
                    "accuracy"
                ],

            "Validation_Loss":
                history[
                    "val_loss"
                ],

            "Validation_Accuracy":
                history[
                    "val_accuracy"
                ],

            "Learning_Rate":
                history[
                    "learning_rate"
                ]
        }
    )


    history_table[
        "Generalization_Gap"
    ] = (
        history_table[
            "Training_Accuracy"
        ]
        -
        history_table[
            "Validation_Accuracy"
        ]
    )


    history_table.to_csv(
        output_dir
        / "training_history.csv",
        index=False
    )


    # =========================================================================
    # LOSS CURVE
    # =========================================================================

    fig, ax = plt.subplots(
        figsize=(
            9,
            5
        )
    )


    ax.plot(
        epochs,
        history[
            "loss"
        ],
        label=
            "Training Loss"
    )


    ax.plot(
        epochs,
        history[
            "val_loss"
        ],
        label=
            "Validation Loss"
    )


    ax.set_xlabel(
        "Epoch"
    )


    ax.set_ylabel(
        "Cross Entropy Loss"
    )


    ax.set_title(
        f"{algorithm_name} Loss Curve"
    )


    ax.legend()

    ax.grid(
        alpha=0.3
    )


    fig.tight_layout()


    fig.savefig(
        output_dir
        / "loss_curve.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # =========================================================================
    # ACCURACY CURVE
    # =========================================================================

    fig, ax = plt.subplots(
        figsize=(
            9,
            5
        )
    )


    ax.plot(
        epochs,
        history[
            "accuracy"
        ],
        label=
            "Training Accuracy"
    )


    ax.plot(
        epochs,
        history[
            "val_accuracy"
        ],
        label=
            "Validation Accuracy"
    )


    ax.set_xlabel(
        "Epoch"
    )


    ax.set_ylabel(
        "Accuracy"
    )


    ax.set_title(
        f"{algorithm_name} Accuracy Curve"
    )


    ax.legend()

    ax.grid(
        alpha=0.3
    )


    fig.tight_layout()


    fig.savefig(
        output_dir
        / "accuracy_curve.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # =========================================================================
    # GENERALIZATION GAP
    # =========================================================================

    fig, ax = plt.subplots(
        figsize=(
            9,
            5
        )
    )


    ax.plot(
        epochs,
        history_table[
            "Generalization_Gap"
        ]
    )


    ax.axhline(
        0.0,
        linewidth=1
    )


    ax.set_xlabel(
        "Epoch"
    )


    ax.set_ylabel(
        "Training Accuracy minus Validation Accuracy"
    )


    ax.set_title(
        f"{algorithm_name} Generalization Gap"
    )


    ax.grid(
        alpha=0.3
    )


    fig.tight_layout()


    fig.savefig(
        output_dir
        / "generalization_gap_curve.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # =========================================================================
    # LEARNING RATE
    # =========================================================================

    fig, ax = plt.subplots(
        figsize=(
            9,
            5
        )
    )


    ax.plot(
        epochs,
        history[
            "learning_rate"
        ]
    )


    ax.set_xlabel(
        "Epoch"
    )


    ax.set_ylabel(
        "Learning Rate"
    )


    ax.set_title(
        f"{algorithm_name} Learning Rate"
    )


    ax.grid(
        alpha=0.3
    )


    fig.tight_layout()


    fig.savefig(
        output_dir
        / "learning_rate_curve.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# =============================================================================
# SAVE CONFUSION MATRICES
# =============================================================================

def save_confusion_matrices(
    y_true,
    y_prediction,
    output_dir,
    algorithm_name
):

    cm = confusion_matrix(
        y_true,
        y_prediction,
        labels=np.arange(
            NUMBER_CLASSES
        )
    )


    pd.DataFrame(
        cm,
        index=
            CLASS_NAMES,
        columns=
            CLASS_NAMES
    ).to_csv(
        output_dir
        / "confusion_matrix.csv"
    )


    # =========================================================================
    # RAW MATRIX
    # =========================================================================

    fig, ax = plt.subplots(
        figsize=(
            8,
            6
        )
    )


    image = ax.imshow(
        cm
    )


    ax.set_xticks(
        np.arange(
            NUMBER_CLASSES
        )
    )


    ax.set_yticks(
        np.arange(
            NUMBER_CLASSES
        )
    )


    ax.set_xticklabels(
        CLASS_NAMES,
        rotation=45,
        ha="right"
    )


    ax.set_yticklabels(
        CLASS_NAMES
    )


    ax.set_xlabel(
        "Predicted Class"
    )


    ax.set_ylabel(
        "True Class"
    )


    ax.set_title(
        f"{algorithm_name} Confusion Matrix"
    )


    for row in range(
        NUMBER_CLASSES
    ):

        for column in range(
            NUMBER_CLASSES
        ):

            ax.text(
                column,
                row,
                str(
                    cm[
                        row,
                        column
                    ]
                ),
                ha="center",
                va="center"
            )


    fig.colorbar(
        image,
        ax=ax
    )


    fig.tight_layout()


    fig.savefig(
        output_dir
        / "confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # =========================================================================
    # NORMALIZED MATRIX
    # =========================================================================

    row_sums = cm.sum(
        axis=1,
        keepdims=True
    )


    normalized_cm = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(
            cm,
            dtype=np.float64
        ),
        where=(
            row_sums
            != 0
        )
    )


    pd.DataFrame(
        normalized_cm,
        index=
            CLASS_NAMES,
        columns=
            CLASS_NAMES
    ).to_csv(
        output_dir
        / "normalized_confusion_matrix.csv"
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            6
        )
    )


    image = ax.imshow(
        normalized_cm
    )


    ax.set_xticks(
        np.arange(
            NUMBER_CLASSES
        )
    )


    ax.set_yticks(
        np.arange(
            NUMBER_CLASSES
        )
    )


    ax.set_xticklabels(
        CLASS_NAMES,
        rotation=45,
        ha="right"
    )


    ax.set_yticklabels(
        CLASS_NAMES
    )


    ax.set_xlabel(
        "Predicted Class"
    )


    ax.set_ylabel(
        "True Class"
    )


    ax.set_title(
        f"Normalized {algorithm_name} Confusion Matrix"
    )


    for row in range(
        NUMBER_CLASSES
    ):

        for column in range(
            NUMBER_CLASSES
        ):

            ax.text(
                column,
                row,
                f"{normalized_cm[row, column]:.2f}",
                ha="center",
                va="center"
            )


    fig.colorbar(
        image,
        ax=ax
    )


    fig.tight_layout()


    fig.savefig(
        output_dir
        / "normalized_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    return cm


# =============================================================================
# OPTUNA RUNNER
#
# One trial is launched at a time.
#
# After a power cut:
#
# Completed trials remain in SQLite.
# An interrupted unfinished trial may be repeated.
# Completed trials are not repeated.
# =============================================================================

def run_optuna(
    spec,
    output_dir
):

    algorithm_name = spec[
        "display_name"
    ]


    study_name = (
        spec[
            "version"
        ]
        + "_"
        + DATASET_SIGNATURE
    )


    database_path = (
        output_dir
        / "optuna.db"
    )


    storage = (
        f"sqlite:///"
        f"{database_path}"
    )


    def objective(
        trial
    ):

        params = spec[
            "search"
        ](
            trial
        )


        print()

        print(
            algorithm_name
        )


        print(
            f"Optuna Trial "
            f"{trial.number + 1}"
        )


        print(
            params
        )


        fold_scores = []


        for fold_number, (
            training_index,
            validation_index

        ) in enumerate(
            group_kfold_iterator(),
            start=1
        ):

            fold_start = time.time()


            (
                X_fold_training,
                X_fold_validation

            ) = prepare_fold(
                training_index,
                validation_index
            )


            (
                model,
                history,
                best_epoch

            ) = fit_model(
                spec,
                params,
                X_fold_training,
                Y_TRAIN[
                    training_index
                ],
                X_fold_validation,
                Y_TRAIN[
                    validation_index
                ],
                seed=
                    SEED
                    + fold_number,
                epochs=
                    spec[
                        "max_epochs"
                    ],
                use_early_stopping=
                    True,
                verbose=
                    False
            )


            probabilities = predict_probabilities(
                spec,
                model,
                X_fold_validation,
                Y_TRAIN[
                    validation_index
                ]
            )


            predictions = np.argmax(
                probabilities,
                axis=1
            )


            fold_accuracy = accuracy_score(
                Y_TRAIN[
                    validation_index
                ],
                predictions
            )


            fold_scores.append(
                float(
                    fold_accuracy
                )
            )


            print(
                f"Trial "
                f"{trial.number + 1}, "
                f"Fold "
                f"{fold_number}/"
                f"{GROUP_K_FOLDS}, "
                f"Accuracy = "
                f"{fold_accuracy:.4f}, "
                f"Best Epoch = "
                f"{best_epoch}, "
                f"Time = "
                f"{(time.time() - fold_start) / 60:.2f} min, "
                f"RAM = "
                f"{memory_mb():.2f} MB"
            )


            del model

            del history

            del probabilities

            del predictions

            del X_fold_training

            del X_fold_validation


            clear_device()


        mean_accuracy = float(
            np.mean(
                fold_scores
            )
        )


        std_accuracy = float(
            np.std(
                fold_scores
            )
        )


        trial.set_user_attr(
            "fold_scores",
            json.dumps(
                fold_scores
            )
        )


        trial.set_user_attr(
            "fold_std",
            std_accuracy
        )


        print(
            f"Trial "
            f"{trial.number + 1} "
            f"Mean Accuracy = "
            f"{mean_accuracy:.4f} "
            f"plus or minus "
            f"{std_accuracy:.4f}"
        )


        return mean_accuracy


    study = optuna.create_study(
        study_name=
            study_name,
        storage=
            storage,
        direction=
            "maximize",
        sampler=
            optuna.samplers.TPESampler(
                seed=SEED
            ),
        load_if_exists=
            True
    )


    def completed_count():

        return sum(
            1

            for trial
            in study.trials

            if trial.state
            == optuna.trial.TrialState.COMPLETE
        )


    print()

    print(
        "=" * 90
    )

    print(
        algorithm_name
    )

    print(
        "OPTUNA"
    )

    print(
        "=" * 90
    )


    print(
        "Previously completed trials:",
        completed_count()
    )


    # =========================================================================
    # Continue until exactly N_TRIALS successful trials exist.
    # =========================================================================

    while completed_count() < N_TRIALS:

        print(
            "Completed:",
            completed_count(),
            "of",
            N_TRIALS
        )


        try:

            study.optimize(
                objective,
                n_trials=1,
                show_progress_bar=False,
                gc_after_trial=True
            )


        except RuntimeError as error:

            print(
                "Trial failed with RuntimeError:"
            )

            print(
                error
            )

            clear_device()

            continue


    study.trials_dataframe().to_csv(
        output_dir
        / "optuna_trials.csv",
        index=False
    )


    best_params = dict(
        study.best_params
    )


    save_json(
        best_params,
        output_dir
        / "best_parameters.json"
    )


    print()

    print(
        "Best GroupKFold Accuracy:",
        f"{study.best_value:.4f}"
    )


    print(
        "Best Parameters:"
    )


    print(
        best_params
    )


    return (
        study,
        best_params
    )


# =============================================================================
# CONFIGURATION ID
#
# Prevents old checkpoints from a different parameter configuration
# from being silently reused.
# =============================================================================

def build_config_id(
    spec,
    best_params
):

    payload = {

        "algorithm":
            spec[
                "display_name"
            ],

        "version":
            spec[
                "version"
            ],

        "dataset_signature":
            DATASET_SIGNATURE,

        "best_params":
            best_params,

        "seed":
            SEED,

        "features":
            NUMBER_FEATURES,

        "classes":
            CLASS_NAMES,

        "modalities":
            (
                MODALITY_INDICES

                if spec[
                    "uses_modalities"
                ]

                else None
            )
    }


    return hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True
        ).encode(
            "utf8"
        )
    ).hexdigest()[
        :16
    ]


# =============================================================================
# FIVE FINAL STOCHASTIC RUNS
#
# Each completed run is saved as its own checkpoint.
#
# If power fails during an unfinished run, only that run repeats.
# =============================================================================

def run_final_runs(
    spec,
    best_params,
    config_id,
    output_dir
):

    algorithm_name = spec[
        "display_name"
    ]


    checkpoint_dir = (
        output_dir
        / "checkpoints"
    )


    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    run_records = []


    print()

    print(
        "=" * 90
    )

    print(
        algorithm_name
    )

    print(
        "FIVE FINAL STOCHASTIC RUNS"
    )

    print(
        "=" * 90
    )


    for run_number in range(
        1,
        N_RUNS
        + 1
    ):

        checkpoint_path = (
            checkpoint_dir
            / f"run_{run_number:02d}.pt"
        )


        valid_checkpoint = False

        bundle = None


        if checkpoint_path.exists():

            try:

                bundle = torch.load(
                    checkpoint_path,
                    map_location="cpu",
                    weights_only=False
                )


                valid_checkpoint = (
                    str(
                        bundle.get(
                            "Config_ID",
                            ""
                        )
                    )
                    == config_id
                )


            except Exception:

                valid_checkpoint = False


        if valid_checkpoint:

            record = bundle[
                "record"
            ]


            run_records.append(
                record
            )


            print(
                f"Run "
                f"{run_number}/"
                f"{N_RUNS} "
                f"already completed."
            )


            continue


        print()

        print(
            f"RUN "
            f"{run_number}/"
            f"{N_RUNS}"
        )


        run_start = time.time()


        run_seed = (
            SEED
            + run_number
            - 1
        )


        (
            model,
            history,
            best_epoch

        ) = fit_model(
            spec,
            best_params,
            X_TRAIN,
            Y_TRAIN,
            X_VALIDATION,
            Y_VALIDATION,
            seed=
                run_seed,
            epochs=
                spec[
                    "max_epochs"
                ],
            use_early_stopping=
                True,
            verbose=
                True
        )


        # =========================================================================
        # TRAIN PERFORMANCE
        # =========================================================================

        train_probabilities = predict_probabilities(
            spec,
            model,
            X_TRAIN,
            Y_TRAIN
        )


        train_predictions = np.argmax(
            train_probabilities,
            axis=1
        )


        training_accuracy = accuracy_score(
            Y_TRAIN,
            train_predictions
        )


        # =========================================================================
        # VALIDATION PERFORMANCE
        # =========================================================================

        validation_probabilities = predict_probabilities(
            spec,
            model,
            X_VALIDATION,
            Y_VALIDATION
        )


        validation_predictions = np.argmax(
            validation_probabilities,
            axis=1
        )


        validation_accuracy = accuracy_score(
            Y_VALIDATION,
            validation_predictions
        )


        generalization_gap = (
            training_accuracy
            - validation_accuracy
        )


        run_time = (
            time.time()
            - run_start
        )


        record = {

            "Config_ID":
                config_id,

            "Run":
                run_number,

            "Seed":
                run_seed,

            "Training_Accuracy":
                float(
                    training_accuracy
                ),

            "Validation_Accuracy":
                float(
                    validation_accuracy
                ),

            "Generalization_Gap":
                float(
                    generalization_gap
                ),

            "Absolute_Generalization_Gap":
                float(
                    abs(
                        generalization_gap
                    )
                ),

            "Best_Epoch":
                int(
                    best_epoch
                ),

            "Epochs_Run":
                int(
                    len(
                        history[
                            "loss"
                        ]
                    )
                ),

            "Execution_Time_Seconds":
                float(
                    run_time
                )
        }


        model_state = {

            key:
                value.detach()
                .cpu()
                .clone()

            for key, value
            in model.state_dict().items()
        }


        atomic_torch_save(
            {

                "Config_ID":
                    config_id,

                "record":
                    record,

                "best_params":
                    best_params,

                "model_state_dict":
                    model_state,

                "history":
                    history
            },
            checkpoint_path
        )


        run_records.append(
            record
        )


        atomic_csv(
            pd.DataFrame(
                run_records
            )
            .sort_values(
                "Run"
            )
            .drop_duplicates(
                subset=[
                    "Run"
                ],
                keep="last"
            ),
            output_dir
            / "five_runs.csv"
        )


        print(
            f"Train = "
            f"{training_accuracy:.4f}, "
            f"Validation = "
            f"{validation_accuracy:.4f}, "
            f"Gap = "
            f"{generalization_gap:.4f}, "
            f"Best Epoch = "
            f"{best_epoch}, "
            f"Time = "
            f"{run_time / 60:.2f} min"
        )


        del model

        del history

        del model_state

        del train_probabilities

        del train_predictions

        del validation_probabilities

        del validation_predictions


        clear_device()


    run_table = pd.DataFrame(
        run_records
    )


    run_table = (
        run_table
        .sort_values(
            "Run"
        )
        .drop_duplicates(
            subset=[
                "Run"
            ],
            keep="last"
        )
    )


    if len(
        run_table
    ) != N_RUNS:

        raise RuntimeError(
            "Five final runs are not complete."
        )


    atomic_csv(
        run_table,
        output_dir
        / "five_runs.csv"
    )


    # =========================================================================
    # SELECT MODEL USING VALIDATION ONLY
    # =========================================================================

    selected_row = (
        run_table
        .sort_values(
            by=[
                "Validation_Accuracy",
                "Absolute_Generalization_Gap",
                "Run"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )
        .iloc[
            0
        ]
    )


    best_run_number = int(
        selected_row[
            "Run"
        ]
    )


    best_checkpoint_path = (
        checkpoint_dir
        / f"run_{best_run_number:02d}.pt"
    )


    best_bundle = torch.load(
        best_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )


    print()

    print(
        "Selected Run:",
        best_run_number
    )


    print(
        "Validation Accuracy:",
        f"{best_bundle['record']['Validation_Accuracy']:.4f}"
    )


    print(
        "Generalization Gap:",
        f"{best_bundle['record']['Generalization_Gap']:.4f}"
    )


    return (
        run_table,
        best_bundle
    )


# =============================================================================
# FULL LOGO
#
# Uses all training participants unless LOGO_MAX_FOLDS is changed.
#
# Held out participant is never used for early stopping.
#
# Epoch count comes from the independently held validation participants.
# =============================================================================

def run_full_logo(
    spec,
    best_params,
    config_id,
    best_record,
    output_dir
):

    algorithm_name = spec[
        "display_name"
    ]


    fixed_epochs = max(
        1,
        int(
            best_record[
                "Best_Epoch"
            ]
        )
    )


    if LOGO_MAX_FOLDS is None:

        logo_folds_to_use = (
            FULL_LOGO_FOLDS
        )


    else:

        logo_folds_to_use = min(
            int(
                LOGO_MAX_FOLDS
            ),
            FULL_LOGO_FOLDS
        )


    logo_configuration = {

        "Config_ID":
            config_id,

        "Fixed_Epochs":
            fixed_epochs,

        "Selected_Seed":
            int(
                best_record[
                    "Seed"
                ]
            )
    }


    logo_config_id = hashlib.sha256(
        json.dumps(
            logo_configuration,
            sort_keys=True
        ).encode(
            "utf8"
        )
    ).hexdigest()[
        :16
    ]


    logo_path = (
        output_dir
        / "full_logo_results.csv"
    )


    logo_records = []

    completed_persons = set()


    # =========================================================================
    # RESTORE PREVIOUS COMPLETED LOGO FOLDS
    # =========================================================================

    if logo_path.exists():

        try:

            previous_logo = pd.read_csv(
                logo_path,
                dtype={
                    "Config_ID":
                        str,

                    "Held_Out_Person":
                        str
                }
            )


            if (
                "Config_ID"
                in previous_logo.columns
            ):

                matching_logo = previous_logo[
                    previous_logo[
                        "Config_ID"
                    ].astype(
                        str
                    )
                    == logo_config_id
                ].copy()


                if len(
                    matching_logo
                ) > 0:

                    logo_records = matching_logo.to_dict(
                        orient="records"
                    )


                    completed_persons = set(
                        matching_logo[
                            "Held_Out_Person"
                        ].astype(
                            str
                        )
                    )


        except Exception:

            logo_records = []

            completed_persons = set()


    print()

    print(
        "=" * 90
    )

    print(
        algorithm_name
    )

    print(
        "FULL LEAVE ONE GROUP OUT"
    )

    print(
        "=" * 90
    )


    print(
        "Available folds:",
        FULL_LOGO_FOLDS
    )


    print(
        "Requested folds:",
        logo_folds_to_use
    )


    print(
        "Fixed epochs:",
        fixed_epochs
    )


    print(
        "Previously completed:",
        len(
            completed_persons
        )
    )


    logo_start = time.time()


    splitter = LeaveOneGroupOut()


    for fold_number, (
        training_index,
        validation_index

    ) in enumerate(
        splitter.split(
            X_TRAIN_RAW,
            Y_TRAIN,
            G_TRAIN
        ),
        start=1
    ):

        if (
            fold_number
            > logo_folds_to_use
        ):

            break


        held_out_values = np.unique(
            G_TRAIN[
                validation_index
            ]
        )


        if len(
            held_out_values
        ) != 1:

            raise RuntimeError(
                "Unexpected LOGO group structure."
            )


        held_out_person = str(
            held_out_values[
                0
            ]
        )


        if (
            held_out_person
            in completed_persons
        ):

            print(
                f"LOGO "
                f"{fold_number}/"
                f"{logo_folds_to_use}, "
                f"Participant "
                f"{held_out_person}, "
                f"already completed."
            )


            continue


        fold_start = time.time()


        # =========================================================================
        # FOLD SPECIFIC SCALING
        # =========================================================================

        (
            X_logo_training,
            X_logo_validation

        ) = prepare_fold(
            training_index,
            validation_index
        )


        # =========================================================================
        # FIXED EPOCH TRAINING
        # =========================================================================

        (
            logo_model,
            _,
            _

        ) = fit_model(
            spec,
            best_params,
            X_logo_training,
            Y_TRAIN[
                training_index
            ],
            X_validation_data=None,
            y_validation_data=None,
            seed=
                int(
                    best_record[
                        "Seed"
                    ]
                ),
            epochs=
                fixed_epochs,
            use_early_stopping=
                False,
            verbose=
                False
        )


        probabilities = predict_probabilities(
            spec,
            logo_model,
            X_logo_validation,
            Y_TRAIN[
                validation_index
            ]
        )


        predictions = np.argmax(
            probabilities,
            axis=1
        )


        y_true = Y_TRAIN[
            validation_index
        ]


        fold_accuracy = accuracy_score(
            y_true,
            predictions
        )


        fold_precision = precision_score(
            y_true,
            predictions,
            average="weighted",
            zero_division=0
        )


        fold_recall = recall_score(
            y_true,
            predictions,
            average="weighted",
            zero_division=0
        )


        fold_f1 = f1_score(
            y_true,
            predictions,
            average="weighted",
            zero_division=0
        )


        fold_macro_f1 = f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        )


        fold_mcc = matthews_corrcoef(
            y_true,
            predictions
        )


        fold_kappa = cohen_kappa_score(
            y_true,
            predictions
        )


        fold_time = (
            time.time()
            - fold_start
        )


        record = {

            "Config_ID":
                logo_config_id,

            "Fold":
                fold_number,

            "Held_Out_Person":
                held_out_person,

            "Rows":
                int(
                    len(
                        validation_index
                    )
                ),

            "Accuracy":
                float(
                    fold_accuracy
                ),

            "Precision_Weighted":
                float(
                    fold_precision
                ),

            "Recall_Weighted":
                float(
                    fold_recall
                ),

            "F1_Weighted":
                float(
                    fold_f1
                ),

            "F1_Macro":
                float(
                    fold_macro_f1
                ),

            "Cohen_Kappa":
                float(
                    fold_kappa
                ),

            "MCC":
                float(
                    fold_mcc
                ),

            "Fixed_Epochs":
                int(
                    fixed_epochs
                ),

            "Execution_Time_Seconds":
                float(
                    fold_time
                )
        }


        logo_records.append(
            record
        )


        atomic_csv(
            pd.DataFrame(
                logo_records
            ),
            logo_path
        )


        completed_persons.add(
            held_out_person
        )


        print(
            f"LOGO "
            f"{fold_number}/"
            f"{logo_folds_to_use}, "
            f"Participant = "
            f"{held_out_person}, "
            f"Accuracy = "
            f"{fold_accuracy:.4f}, "
            f"F1 = "
            f"{fold_f1:.4f}, "
            f"MCC = "
            f"{fold_mcc:.4f}, "
            f"Time = "
            f"{fold_time / 60:.2f} min"
        )


        del logo_model

        del probabilities

        del predictions

        del X_logo_training

        del X_logo_validation


        clear_device()


    logo_table = pd.read_csv(
        logo_path,
        dtype={
            "Config_ID":
                str,

            "Held_Out_Person":
                str
        }
    )


    logo_table = logo_table[
        logo_table[
            "Config_ID"
        ].astype(
            str
        )
        == logo_config_id
    ].copy()


    logo_table = (
        logo_table
        .sort_values(
            "Fold"
        )
        .drop_duplicates(
            subset=[
                "Held_Out_Person"
            ],
            keep="last"
        )
    )


    logo_accuracy = (
        logo_table[
            "Accuracy"
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    logo_mean = float(
        np.mean(
            logo_accuracy
        )
    )


    logo_std = float(
        np.std(
            logo_accuracy
        )
    )


    logo_time = (
        time.time()
        - logo_start
    )


    print()

    print(
        "Full LOGO Accuracy:",
        f"{logo_mean:.4f}",
        "plus or minus",
        f"{logo_std:.4f}"
    )


    print(
        "Completed LOGO folds:",
        len(
            logo_table
        )
    )


    return (
        logo_table,
        logo_mean,
        logo_std,
        logo_time
    )


# =============================================================================
# FINAL UNTOUCHED TEST EVALUATION
# =============================================================================

def final_test_evaluation(
    spec,
    best_params,
    best_bundle,
    output_dir
):

    model = instantiate_model(
        spec,
        best_params,
        seed=
            int(
                best_bundle[
                    "record"
                ][
                    "Seed"
                ]
            )
    )


    model.load_state_dict(
        best_bundle[
            "model_state_dict"
        ]
    )


    model.eval()


    test_start = time.time()


    probabilities = predict_probabilities(
        spec,
        model,
        X_TEST,
        Y_TEST
    )


    predictions = np.argmax(
        probabilities,
        axis=1
    )


    test_time = (
        time.time()
        - test_start
    )


    # =========================================================================
    # METRICS
    # =========================================================================

    accuracy = accuracy_score(
        Y_TEST,
        predictions
    )


    precision = precision_score(
        Y_TEST,
        predictions,
        average="weighted",
        zero_division=0
    )


    recall = recall_score(
        Y_TEST,
        predictions,
        average="weighted",
        zero_division=0
    )


    weighted_f1 = f1_score(
        Y_TEST,
        predictions,
        average="weighted",
        zero_division=0
    )


    macro_f1 = f1_score(
        Y_TEST,
        predictions,
        average="macro",
        zero_division=0
    )


    kappa = cohen_kappa_score(
        Y_TEST,
        predictions
    )


    mcc = matthews_corrcoef(
        Y_TEST,
        predictions
    )


    try:

        roc_auc = roc_auc_score(
            Y_TEST,
            probabilities,
            multi_class="ovr",
            average="weighted"
        )


    except Exception:

        roc_auc = np.nan


    # =========================================================================
    # CLASSIFICATION REPORT
    # =========================================================================

    report_text = classification_report(
        Y_TEST,
        predictions,
        labels=np.arange(
            NUMBER_CLASSES
        ),
        target_names=
            CLASS_NAMES,
        digits=4,
        zero_division=0
    )


    with open(
        output_dir
        / "classification_report.txt",
        "w",
        encoding="utf8"
    ) as file:

        file.write(
            report_text
        )


    report_table = pd.DataFrame(
        classification_report(
            Y_TEST,
            predictions,
            labels=np.arange(
                NUMBER_CLASSES
            ),
            target_names=
                CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    ).transpose()


    report_table.to_csv(
        output_dir
        / "classification_report.csv"
    )


    save_confusion_matrices(
        Y_TEST,
        predictions,
        output_dir,
        spec[
            "display_name"
        ]
    )


    # =========================================================================
    # TEST PREDICTIONS
    # =========================================================================

    prediction_table = pd.DataFrame(
        {

            "PersonSrial":
                G_TEST,

            "True_Class":
                LABEL_ENCODER.inverse_transform(
                    Y_TEST
                ),

            "Predicted_Class":
                LABEL_ENCODER.inverse_transform(
                    predictions.astype(
                        np.int32
                    )
                )
        }
    )


    for class_index, class_name in enumerate(
        CLASS_NAMES
    ):

        prediction_table[
            f"Probability_{class_name}"
        ] = probabilities[
            :,
            class_index
        ]


    prediction_table.to_csv(
        output_dir
        / "test_predictions.csv",
        index=False
    )


    print()

    print(
        "CLASSIFICATION REPORT"
    )

    print()

    print(
        report_text
    )


    del model


    clear_device()


    return {

        "Testing_Accuracy":
            float(
                accuracy
            ),

        "Precision_Weighted":
            float(
                precision
            ),

        "Recall_Weighted":
            float(
                recall
            ),

        "F1_Weighted":
            float(
                weighted_f1
            ),

        "F1_Macro":
            float(
                macro_f1
            ),

        "Cohen_Kappa":
            float(
                kappa
            ),

        "MCC":
            float(
                mcc
            ),

        "ROC_AUC_Weighted_OVR":
            (
                None

                if np.isnan(
                    roc_auc
                )

                else float(
                    roc_auc
                )
            ),

        "Test_Time_Seconds":
            float(
                test_time
            )
    }


# =============================================================================
# COMPLETE ALGORITHM RUNNER
# =============================================================================

def run_algorithm(
    algorithm_key
):

    spec = ALGORITHM_SPECS[
        algorithm_key
    ]


    algorithm_name = spec[
        "display_name"
    ]


    output_dir = (
        OUTPUT_ROOT
        / spec[
            "folder"
        ]
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    (
        output_dir
        / "checkpoints"
    ).mkdir(
        parents=True,
        exist_ok=True
    )


    algorithm_start = time.time()


    print()

    print(
        "#" * 90
    )

    print(
        algorithm_name
    )

    print(
        "MIN DATASET"
    )

    print(
        "#" * 90
    )


    print(
        "Output folder:"
    )

    print(
        output_dir
    )


    print(
        "Batch size:",
        spec[
            "batch_size"
        ]
    )


    print(
        "Maximum epochs:",
        spec[
            "max_epochs"
        ]
    )


    print(
        "Optuna trials:",
        N_TRIALS
    )


    print(
        "Final runs:",
        N_RUNS
    )


    print(
        "GroupKFold folds:",
        GROUP_K_FOLDS
    )


    print(
        "Full LOGO folds:",
        FULL_LOGO_FOLDS
    )


    # =========================================================================
    # COMMON FILES
    # =========================================================================

    save_common_files(
        output_dir,
        spec[
            "uses_modalities"
        ]
    )


    # =========================================================================
    # REAL GPU TRAINING TEST
    # =========================================================================

    run_smoke_test(
        spec
    )


    # =========================================================================
    # OPTUNA
    # =========================================================================

    optuna_start = time.time()


    (
        study,
        best_params

    ) = run_optuna(
        spec,
        output_dir
    )


    optuna_time = (
        time.time()
        - optuna_start
    )


    config_id = build_config_id(
        spec,
        best_params
    )


    print(
        "Configuration ID:",
        config_id
    )


    # =========================================================================
    # MODEL ARCHITECTURE
    # =========================================================================

    temporary_model = instantiate_model(
        spec,
        best_params,
        SEED
    )


    trainable_parameters = count_trainable_parameters(
        temporary_model
    )


    with open(
        output_dir
        / "model_architecture.txt",
        "w",
        encoding="utf8"
    ) as file:

        file.write(
            str(
                temporary_model
            )
        )


    del temporary_model


    clear_device()


    print(
        "Trainable parameters:",
        f"{trainable_parameters:,}"
    )


    # =========================================================================
    # FIVE FINAL RUNS
    # =========================================================================

    final_run_start = time.time()


    (
        run_table,
        best_bundle

    ) = run_final_runs(
        spec,
        best_params,
        config_id,
        output_dir
    )


    final_run_time = (
        time.time()
        - final_run_start
    )


    best_record = best_bundle[
        "record"
    ]


    best_history = best_bundle[
        "history"
    ]


    save_training_curves(
        best_history,
        output_dir,
        algorithm_name
    )


    # =========================================================================
    # SAVE BEST MODEL
    # =========================================================================

    atomic_torch_save(
        {

            "Algorithm":
                algorithm_name,

            "Algorithm_Key":
                algorithm_key,

            "Version":
                spec[
                    "version"
                ],

            "Dataset":
                "Min Dataset",

            "Dataset_Signature":
                DATASET_SIGNATURE,

            "Config_ID":
                config_id,

            "Device_Type":
                DEVICE_TYPE,

            "Device_Name":
                DEVICE_NAME,

            "Best_Params":
                best_params,

            "Selected_Run":
                int(
                    best_record[
                        "Run"
                    ]
                ),

            "Selected_Seed":
                int(
                    best_record[
                        "Seed"
                    ]
                ),

            "Best_Epoch":
                int(
                    best_record[
                        "Best_Epoch"
                    ]
                ),

            "Model_State_Dict":
                best_bundle[
                    "model_state_dict"
                ],

            "Feature_Columns":
                FEATURE_COLUMNS,

            "Class_Names":
                CLASS_NAMES,

            "Modality_Indices":
                (
                    MODALITY_INDICES

                    if spec[
                        "uses_modalities"
                    ]

                    else None
                )
        },
        output_dir
        / "best_model.pt"
    )


    # =========================================================================
    # FULL LOGO
    # =========================================================================

    (
        logo_table,
        logo_mean,
        logo_std,
        logo_time

    ) = run_full_logo(
        spec,
        best_params,
        config_id,
        best_record,
        output_dir
    )


    # =========================================================================
    # FINAL UNTOUCHED TEST
    # =========================================================================

    print()

    print(
        "=" * 90
    )

    print(
        algorithm_name
    )

    print(
        "FINAL UNTOUCHED TEST EVALUATION"
    )

    print(
        "=" * 90
    )


    test_metrics = final_test_evaluation(
        spec,
        best_params,
        best_bundle,
        output_dir
    )


    # =========================================================================
    # FIVE RUN STATISTICS
    # =========================================================================

    train_mean = float(
        run_table[
            "Training_Accuracy"
        ].mean()
    )


    train_std = float(
        run_table[
            "Training_Accuracy"
        ].std(
            ddof=0
        )
    )


    validation_mean = float(
        run_table[
            "Validation_Accuracy"
        ].mean()
    )


    validation_std = float(
        run_table[
            "Validation_Accuracy"
        ].std(
            ddof=0
        )
    )


    gap_mean = float(
        run_table[
            "Generalization_Gap"
        ].mean()
    )


    gap_std = float(
        run_table[
            "Generalization_Gap"
        ].std(
            ddof=0
        )
    )


    algorithm_time = (
        time.time()
        - algorithm_start
    )


    # =========================================================================
    # FINAL METRICS
    # =========================================================================

    final_metrics = {

        "Algorithm":
            algorithm_name,

        "Algorithm_Key":
            algorithm_key,

        "Version":
            spec[
                "version"
            ],

        "Dataset":
            "Min Dataset",

        "Dataset_Signature":
            DATASET_SIGNATURE,

        "Config_ID":
            config_id,

        "Input_File":
            str(
                FILE_PATH
            ),

        "Output_Directory":
            str(
                output_dir
            ),

        "Framework":
            "PyTorch",

        "Device_Type":
            DEVICE_TYPE,

        "Device_Name":
            DEVICE_NAME,

        "Feature_Count":
            int(
                NUMBER_FEATURES
            ),

        "Classes":
            CLASS_NAMES,

        "Training_Participants":
            int(
                len(
                    TRAIN_PERSONS
                )
            ),

        "Validation_Participants":
            int(
                len(
                    VALIDATION_PERSONS
                )
            ),

        "Testing_Participants":
            int(
                len(
                    TEST_PERSONS
                )
            ),

        "Training_Samples":
            int(
                TRAIN_SAMPLES
            ),

        "Validation_Samples":
            int(
                VALIDATION_SAMPLES
            ),

        "Testing_Samples":
            int(
                TEST_SAMPLES
            ),

        "Trainable_Parameters":
            int(
                trainable_parameters
            ),

        "Optuna_Trials":
            int(
                N_TRIALS
            ),

        "Optuna_GroupKFold_Accuracy":
            float(
                study.best_value
            ),

        "Best_Parameters":
            best_params,

        "Selected_Run":
            int(
                best_record[
                    "Run"
                ]
            ),

        "Selected_Seed":
            int(
                best_record[
                    "Seed"
                ]
            ),

        "Selected_Best_Epoch":
            int(
                best_record[
                    "Best_Epoch"
                ]
            ),

        "Training_Accuracy":
            float(
                best_record[
                    "Training_Accuracy"
                ]
            ),

        "Validation_Accuracy":
            float(
                best_record[
                    "Validation_Accuracy"
                ]
            ),

        "Generalization_Gap":
            float(
                best_record[
                    "Generalization_Gap"
                ]
            ),

        "Five_Run_Training_Mean":
            train_mean,

        "Five_Run_Training_STD":
            train_std,

        "Five_Run_Validation_Mean":
            validation_mean,

        "Five_Run_Validation_STD":
            validation_std,

        "Five_Run_Gap_Mean":
            gap_mean,

        "Five_Run_Gap_STD":
            gap_std,

        "Full_LOGO_Accuracy":
            float(
                logo_mean
            ),

        "Full_LOGO_STD":
            float(
                logo_std
            ),

        "Full_LOGO_Folds_Available":
            int(
                FULL_LOGO_FOLDS
            ),

        "Full_LOGO_Folds_Completed":
            int(
                len(
                    logo_table
                )
            ),

        **test_metrics,

        "Optuna_Time_Seconds":
            float(
                optuna_time
            ),

        "Five_Run_Time_Seconds":
            float(
                final_run_time
            ),

        "LOGO_Time_Seconds":
            float(
                logo_time
            ),

        "Algorithm_Total_Time_Seconds":
            float(
                algorithm_time
            ),

        "Current_RAM_MB":
            float(
                memory_mb()
            )
    }


    if spec[
        "uses_modalities"
    ]:

        final_metrics[
            "Modality_Grouping_Mode"
        ] = MODALITY_GROUPING_MODE


        final_metrics[
            "Modality_Names"
        ] = MODALITY_NAMES


    save_json(
        final_metrics,
        output_dir
        / "final_metrics.json"
    )


    # =========================================================================
    # SUMMARY TEXT
    # =========================================================================

    summary_lines = [

        algorithm_name,

        "MIN DATASET",

        "",

        f"Input File: "
        f"{FILE_PATH}",

        f"Output Directory: "
        f"{output_dir}",

        "",

        f"Device: "
        f"{DEVICE_NAME}",

        f"Features: "
        f"{NUMBER_FEATURES}",

        f"Classes: "
        f"{CLASS_NAMES}",

        f"Trainable Parameters: "
        f"{trainable_parameters}",

        "",

        f"Training Participants: "
        f"{len(TRAIN_PERSONS)}",

        f"Validation Participants: "
        f"{len(VALIDATION_PERSONS)}",

        f"Testing Participants: "
        f"{len(TEST_PERSONS)}",

        "",

        f"Best Parameters: "
        f"{best_params}",

        "",

        f"Selected Run: "
        f"{best_record['Run']}",

        f"Selected Seed: "
        f"{best_record['Seed']}",

        f"Best Epoch: "
        f"{best_record['Best_Epoch']}",

        "",

        f"Optuna GroupKFold Accuracy: "
        f"{study.best_value:.4f}",

        f"Training Accuracy: "
        f"{best_record['Training_Accuracy']:.4f}",

        f"Validation Accuracy: "
        f"{best_record['Validation_Accuracy']:.4f}",

        f"Generalization Gap: "
        f"{best_record['Generalization_Gap']:.4f}",

        f"Testing Accuracy: "
        f"{test_metrics['Testing_Accuracy']:.4f}",

        "",

        f"Five Run Training: "
        f"{train_mean:.4f} "
        f"plus or minus "
        f"{train_std:.4f}",

        f"Five Run Validation: "
        f"{validation_mean:.4f} "
        f"plus or minus "
        f"{validation_std:.4f}",

        f"Five Run Gap: "
        f"{gap_mean:.4f} "
        f"plus or minus "
        f"{gap_std:.4f}",

        "",

        f"Full LOGO Accuracy: "
        f"{logo_mean:.4f} "
        f"plus or minus "
        f"{logo_std:.4f}",

        f"Full LOGO Folds: "
        f"{len(logo_table)}",

        "",

        f"Precision Weighted: "
        f"{test_metrics['Precision_Weighted']:.4f}",

        f"Recall Weighted: "
        f"{test_metrics['Recall_Weighted']:.4f}",

        f"F1 Weighted: "
        f"{test_metrics['F1_Weighted']:.4f}",

        f"F1 Macro: "
        f"{test_metrics['F1_Macro']:.4f}",

        f"Cohen Kappa: "
        f"{test_metrics['Cohen_Kappa']:.4f}",

        f"MCC: "
        f"{test_metrics['MCC']:.4f}",

        (
            "ROC AUC Weighted OVR: N/A"

            if test_metrics[
                "ROC_AUC_Weighted_OVR"
            ] is None

            else

            f"ROC AUC Weighted OVR: "
            f"{test_metrics['ROC_AUC_Weighted_OVR']:.4f}"
        ),

        "",

        f"Optuna Time: "
        f"{optuna_time / 60:.2f} minutes",

        f"Five Run Time: "
        f"{final_run_time / 60:.2f} minutes",

        f"LOGO Time: "
        f"{logo_time / 60:.2f} minutes",

        f"Algorithm Total Time: "
        f"{algorithm_time / 60:.2f} minutes"
    ]


    if spec[
        "uses_modalities"
    ]:

        summary_lines.extend(
            [

                "",

                f"Modality Grouping: "
                f"{MODALITY_GROUPING_MODE}",

                f"Modalities: "
                f"{MODALITY_NAMES}"
            ]
        )


    with open(
        output_dir
        / "final_summary.txt",
        "w",
        encoding="utf8"
    ) as file:

        file.write(
            "\n".join(
                summary_lines
            )
        )


    # =========================================================================
    # FINAL SCREEN OUTPUT
    # =========================================================================

    print()

    print(
        "#" * 90
    )

    print(
        algorithm_name
    )

    print(
        "FINAL RESULTS"
    )

    print(
        "#" * 90
    )


    print(
        "Optuna GroupKFold         :",
        f"{study.best_value:.4f}"
    )


    print(
        "Training Accuracy         :",
        f"{best_record['Training_Accuracy']:.4f}"
    )


    print(
        "Validation Accuracy       :",
        f"{best_record['Validation_Accuracy']:.4f}"
    )


    print(
        "Testing Accuracy          :",
        f"{test_metrics['Testing_Accuracy']:.4f}"
    )


    print(
        "Generalization Gap        :",
        f"{best_record['Generalization_Gap']:.4f}"
    )


    print(
        "Full LOGO Accuracy        :",
        f"{logo_mean:.4f}",
        "plus or minus",
        f"{logo_std:.4f}"
    )


    print(
        "Full LOGO Folds           :",
        len(
            logo_table
        ),
        "of",
        FULL_LOGO_FOLDS
    )


    print(
        "Precision Weighted        :",
        f"{test_metrics['Precision_Weighted']:.4f}"
    )


    print(
        "Recall Weighted           :",
        f"{test_metrics['Recall_Weighted']:.4f}"
    )


    print(
        "F1 Weighted               :",
        f"{test_metrics['F1_Weighted']:.4f}"
    )


    print(
        "F1 Macro                  :",
        f"{test_metrics['F1_Macro']:.4f}"
    )


    print(
        "Cohen Kappa               :",
        f"{test_metrics['Cohen_Kappa']:.4f}"
    )


    print(
        "MCC                       :",
        f"{test_metrics['MCC']:.4f}"
    )


    if (
        test_metrics[
            "ROC_AUC_Weighted_OVR"
        ]
        is None
    ):

        print(
            "ROC AUC Weighted OVR      : N/A"
        )


    else:

        print(
            "ROC AUC Weighted OVR      :",
            f"{test_metrics['ROC_AUC_Weighted_OVR']:.4f}"
        )


    print(
        "Algorithm Time            :",
        f"{algorithm_time / 60:.2f}",
        "minutes"
    )


    print()

    print(
        "RESULTS SAVED IN:"
    )


    print(
        output_dir
    )


    print(
        "#" * 90
    )


    del study

    del best_bundle


    clear_device()


    return final_metrics


# =============================================================================
# RUN ALL FOUR ALGORITHMS
# =============================================================================

ALL_RESULTS = []


for algorithm_key in ALGORITHMS_TO_RUN:

    try:

        result = run_algorithm(
            algorithm_key
        )


        ALL_RESULTS.append(
            result
        )


    except KeyboardInterrupt:

        print()

        print(
            "Execution was manually interrupted."
        )


        print(
            "Completed Optuna trials, final runs "
            "and LOGO folds remain saved."
        )


        raise


    except Exception:

        spec = ALGORITHM_SPECS[
            algorithm_key
        ]


        error_output = (
            OUTPUT_ROOT
            / spec[
                "folder"
            ]
        )


        error_output.mkdir(
            parents=True,
            exist_ok=True
        )


        error_text = traceback.format_exc()


        with open(
            error_output
            / "last_error.txt",
            "w",
            encoding="utf8"
        ) as file:

            file.write(
                error_text
            )


        print()

        print(
            "=" * 90
        )

        print(
            "ERROR IN:"
        )


        print(
            spec[
                "display_name"
            ]
        )


        print(
            "=" * 90
        )


        print(
            error_text
        )


        print(
            "Error log saved in:"
        )


        print(
            error_output
            / "last_error.txt"
        )


        raise


# =============================================================================
# PROGRAM FINISHED
# =============================================================================

program_time = (
    time.time()
    - PROGRAM_START
)


print()

print(
    "=" * 90
)

print(
    "ALL FOUR MIN DATASET DL EXPERIMENTS COMPLETED"
)

print(
    "=" * 90
)


print(
    "Total program time:",
    f"{program_time / 60:.2f}",
    "minutes"
)


print(
    "Final RAM:",
    f"{memory_mb():.2f} MB"
)


print()

print(
    "RESULT FOLDERS"
)


for algorithm_key in ALGORITHMS_TO_RUN:

    spec = ALGORITHM_SPECS[
        algorithm_key
    ]


    print(
        OUTPUT_ROOT
        / spec[
            "folder"
        ]
    )


print(
    "=" * 90
)

/home/rakib/Desktop/ml_apply/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE CONFIGURATION
PyTorch version : 2.14.1+xpu
Selected device : xpu:0
Device type     : xpu
Device name     : Intel(R) Arc(TM) A750 Graphics
CPU threads     : 8

LOADING MIN DATASET
Raw dataset shape: (211032, 31)
RAM after loading: 764.75 MB
Feature count: 25
Invalid feature cells: 0

Feature matrix: (211032, 25)
Classes: ['class_lecture', 'music', 'normal', 'reading']
Number of classes: 4
Participants: 60

CLASS DISTRIBUTION
class_lecture: 52261 (24.76%)
music: 53336 (25.27%)
normal: 53840 (25.51%)
reading: 51595 (24.45%)
Dataset signature: 4cdad3b2a0d6512b
Loading time: 1.14 seconds

FEATURE MODALITY CONSTRUCTION
Grouping mode: Balanced feature view fallback
feature_view_1 : 9 features
feature_view_2 : 8 features
feature_view_3 : 8 features

PERSON WISE 70 / 20 / 10 SPLIT
Training participants: 42
Validation participants: 6
Testing participants: 12

Training: (147945, 25)
Validation: (20852, 25)
Testing: (42235, 25)

GroupKFold folds: 5
Full LOGO folds: 42
RAM after split: 790.7

**END of Deep Learning Aplly**